### Sice all the final corrections are done now I am starting to do forecast preparation

In [1]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 1
# Cell 1: Create folders and define project paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Locate the eden_datasets folder
# ------------------------------------------------------------

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "eden_datasets":
    EDEN_DATASETS_DIR = CURRENT_DIRECTORY

elif (CURRENT_DIRECTORY / "eden_datasets").exists():
    EDEN_DATASETS_DIR = CURRENT_DIRECTORY / "eden_datasets"

else:
    raise FileNotFoundError(
        "The 'eden_datasets' folder could not be found.\n"
        f"Current notebook directory: {CURRENT_DIRECTORY}\n\n"
        "Make sure the notebook is running from the project folder "
        "that contains the eden_datasets directory."
    )

# ------------------------------------------------------------
# 2. Create the new forecasting-preparation folder
# ------------------------------------------------------------

FORECAST_PREPARATION_DIR = (
    EDEN_DATASETS_DIR / "forceast_preparation"
)

FORECAST_PREPARATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Reserve the two final output paths
# ------------------------------------------------------------

FINAL_FORECAST_FILE_IN_FOLDER = (
    FORECAST_PREPARATION_DIR
    / "UL_EDEN_forecast_preparation_final.csv"
)

FINAL_FORECAST_FILE_IN_EDEN_DATASETS = (
    EDEN_DATASETS_DIR
    / "UL_EDEN_forecast_preparation_final.csv"
)

# ------------------------------------------------------------
# 4. Display the folder setup
# ------------------------------------------------------------

print("Folder setup completed successfully.")
print()
print("Current notebook directory:")
print(CURRENT_DIRECTORY)
print()
print("Eden datasets directory:")
print(EDEN_DATASETS_DIR)
print()
print("Forecast preparation directory:")
print(FORECAST_PREPARATION_DIR)
print()
print("Final dataset will eventually be saved to:")
print(f"1. {FINAL_FORECAST_FILE_IN_FOLDER}")
print(f"2. {FINAL_FORECAST_FILE_IN_EDEN_DATASETS}")

assert EDEN_DATASETS_DIR.exists()
assert FORECAST_PREPARATION_DIR.exists()

print()
print("All folder checks passed.")

Folder setup completed successfully.

Current notebook directory:
/Users/ryansmac/Desktop/Meng Project

Eden datasets directory:
/Users/ryansmac/Desktop/Meng Project/eden_datasets

Forecast preparation directory:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation

Final dataset will eventually be saved to:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/UL_EDEN_forecast_preparation_final.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/UL_EDEN_forecast_preparation_final.csv

All folder checks passed.


In [3]:
# ============================================================
# STEP 1 — PART 1
# Cell 2: Locate the daily product-demand dataset
# ============================================================

# ------------------------------------------------------------
# Manual selection option
# ------------------------------------------------------------
# Leave this as None for automatic detection.
#
# If automatic detection selects the wrong file, enter the
# exact filename here, for example:
#
# MANUAL_DAILY_DATASET_FILENAME = "UL_EDEN_daily_product_demand.csv"
# ------------------------------------------------------------

MANUAL_DAILY_DATASET_FILENAME = (
    "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"
)


if MANUAL_DAILY_DATASET_FILENAME is not None:

    SOURCE_DAILY_DEMAND_FILE = (
        EDEN_DATASETS_DIR / MANUAL_DAILY_DATASET_FILENAME
    )

    if not SOURCE_DAILY_DEMAND_FILE.exists():
        raise FileNotFoundError(
            "The manually selected dataset does not exist:\n"
            f"{SOURCE_DAILY_DEMAND_FILE}"
        )

    candidate_files = [SOURCE_DAILY_DEMAND_FILE]

else:

    # Find all CSV files outside the new forecasting folder
    all_csv_files = [
        file_path
        for file_path in EDEN_DATASETS_DIR.rglob("*.csv")
        if FORECAST_PREPARATION_DIR not in file_path.parents
    ]

    # Search for files that look like daily demand datasets
    candidate_files = []

    for file_path in all_csv_files:

        filename_lower = file_path.stem.lower()

        contains_daily = "daily" in filename_lower

        contains_demand_information = (
            "demand" in filename_lower
            or "product" in filename_lower
            or "unitsold" in filename_lower
        )

        excluded_file = any(
            excluded_word in filename_lower
            for excluded_word in [
                "forecast",
                "forceast",
                "audit",
                "refund",
                "own_cup",
                "drs"
            ]
        )

        if (
            contains_daily
            and contains_demand_information
            and not excluded_file
        ):
            candidate_files.append(file_path)

    if len(candidate_files) == 0:

        available_files = sorted(
            all_csv_files,
            key=lambda file_path: file_path.stat().st_mtime,
            reverse=True
        )

        print("Most recently modified CSV files:")
        print()

        for file_path in available_files[:15]:
            print(f"- {file_path.relative_to(EDEN_DATASETS_DIR)}")

        raise FileNotFoundError(
            "\nNo daily product-demand dataset was detected.\n"
            "Set MANUAL_DAILY_DATASET_FILENAME to the exact filename."
        )

    # Put the most recently modified candidate first
    candidate_files = sorted(
        candidate_files,
        key=lambda file_path: file_path.stat().st_mtime,
        reverse=True
    )

    SOURCE_DAILY_DEMAND_FILE = candidate_files[0]


# ------------------------------------------------------------
# Display all detected candidates
# ------------------------------------------------------------

print("Daily-demand dataset candidates:")
print()

for position, file_path in enumerate(candidate_files, start=1):

    selected_marker = (
        "  <-- SELECTED"
        if file_path == SOURCE_DAILY_DEMAND_FILE
        else ""
    )

    print(
        f"{position}. "
        f"{file_path.relative_to(EDEN_DATASETS_DIR)}"
        f"{selected_marker}"
    )

print()
print("Selected source dataset:")
print(SOURCE_DAILY_DEMAND_FILE)

assert SOURCE_DAILY_DEMAND_FILE.exists()

print()
print("Source-file check passed.")

Daily-demand dataset candidates:

1. UL_EDEN_canonical_product_daily_demand_forecasting_final.csv  <-- SELECTED

Selected source dataset:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv

Source-file check passed.


In [5]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 1
# Corrected Cell 3:
# Load and validate the final daily forecasting dataset
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load the selected final dataset
# ------------------------------------------------------------

daily_demand_df = pd.read_csv(
    SOURCE_DAILY_DEMAND_FILE,
    low_memory=False
)

# Remove accidental spaces from column headings
daily_demand_df.columns = (
    daily_demand_df.columns
    .astype(str)
    .str.strip()
)

print("=" * 75)
print("DAILY PRODUCT-DEMAND DATASET LOADED")
print("=" * 75)

print()
print(f"Source file: {SOURCE_DAILY_DEMAND_FILE.name}")
print(f"Number of rows: {len(daily_demand_df):,}")
print(f"Number of columns: {daily_demand_df.shape[1]:,}")


# ------------------------------------------------------------
# 2. Define the confirmed forecasting columns
# ------------------------------------------------------------

DATE_COLUMN = "Date"
PRODUCT_ID_COLUMN = "CanonicalProductID"
PRODUCT_COLUMN = "CanonicalProductName"
DEMAND_COLUMN = "TotalDemand"

NORMAL_DEMAND_COLUMN = "NormalDemand"
BULK_DEMAND_COLUMN = "BulkDemand"

required_columns = [
    DATE_COLUMN,
    PRODUCT_ID_COLUMN,
    PRODUCT_COLUMN,
    NORMAL_DEMAND_COLUMN,
    BULK_DEMAND_COLUMN,
    DEMAND_COLUMN,
    "IsObservedProductDate",
    "IsZeroDemandRow"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in daily_demand_df.columns
]

if missing_required_columns:
    raise KeyError(
        "The following required columns are missing:\n"
        f"{missing_required_columns}"
    )

print()
print("Confirmed forecasting columns:")
print(f"Date column:          {DATE_COLUMN}")
print(f"Product ID column:    {PRODUCT_ID_COLUMN}")
print(f"Product name column:  {PRODUCT_COLUMN}")
print(f"Forecast target:      {DEMAND_COLUMN}")


# ------------------------------------------------------------
# 3. Parse ISO dates correctly
# ------------------------------------------------------------

original_date_values = daily_demand_df[DATE_COLUMN].copy()

daily_demand_df[DATE_COLUMN] = pd.to_datetime(
    daily_demand_df[DATE_COLUMN],
    format="%Y-%m-%d",
    errors="coerce"
)

invalid_date_count = int(
    daily_demand_df[DATE_COLUMN].isna().sum()
)

print()
print("Date validation:")
print(f"Invalid or missing dates: {invalid_date_count:,}")

if invalid_date_count > 0:

    invalid_examples = (
        original_date_values[
            daily_demand_df[DATE_COLUMN].isna()
        ]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    print("Invalid date examples:")
    print(invalid_examples)

else:

    print(
        "Date range:",
        daily_demand_df[DATE_COLUMN].min().date(),
        "to",
        daily_demand_df[DATE_COLUMN].max().date()
    )

    print(
        "Unique operating dates:",
        f"{daily_demand_df[DATE_COLUMN].nunique():,}"
    )


# ------------------------------------------------------------
# 4. Clean and validate product identifiers
# ------------------------------------------------------------

daily_demand_df[PRODUCT_ID_COLUMN] = (
    daily_demand_df[PRODUCT_ID_COLUMN]
    .astype("string")
    .str.strip()
)

daily_demand_df[PRODUCT_COLUMN] = (
    daily_demand_df[PRODUCT_COLUMN]
    .astype("string")
    .str.strip()
)

unique_product_ids = (
    daily_demand_df[PRODUCT_ID_COLUMN]
    .dropna()
    .nunique()
)

unique_product_names = (
    daily_demand_df[PRODUCT_COLUMN]
    .dropna()
    .nunique()
)

missing_product_ids = int(
    daily_demand_df[PRODUCT_ID_COLUMN].isna().sum()
)

missing_product_names = int(
    daily_demand_df[PRODUCT_COLUMN].isna().sum()
)

print()
print("Product validation:")
print(f"Unique canonical product IDs: {unique_product_ids:,}")
print(f"Unique canonical product names: {unique_product_names:,}")
print(f"Missing product IDs: {missing_product_ids:,}")
print(f"Missing product names: {missing_product_names:,}")


# ------------------------------------------------------------
# 5. Convert and validate demand columns
# ------------------------------------------------------------

for column in [
    NORMAL_DEMAND_COLUMN,
    BULK_DEMAND_COLUMN,
    DEMAND_COLUMN
]:
    daily_demand_df[column] = pd.to_numeric(
        daily_demand_df[column],
        errors="coerce"
    )

missing_total_demand = int(
    daily_demand_df[DEMAND_COLUMN].isna().sum()
)

negative_total_demand = int(
    (daily_demand_df[DEMAND_COLUMN] < 0).sum()
)

zero_total_demand = int(
    (daily_demand_df[DEMAND_COLUMN] == 0).sum()
)

normal_demand_units = daily_demand_df[
    NORMAL_DEMAND_COLUMN
].sum()

bulk_demand_units = daily_demand_df[
    BULK_DEMAND_COLUMN
].sum()

total_demand_units = daily_demand_df[
    DEMAND_COLUMN
].sum()

demand_component_mismatch = int(
    (
        daily_demand_df[DEMAND_COLUMN]
        != (
            daily_demand_df[NORMAL_DEMAND_COLUMN]
            + daily_demand_df[BULK_DEMAND_COLUMN]
        )
    ).sum()
)

print()
print("Demand validation:")
print(f"Missing TotalDemand values: {missing_total_demand:,}")
print(f"Negative TotalDemand values: {negative_total_demand:,}")
print(f"Zero-demand rows: {zero_total_demand:,}")
print(f"Normal demand units: {normal_demand_units:,.0f}")
print(f"Bulk demand units: {bulk_demand_units:,.0f}")
print(f"Total demand units: {total_demand_units:,.0f}")
print(
    "Rows where TotalDemand differs from "
    "NormalDemand + BulkDemand:",
    f"{demand_component_mismatch:,}"
)


# ------------------------------------------------------------
# 6. Validate panel-status columns
# ------------------------------------------------------------

observed_product_date_rows = int(
    daily_demand_df["IsObservedProductDate"].sum()
)

zero_demand_flag_rows = int(
    daily_demand_df["IsZeroDemandRow"].sum()
)

print()
print("Daily panel validation:")
print(
    f"Observed product-date rows: "
    f"{observed_product_date_rows:,}"
)

print(
    f"Zero-demand product-date rows: "
    f"{zero_demand_flag_rows:,}"
)


# ------------------------------------------------------------
# 7. Check duplicates using the canonical product ID
# ------------------------------------------------------------

complete_duplicate_count = int(
    daily_demand_df.duplicated().sum()
)

duplicate_product_date_mask = daily_demand_df.duplicated(
    subset=[DATE_COLUMN, PRODUCT_ID_COLUMN],
    keep=False
)

duplicate_product_date_count = int(
    duplicate_product_date_mask.sum()
)

print()
print("Duplicate validation:")
print(f"Completely duplicated rows: {complete_duplicate_count:,}")
print(
    "Rows involved in duplicate Date–CanonicalProductID pairs:",
    f"{duplicate_product_date_count:,}"
)


# ------------------------------------------------------------
# 8. Check names shared by multiple canonical IDs
# ------------------------------------------------------------

name_id_counts = (
    daily_demand_df[
        [PRODUCT_COLUMN, PRODUCT_ID_COLUMN]
    ]
    .drop_duplicates()
    .groupby(PRODUCT_COLUMN)[PRODUCT_ID_COLUMN]
    .nunique()
    .sort_values(ascending=False)
)

shared_name_summary = (
    name_id_counts[name_id_counts > 1]
    .rename("CanonicalIDCount")
    .reset_index()
)

print()
print("Product names associated with multiple canonical IDs:")
print(f"Number of shared names: {len(shared_name_summary):,}")

display(shared_name_summary)


# ------------------------------------------------------------
# 9. Final expected-value checks
# ------------------------------------------------------------

assert len(daily_demand_df) == 25_405
assert daily_demand_df.shape[1] == 38

assert invalid_date_count == 0
assert daily_demand_df[DATE_COLUMN].nunique() == 245

assert unique_product_ids == 227
assert unique_product_names == 218

assert missing_product_ids == 0
assert missing_product_names == 0
assert missing_total_demand == 0
assert negative_total_demand == 0

assert observed_product_date_rows == 15_138
assert zero_demand_flag_rows == 10_267
assert zero_total_demand == 10_267

assert normal_demand_units == 114_186
assert bulk_demand_units == 1_972
assert total_demand_units == 116_158

assert demand_component_mismatch == 0
assert complete_duplicate_count == 0
assert duplicate_product_date_count == 0

print()
print("=" * 75)
print("ALL STEP 1, PART 1 DATASET VALIDATION CHECKS PASSED")
print("=" * 75)


# ------------------------------------------------------------
# 10. Display key data types and sample rows
# ------------------------------------------------------------

key_columns = [
    DATE_COLUMN,
    PRODUCT_ID_COLUMN,
    PRODUCT_COLUMN,
    NORMAL_DEMAND_COLUMN,
    BULK_DEMAND_COLUMN,
    DEMAND_COLUMN,
    "IsObservedProductDate",
    "IsZeroDemandRow"
]

print()
print("Key column data types:")
display(
    daily_demand_df[key_columns]
    .dtypes
    .rename("DataType")
    .to_frame()
)

print()
print("First 10 rows:")
display(
    daily_demand_df[key_columns].head(10)
)

DAILY PRODUCT-DEMAND DATASET LOADED

Source file: UL_EDEN_canonical_product_daily_demand_forecasting_final.csv
Number of rows: 25,405
Number of columns: 38

Confirmed forecasting columns:
Date column:          Date
Product ID column:    CanonicalProductID
Product name column:  CanonicalProductName
Forecast target:      TotalDemand

Date validation:
Invalid or missing dates: 0
Date range: 2025-04-01 to 2026-03-30
Unique operating dates: 245

Product validation:
Unique canonical product IDs: 227
Unique canonical product names: 218
Missing product IDs: 0
Missing product names: 0

Demand validation:
Missing TotalDemand values: 0
Negative TotalDemand values: 0
Zero-demand rows: 10,267
Normal demand units: 114,186
Bulk demand units: 1,972
Total demand units: 116,158
Rows where TotalDemand differs from NormalDemand + BulkDemand: 0

Daily panel validation:
Observed product-date rows: 15,138
Zero-demand product-date rows: 10,267

Duplicate validation:
Completely duplicated rows: 0
Rows involved

,CanonicalProductName,CanonicalIDCount
0,EXTRA SHOT,2
1,ICED LATTE 16OZ,2
2,SINGLE ESPRESSO,2
3,SCONE & BUTTER JAM,2
4,SYRUP,2
5,FILTER COFFEE SM,2
6,FLAT WHITE,2
7,FRUIT SALAD,2
8,FRUIT SALAD & TOPPINGS,2



ALL STEP 1, PART 1 DATASET VALIDATION CHECKS PASSED

Key column data types:


,DataType
Date,datetime64[ns]
CanonicalProductID,string[python]
CanonicalProductName,string[python]
NormalDemand,int64
BulkDemand,int64
TotalDemand,int64
IsObservedProductDate,bool
IsZeroDemandRow,bool



First 10 rows:


,Date,CanonicalProductID,CanonicalProductName,NormalDemand,BulkDemand,TotalDemand,IsObservedProductDate,IsZeroDemandRow
0,2025-04-01,BEV_BRANDED_AMERICANO,B-AMERICANO,13,0,13,True,False
1,2025-04-01,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,1,0,1,True,False
2,2025-04-01,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,1,0,1,True,False
3,2025-04-01,BEV_BRANDED_LATTE,B-LATTE,1,0,1,True,False
4,2025-04-01,BEV_BRANDED_MOCHA,B-MOCHA,1,0,1,True,False
5,2025-04-01,PLU_12304,KOMBUCHA,1,0,1,True,False
6,2025-04-01,PLU_125038,FILTER COFFEE SM,5,0,5,True,False
7,2025-04-01,PLU_2000000019,FULL FAT CAN,15,0,15,True,False
8,2025-04-01,PLU_2000000023,COKE ZERO 330ML,21,0,21,True,False
9,2025-04-01,PLU_2000000027,DIET/ZERO COKE 330ML CAN,4,0,4,True,False


In [6]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 1
# Cell 4: Save the controlled working copy
# ============================================================

forecast_working_df = (
    daily_demand_df
    .copy()
    .sort_values(
        by=[
            DATE_COLUMN,
            PRODUCT_ID_COLUMN
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

STEP_1_PART_1_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "01_daily_product_demand_loaded.csv"
)

forecast_working_df.to_csv(
    STEP_1_PART_1_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

# Reload the saved dataset for verification
saved_check_df = pd.read_csv(
    STEP_1_PART_1_OUTPUT,
    low_memory=False
)

# ------------------------------------------------------------
# Saved-file validation
# ------------------------------------------------------------

assert STEP_1_PART_1_OUTPUT.exists()

assert saved_check_df.shape == forecast_working_df.shape, (
    "The saved dataset dimensions do not match the working dataset."
)

assert len(saved_check_df) == 25_405, (
    f"Expected 25,405 rows, but saved {len(saved_check_df):,}."
)

assert saved_check_df.shape[1] == 38, (
    f"Expected 38 columns, but saved {saved_check_df.shape[1]}."
)

assert saved_check_df[
    ["Date", "CanonicalProductID"]
].duplicated().sum() == 0, (
    "Duplicate product-date rows were found after saving."
)

assert saved_check_df["TotalDemand"].sum() == 116_158, (
    "The TotalDemand sum changed during saving."
)

print("=" * 75)
print("STEP 1, PART 1 OUTPUT SAVED SUCCESSFULLY")
print("=" * 75)

print()
print(f"Saved file: {STEP_1_PART_1_OUTPUT}")
print(f"Rows saved: {len(saved_check_df):,}")
print(f"Columns saved: {saved_check_df.shape[1]:,}")
print(
    f"Total demand units saved: "
    f"{saved_check_df['TotalDemand'].sum():,.0f}"
)
print(
    "Duplicate Date–CanonicalProductID rows:",
    saved_check_df[
        ["Date", "CanonicalProductID"]
    ].duplicated().sum()
)

STEP 1, PART 1 OUTPUT SAVED SUCCESSFULLY

Saved file: /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/01_daily_product_demand_loaded.csv
Rows saved: 25,405
Columns saved: 38
Total demand units saved: 116,158
Duplicate Date–CanonicalProductID rows: 0


In [7]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 2
# Cell 5: Create and maintain the Markdown project handoff
# ============================================================

from pathlib import Path

HANDOFF_FILE = (
    FORECAST_PREPARATION_DIR
    / "FORECASTING_PREPARATION_HANDOFF.md"
)


def upsert_markdown_section(
    file_path,
    section_id,
    section_title,
    section_body
):
    """
    Add or replace a controlled section in the Markdown handoff.

    Re-running the cell will update the existing section rather
    than creating duplicate entries.
    """

    start_marker = f"<!-- START:{section_id} -->"
    end_marker = f"<!-- END:{section_id} -->"

    section_block = (
        f"{start_marker}\n"
        f"## {section_title}\n\n"
        f"{section_body.strip()}\n"
        f"{end_marker}\n"
    )

    if file_path.exists():
        existing_text = file_path.read_text(encoding="utf-8")
    else:
        existing_text = (
            "# Eden Forecasting Preparation Handoff\n\n"
            "This file records completed and validated stages of "
            "the forecasting-preparation workflow.\n\n"
        )

    if (
        start_marker in existing_text
        and end_marker in existing_text
    ):

        section_start = existing_text.index(start_marker)
        section_end = (
            existing_text.index(end_marker)
            + len(end_marker)
        )

        updated_text = (
            existing_text[:section_start]
            + section_block.rstrip()
            + existing_text[section_end:]
        )

    else:

        updated_text = (
            existing_text.rstrip()
            + "\n\n"
            + section_block
        )

    file_path.write_text(
        updated_text.rstrip() + "\n",
        encoding="utf-8"
    )


# ------------------------------------------------------------
# Record the already completed Part 1
# ------------------------------------------------------------

part_1_summary = """
**Status:** Completed and validated

### Source dataset

`UL_EDEN_canonical_product_daily_demand_forecasting_final.csv`

### Confirmed structure

- Rows: 25,405
- Columns: 38
- Operating dates: 245
- Date range: 2025-04-01 to 2026-03-30
- Canonical product IDs: 227
- Canonical product names: 218
- Observed product-date rows: 15,138
- Zero-demand product-date rows: 10,267
- Normal demand units: 114,186
- Bulk demand units: 1,972
- Total demand units: 116,158
- Duplicate Date–CanonicalProductID rows: 0

### Locked dataset decisions

- Forecasting product key: `CanonicalProductID`
- Human-readable product label: `CanonicalProductName`
- Forecast target: `TotalDemand`

### Controlled output

`forceast_preparation/01_daily_product_demand_loaded.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_1",
    section_title=(
        "Forecasting Preparation — Step 1, Part 1"
    ),
    section_body=part_1_summary
)

print("Markdown handoff created or updated successfully.")
print()
print(f"Handoff file: {HANDOFF_FILE}")
print(f"File exists: {HANDOFF_FILE.exists()}")

assert HANDOFF_FILE.exists()

Markdown handoff created or updated successfully.

Handoff file: /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/FORECASTING_PREPARATION_HANDOFF.md
File exists: True


In [8]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 2
# Cell 6: Create the forecasting column-role register
# ============================================================

column_roles = [
    (
        "Date",
        "KEY",
        "RETAIN",
        "Operating-date key; known before forecasting."
    ),
    (
        "CanonicalProductID",
        "SERIES_KEY",
        "RETAIN",
        "Stable product-series identifier and primary product key."
    ),
    (
        "CanonicalProductName",
        "LABEL",
        "SUPPORT_ONLY",
        "Readable product label; not sufficiently unique as the key."
    ),
    (
        "ProductFirstObservedDate",
        "LIFECYCLE_SUPPORT",
        "SUPPORT_ONLY",
        "Used for lifecycle and cold-start checks; not used directly "
        "until leakage-safe handling is confirmed."
    ),
    (
        "ProductAgeOperatingDays",
        "TEMPORAL_FEATURE",
        "CANDIDATE",
        "Product age measured in operating days."
    ),
    (
        "SourcePLUCount",
        "PRODUCT_METADATA",
        "CANDIDATE",
        "Number of source PLUs mapped to the canonical product."
    ),
    (
        "SourcePLUCodes",
        "PRODUCT_METADATA_TEXT",
        "EXCLUDE_HIGH_CARDINALITY",
        "Raw source identifiers retained for audit only."
    ),
    (
        "SourcePLUNames",
        "PRODUCT_METADATA_TEXT",
        "EXCLUDE_HIGH_CARDINALITY",
        "Raw source names retained for audit only."
    ),
    (
        "SourceGroupCodes",
        "PRODUCT_METADATA",
        "CANDIDATE",
        "Product-group code metadata."
    ),
    (
        "SourceGroupNames",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Readable product-group metadata; select either code or name "
        "later to avoid duplicate representations."
    ),
    (
        "BeverageSeries",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Beverage metadata; missing values may identify non-beverages."
    ),
    (
        "BeverageType",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Beverage subtype where applicable."
    ),
    (
        "SupplierLabelsObserved",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Supplier metadata where applicable."
    ),
    (
        "TierProductFamily",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Product-family tier where available."
    ),
    (
        "NominalPriceTier",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Nominal price tier where available."
    ),
    (
        "MenuGeneration",
        "PRODUCT_METADATA",
        "CANDIDATE_CONDITIONAL",
        "Menu-generation metadata where available."
    ),
    (
        "IsMultiPLUCanonicalProduct",
        "PRODUCT_METADATA",
        "CANDIDATE",
        "Identifies products composed of multiple source PLUs."
    ),
    (
        "OperatingDaySequence",
        "TEMPORAL_FEATURE",
        "CANDIDATE",
        "Sequential operating-day trend feature."
    ),
    (
        "Year",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "Calendar year."
    ),
    (
        "Month",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "Numeric calendar month."
    ),
    (
        "MonthName",
        "CALENDAR_LABEL",
        "EXCLUDE_REDUNDANT",
        "Readable duplicate of the Month column."
    ),
    (
        "Quarter",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "Calendar quarter."
    ),
    (
        "DayOfWeekNumber",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "Numeric day-of-week representation."
    ),
    (
        "DayOfWeek",
        "CALENDAR_LABEL",
        "EXCLUDE_REDUNDANT",
        "Readable duplicate of DayOfWeekNumber."
    ),
    (
        "ISOYear",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "ISO calendar year."
    ),
    (
        "ISOWeek",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "ISO week number."
    ),
    (
        "DayOfYear",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "Calendar-day position within the year."
    ),
    (
        "IsWeekend",
        "CALENDAR_FEATURE",
        "CANDIDATE",
        "Weekend indicator."
    ),
    (
        "DaysSincePreviousOperatingDate",
        "OPERATING_CALENDAR_FEATURE",
        "CANDIDATE",
        "Known gap from the previous Eden operating date."
    ),
    (
        "IsConsecutiveCalendarDay",
        "OPERATING_CALENDAR_FEATURE",
        "CANDIDATE",
        "Indicates consecutive calendar-day operation."
    ),
    (
        "NormalDemand",
        "TARGET_COMPONENT",
        "EXCLUDE_LEAKAGE",
        "Same-day component of TotalDemand and unavailable when "
        "forecasting."
    ),
    (
        "BulkDemand",
        "TARGET_COMPONENT",
        "EXCLUDE_LEAKAGE",
        "Same-day component of TotalDemand and unavailable when "
        "forecasting."
    ),
    (
        "TotalDemand",
        "TARGET",
        "TARGET",
        "Final daily product-demand forecasting target."
    ),
    (
        "IsObservedProductDate",
        "OUTCOME_DERIVED",
        "EXCLUDE_LEAKAGE",
        "Directly indicates whether positive demand occurred."
    ),
    (
        "IsZeroDemandRow",
        "OUTCOME_DERIVED",
        "EXCLUDE_LEAKAGE",
        "Directly identifies whether TotalDemand equals zero."
    ),
    (
        "DemandRecordSource",
        "OUTCOME_DERIVED",
        "EXCLUDE_LEAKAGE",
        "Identifies observed versus zero-filled outcomes."
    ),
    (
        "DailyPanelVersion",
        "VERSION_METADATA",
        "EXCLUDE_VERSION",
        "Dataset-version label retained for reproducibility."
    ),
    (
        "ProductMetadataVersion",
        "VERSION_METADATA",
        "EXCLUDE_VERSION",
        "Product-metadata version retained for reproducibility."
    )
]

column_role_df = pd.DataFrame(
    column_roles,
    columns=[
        "Column",
        "Role",
        "ModelUse",
        "Reason"
    ]
)

# ------------------------------------------------------------
# Confirm every dataset column is classified exactly once
# ------------------------------------------------------------

dataset_columns = set(daily_demand_df.columns)
classified_columns = set(column_role_df["Column"])

unclassified_columns = (
    dataset_columns - classified_columns
)

unknown_classified_columns = (
    classified_columns - dataset_columns
)

duplicated_role_columns = (
    column_role_df["Column"]
    .duplicated()
    .sum()
)

assert len(column_role_df) == 38, (
    f"Expected 38 role records, found {len(column_role_df)}."
)

assert not unclassified_columns, (
    "The following dataset columns were not classified:\n"
    f"{sorted(unclassified_columns)}"
)

assert not unknown_classified_columns, (
    "The role register contains columns not found in the dataset:\n"
    f"{sorted(unknown_classified_columns)}"
)

assert duplicated_role_columns == 0, (
    "A dataset column was classified more than once."
)

print("=" * 75)
print("COLUMN-ROLE REGISTER CREATED")
print("=" * 75)

print()
print(f"Dataset columns: {len(dataset_columns)}")
print(f"Classified columns: {len(classified_columns)}")
print(f"Unclassified columns: {len(unclassified_columns)}")
print(f"Duplicated classifications: {duplicated_role_columns}")

print()
print("Columns by proposed model use:")

display(
    column_role_df["ModelUse"]
    .value_counts()
    .rename_axis("ModelUse")
    .reset_index(name="ColumnCount")
)

display(column_role_df)

COLUMN-ROLE REGISTER CREATED

Dataset columns: 38
Classified columns: 38
Unclassified columns: 0
Duplicated classifications: 0

Columns by proposed model use:


,ModelUse,ColumnCount
0,CANDIDATE,15
1,CANDIDATE_CONDITIONAL,7
2,EXCLUDE_LEAKAGE,5
3,RETAIN,2
4,SUPPORT_ONLY,2
5,EXCLUDE_HIGH_CARDINALITY,2
6,EXCLUDE_REDUNDANT,2
7,EXCLUDE_VERSION,2
8,TARGET,1


,Column,Role,ModelUse,Reason
0,Date,KEY,RETAIN,Operating-date key; known before forecasting.
1,CanonicalProductID,SERIES_KEY,RETAIN,Stable product-series identifier and primary p...
2,CanonicalProductName,LABEL,SUPPORT_ONLY,Readable product label; not sufficiently uniqu...
3,ProductFirstObservedDate,LIFECYCLE_SUPPORT,SUPPORT_ONLY,Used for lifecycle and cold-start checks; not ...
4,ProductAgeOperatingDays,TEMPORAL_FEATURE,CANDIDATE,Product age measured in operating days.
5,SourcePLUCount,PRODUCT_METADATA,CANDIDATE,Number of source PLUs mapped to the canonical ...
6,SourcePLUCodes,PRODUCT_METADATA_TEXT,EXCLUDE_HIGH_CARDINALITY,Raw source identifiers retained for audit only.
7,SourcePLUNames,PRODUCT_METADATA_TEXT,EXCLUDE_HIGH_CARDINALITY,Raw source names retained for audit only.
8,SourceGroupCodes,PRODUCT_METADATA,CANDIDATE,Product-group code metadata.
9,SourceGroupNames,PRODUCT_METADATA,CANDIDATE_CONDITIONAL,Readable product-group metadata; select either...


In [9]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 2
# Cell 7: Profile all columns
# ============================================================

def create_sample_values(series, maximum_values=3):
    """
    Return a short readable sample of unique non-null values.
    """

    unique_values = (
        series
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(maximum_values)
        .tolist()
    )

    shortened_values = []

    for value in unique_values:

        if len(value) > 60:
            value = value[:57] + "..."

        shortened_values.append(value)

    return " | ".join(shortened_values)


profile_records = []

for column in daily_demand_df.columns:

    series = daily_demand_df[column]

    missing_count = int(series.isna().sum())

    missing_percentage = (
        missing_count
        / len(daily_demand_df)
        * 100
    )

    unique_non_null = int(
        series.nunique(dropna=True)
    )

    unique_including_null = int(
        series.nunique(dropna=False)
    )

    profile_records.append({
        "Column": column,
        "DataType": str(series.dtype),
        "NonNullCount": int(series.notna().sum()),
        "MissingCount": missing_count,
        "MissingPercentage": round(
            missing_percentage,
            2
        ),
        "UniqueNonNullValues": unique_non_null,
        "UniqueIncludingNull": unique_including_null,
        "ExampleValues": create_sample_values(series)
    })


column_profile_df = pd.DataFrame(profile_records)

# Add the agreed forecasting role beside each column
column_profile_df = column_profile_df.merge(
    column_role_df,
    on="Column",
    how="left",
    validate="one_to_one"
)

assert column_profile_df["Role"].isna().sum() == 0
assert len(column_profile_df) == 38

print("=" * 75)
print("COLUMN PROFILE COMPLETED")
print("=" * 75)

print()
print("Columns containing missing values:")

missing_column_profile = (
    column_profile_df[
        column_profile_df["MissingCount"] > 0
    ]
    .sort_values(
        by="MissingPercentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    missing_column_profile[
        [
            "Column",
            "MissingCount",
            "MissingPercentage",
            "UniqueNonNullValues",
            "ModelUse",
            "ExampleValues"
        ]
    ]
)

print()
print("Complete column profile:")

display(column_profile_df)

COLUMN PROFILE COMPLETED

Columns containing missing values:


,Column,MissingCount,MissingPercentage,UniqueNonNullValues,ModelUse,ExampleValues
0,TierProductFamily,23926,94.18,5,CANDIDATE_CONDITIONAL,KIMBOX | MAINS | VEGETARIAN MAINS
1,NominalPriceTier,23926,94.18,3,CANDIDATE_CONDITIONAL,5.0 | 7.0 | 9.0
2,MenuGeneration,23926,94.18,2,CANDIDATE_CONDITIONAL,LEGACY_LEVEL_LABELLED | CURRENT_PRICE_LABELLED
3,BeverageSeries,23827,93.79,1,CANDIDATE_CONDITIONAL,BRANDED
4,BeverageType,23827,93.79,9,CANDIDATE_CONDITIONAL,AMERICANO | CAPPUCCINO | FLAT WHITE
5,SupplierLabelsObserved,23827,93.79,4,CANDIDATE_CONDITIONAL,BEWLEYS | CT | RN | BEWLEYS | RN | CT | RN



Complete column profile:


,Column,DataType,NonNullCount,MissingCount,MissingPercentage,UniqueNonNullValues,UniqueIncludingNull,ExampleValues,Role,ModelUse,Reason
0,Date,datetime64[ns],25405,0,0.00,245,245,2025-04-01 | 2025-04-02 | 2025-04-03,KEY,RETAIN,Operating-date key; known before forecasting.
1,CanonicalProductID,string,25405,0,0.00,227,227,BEV_BRANDED_AMERICANO | BEV_BRANDED_CAPPUCCINO...,SERIES_KEY,RETAIN,Stable product-series identifier and primary p...
2,CanonicalProductName,string,25405,0,0.00,218,218,B-AMERICANO | B-CAPPUCCINO | B-FLAT WHITE,LABEL,SUPPORT_ONLY,Readable product label; not sufficiently uniqu...
3,ProductFirstObservedDate,object,25405,0,0.00,53,53,2025-04-01 | 2025-04-02 | 2025-04-03,LIFECYCLE_SUPPORT,SUPPORT_ONLY,Used for lifecycle and cold-start checks; not ...
4,ProductAgeOperatingDays,int64,25405,0,0.00,245,245,0 | 1 | 2,TEMPORAL_FEATURE,CANDIDATE,Product age measured in operating days.
5,SourcePLUCount,int64,25405,0,0.00,3,3,3 | 2 | 1,PRODUCT_METADATA,CANDIDATE,Number of source PLUs mapped to the canonical ...
6,SourcePLUCodes,object,25405,0,0.00,227,227,"44381, 425531, 42527101 | 44382, 42527103 | 41...",PRODUCT_METADATA_TEXT,EXCLUDE_HIGH_CARDINALITY,Raw source identifiers retained for audit only.
7,SourcePLUNames,object,25405,0,0.00,218,218,BEWLEYS AMERICANO | RN AMERICANO | ROASTED NOT...,PRODUCT_METADATA_TEXT,EXCLUDE_HIGH_CARDINALITY,Raw source names retained for audit only.
8,SourceGroupCodes,int64,25405,0,0.00,10,10,1 | 2 | 15,PRODUCT_METADATA,CANDIDATE,Product-group code metadata.
9,SourceGroupNames,object,25405,0,0.00,10,10,HOT BEVS | COLD BEVS | HOSPITALITY,PRODUCT_METADATA,CANDIDATE_CONDITIONAL,Readable product-group metadata; select either...


In [10]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 2
# Cell 8: Validate leakage decisions, save audits,
# and update the Markdown handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm target-component relationship
# ------------------------------------------------------------

target_component_mismatches = int(
    (
        daily_demand_df["TotalDemand"]
        != (
            daily_demand_df["NormalDemand"]
            + daily_demand_df["BulkDemand"]
        )
    ).sum()
)

# ------------------------------------------------------------
# 2. Confirm same-day outcome leakage
# ------------------------------------------------------------

zero_flag_mismatches = int(
    (
        daily_demand_df["IsZeroDemandRow"]
        != daily_demand_df["TotalDemand"].eq(0)
    ).sum()
)

observed_flag_mismatches = int(
    (
        daily_demand_df["IsObservedProductDate"]
        != daily_demand_df["TotalDemand"].gt(0)
    ).sum()
)

expected_record_source = np.where(
    daily_demand_df["TotalDemand"].gt(0),
    "OBSERVED_TRANSACTION_AGGREGATION",
    "ZERO_FILLED_ACTIVE_OPERATING_DATE"
)

record_source_mismatches = int(
    (
        daily_demand_df["DemandRecordSource"]
        != expected_record_source
    ).sum()
)


# ------------------------------------------------------------
# 3. Validate the leakage findings
# ------------------------------------------------------------

assert target_component_mismatches == 0, (
    "TotalDemand does not consistently equal "
    "NormalDemand + BulkDemand."
)

assert zero_flag_mismatches == 0, (
    "IsZeroDemandRow does not consistently encode TotalDemand."
)

assert observed_flag_mismatches == 0, (
    "IsObservedProductDate does not consistently encode "
    "positive demand."
)

assert record_source_mismatches == 0, (
    "DemandRecordSource does not consistently encode the "
    "same-day demand outcome."
)


# ------------------------------------------------------------
# 4. Create explicit model lists
# ------------------------------------------------------------

FORECAST_KEYS = [
    "Date",
    "CanonicalProductID"
]

FORECAST_LABEL_COLUMNS = [
    "CanonicalProductName"
]

FORECAST_TARGET = [
    "TotalDemand"
]

LEAKAGE_COLUMNS = [
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
]

DIRECT_CANDIDATE_FEATURES = (
    column_role_df.loc[
        column_role_df["ModelUse"] == "CANDIDATE",
        "Column"
    ]
    .tolist()
)

CONDITIONAL_CANDIDATE_FEATURES = (
    column_role_df.loc[
        column_role_df["ModelUse"]
        == "CANDIDATE_CONDITIONAL",
        "Column"
    ]
    .tolist()
)

NON_MODEL_COLUMNS = (
    column_role_df.loc[
        column_role_df["ModelUse"].isin([
            "SUPPORT_ONLY",
            "EXCLUDE_HIGH_CARDINALITY",
            "EXCLUDE_REDUNDANT",
            "EXCLUDE_VERSION"
        ]),
        "Column"
    ]
    .tolist()
)


# ------------------------------------------------------------
# 5. Save the audit outputs
# ------------------------------------------------------------

COLUMN_ROLE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "02_column_role_audit.csv"
)

COLUMN_PROFILE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "02_column_profile_audit.csv"
)

LEAKAGE_REGISTER_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "02_model_leakage_exclusion_register.csv"
)

column_role_df.to_csv(
    COLUMN_ROLE_OUTPUT,
    index=False
)

column_profile_df.to_csv(
    COLUMN_PROFILE_OUTPUT,
    index=False
)

leakage_register_df = column_role_df[
    column_role_df["ModelUse"]
    == "EXCLUDE_LEAKAGE"
].copy()

leakage_register_df.to_csv(
    LEAKAGE_REGISTER_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 6. Reload outputs for save verification
# ------------------------------------------------------------

saved_role_check = pd.read_csv(
    COLUMN_ROLE_OUTPUT
)

saved_profile_check = pd.read_csv(
    COLUMN_PROFILE_OUTPUT
)

saved_leakage_check = pd.read_csv(
    LEAKAGE_REGISTER_OUTPUT
)

assert len(saved_role_check) == 38
assert len(saved_profile_check) == 38
assert len(saved_leakage_check) == 5

assert set(
    saved_leakage_check["Column"]
) == set(LEAKAGE_COLUMNS)


# ------------------------------------------------------------
# 7. Update the Markdown handoff
# ------------------------------------------------------------

part_2_summary = f"""
**Status:** Completed and validated

### Locked modelling definitions

- Date key: `Date`
- Product-series key: `CanonicalProductID`
- Product label: `CanonicalProductName`
- Forecast target: `TotalDemand`

### Target leakage exclusions

The following columns must not be used as predictive inputs:

{chr(10).join(f"- `{column}`" for column in LEAKAGE_COLUMNS)}

Validation confirmed:

- `TotalDemand = NormalDemand + BulkDemand`
- `IsZeroDemandRow` directly identifies zero TotalDemand
- `IsObservedProductDate` directly identifies positive TotalDemand
- `DemandRecordSource` identifies observed versus zero-filled outcomes
- Leakage validation mismatches: 0

### Column-role results

- Total classified columns: 38
- Direct candidate features: {len(DIRECT_CANDIDATE_FEATURES)}
- Conditional candidate features: {len(CONDITIONAL_CANDIDATE_FEATURES)}
- Leakage columns excluded: {len(LEAKAGE_COLUMNS)}

### Saved audit outputs

- `02_column_role_audit.csv`
- `02_column_profile_audit.csv`
- `02_model_leakage_exclusion_register.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_2",
    section_title=(
        "Forecasting Preparation — Step 1, Part 2"
    ),
    section_body=part_2_summary
)


# ------------------------------------------------------------
# 8. Print final Part 2 summary
# ------------------------------------------------------------

print("=" * 75)
print("FORECASTING PREPARATION — STEP 1, PART 2 COMPLETED")
print("=" * 75)

print()
print(f"Columns classified: {len(column_role_df)}")
print(
    f"Direct candidate features: "
    f"{len(DIRECT_CANDIDATE_FEATURES)}"
)
print(
    f"Conditional candidate features: "
    f"{len(CONDITIONAL_CANDIDATE_FEATURES)}"
)
print(f"Leakage columns excluded: {len(LEAKAGE_COLUMNS)}")

print()
print("Leakage validation mismatches:")
print(
    f"Target-component mismatches: "
    f"{target_component_mismatches}"
)
print(
    f"Zero-demand flag mismatches: "
    f"{zero_flag_mismatches}"
)
print(
    f"Observed-demand flag mismatches: "
    f"{observed_flag_mismatches}"
)
print(
    f"Record-source mismatches: "
    f"{record_source_mismatches}"
)

print()
print("Saved files:")
print(COLUMN_ROLE_OUTPUT)
print(COLUMN_PROFILE_OUTPUT)
print(LEAKAGE_REGISTER_OUTPUT)
print(HANDOFF_FILE)

print()
print("All Step 1, Part 2 validation checks passed.")

FORECASTING PREPARATION — STEP 1, PART 2 COMPLETED

Columns classified: 38
Direct candidate features: 15
Conditional candidate features: 7
Leakage columns excluded: 5

Leakage validation mismatches:
Target-component mismatches: 0
Zero-demand flag mismatches: 0
Observed-demand flag mismatches: 0
Record-source mismatches: 0

Saved files:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/02_column_role_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/02_column_profile_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/02_model_leakage_exclusion_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/FORECASTING_PREPARATION_HANDOFF.md

All Step 1, Part 2 validation checks passed.


In [11]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 2
# Cell 8: Validate leakage decisions, save audit outputs,
# and update the Markdown handoff
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Confirm that the required earlier objects exist
# ------------------------------------------------------------

required_objects = [
    "daily_demand_df",
    "column_role_df",
    "column_profile_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "The following required objects are missing:\n"
        f"{missing_objects}\n\n"
        "Run Cells 5, 6 and 7 before running Cell 8."
    )


# ------------------------------------------------------------
# 2. Validate the target-component relationship
# ------------------------------------------------------------

target_component_mismatches = int(
    (
        daily_demand_df["TotalDemand"]
        != (
            daily_demand_df["NormalDemand"]
            + daily_demand_df["BulkDemand"]
        )
    ).sum()
)


# ------------------------------------------------------------
# 3. Validate same-day outcome leakage
# ------------------------------------------------------------

zero_flag_mismatches = int(
    (
        daily_demand_df["IsZeroDemandRow"]
        != daily_demand_df["TotalDemand"].eq(0)
    ).sum()
)

observed_flag_mismatches = int(
    (
        daily_demand_df["IsObservedProductDate"]
        != daily_demand_df["TotalDemand"].gt(0)
    ).sum()
)

expected_record_source = np.where(
    daily_demand_df["TotalDemand"].gt(0),
    "OBSERVED_TRANSACTION_AGGREGATION",
    "ZERO_FILLED_ACTIVE_OPERATING_DATE"
)

record_source_mismatches = int(
    (
        daily_demand_df["DemandRecordSource"]
        .astype(str)
        != expected_record_source
    ).sum()
)


# ------------------------------------------------------------
# 4. Stop immediately if any leakage relationship fails
# ------------------------------------------------------------

assert target_component_mismatches == 0, (
    "TotalDemand does not consistently equal "
    "NormalDemand + BulkDemand."
)

assert zero_flag_mismatches == 0, (
    "IsZeroDemandRow does not consistently identify "
    "rows where TotalDemand equals zero."
)

assert observed_flag_mismatches == 0, (
    "IsObservedProductDate does not consistently identify "
    "rows where TotalDemand is greater than zero."
)

assert record_source_mismatches == 0, (
    "DemandRecordSource does not consistently represent "
    "observed and zero-filled product-date rows."
)


# ------------------------------------------------------------
# 5. Create the official modelling column lists
# ------------------------------------------------------------

FORECAST_KEYS = [
    "Date",
    "CanonicalProductID"
]

FORECAST_LABEL_COLUMNS = [
    "CanonicalProductName"
]

FORECAST_TARGET = "TotalDemand"

LEAKAGE_COLUMNS = [
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
]

DIRECT_CANDIDATE_FEATURES = (
    column_role_df.loc[
        column_role_df["ModelUse"].eq("CANDIDATE"),
        "Column"
    ]
    .tolist()
)

CONDITIONAL_CANDIDATE_FEATURES = (
    column_role_df.loc[
        column_role_df["ModelUse"].eq(
            "CANDIDATE_CONDITIONAL"
        ),
        "Column"
    ]
    .tolist()
)

NON_MODEL_COLUMNS = (
    column_role_df.loc[
        column_role_df["ModelUse"].isin([
            "SUPPORT_ONLY",
            "EXCLUDE_HIGH_CARDINALITY",
            "EXCLUDE_REDUNDANT",
            "EXCLUDE_VERSION"
        ]),
        "Column"
    ]
    .tolist()
)


# ------------------------------------------------------------
# 6. Validate the official column lists
# ------------------------------------------------------------

assert FORECAST_TARGET in daily_demand_df.columns

assert set(FORECAST_KEYS).issubset(
    daily_demand_df.columns
)

assert set(FORECAST_LABEL_COLUMNS).issubset(
    daily_demand_df.columns
)

assert set(LEAKAGE_COLUMNS).issubset(
    daily_demand_df.columns
)

assert len(DIRECT_CANDIDATE_FEATURES) == 15, (
    "Expected 15 direct candidate features, but found "
    f"{len(DIRECT_CANDIDATE_FEATURES)}."
)

assert len(CONDITIONAL_CANDIDATE_FEATURES) == 7, (
    "Expected 7 conditional candidate features, but found "
    f"{len(CONDITIONAL_CANDIDATE_FEATURES)}."
)

assert len(LEAKAGE_COLUMNS) == 5


# ------------------------------------------------------------
# 7. Create the leakage-exclusion register
# ------------------------------------------------------------

leakage_register_df = (
    column_role_df.loc[
        column_role_df["Column"].isin(LEAKAGE_COLUMNS)
    ]
    .copy()
    .sort_values("Column")
    .reset_index(drop=True)
)

assert len(leakage_register_df) == 5

assert set(
    leakage_register_df["Column"]
) == set(LEAKAGE_COLUMNS)


# ------------------------------------------------------------
# 8. Define Part 2 output paths
# ------------------------------------------------------------

COLUMN_ROLE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "02_column_role_audit.csv"
)

COLUMN_PROFILE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "02_column_profile_audit.csv"
)

LEAKAGE_REGISTER_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "02_model_leakage_exclusion_register.csv"
)


# ------------------------------------------------------------
# 9. Save all Part 2 audit outputs
# ------------------------------------------------------------

column_role_df.to_csv(
    COLUMN_ROLE_OUTPUT,
    index=False
)

column_profile_df.to_csv(
    COLUMN_PROFILE_OUTPUT,
    index=False
)

leakage_register_df.to_csv(
    LEAKAGE_REGISTER_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 10. Reload the saved files and verify them
# ------------------------------------------------------------

saved_role_check = pd.read_csv(
    COLUMN_ROLE_OUTPUT,
    low_memory=False
)

saved_profile_check = pd.read_csv(
    COLUMN_PROFILE_OUTPUT,
    low_memory=False
)

saved_leakage_check = pd.read_csv(
    LEAKAGE_REGISTER_OUTPUT,
    low_memory=False
)

assert COLUMN_ROLE_OUTPUT.exists()
assert COLUMN_PROFILE_OUTPUT.exists()
assert LEAKAGE_REGISTER_OUTPUT.exists()

assert len(saved_role_check) == 38
assert len(saved_profile_check) == 38
assert len(saved_leakage_check) == 5

assert saved_role_check["Column"].nunique() == 38
assert saved_profile_check["Column"].nunique() == 38

assert set(
    saved_leakage_check["Column"]
) == set(LEAKAGE_COLUMNS)


# ------------------------------------------------------------
# 11. Update the Markdown project handoff
# ------------------------------------------------------------

leakage_markdown = "\n".join(
    f"- `{column}`"
    for column in LEAKAGE_COLUMNS
)

direct_feature_markdown = "\n".join(
    f"- `{column}`"
    for column in DIRECT_CANDIDATE_FEATURES
)

conditional_feature_markdown = "\n".join(
    f"- `{column}`"
    for column in CONDITIONAL_CANDIDATE_FEATURES
)

part_2_summary = f"""
**Status:** Completed and validated

### Locked modelling definitions

- Date key: `Date`
- Product-series key: `CanonicalProductID`
- Human-readable product label: `CanonicalProductName`
- Forecasting target: `TotalDemand`

### Target leakage exclusions

The following same-day outcome columns must not be used as
predictive model inputs:

{leakage_markdown}

### Leakage validation results

- `TotalDemand = NormalDemand + BulkDemand`
- Target-component mismatches: {target_component_mismatches}
- Zero-demand flag mismatches: {zero_flag_mismatches}
- Observed-demand flag mismatches: {observed_flag_mismatches}
- Demand-record-source mismatches: {record_source_mismatches}

### Direct candidate features

{direct_feature_markdown}

### Conditional candidate features

{conditional_feature_markdown}

The conditional features contain substantial not-applicable
missingness and will be evaluated separately rather than being
automatically included in the primary model.

### Column-role results

- Dataset columns classified: {len(column_role_df)}
- Direct candidate features: {len(DIRECT_CANDIDATE_FEATURES)}
- Conditional candidate features: {len(CONDITIONAL_CANDIDATE_FEATURES)}
- Leakage columns excluded: {len(LEAKAGE_COLUMNS)}

### Saved Part 2 outputs

- `02_column_role_audit.csv`
- `02_column_profile_audit.csv`
- `02_model_leakage_exclusion_register.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_2",
    section_title=(
        "Forecasting Preparation — Step 1, Part 2"
    ),
    section_body=part_2_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 12. Print final Part 2 results
# ------------------------------------------------------------

print("=" * 75)
print("FORECASTING PREPARATION — STEP 1, PART 2 COMPLETED")
print("=" * 75)

print()
print("Locked modelling definitions:")
print(f"Date key: {FORECAST_KEYS[0]}")
print(f"Product key: {FORECAST_KEYS[1]}")
print(
    f"Product label: "
    f"{FORECAST_LABEL_COLUMNS[0]}"
)
print(f"Forecast target: {FORECAST_TARGET}")

print()
print("Column-role summary:")
print(f"Columns classified: {len(column_role_df)}")
print(
    "Direct candidate features:",
    len(DIRECT_CANDIDATE_FEATURES)
)
print(
    "Conditional candidate features:",
    len(CONDITIONAL_CANDIDATE_FEATURES)
)
print(
    "Leakage columns excluded:",
    len(LEAKAGE_COLUMNS)
)

print()
print("Leakage validation mismatches:")
print(
    "Target-component mismatches:",
    target_component_mismatches
)
print(
    "Zero-demand flag mismatches:",
    zero_flag_mismatches
)
print(
    "Observed-demand flag mismatches:",
    observed_flag_mismatches
)
print(
    "Record-source mismatches:",
    record_source_mismatches
)

print()
print("Saved files:")
print(f"1. {COLUMN_ROLE_OUTPUT}")
print(f"2. {COLUMN_PROFILE_OUTPUT}")
print(f"3. {LEAKAGE_REGISTER_OUTPUT}")
print(f"4. {HANDOFF_FILE}")

print()
print(
    "All Step 1, Part 2 validation checks passed."
)

FORECASTING PREPARATION — STEP 1, PART 2 COMPLETED

Locked modelling definitions:
Date key: Date
Product key: CanonicalProductID
Product label: CanonicalProductName
Forecast target: TotalDemand

Column-role summary:
Columns classified: 38
Direct candidate features: 15
Conditional candidate features: 7
Leakage columns excluded: 5

Leakage validation mismatches:
Target-component mismatches: 0
Zero-demand flag mismatches: 0
Observed-demand flag mismatches: 0
Record-source mismatches: 0

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/02_column_role_audit.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/02_column_profile_audit.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/02_model_leakage_exclusion_register.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/FORECASTING_PREPARATION_HANDOFF.md

All Step 1, Part 2 validation checks passed.


In [17]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 3
# Cell 9: Prepare and validate the temporal panel
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Confirm required objects exist
# ------------------------------------------------------------

required_objects = [
    "daily_demand_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "The following required objects are missing:\n"
        f"{missing_objects}\n\n"
        "Run the previous forecasting-preparation cells "
        "before running Cell 9."
    )


# ------------------------------------------------------------
# 2. Create a separate temporal working copy
# ------------------------------------------------------------

temporal_panel_df = daily_demand_df.copy()

required_temporal_columns = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "ProductAgeOperatingDays",
    "OperatingDaySequence",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay",
    "TotalDemand"
]

missing_temporal_columns = [
    column
    for column in required_temporal_columns
    if column not in temporal_panel_df.columns
]

if missing_temporal_columns:
    raise KeyError(
        "The following temporal columns are missing:\n"
        f"{missing_temporal_columns}"
    )


# ------------------------------------------------------------
# 3. Parse date columns safely
# ------------------------------------------------------------

temporal_panel_df["Date"] = pd.to_datetime(
    temporal_panel_df["Date"],
    errors="coerce"
)

temporal_panel_df["ProductFirstObservedDate"] = pd.to_datetime(
    temporal_panel_df["ProductFirstObservedDate"],
    errors="coerce"
)

invalid_date_count = int(
    temporal_panel_df["Date"].isna().sum()
)

invalid_first_observed_date_count = int(
    temporal_panel_df["ProductFirstObservedDate"]
    .isna()
    .sum()
)

assert invalid_date_count == 0, (
    f"Found {invalid_date_count} invalid Date values."
)

assert invalid_first_observed_date_count == 0, (
    "Found "
    f"{invalid_first_observed_date_count} invalid "
    "ProductFirstObservedDate values."
)


# ------------------------------------------------------------
# 4. Convert temporal numeric columns
# ------------------------------------------------------------

numeric_temporal_columns = [
    "ProductAgeOperatingDays",
    "OperatingDaySequence",
    "DaysSincePreviousOperatingDate",
    "TotalDemand"
]

for column in numeric_temporal_columns:
    temporal_panel_df[column] = pd.to_numeric(
        temporal_panel_df[column],
        errors="coerce"
    )

missing_numeric_counts = {
    column: int(temporal_panel_df[column].isna().sum())
    for column in numeric_temporal_columns
}

assert all(
    missing_count == 0
    for missing_count in missing_numeric_counts.values()
), (
    "Missing or non-numeric temporal values were detected:\n"
    f"{missing_numeric_counts}"
)


# ------------------------------------------------------------
# 5. Standardise product identifiers
# ------------------------------------------------------------

temporal_panel_df["CanonicalProductID"] = (
    temporal_panel_df["CanonicalProductID"]
    .astype("string")
    .str.strip()
)

temporal_panel_df["CanonicalProductName"] = (
    temporal_panel_df["CanonicalProductName"]
    .astype("string")
    .str.strip()
)

assert temporal_panel_df[
    "CanonicalProductID"
].isna().sum() == 0

assert temporal_panel_df[
    "CanonicalProductName"
].isna().sum() == 0


# ------------------------------------------------------------
# 6. Sort by product and chronological operating sequence
# ------------------------------------------------------------

temporal_panel_df = (
    temporal_panel_df
    .sort_values(
        by=[
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Validate unique product-date records
# ------------------------------------------------------------

duplicate_product_date_rows = int(
    temporal_panel_df.duplicated(
        subset=[
            "CanonicalProductID",
            "Date"
        ],
        keep=False
    ).sum()
)

duplicate_product_sequence_rows = int(
    temporal_panel_df.duplicated(
        subset=[
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        keep=False
    ).sum()
)

assert duplicate_product_date_rows == 0, (
    "Duplicate CanonicalProductID-Date rows were found."
)

assert duplicate_product_sequence_rows == 0, (
    "Duplicate CanonicalProductID-OperatingDaySequence "
    "rows were found."
)


# ------------------------------------------------------------
# 8. Validate global date-to-operating-sequence mapping
# ------------------------------------------------------------

sequence_counts_per_date = (
    temporal_panel_df
    .groupby("Date")["OperatingDaySequence"]
    .nunique()
)

date_counts_per_sequence = (
    temporal_panel_df
    .groupby("OperatingDaySequence")["Date"]
    .nunique()
)

dates_with_multiple_sequences = int(
    (sequence_counts_per_date > 1).sum()
)

sequences_with_multiple_dates = int(
    (date_counts_per_sequence > 1).sum()
)

assert dates_with_multiple_sequences == 0, (
    "At least one date is linked to multiple "
    "OperatingDaySequence values."
)

assert sequences_with_multiple_dates == 0, (
    "At least one OperatingDaySequence value is linked "
    "to multiple dates."
)


# ------------------------------------------------------------
# 9. Validate chronological ordering within products
# ------------------------------------------------------------

date_difference = (
    temporal_panel_df
    .groupby("CanonicalProductID")["Date"]
    .diff()
)

sequence_difference = (
    temporal_panel_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .diff()
)

non_increasing_date_rows = int(
    (date_difference.dt.days <= 0).sum()
)

non_increasing_sequence_rows = int(
    (sequence_difference <= 0).sum()
)

assert non_increasing_date_rows == 0, (
    "One or more product series are not ordered "
    "chronologically by Date."
)

assert non_increasing_sequence_rows == 0, (
    "One or more product series have non-increasing "
    "OperatingDaySequence values."
)


# ------------------------------------------------------------
# 10. Validate product metadata stability
# ------------------------------------------------------------

product_name_counts = (
    temporal_panel_df
    .groupby("CanonicalProductID")[
        "CanonicalProductName"
    ]
    .nunique()
)

first_observed_date_counts = (
    temporal_panel_df
    .groupby("CanonicalProductID")[
        "ProductFirstObservedDate"
    ]
    .nunique()
)

product_ids_with_multiple_names = int(
    (product_name_counts > 1).sum()
)

product_ids_with_multiple_first_dates = int(
    (first_observed_date_counts > 1).sum()
)

assert product_ids_with_multiple_names == 0, (
    "At least one CanonicalProductID has multiple "
    "CanonicalProductName values."
)

assert product_ids_with_multiple_first_dates == 0, (
    "At least one CanonicalProductID has multiple "
    "ProductFirstObservedDate values."
)


# ------------------------------------------------------------
# 11. Validate lifecycle dates
# ------------------------------------------------------------

rows_before_first_observed_date = int(
    (
        temporal_panel_df["Date"]
        < temporal_panel_df["ProductFirstObservedDate"]
    ).sum()
)

assert rows_before_first_observed_date == 0, (
    "Product rows were found before ProductFirstObservedDate."
)

actual_first_panel_date = (
    temporal_panel_df
    .groupby("CanonicalProductID")["Date"]
    .transform("min")
)

first_date_mismatches = int(
    (
        temporal_panel_df["ProductFirstObservedDate"]
        != actual_first_panel_date
    ).sum()
)

assert first_date_mismatches == 0, (
    "ProductFirstObservedDate does not match the first "
    "panel date for one or more products."
)


# ------------------------------------------------------------
# 12. Validate ProductAgeOperatingDays
# ------------------------------------------------------------

first_product_sequence = (
    temporal_panel_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .transform("min")
)

expected_product_age = (
    temporal_panel_df["OperatingDaySequence"]
    - first_product_sequence
)

product_age_mismatches = int(
    (
        temporal_panel_df["ProductAgeOperatingDays"]
        != expected_product_age
    ).sum()
)

negative_product_age_rows = int(
    (
        temporal_panel_df["ProductAgeOperatingDays"] < 0
    ).sum()
)

assert product_age_mismatches == 0, (
    "ProductAgeOperatingDays is inconsistent with "
    "OperatingDaySequence."
)

assert negative_product_age_rows == 0, (
    "Negative ProductAgeOperatingDays values were found."
)


# ------------------------------------------------------------
# 13. Detect missing operating dates inside product series
# ------------------------------------------------------------

temporal_panel_df[
    "PreviousProductOperatingSequence"
] = (
    temporal_panel_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .shift(1)
)

temporal_panel_df[
    "ProductOperatingSequenceGap"
] = (
    temporal_panel_df["OperatingDaySequence"]
    - temporal_panel_df[
        "PreviousProductOperatingSequence"
    ]
)

internal_gap_mask = (
    temporal_panel_df[
        "PreviousProductOperatingSequence"
    ].notna()
    & (
        temporal_panel_df[
            "ProductOperatingSequenceGap"
        ] != 1
    )
)

internal_gap_row_count = int(
    internal_gap_mask.sum()
)

products_with_internal_gaps = int(
    temporal_panel_df.loc[
        internal_gap_mask,
        "CanonicalProductID"
    ].nunique()
)


# ------------------------------------------------------------
# 14. Build product-series summary
# ------------------------------------------------------------

product_series_summary_df = (
    temporal_panel_df
    .groupby(
        [
            "CanonicalProductID",
            "CanonicalProductName"
        ],
        as_index=False
    )
    .agg(
        ProductFirstDate=("Date", "min"),
        ProductLastDate=("Date", "max"),
        FirstOperatingDaySequence=(
            "OperatingDaySequence",
            "min"
        ),
        LastOperatingDaySequence=(
            "OperatingDaySequence",
            "max"
        ),
        ProductSeriesRows=("Date", "size"),
        PositiveDemandDays=(
            "TotalDemand",
            lambda series: int((series > 0).sum())
        ),
        ZeroDemandDays=(
            "TotalDemand",
            lambda series: int((series == 0).sum())
        ),
        TotalDemandUnits=("TotalDemand", "sum"),
        MeanDailyDemand=("TotalDemand", "mean"),
        MaximumDailyDemand=("TotalDemand", "max")
    )
)

product_series_summary_df[
    "ExpectedSeriesRows"
] = (
    product_series_summary_df[
        "LastOperatingDaySequence"
    ]
    - product_series_summary_df[
        "FirstOperatingDaySequence"
    ]
    + 1
)

product_series_summary_df[
    "MissingInternalOperatingDates"
] = (
    product_series_summary_df[
        "ExpectedSeriesRows"
    ]
    - product_series_summary_df[
        "ProductSeriesRows"
    ]
)

product_series_summary_df[
    "PositiveDemandRate"
] = (
    product_series_summary_df[
        "PositiveDemandDays"
    ]
    / product_series_summary_df[
        "ProductSeriesRows"
    ]
).round(4)

products_with_missing_panel_rows = int(
    (
        product_series_summary_df[
            "MissingInternalOperatingDates"
        ] != 0
    ).sum()
)


# ------------------------------------------------------------
# 15. Create internal-gap audit table
# ------------------------------------------------------------

product_series_gap_audit_df = (
    temporal_panel_df.loc[
        internal_gap_mask,
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "Date",
            "OperatingDaySequence",
            "PreviousProductOperatingSequence",
            "ProductOperatingSequenceGap"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 16. Final panel-level validation
# ------------------------------------------------------------

assert len(temporal_panel_df) == 25_405
assert temporal_panel_df["Date"].nunique() == 245
assert temporal_panel_df[
    "CanonicalProductID"
].nunique() == 227

assert internal_gap_row_count == 0, (
    f"Found {internal_gap_row_count} internal product-series gaps."
)

assert products_with_internal_gaps == 0, (
    f"Found {products_with_internal_gaps} products with "
    "internal operating-date gaps."
)

assert products_with_missing_panel_rows == 0, (
    f"Found {products_with_missing_panel_rows} products whose "
    "panel row count does not match the expected operating span."
)


# ------------------------------------------------------------
# 17. Remove temporary calculation columns
# ------------------------------------------------------------

temporal_panel_df = temporal_panel_df.drop(
    columns=[
        "PreviousProductOperatingSequence",
        "ProductOperatingSequenceGap"
    ]
)


# ------------------------------------------------------------
# 18. Print the Cell 9 validation summary
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 3 — TEMPORAL PANEL PREPARATION PASSED")
print("=" * 75)

print()
print("Panel structure:")
print(f"Rows: {len(temporal_panel_df):,}")
print(
    "Canonical products:",
    f"{temporal_panel_df['CanonicalProductID'].nunique():,}"
)
print(
    "Operating dates:",
    f"{temporal_panel_df['Date'].nunique():,}"
)
print(
    "Date range:",
    temporal_panel_df["Date"].min().date(),
    "to",
    temporal_panel_df["Date"].max().date()
)

print()
print("Temporal validation:")
print(
    "Duplicate product-date rows:",
    duplicate_product_date_rows
)
print(
    "Duplicate product-sequence rows:",
    duplicate_product_sequence_rows
)
print(
    "Dates linked to multiple sequences:",
    dates_with_multiple_sequences
)
print(
    "Sequences linked to multiple dates:",
    sequences_with_multiple_dates
)
print(
    "Non-increasing product dates:",
    non_increasing_date_rows
)
print(
    "Non-increasing product sequences:",
    non_increasing_sequence_rows
)

print()
print("Product lifecycle validation:")
print(
    "Products with multiple names:",
    product_ids_with_multiple_names
)
print(
    "Products with multiple first-observed dates:",
    product_ids_with_multiple_first_dates
)
print(
    "Rows before first-observed date:",
    rows_before_first_observed_date
)
print(
    "First-observed-date mismatches:",
    first_date_mismatches
)
print(
    "Product-age mismatches:",
    product_age_mismatches
)

print()
print("Series continuity:")
print(
    "Internal gap rows:",
    internal_gap_row_count
)
print(
    "Products with internal gaps:",
    products_with_internal_gaps
)
print(
    "Products with missing panel rows:",
    products_with_missing_panel_rows
)

print()
print("Cell 9 completed successfully.")

STEP 1, PART 3 — TEMPORAL PANEL PREPARATION PASSED

Panel structure:
Rows: 25,405
Canonical products: 227
Operating dates: 245
Date range: 2025-04-01 to 2026-03-30

Temporal validation:
Duplicate product-date rows: 0
Duplicate product-sequence rows: 0
Dates linked to multiple sequences: 0
Sequences linked to multiple dates: 0
Non-increasing product dates: 0
Non-increasing product sequences: 0

Product lifecycle validation:
Products with multiple names: 0
Products with multiple first-observed dates: 0
Rows before first-observed date: 0
First-observed-date mismatches: 0
Product-age mismatches: 0

Series continuity:
Internal gap rows: 0
Products with internal gaps: 0
Products with missing panel rows: 0

Cell 9 completed successfully.


In [18]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 3
# Cell 10: Validate operating calendar and product lifecycles
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Confirm Cell 9 objects exist
# ------------------------------------------------------------

required_cell_9_objects = [
    "temporal_panel_df",
    "product_series_summary_df",
    "product_series_gap_audit_df"
]

missing_cell_9_objects = [
    object_name
    for object_name in required_cell_9_objects
    if object_name not in globals()
]

if missing_cell_9_objects:
    raise NameError(
        "The following Cell 9 objects are missing:\n"
        f"{missing_cell_9_objects}\n\n"
        "Run Cell 9 before running Cell 10."
    )


# ------------------------------------------------------------
# 2. Confirm calendar columns are stable within each date
# ------------------------------------------------------------

calendar_columns = [
    "OperatingDaySequence",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay"
]

calendar_values_per_date = (
    temporal_panel_df
    .groupby("Date")[calendar_columns]
    .nunique(dropna=False)
)

unstable_calendar_dates = int(
    calendar_values_per_date.gt(1).any(axis=1).sum()
)

assert unstable_calendar_dates == 0, (
    f"Found {unstable_calendar_dates} dates with inconsistent "
    "operating-calendar metadata."
)


# ------------------------------------------------------------
# 3. Create one-row-per-operating-date calendar
# ------------------------------------------------------------

operating_calendar_audit_df = (
    temporal_panel_df[
        [
            "Date",
            "OperatingDaySequence",
            "DaysSincePreviousOperatingDate",
            "IsConsecutiveCalendarDay",
            "Year",
            "Month",
            "MonthName",
            "Quarter",
            "DayOfWeekNumber",
            "DayOfWeek",
            "ISOYear",
            "ISOWeek",
            "DayOfYear",
            "IsWeekend"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=[
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert len(operating_calendar_audit_df) == 245, (
    "Expected 245 operating-calendar rows, but found "
    f"{len(operating_calendar_audit_df)}."
)

assert operating_calendar_audit_df["Date"].nunique() == 245

assert (
    operating_calendar_audit_df[
        "OperatingDaySequence"
    ].nunique()
    == 245
)


# ------------------------------------------------------------
# 4. Recalculate the expected operating sequence
# ------------------------------------------------------------

operating_calendar_audit_df[
    "ExpectedOperatingDaySequence"
] = np.arange(
    1,
    len(operating_calendar_audit_df) + 1
)

operating_sequence_mismatches = int(
    (
        operating_calendar_audit_df[
            "OperatingDaySequence"
        ]
        != operating_calendar_audit_df[
            "ExpectedOperatingDaySequence"
        ]
    ).sum()
)

assert operating_sequence_mismatches == 0, (
    "OperatingDaySequence is not a continuous sequence "
    "from 1 to 245."
)


# ------------------------------------------------------------
# 5. Recalculate calendar-day gaps
# ------------------------------------------------------------

operating_calendar_audit_df[
    "ExpectedDaysSincePreviousOperatingDate"
] = (
    operating_calendar_audit_df["Date"]
    .diff()
    .dt.days
    .fillna(0)
    .astype(int)
)

actual_gap_values = pd.to_numeric(
    operating_calendar_audit_df[
        "DaysSincePreviousOperatingDate"
    ],
    errors="coerce"
)

calendar_gap_mismatches = int(
    (
        actual_gap_values
        != operating_calendar_audit_df[
            "ExpectedDaysSincePreviousOperatingDate"
        ]
    ).sum()
)

assert actual_gap_values.isna().sum() == 0

assert calendar_gap_mismatches == 0, (
    "DaysSincePreviousOperatingDate contains incorrect values."
)


# ------------------------------------------------------------
# 6. Recalculate consecutive-calendar-day flags
# ------------------------------------------------------------

expected_consecutive_flag = (
    operating_calendar_audit_df[
        "ExpectedDaysSincePreviousOperatingDate"
    ].eq(1)
)

actual_consecutive_flag = (
    operating_calendar_audit_df[
        "IsConsecutiveCalendarDay"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
)

unrecognised_consecutive_flags = int(
    actual_consecutive_flag.isna().sum()
)

assert unrecognised_consecutive_flags == 0, (
    "Unrecognised values were found in "
    "IsConsecutiveCalendarDay."
)

consecutive_flag_mismatches = int(
    (
        actual_consecutive_flag
        != expected_consecutive_flag
    ).sum()
)

assert consecutive_flag_mismatches == 0, (
    "IsConsecutiveCalendarDay contains incorrect values."
)

operating_calendar_audit_df[
    "ExpectedIsConsecutiveCalendarDay"
] = expected_consecutive_flag


# ------------------------------------------------------------
# 7. Summarise calendar closure gaps
# ------------------------------------------------------------

calendar_gap_event_mask = (
    operating_calendar_audit_df[
        "ExpectedDaysSincePreviousOperatingDate"
    ] > 1
)

calendar_gap_event_count = int(
    calendar_gap_event_mask.sum()
)

maximum_calendar_gap = int(
    operating_calendar_audit_df[
        "ExpectedDaysSincePreviousOperatingDate"
    ].max()
)

operating_calendar_audit_df[
    "IsCalendarGapEvent"
] = calendar_gap_event_mask


# ------------------------------------------------------------
# 8. Calculate first and last positive-demand dates
# ------------------------------------------------------------

positive_demand_df = temporal_panel_df.loc[
    temporal_panel_df["TotalDemand"] > 0,
    [
        "CanonicalProductID",
        "Date"
    ]
].copy()

positive_date_summary_df = (
    positive_demand_df
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        FirstPositiveDemandDate=("Date", "min"),
        LastPositiveDemandDate=("Date", "max")
    )
)

products_without_positive_demand = (
    temporal_panel_df["CanonicalProductID"].nunique()
    - positive_date_summary_df[
        "CanonicalProductID"
    ].nunique()
)

assert products_without_positive_demand == 0, (
    f"Found {products_without_positive_demand} products "
    "without any positive demand."
)


# ------------------------------------------------------------
# 9. Add lifecycle information to the product summary
# ------------------------------------------------------------

product_temporal_summary_df = (
    product_series_summary_df
    .merge(
        positive_date_summary_df,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
    .copy()
)

global_first_operating_date = (
    operating_calendar_audit_df["Date"].min()
)

global_last_operating_date = (
    operating_calendar_audit_df["Date"].max()
)

product_temporal_summary_df[
    "IntroducedAfterDatasetStart"
] = (
    product_temporal_summary_df["ProductFirstDate"]
    > global_first_operating_date
)

product_temporal_summary_df[
    "IsActiveAtDatasetEnd"
] = (
    product_temporal_summary_df["ProductLastDate"]
    == global_last_operating_date
)

product_temporal_summary_df[
    "EndedBeforeDatasetEnd"
] = (
    product_temporal_summary_df["ProductLastDate"]
    < global_last_operating_date
)

product_temporal_summary_df[
    "SeriesCalendarSpanDays"
] = (
    product_temporal_summary_df["ProductLastDate"]
    - product_temporal_summary_df["ProductFirstDate"]
).dt.days + 1

product_temporal_summary_df[
    "ZeroDemandRate"
] = (
    product_temporal_summary_df["ZeroDemandDays"]
    / product_temporal_summary_df["ProductSeriesRows"]
).round(4)


# ------------------------------------------------------------
# 10. Validate product lifecycle windows
# ------------------------------------------------------------

first_positive_date_mismatches = int(
    (
        product_temporal_summary_df[
            "FirstPositiveDemandDate"
        ]
        != product_temporal_summary_df[
            "ProductFirstDate"
        ]
    ).sum()
)

last_positive_date_mismatches = int(
    (
        product_temporal_summary_df[
            "LastPositiveDemandDate"
        ]
        != product_temporal_summary_df[
            "ProductLastDate"
        ]
    ).sum()
)

demand_day_count_mismatches = int(
    (
        product_temporal_summary_df[
            "PositiveDemandDays"
        ]
        + product_temporal_summary_df[
            "ZeroDemandDays"
        ]
        != product_temporal_summary_df[
            "ProductSeriesRows"
        ]
    ).sum()
)

non_positive_product_total_count = int(
    (
        product_temporal_summary_df[
            "TotalDemandUnits"
        ] <= 0
    ).sum()
)

assert first_positive_date_mismatches == 0, (
    "At least one product panel begins before its first "
    "positive-demand date."
)

assert last_positive_date_mismatches == 0, (
    "At least one product panel continues beyond its last "
    "positive-demand date."
)

assert demand_day_count_mismatches == 0, (
    "PositiveDemandDays + ZeroDemandDays does not match "
    "ProductSeriesRows for one or more products."
)

assert non_positive_product_total_count == 0, (
    "A product with zero total historical demand was found."
)


# ------------------------------------------------------------
# 11. Calculate lifecycle counts
# ------------------------------------------------------------

products_present_at_dataset_start = int(
    (
        product_temporal_summary_df[
            "ProductFirstDate"
        ]
        == global_first_operating_date
    ).sum()
)

products_introduced_later = int(
    product_temporal_summary_df[
        "IntroducedAfterDatasetStart"
    ].sum()
)

products_active_at_dataset_end = int(
    product_temporal_summary_df[
        "IsActiveAtDatasetEnd"
    ].sum()
)

products_ended_before_dataset_end = int(
    product_temporal_summary_df[
        "EndedBeforeDatasetEnd"
    ].sum()
)

assert (
    products_present_at_dataset_start
    + products_introduced_later
    == 227
)

assert (
    products_active_at_dataset_end
    + products_ended_before_dataset_end
    == 227
)


# ------------------------------------------------------------
# 12. Create lifecycle-status labels
# ------------------------------------------------------------

product_temporal_summary_df[
    "LifecycleStatus"
] = np.select(
    [
        (
            ~product_temporal_summary_df[
                "IntroducedAfterDatasetStart"
            ]
            & product_temporal_summary_df[
                "IsActiveAtDatasetEnd"
            ]
        ),
        (
            product_temporal_summary_df[
                "IntroducedAfterDatasetStart"
            ]
            & product_temporal_summary_df[
                "IsActiveAtDatasetEnd"
            ]
        ),
        (
            ~product_temporal_summary_df[
                "IntroducedAfterDatasetStart"
            ]
            & product_temporal_summary_df[
                "EndedBeforeDatasetEnd"
            ]
        ),
        (
            product_temporal_summary_df[
                "IntroducedAfterDatasetStart"
            ]
            & product_temporal_summary_df[
                "EndedBeforeDatasetEnd"
            ]
        )
    ],
    [
        "PRESENT_AT_START_AND_ACTIVE_AT_END",
        "INTRODUCED_LATER_AND_ACTIVE_AT_END",
        "PRESENT_AT_START_AND_ENDED_EARLIER",
        "INTRODUCED_LATER_AND_ENDED_EARLIER"
    ],
    default="UNCLASSIFIED"
)

unclassified_lifecycle_products = int(
    (
        product_temporal_summary_df[
            "LifecycleStatus"
        ]
        == "UNCLASSIFIED"
    ).sum()
)

assert unclassified_lifecycle_products == 0


# ------------------------------------------------------------
# 13. Create lifecycle summary table
# ------------------------------------------------------------

lifecycle_status_summary_df = (
    product_temporal_summary_df[
        "LifecycleStatus"
    ]
    .value_counts()
    .rename_axis("LifecycleStatus")
    .reset_index(name="ProductCount")
)

lifecycle_status_summary_df[
    "ProductPercentage"
] = (
    lifecycle_status_summary_df["ProductCount"]
    / len(product_temporal_summary_df)
    * 100
).round(2)


# ------------------------------------------------------------
# 14. Final Cell 10 validation
# ------------------------------------------------------------

assert len(product_temporal_summary_df) == 227
assert product_temporal_summary_df[
    "CanonicalProductID"
].nunique() == 227

assert product_temporal_summary_df[
    "MissingInternalOperatingDates"
].sum() == 0

assert operating_sequence_mismatches == 0
assert calendar_gap_mismatches == 0
assert consecutive_flag_mismatches == 0


# ------------------------------------------------------------
# 15. Print Cell 10 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 3 — OPERATING CALENDAR VALIDATION PASSED")
print("=" * 75)

print()
print("Operating calendar:")
print(
    f"Operating dates: "
    f"{len(operating_calendar_audit_df):,}"
)
print(
    "Operating-day sequence:",
    int(
        operating_calendar_audit_df[
            "OperatingDaySequence"
        ].min()
    ),
    "to",
    int(
        operating_calendar_audit_df[
            "OperatingDaySequence"
        ].max()
    )
)
print(
    f"Calendar-gap events: "
    f"{calendar_gap_event_count:,}"
)
print(
    f"Maximum calendar gap: "
    f"{maximum_calendar_gap} days"
)

print()
print("Operating-calendar mismatches:")
print(
    "Operating-sequence mismatches:",
    operating_sequence_mismatches
)
print(
    "Days-since-previous-date mismatches:",
    calendar_gap_mismatches
)
print(
    "Consecutive-day flag mismatches:",
    consecutive_flag_mismatches
)

print()
print("Product lifecycle:")
print(
    "Products present at dataset start:",
    products_present_at_dataset_start
)
print(
    "Products introduced later:",
    products_introduced_later
)
print(
    "Products active at dataset end:",
    products_active_at_dataset_end
)
print(
    "Products ending before dataset end:",
    products_ended_before_dataset_end
)

print()
print("Lifecycle validation mismatches:")
print(
    "First-positive-date mismatches:",
    first_positive_date_mismatches
)
print(
    "Last-positive-date mismatches:",
    last_positive_date_mismatches
)
print(
    "Demand-day-count mismatches:",
    demand_day_count_mismatches
)

print()
print("Lifecycle status summary:")
display(lifecycle_status_summary_df)

print()
print("Cell 10 completed successfully.")

STEP 1, PART 3 — OPERATING CALENDAR VALIDATION PASSED

Operating calendar:
Operating dates: 245
Operating-day sequence: 1 to 245
Calendar-gap events: 52
Maximum calendar gap: 13 days

Operating-calendar mismatches:
Operating-sequence mismatches: 0
Days-since-previous-date mismatches: 0
Consecutive-day flag mismatches: 0

Product lifecycle:
Products present at dataset start: 84
Products introduced later: 143
Products active at dataset end: 57
Products ending before dataset end: 170

Lifecycle validation mismatches:
First-positive-date mismatches: 0
Last-positive-date mismatches: 0
Demand-day-count mismatches: 0

Lifecycle status summary:


,LifecycleStatus,ProductCount,ProductPercentage
0,INTRODUCED_LATER_AND_ENDED_EARLIER,103,45.37
1,PRESENT_AT_START_AND_ENDED_EARLIER,67,29.52
2,INTRODUCED_LATER_AND_ACTIVE_AT_END,40,17.62
3,PRESENT_AT_START_AND_ACTIVE_AT_END,17,7.49



Cell 10 completed successfully.


In [19]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 3
# Cell 11: Save temporal audits and update the handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 3 objects exist
# ------------------------------------------------------------

required_part_3_objects = [
    "temporal_panel_df",
    "operating_calendar_audit_df",
    "product_temporal_summary_df",
    "product_series_gap_audit_df",
    "lifecycle_status_summary_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_part_3_objects = [
    object_name
    for object_name in required_part_3_objects
    if object_name not in globals()
]

if missing_part_3_objects:
    raise NameError(
        "The following Part 3 objects are missing:\n"
        f"{missing_part_3_objects}\n\n"
        "Run Cells 9 and 10 before running Cell 11."
    )


# ------------------------------------------------------------
# 2. Define Part 3 output paths
# ------------------------------------------------------------

TEMPORAL_PANEL_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_temporal_panel_validated.csv"
)

OPERATING_CALENDAR_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_operating_calendar_audit.csv"
)

PRODUCT_TEMPORAL_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_product_temporal_summary.csv"
)

PRODUCT_GAP_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_product_series_gap_audit.csv"
)

LIFECYCLE_STATUS_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_lifecycle_status_summary.csv"
)


# ------------------------------------------------------------
# 3. Save Part 3 outputs
# ------------------------------------------------------------

temporal_panel_df.to_csv(
    TEMPORAL_PANEL_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

operating_calendar_audit_df.to_csv(
    OPERATING_CALENDAR_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

product_temporal_summary_df.to_csv(
    PRODUCT_TEMPORAL_SUMMARY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

product_series_gap_audit_df.to_csv(
    PRODUCT_GAP_AUDIT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

lifecycle_status_summary_df.to_csv(
    LIFECYCLE_STATUS_SUMMARY_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 4. Reload and validate saved outputs
# ------------------------------------------------------------

saved_temporal_panel = pd.read_csv(
    TEMPORAL_PANEL_OUTPUT,
    low_memory=False
)

saved_operating_calendar = pd.read_csv(
    OPERATING_CALENDAR_OUTPUT,
    low_memory=False
)

saved_product_summary = pd.read_csv(
    PRODUCT_TEMPORAL_SUMMARY_OUTPUT,
    low_memory=False
)

saved_gap_audit = pd.read_csv(
    PRODUCT_GAP_AUDIT_OUTPUT,
    low_memory=False
)

saved_lifecycle_summary = pd.read_csv(
    LIFECYCLE_STATUS_SUMMARY_OUTPUT,
    low_memory=False
)

assert saved_temporal_panel.shape == temporal_panel_df.shape
assert len(saved_temporal_panel) == 25_405

assert len(saved_operating_calendar) == 245
assert saved_operating_calendar["Date"].nunique() == 245

assert len(saved_product_summary) == 227
assert (
    saved_product_summary[
        "CanonicalProductID"
    ].nunique()
    == 227
)

assert len(saved_gap_audit) == 0

assert (
    saved_lifecycle_summary[
        "ProductCount"
    ].sum()
    == 227
)

assert saved_temporal_panel[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_temporal_panel[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 5. Update the Markdown handoff
# ------------------------------------------------------------

lifecycle_markdown_rows = "\n".join(
    (
        f"- `{row.LifecycleStatus}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%)"
    )
    for row in lifecycle_status_summary_df.itertuples()
)

part_3_summary = f"""
**Status:** Completed and validated

### Temporal-panel structure

- Rows: {len(temporal_panel_df):,}
- Canonical products: {temporal_panel_df["CanonicalProductID"].nunique():,}
- Operating dates: {temporal_panel_df["Date"].nunique():,}
- Date range: {temporal_panel_df["Date"].min().date()} to {temporal_panel_df["Date"].max().date()}
- Duplicate product-date rows: {duplicate_product_date_rows}
- Duplicate product-sequence rows: {duplicate_product_sequence_rows}
- Internal product-series gaps: {internal_gap_row_count}
- Products with missing panel rows: {products_with_missing_panel_rows}

### Operating-calendar validation

- Operating-day sequence: 1 to {len(operating_calendar_audit_df)}
- Calendar-gap events: {calendar_gap_event_count}
- Maximum gap between operating dates: {maximum_calendar_gap} days
- Operating-sequence mismatches: {operating_sequence_mismatches}
- Days-since-previous-date mismatches: {calendar_gap_mismatches}
- Consecutive-day flag mismatches: {consecutive_flag_mismatches}

### Product lifecycle findings

- Products present at the dataset start: {products_present_at_dataset_start}
- Products introduced after the dataset start: {products_introduced_later}
- Products active on the final dataset date: {products_active_at_dataset_end}
- Products ending before the final dataset date: {products_ended_before_dataset_end}
- First-positive-date mismatches: {first_positive_date_mismatches}
- Last-positive-date mismatches: {last_positive_date_mismatches}

The daily panel represents each product only within its observed
active window, from its first positive-demand date to its last
positive-demand date. Products ending before the dataset end are
not automatically treated as data errors. Forecast eligibility will
be decided later using lifecycle and history-length criteria.

### Lifecycle status distribution

{lifecycle_markdown_rows}

### Saved Part 3 outputs

- `03_temporal_panel_validated.csv`
- `03_operating_calendar_audit.csv`
- `03_product_temporal_summary.csv`
- `03_product_series_gap_audit.csv`
- `03_lifecycle_status_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_3",
    section_title=(
        "Forecasting Preparation — Step 1, Part 3"
    ),
    section_body=part_3_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 6. Print final Part 3 completion summary
# ------------------------------------------------------------

print("=" * 75)
print("FORECASTING PREPARATION — STEP 1, PART 3 COMPLETED")
print("=" * 75)

print()
print("Temporal panel:")
print(f"Rows saved: {len(saved_temporal_panel):,}")
print(
    "Canonical products:",
    saved_temporal_panel[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_temporal_panel["Date"].nunique()
)
print(
    "Duplicate product-date rows:",
    saved_temporal_panel[
        [
            "Date",
            "CanonicalProductID"
        ]
    ].duplicated().sum()
)

print()
print("Continuity:")
print(
    "Internal product-series gaps:",
    len(saved_gap_audit)
)
print(
    "Operating-calendar mismatches:",
    (
        operating_sequence_mismatches
        + calendar_gap_mismatches
        + consecutive_flag_mismatches
    )
)

print()
print("Lifecycle:")
print(
    "Products active at dataset end:",
    products_active_at_dataset_end
)
print(
    "Products ending before dataset end:",
    products_ended_before_dataset_end
)

print()
print("Saved files:")
print(f"1. {TEMPORAL_PANEL_OUTPUT}")
print(f"2. {OPERATING_CALENDAR_OUTPUT}")
print(f"3. {PRODUCT_TEMPORAL_SUMMARY_OUTPUT}")
print(f"4. {PRODUCT_GAP_AUDIT_OUTPUT}")
print(f"5. {LIFECYCLE_STATUS_SUMMARY_OUTPUT}")
print(f"6. {HANDOFF_FILE}")

print()
print(
    "All Step 1, Part 3 validation checks passed."
)

FORECASTING PREPARATION — STEP 1, PART 3 COMPLETED

Temporal panel:
Rows saved: 25,405
Canonical products: 227
Operating dates: 245
Duplicate product-date rows: 0

Continuity:
Internal product-series gaps: 0
Operating-calendar mismatches: 0

Lifecycle:
Products active at dataset end: 57
Products ending before dataset end: 170

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_temporal_panel_validated.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_operating_calendar_audit.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_product_temporal_summary.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_product_series_gap_audit.csv
5. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_lifecycle_status_summary.csv
6. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/FORECASTING_PREPARATION_HANDOFF.md

All Step 1, 

In [20]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 3 CORRECTION
# Cell 12: Expand all continuing products through the final
# operating date
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Confirm required objects exist
# ------------------------------------------------------------

required_objects = [
    "daily_demand_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "The following required objects are missing:\n"
        f"{missing_objects}\n\n"
        "Run the previous forecasting-preparation cells first."
    )


# ------------------------------------------------------------
# 2. Start from the original validated 25,405-row panel
# ------------------------------------------------------------

continuation_source_df = daily_demand_df.copy()

assert len(continuation_source_df) == 25_405, (
    "Cell 12 must start from the original 25,405-row panel. "
    f"Current rows: {len(continuation_source_df):,}"
)

original_column_order = (
    continuation_source_df.columns.tolist()
)

continuation_source_df["Date"] = pd.to_datetime(
    continuation_source_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

continuation_source_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    continuation_source_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

assert continuation_source_df["Date"].isna().sum() == 0

assert continuation_source_df[
    "ProductFirstObservedDate"
].isna().sum() == 0


# ------------------------------------------------------------
# 3. Define product metadata columns
# ------------------------------------------------------------

product_metadata_columns = [
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "SourcePLUCount",
    "SourcePLUCodes",
    "SourcePLUNames",
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
    "IsMultiPLUCanonicalProduct",
    "ProductMetadataVersion"
]


# ------------------------------------------------------------
# 4. Validate product metadata stability
# ------------------------------------------------------------

metadata_stability_df = (
    continuation_source_df
    .groupby("CanonicalProductID")[
        [
            column
            for column in product_metadata_columns
            if column != "CanonicalProductID"
        ]
    ]
    .nunique(dropna=False)
)

unstable_metadata_mask = (
    metadata_stability_df > 1
)

unstable_product_metadata_cells = int(
    unstable_metadata_mask.sum().sum()
)

assert unstable_product_metadata_cells == 0, (
    "Product metadata is not stable within one or more "
    "CanonicalProductID series."
)

product_metadata_df = (
    continuation_source_df[
        product_metadata_columns
    ]
    .sort_values(
        [
            "CanonicalProductID",
            "ProductFirstObservedDate"
        ],
        kind="stable"
    )
    .drop_duplicates(
        subset=["CanonicalProductID"],
        keep="first"
    )
    .reset_index(drop=True)
)

assert len(product_metadata_df) == 227
assert product_metadata_df[
    "CanonicalProductID"
].nunique() == 227


# ------------------------------------------------------------
# 5. Define and reconstruct the operating calendar
# ------------------------------------------------------------

calendar_columns = [
    "Date",
    "OperatingDaySequence",
    "Year",
    "Month",
    "MonthName",
    "Quarter",
    "DayOfWeekNumber",
    "DayOfWeek",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay"
]

calendar_stability_df = (
    continuation_source_df
    .groupby("Date")[
        [
            column
            for column in calendar_columns
            if column != "Date"
        ]
    ]
    .nunique(dropna=False)
)

unstable_calendar_cells = int(
    (calendar_stability_df > 1).sum().sum()
)

assert unstable_calendar_cells == 0, (
    "Operating-calendar metadata is inconsistent "
    "within one or more dates."
)

continued_operating_calendar_df = (
    continuation_source_df[
        calendar_columns
    ]
    .drop_duplicates(subset=["Date"])
    .sort_values(
        [
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert len(continued_operating_calendar_df) == 245
assert continued_operating_calendar_df[
    "Date"
].nunique() == 245

assert continued_operating_calendar_df[
    "OperatingDaySequence"
].nunique() == 245

FINAL_OPERATING_DATE = (
    continued_operating_calendar_df["Date"].max()
)

FINAL_OPERATING_SEQUENCE = int(
    continued_operating_calendar_df[
        "OperatingDaySequence"
    ].max()
)

assert FINAL_OPERATING_DATE == pd.Timestamp(
    "2026-03-30"
)

assert FINAL_OPERATING_SEQUENCE == 245


# ------------------------------------------------------------
# 6. Find each product's first operating-day sequence
# ------------------------------------------------------------

first_sequence_lookup_df = (
    continued_operating_calendar_df[
        [
            "Date",
            "OperatingDaySequence"
        ]
    ]
    .rename(
        columns={
            "Date": "ProductFirstObservedDate",
            "OperatingDaySequence":
                "FirstObservedOperatingSequence"
        }
    )
)

product_metadata_df = (
    product_metadata_df
    .merge(
        first_sequence_lookup_df,
        on="ProductFirstObservedDate",
        how="left",
        validate="many_to_one"
    )
)

assert product_metadata_df[
    "FirstObservedOperatingSequence"
].isna().sum() == 0


# ------------------------------------------------------------
# 7. Create the complete product-operating-date grid
# ------------------------------------------------------------

complete_product_calendar_df = (
    product_metadata_df
    .merge(
        continued_operating_calendar_df,
        how="cross"
    )
)

# Preserve later product introductions, but continue every
# product through the final Eden operating date.
complete_product_calendar_df = (
    complete_product_calendar_df.loc[
        complete_product_calendar_df[
            "OperatingDaySequence"
        ]
        >= complete_product_calendar_df[
            "FirstObservedOperatingSequence"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

expected_complete_panel_rows = int(
    (
        FINAL_OPERATING_SEQUENCE
        - product_metadata_df[
            "FirstObservedOperatingSequence"
        ]
        + 1
    ).sum()
)

assert expected_complete_panel_rows == 43_774

assert len(
    complete_product_calendar_df
) == expected_complete_panel_rows


# ------------------------------------------------------------
# 8. Extract the original demand values
# ------------------------------------------------------------

original_demand_df = (
    continuation_source_df[
        [
            "Date",
            "CanonicalProductID",
            "NormalDemand",
            "BulkDemand",
            "TotalDemand"
        ]
    ]
    .copy()
)

assert original_demand_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0


# ------------------------------------------------------------
# 9. Merge original demand into the complete panel
# ------------------------------------------------------------

expanded_panel_working_df = (
    complete_product_calendar_df
    .merge(
        original_demand_df,
        on=[
            "Date",
            "CanonicalProductID"
        ],
        how="left",
        validate="one_to_one",
        indicator="_OriginalPanelMerge"
    )
)

expanded_panel_working_df[
    "AddedByContinuationCorrection"
] = (
    expanded_panel_working_df[
        "_OriginalPanelMerge"
    ]
    .eq("left_only")
)

added_trailing_zero_row_count = int(
    expanded_panel_working_df[
        "AddedByContinuationCorrection"
    ].sum()
)

original_rows_preserved = int(
    expanded_panel_working_df[
        "_OriginalPanelMerge"
    ].eq("both").sum()
)

assert original_rows_preserved == 25_405
assert added_trailing_zero_row_count == 18_369


# ------------------------------------------------------------
# 10. Fill missing demand for continued-product dates
# ------------------------------------------------------------

demand_columns = [
    "NormalDemand",
    "BulkDemand",
    "TotalDemand"
]

for column in demand_columns:

    expanded_panel_working_df[column] = (
        expanded_panel_working_df[column]
        .fillna(0)
        .astype(int)
    )

expanded_panel_working_df[
    "IsObservedProductDate"
] = (
    expanded_panel_working_df[
        "TotalDemand"
    ] > 0
)

expanded_panel_working_df[
    "IsZeroDemandRow"
] = (
    expanded_panel_working_df[
        "TotalDemand"
    ] == 0
)

# "ACTIVE_OPERATING_DATE" refers to Eden being open on
# the date. It does not mean the product was active/inactive.
expanded_panel_working_df[
    "DemandRecordSource"
] = np.where(
    expanded_panel_working_df[
        "TotalDemand"
    ] > 0,
    "OBSERVED_TRANSACTION_AGGREGATION",
    "ZERO_FILLED_ACTIVE_OPERATING_DATE"
)


# ------------------------------------------------------------
# 11. Recalculate product age
# ------------------------------------------------------------

expanded_panel_working_df[
    "ProductAgeOperatingDays"
] = (
    expanded_panel_working_df[
        "OperatingDaySequence"
    ]
    - expanded_panel_working_df[
        "FirstObservedOperatingSequence"
    ]
).astype(int)

assert (
    expanded_panel_working_df[
        "ProductAgeOperatingDays"
    ] < 0
).sum() == 0


# ------------------------------------------------------------
# 12. Apply the corrected panel version
# ------------------------------------------------------------

expanded_panel_working_df[
    "DailyPanelVersion"
] = (
    "FORECAST_PREP_STEP1_PART3_"
    "CONTINUED_PRODUCT_PANEL_V2"
)


# ------------------------------------------------------------
# 13. Create the added-row audit before removing helper fields
# ------------------------------------------------------------

trailing_zero_rows_added_audit_df = (
    expanded_panel_working_df.loc[
        expanded_panel_working_df[
            "AddedByContinuationCorrection"
        ],
        [
            "Date",
            "CanonicalProductID",
            "CanonicalProductName",
            "ProductFirstObservedDate",
            "OperatingDaySequence",
            "ProductAgeOperatingDays",
            "NormalDemand",
            "BulkDemand",
            "TotalDemand",
            "IsObservedProductDate",
            "IsZeroDemandRow",
            "DemandRecordSource"
        ]
    ]
    .copy()
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

trailing_zero_rows_added_audit_df[
    "RowsAddedReason"
] = (
    "PRODUCT_CONTINUED_AFTER_LAST_OBSERVED_"
    "POSITIVE_DEMAND_DATE"
)

assert len(
    trailing_zero_rows_added_audit_df
) == 18_369

assert trailing_zero_rows_added_audit_df[
    "TotalDemand"
].sum() == 0


# ------------------------------------------------------------
# 14. Create the official corrected 38-column panel
# ------------------------------------------------------------

expanded_continued_panel_df = (
    expanded_panel_working_df[
        original_column_order
    ]
    .copy()
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert expanded_continued_panel_df.shape == (
    43_774,
    38
)

assert expanded_continued_panel_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0


# ------------------------------------------------------------
# 15. Confirm original demand totals were preserved
# ------------------------------------------------------------

original_normal_demand = int(
    continuation_source_df[
        "NormalDemand"
    ].sum()
)

original_bulk_demand = int(
    continuation_source_df[
        "BulkDemand"
    ].sum()
)

original_total_demand = int(
    continuation_source_df[
        "TotalDemand"
    ].sum()
)

expanded_normal_demand = int(
    expanded_continued_panel_df[
        "NormalDemand"
    ].sum()
)

expanded_bulk_demand = int(
    expanded_continued_panel_df[
        "BulkDemand"
    ].sum()
)

expanded_total_demand = int(
    expanded_continued_panel_df[
        "TotalDemand"
    ].sum()
)

assert original_normal_demand == 114_186
assert original_bulk_demand == 1_972
assert original_total_demand == 116_158

assert expanded_normal_demand == original_normal_demand
assert expanded_bulk_demand == original_bulk_demand
assert expanded_total_demand == original_total_demand


# ------------------------------------------------------------
# 16. Confirm every product continues to the final date
# ------------------------------------------------------------

product_final_dates = (
    expanded_continued_panel_df
    .groupby("CanonicalProductID")[
        "Date"
    ]
    .max()
)

products_not_extended_to_final_date = int(
    (
        product_final_dates
        != FINAL_OPERATING_DATE
    ).sum()
)

assert products_not_extended_to_final_date == 0


# ------------------------------------------------------------
# 17. Print Cell 12 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 3 CORRECTION — PANEL EXPANSION PASSED")
print("=" * 75)

print()
print("Original panel:")
print(f"Rows: {len(continuation_source_df):,}")
print(
    "Products:",
    continuation_source_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Total demand units:",
    f"{original_total_demand:,}"
)

print()
print("Corrected continuing-product panel:")
print(f"Rows: {len(expanded_continued_panel_df):,}")
print(
    "Products:",
    expanded_continued_panel_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    expanded_continued_panel_df[
        "Date"
    ].nunique()
)
print(
    "Final operating date:",
    FINAL_OPERATING_DATE.date()
)

print()
print(
    "Original rows preserved:",
    f"{original_rows_preserved:,}"
)
print(
    "Trailing zero-demand rows added:",
    f"{added_trailing_zero_row_count:,}"
)
print(
    "Products not extended to final date:",
    products_not_extended_to_final_date
)

print()
print("Demand preservation:")
print(
    "Normal demand:",
    f"{expanded_normal_demand:,}"
)
print(
    "Bulk demand:",
    f"{expanded_bulk_demand:,}"
)
print(
    "Total demand:",
    f"{expanded_total_demand:,}"
)

print()
print("Cell 12 completed successfully.")

STEP 1, PART 3 CORRECTION — PANEL EXPANSION PASSED

Original panel:
Rows: 25,405
Products: 227
Total demand units: 116,158

Corrected continuing-product panel:
Rows: 43,774
Products: 227
Operating dates: 245
Final operating date: 2026-03-30

Original rows preserved: 25,405
Trailing zero-demand rows added: 18,369
Products not extended to final date: 0

Demand preservation:
Normal demand: 114,186
Bulk demand: 1,972
Total demand: 116,158

Cell 12 completed successfully.


In [21]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 3 CORRECTION
# Cell 13: Revalidate the corrected temporal panel and replace
# lifecycle labels with sales-recency measures
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 12 objects exist
# ------------------------------------------------------------

required_cell_12_objects = [
    "expanded_continued_panel_df",
    "continued_operating_calendar_df",
    "trailing_zero_rows_added_audit_df",
    "FINAL_OPERATING_DATE",
    "FINAL_OPERATING_SEQUENCE"
]

missing_cell_12_objects = [
    object_name
    for object_name in required_cell_12_objects
    if object_name not in globals()
]

if missing_cell_12_objects:
    raise NameError(
        "The following Cell 12 objects are missing:\n"
        f"{missing_cell_12_objects}\n\n"
        "Run Cell 12 before running Cell 13."
    )


# ------------------------------------------------------------
# 2. Sort temporarily for continuity calculations
# ------------------------------------------------------------

continuity_check_df = (
    expanded_continued_panel_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

continuity_check_df[
    "PreviousOperatingSequence"
] = (
    continuity_check_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .shift(1)
)

continuity_check_df[
    "OperatingSequenceDifference"
] = (
    continuity_check_df[
        "OperatingDaySequence"
    ]
    - continuity_check_df[
        "PreviousOperatingSequence"
    ]
)

corrected_gap_mask = (
    continuity_check_df[
        "PreviousOperatingSequence"
    ].notna()
    & (
        continuity_check_df[
            "OperatingSequenceDifference"
        ] != 1
    )
)

corrected_internal_gap_rows = int(
    corrected_gap_mask.sum()
)

corrected_products_with_gaps = int(
    continuity_check_df.loc[
        corrected_gap_mask,
        "CanonicalProductID"
    ].nunique()
)

corrected_product_series_gap_audit_df = (
    continuity_check_df.loc[
        corrected_gap_mask,
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "Date",
            "OperatingDaySequence",
            "PreviousOperatingSequence",
            "OperatingSequenceDifference"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert corrected_internal_gap_rows == 0
assert corrected_products_with_gaps == 0


# ------------------------------------------------------------
# 3. Obtain first and last positive-demand observations
# ------------------------------------------------------------

positive_demand_rows_df = (
    expanded_continued_panel_df.loc[
        expanded_continued_panel_df[
            "TotalDemand"
        ] > 0
    ]
    .copy()
)

positive_demand_summary_df = (
    positive_demand_rows_df
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        FirstPositiveDemandDate=(
            "Date",
            "min"
        ),
        LastObservedPositiveDemandDate=(
            "Date",
            "max"
        ),
        FirstPositiveDemandOperatingSequence=(
            "OperatingDaySequence",
            "min"
        ),
        LastPositiveDemandOperatingSequence=(
            "OperatingDaySequence",
            "max"
        )
    )
)

assert len(positive_demand_summary_df) == 227


# ------------------------------------------------------------
# 4. Create the corrected product temporal summary
# ------------------------------------------------------------

corrected_product_temporal_summary_df = (
    expanded_continued_panel_df
    .groupby(
        [
            "CanonicalProductID",
            "CanonicalProductName"
        ],
        as_index=False
    )
    .agg(
        ProductFirstObservedDate=(
            "ProductFirstObservedDate",
            "first"
        ),
        PanelFirstDate=("Date", "min"),
        PanelFinalDate=("Date", "max"),
        FirstOperatingDaySequence=(
            "OperatingDaySequence",
            "min"
        ),
        FinalOperatingDaySequence=(
            "OperatingDaySequence",
            "max"
        ),
        ProductSeriesRows=("Date", "size"),
        PositiveDemandDays=(
            "TotalDemand",
            lambda values: int(
                (values > 0).sum()
            )
        ),
        ZeroDemandDays=(
            "TotalDemand",
            lambda values: int(
                (values == 0).sum()
            )
        ),
        NormalDemandUnits=(
            "NormalDemand",
            "sum"
        ),
        BulkDemandUnits=(
            "BulkDemand",
            "sum"
        ),
        TotalDemandUnits=(
            "TotalDemand",
            "sum"
        ),
        MeanDemandAcrossOperatingDates=(
            "TotalDemand",
            "mean"
        ),
        MaximumDailyDemand=(
            "TotalDemand",
            "max"
        )
    )
)

corrected_product_temporal_summary_df = (
    corrected_product_temporal_summary_df
    .merge(
        positive_demand_summary_df,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 5. Add sales-recency measures
# ------------------------------------------------------------

corrected_product_temporal_summary_df[
    "OperatingDaysSinceLastPositiveDemand"
] = (
    FINAL_OPERATING_SEQUENCE
    - corrected_product_temporal_summary_df[
        "LastPositiveDemandOperatingSequence"
    ]
).astype(int)

corrected_product_temporal_summary_df[
    "CalendarDaysSinceLastPositiveDemand"
] = (
    FINAL_OPERATING_DATE
    - corrected_product_temporal_summary_df[
        "LastObservedPositiveDemandDate"
    ]
).dt.days.astype(int)

corrected_product_temporal_summary_df[
    "SoldOnFinalOperatingDate"
] = (
    corrected_product_temporal_summary_df[
        "OperatingDaysSinceLastPositiveDemand"
    ] == 0
)

corrected_product_temporal_summary_df[
    "ContinuedThroughFinalOperatingDate"
] = True

corrected_product_temporal_summary_df[
    "ExpectedProductSeriesRows"
] = (
    FINAL_OPERATING_SEQUENCE
    - corrected_product_temporal_summary_df[
        "FirstOperatingDaySequence"
    ]
    + 1
)

corrected_product_temporal_summary_df[
    "MissingOperatingDates"
] = (
    corrected_product_temporal_summary_df[
        "ExpectedProductSeriesRows"
    ]
    - corrected_product_temporal_summary_df[
        "ProductSeriesRows"
    ]
)

corrected_product_temporal_summary_df[
    "PositiveDemandRate"
] = (
    corrected_product_temporal_summary_df[
        "PositiveDemandDays"
    ]
    / corrected_product_temporal_summary_df[
        "ProductSeriesRows"
    ]
).round(4)

corrected_product_temporal_summary_df[
    "ZeroDemandRate"
] = (
    corrected_product_temporal_summary_df[
        "ZeroDemandDays"
    ]
    / corrected_product_temporal_summary_df[
        "ProductSeriesRows"
    ]
).round(4)


# ------------------------------------------------------------
# 6. Count added continuation rows per product
# ------------------------------------------------------------

added_rows_by_product_df = (
    trailing_zero_rows_added_audit_df
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "ContinuationZeroRowsAdded"
        }
    )
)

corrected_product_temporal_summary_df = (
    corrected_product_temporal_summary_df
    .merge(
        added_rows_by_product_df,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
)

corrected_product_temporal_summary_df[
    "ContinuationZeroRowsAdded"
] = (
    corrected_product_temporal_summary_df[
        "ContinuationZeroRowsAdded"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 7. Validate the corrected summary
# ------------------------------------------------------------

first_date_mismatches = int(
    (
        corrected_product_temporal_summary_df[
            "PanelFirstDate"
        ]
        != corrected_product_temporal_summary_df[
            "ProductFirstObservedDate"
        ]
    ).sum()
)

first_positive_date_mismatches = int(
    (
        corrected_product_temporal_summary_df[
            "FirstPositiveDemandDate"
        ]
        != corrected_product_temporal_summary_df[
            "ProductFirstObservedDate"
        ]
    ).sum()
)

products_not_continued_to_final_date = int(
    (
        corrected_product_temporal_summary_df[
            "PanelFinalDate"
        ]
        != FINAL_OPERATING_DATE
    ).sum()
)

products_with_missing_operating_dates = int(
    (
        corrected_product_temporal_summary_df[
            "MissingOperatingDates"
        ] != 0
    ).sum()
)

demand_day_count_mismatches = int(
    (
        corrected_product_temporal_summary_df[
            "PositiveDemandDays"
        ]
        + corrected_product_temporal_summary_df[
            "ZeroDemandDays"
        ]
        != corrected_product_temporal_summary_df[
            "ProductSeriesRows"
        ]
    ).sum()
)

added_row_count_mismatches = int(
    (
        corrected_product_temporal_summary_df[
            "ContinuationZeroRowsAdded"
        ]
        != corrected_product_temporal_summary_df[
            "OperatingDaysSinceLastPositiveDemand"
        ]
    ).sum()
)

assert first_date_mismatches == 0
assert first_positive_date_mismatches == 0
assert products_not_continued_to_final_date == 0
assert products_with_missing_operating_dates == 0
assert demand_day_count_mismatches == 0
assert added_row_count_mismatches == 0

assert len(
    corrected_product_temporal_summary_df
) == 227


# ------------------------------------------------------------
# 8. Create a descriptive sales-recency summary
# ------------------------------------------------------------

recency_bins = [
    -1,
    0,
    10,
    30,
    60,
    90,
    120,
    180,
    np.inf
]

recency_labels = [
    "SOLD_ON_FINAL_OPERATING_DATE",
    "1_TO_10_OPERATING_DAYS",
    "11_TO_30_OPERATING_DAYS",
    "31_TO_60_OPERATING_DAYS",
    "61_TO_90_OPERATING_DAYS",
    "91_TO_120_OPERATING_DAYS",
    "121_TO_180_OPERATING_DAYS",
    "181_PLUS_OPERATING_DAYS"
]

corrected_product_temporal_summary_df[
    "LastPositiveDemandRecencyBand"
] = pd.cut(
    corrected_product_temporal_summary_df[
        "OperatingDaysSinceLastPositiveDemand"
    ],
    bins=recency_bins,
    labels=recency_labels,
    include_lowest=True
)

sales_recency_summary_df = (
    corrected_product_temporal_summary_df[
        "LastPositiveDemandRecencyBand"
    ]
    .value_counts(sort=False)
    .rename_axis(
        "LastPositiveDemandRecencyBand"
    )
    .reset_index(name="ProductCount")
)

sales_recency_summary_df[
    "ProductPercentage"
] = (
    sales_recency_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)

products_sold_on_final_date = int(
    corrected_product_temporal_summary_df[
        "SoldOnFinalOperatingDate"
    ].sum()
)

products_with_last_positive_sale_earlier = (
    227 - products_sold_on_final_date
)

assert products_sold_on_final_date == 57
assert products_with_last_positive_sale_earlier == 170


# ------------------------------------------------------------
# 9. Print Cell 13 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 3 CORRECTION — REVALIDATION PASSED")
print("=" * 75)

print()
print("Corrected panel continuity:")
print(
    "Internal product-series gap rows:",
    corrected_internal_gap_rows
)
print(
    "Products with internal gaps:",
    corrected_products_with_gaps
)
print(
    "Products missing operating dates:",
    products_with_missing_operating_dates
)
print(
    "Products not continued to final date:",
    products_not_continued_to_final_date
)

print()
print("Product-date interpretation:")
print(
    "Products continued through final date:",
    227
)
print(
    "Products sold on final operating date:",
    products_sold_on_final_date
)
print(
    "Products whose last positive sale was earlier:",
    products_with_last_positive_sale_earlier
)

print()
print(
    "The 170 products are not classified as inactive, "
    "ended or discontinued."
)

print()
print("Sales-recency summary:")
display(sales_recency_summary_df)

print()
print("Cell 13 completed successfully.")

STEP 1, PART 3 CORRECTION — REVALIDATION PASSED

Corrected panel continuity:
Internal product-series gap rows: 0
Products with internal gaps: 0
Products missing operating dates: 0
Products not continued to final date: 0

Product-date interpretation:
Products continued through final date: 227
Products sold on final operating date: 57
Products whose last positive sale was earlier: 170

The 170 products are not classified as inactive, ended or discontinued.

Sales-recency summary:


,LastPositiveDemandRecencyBand,ProductCount,ProductPercentage
0,SOLD_ON_FINAL_OPERATING_DATE,57,25.11
1,1_TO_10_OPERATING_DAYS,29,12.78
2,11_TO_30_OPERATING_DAYS,4,1.76
3,31_TO_60_OPERATING_DAYS,2,0.88
4,61_TO_90_OPERATING_DAYS,0,0.00
5,91_TO_120_OPERATING_DAYS,54,23.79
6,121_TO_180_OPERATING_DAYS,77,33.92
7,181_PLUS_OPERATING_DAYS,4,1.76



Cell 13 completed successfully.


In [22]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 3 CORRECTION
# Cell 14: Back up superseded outputs, save corrected outputs,
# and replace the Markdown handoff section
# ============================================================

from pathlib import Path
import shutil

# ------------------------------------------------------------
# 1. Confirm required corrected objects exist
# ------------------------------------------------------------

required_corrected_objects = [
    "expanded_continued_panel_df",
    "continued_operating_calendar_df",
    "corrected_product_temporal_summary_df",
    "corrected_product_series_gap_audit_df",
    "trailing_zero_rows_added_audit_df",
    "sales_recency_summary_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_corrected_objects = [
    object_name
    for object_name in required_corrected_objects
    if object_name not in globals()
]

if missing_corrected_objects:
    raise NameError(
        "The following corrected Part 3 objects are missing:\n"
        f"{missing_corrected_objects}\n\n"
        "Run Cells 12 and 13 before running Cell 14."
    )


# ------------------------------------------------------------
# 2. Back up the previously saved incorrect Part 3 outputs
# ------------------------------------------------------------

SUPERSEDED_PART_3_DIR = (
    FORECAST_PREPARATION_DIR
    / "superseded_step1_part3_before_continuation_correction"
)

SUPERSEDED_PART_3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

old_part_3_filenames = [
    "03_temporal_panel_validated.csv",
    "03_operating_calendar_audit.csv",
    "03_product_temporal_summary.csv",
    "03_product_series_gap_audit.csv",
    "03_lifecycle_status_summary.csv"
]

backed_up_files = []

for filename in old_part_3_filenames:

    original_path = (
        FORECAST_PREPARATION_DIR / filename
    )

    backup_path = (
        SUPERSEDED_PART_3_DIR / filename
    )

    if (
        original_path.exists()
        and not backup_path.exists()
    ):
        shutil.copy2(
            original_path,
            backup_path
        )

        backed_up_files.append(filename)


# ------------------------------------------------------------
# 3. Remove the misleading active lifecycle summary
# ------------------------------------------------------------

OLD_LIFECYCLE_FILE = (
    FORECAST_PREPARATION_DIR
    / "03_lifecycle_status_summary.csv"
)

if OLD_LIFECYCLE_FILE.exists():
    OLD_LIFECYCLE_FILE.unlink()


# ------------------------------------------------------------
# 4. Define corrected official output paths
# ------------------------------------------------------------

CORRECTED_TEMPORAL_PANEL_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_temporal_panel_validated.csv"
)

CORRECTED_OPERATING_CALENDAR_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_operating_calendar_audit.csv"
)

CORRECTED_PRODUCT_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_product_temporal_summary.csv"
)

CORRECTED_GAP_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_product_series_gap_audit.csv"
)

TRAILING_ZERO_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_trailing_zero_rows_added_audit.csv"
)

SALES_RECENCY_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_sales_recency_summary.csv"
)

PANEL_EXPANSION_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "03_panel_expansion_summary.csv"
)


# ------------------------------------------------------------
# 5. Create the expansion summary
# ------------------------------------------------------------

panel_expansion_summary_df = pd.DataFrame({
    "Metric": [
        "OriginalPanelRows",
        "CorrectedPanelRows",
        "TrailingZeroRowsAdded",
        "CanonicalProducts",
        "OperatingDates",
        "ProductsContinuedToFinalDate",
        "ProductsSoldOnFinalOperatingDate",
        "ProductsWithEarlierLastPositiveSale",
        "NormalDemandUnits",
        "BulkDemandUnits",
        "TotalDemandUnits",
        "InternalProductSeriesGaps"
    ],
    "Value": [
        25_405,
        len(expanded_continued_panel_df),
        len(trailing_zero_rows_added_audit_df),
        expanded_continued_panel_df[
            "CanonicalProductID"
        ].nunique(),
        expanded_continued_panel_df[
            "Date"
        ].nunique(),
        227,
        products_sold_on_final_date,
        products_with_last_positive_sale_earlier,
        int(
            expanded_continued_panel_df[
                "NormalDemand"
            ].sum()
        ),
        int(
            expanded_continued_panel_df[
                "BulkDemand"
            ].sum()
        ),
        int(
            expanded_continued_panel_df[
                "TotalDemand"
            ].sum()
        ),
        corrected_internal_gap_rows
    ]
})


# ------------------------------------------------------------
# 6. Save corrected official Part 3 outputs
# ------------------------------------------------------------

expanded_continued_panel_df.to_csv(
    CORRECTED_TEMPORAL_PANEL_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

continued_operating_calendar_df.to_csv(
    CORRECTED_OPERATING_CALENDAR_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

corrected_product_temporal_summary_df.to_csv(
    CORRECTED_PRODUCT_SUMMARY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

corrected_product_series_gap_audit_df.to_csv(
    CORRECTED_GAP_AUDIT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

trailing_zero_rows_added_audit_df.to_csv(
    TRAILING_ZERO_AUDIT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

sales_recency_summary_df.to_csv(
    SALES_RECENCY_SUMMARY_OUTPUT,
    index=False
)

panel_expansion_summary_df.to_csv(
    PANEL_EXPANSION_SUMMARY_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 7. Reload and validate saved files
# ------------------------------------------------------------

saved_corrected_panel = pd.read_csv(
    CORRECTED_TEMPORAL_PANEL_OUTPUT,
    low_memory=False
)

saved_calendar = pd.read_csv(
    CORRECTED_OPERATING_CALENDAR_OUTPUT,
    low_memory=False
)

saved_product_summary = pd.read_csv(
    CORRECTED_PRODUCT_SUMMARY_OUTPUT,
    low_memory=False
)

saved_gap_audit = pd.read_csv(
    CORRECTED_GAP_AUDIT_OUTPUT,
    low_memory=False
)

saved_trailing_zero_audit = pd.read_csv(
    TRAILING_ZERO_AUDIT_OUTPUT,
    low_memory=False
)

saved_sales_recency = pd.read_csv(
    SALES_RECENCY_SUMMARY_OUTPUT,
    low_memory=False
)

saved_expansion_summary = pd.read_csv(
    PANEL_EXPANSION_SUMMARY_OUTPUT,
    low_memory=False
)

assert saved_corrected_panel.shape == (
    43_774,
    38
)

assert saved_corrected_panel[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_corrected_panel[
    "CanonicalProductID"
].nunique() == 227

assert saved_corrected_panel[
    "Date"
].nunique() == 245

assert saved_corrected_panel[
    "TotalDemand"
].sum() == 116_158

assert len(saved_calendar) == 245
assert len(saved_product_summary) == 227
assert len(saved_gap_audit) == 0

assert len(
    saved_trailing_zero_audit
) == 18_369

assert saved_trailing_zero_audit[
    "TotalDemand"
].sum() == 0

assert saved_sales_recency[
    "ProductCount"
].sum() == 227


# ------------------------------------------------------------
# 8. Replace the incorrect Markdown Part 3 section
# ------------------------------------------------------------

sales_recency_markdown = "\n".join(
    (
        f"- `{row.LastPositiveDemandRecencyBand}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%)"
    )
    for row in sales_recency_summary_df.itertuples()
)

corrected_part_3_summary = f"""
**Status:** Completed and validated after continuation correction

### Important interpretation correction

All 227 canonical products are treated as continuing products for
the forecasting objective.

The original panel ended each product at its last observed
positive-demand date. This did not prove that products were
inactive, ended or discontinued. The panel was therefore expanded
from each product's `ProductFirstObservedDate` through the final
Eden operating date.

### Corrected temporal panel

- Original rows: 25,405
- Trailing zero-demand rows added: 18,369
- Corrected rows: 43,774
- Columns: 38
- Canonical products: 227
- Operating dates: 245
- Final operating date: 2026-03-30
- Duplicate product-date rows: 0
- Internal product-series gaps: 0
- Products not continued to final operating date: 0

### Demand preservation

- Normal demand units: 114,186
- Bulk demand units: 1,972
- Total demand units: 116,158

The inserted continuation rows contain zero demand and therefore
do not change the historical demand totals.

### Sales-recency terminology

- Products continued through the final operating date: 227
- Products sold on the final operating date: {products_sold_on_final_date}
- Products whose last positive sale occurred earlier: {products_with_last_positive_sale_earlier}

A product whose last positive sale occurred earlier is not
classified as inactive, ended or discontinued.

### Last-positive-demand recency distribution

{sales_recency_markdown}

### Corrected Part 3 outputs

- `03_temporal_panel_validated.csv`
- `03_operating_calendar_audit.csv`
- `03_product_temporal_summary.csv`
- `03_product_series_gap_audit.csv`
- `03_trailing_zero_rows_added_audit.csv`
- `03_sales_recency_summary.csv`
- `03_panel_expansion_summary.csv`

### Superseded output

The original lifecycle interpretation and its outputs were backed
up inside:

`superseded_step1_part3_before_continuation_correction/`

The file `03_lifecycle_status_summary.csv` is no longer an active
forecasting-preparation output.
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_3",
    section_title=(
        "Forecasting Preparation — Step 1, Part 3"
    ),
    section_body=corrected_part_3_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 9. Print final corrected completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 1, PART 3 CORRECTED AND COMPLETED"
)
print("=" * 75)

print()
print("Corrected temporal panel:")
print(
    f"Rows saved: "
    f"{len(saved_corrected_panel):,}"
)
print(
    "Columns saved:",
    saved_corrected_panel.shape[1]
)
print(
    "Canonical products:",
    saved_corrected_panel[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_corrected_panel[
        "Date"
    ].nunique()
)

print()
print("Continuation correction:")
print(
    "Trailing zero-demand rows added:",
    f"{len(saved_trailing_zero_audit):,}"
)
print(
    "Products continued to final date:",
    227
)
print(
    "Internal product-series gaps:",
    len(saved_gap_audit)
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_corrected_panel['TotalDemand'].sum():,}"
)

print()
print("Sales recency:")
print(
    "Products sold on final operating date:",
    products_sold_on_final_date
)
print(
    "Products with earlier last positive sale:",
    products_with_last_positive_sale_earlier
)
print(
    "No products were labelled inactive or discontinued."
)

print()
print("Superseded files backed up to:")
print(SUPERSEDED_PART_3_DIR)

print()
print("Corrected official files:")
print(f"1. {CORRECTED_TEMPORAL_PANEL_OUTPUT}")
print(f"2. {CORRECTED_OPERATING_CALENDAR_OUTPUT}")
print(f"3. {CORRECTED_PRODUCT_SUMMARY_OUTPUT}")
print(f"4. {CORRECTED_GAP_AUDIT_OUTPUT}")
print(f"5. {TRAILING_ZERO_AUDIT_OUTPUT}")
print(f"6. {SALES_RECENCY_SUMMARY_OUTPUT}")
print(f"7. {PANEL_EXPANSION_SUMMARY_OUTPUT}")
print(f"8. {HANDOFF_FILE}")

print()
print(
    "All corrected Step 1, Part 3 "
    "validation checks passed."
)

FORECASTING PREPARATION — STEP 1, PART 3 CORRECTED AND COMPLETED

Corrected temporal panel:
Rows saved: 43,774
Columns saved: 38
Canonical products: 227
Operating dates: 245

Continuation correction:
Trailing zero-demand rows added: 18,369
Products continued to final date: 227
Internal product-series gaps: 0

Demand preserved:
Total demand units: 116,158

Sales recency:
Products sold on final operating date: 57
Products with earlier last positive sale: 170
No products were labelled inactive or discontinued.

Superseded files backed up to:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/superseded_step1_part3_before_continuation_correction

Corrected official files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_temporal_panel_validated.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_operating_calendar_audit.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/03_product_te

In [23]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 4
# Cell 15: Create product demand-history and intermittency profile
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Load the corrected Part 3 panel
# ------------------------------------------------------------

CORRECTED_PANEL_FILE = (
    FORECAST_PREPARATION_DIR
    / "03_temporal_panel_validated.csv"
)

if "expanded_continued_panel_df" in globals():

    demand_analysis_panel_df = (
        expanded_continued_panel_df.copy()
    )

elif CORRECTED_PANEL_FILE.exists():

    demand_analysis_panel_df = pd.read_csv(
        CORRECTED_PANEL_FILE,
        low_memory=False
    )

else:

    raise FileNotFoundError(
        "The corrected Part 3 temporal panel could not be found.\n"
        f"Expected file:\n{CORRECTED_PANEL_FILE}"
    )


# ------------------------------------------------------------
# 2. Parse and validate the corrected panel
# ------------------------------------------------------------

demand_analysis_panel_df["Date"] = pd.to_datetime(
    demand_analysis_panel_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

demand_analysis_panel_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    demand_analysis_panel_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

numeric_columns = [
    "OperatingDaySequence",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand"
]

for column in numeric_columns:

    demand_analysis_panel_df[column] = pd.to_numeric(
        demand_analysis_panel_df[column],
        errors="coerce"
    )

assert demand_analysis_panel_df["Date"].isna().sum() == 0

assert demand_analysis_panel_df[
    "ProductFirstObservedDate"
].isna().sum() == 0

assert demand_analysis_panel_df[
    numeric_columns
].isna().sum().sum() == 0

assert demand_analysis_panel_df.shape == (
    43_774,
    38
)

assert demand_analysis_panel_df[
    "CanonicalProductID"
].nunique() == 227

assert demand_analysis_panel_df[
    "Date"
].nunique() == 245

assert demand_analysis_panel_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert demand_analysis_panel_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 3. Sort each product chronologically
# ------------------------------------------------------------

demand_analysis_panel_df = (
    demand_analysis_panel_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

FINAL_OPERATING_DATE = (
    demand_analysis_panel_df["Date"].max()
)

FINAL_OPERATING_SEQUENCE = int(
    demand_analysis_panel_df[
        "OperatingDaySequence"
    ].max()
)


# ------------------------------------------------------------
# 4. Helper function for consecutive zero-demand runs
# ------------------------------------------------------------

def calculate_longest_true_run(boolean_values):
    """
    Calculate the longest consecutive run of True values.
    """

    maximum_run = 0
    current_run = 0

    for value in boolean_values:

        if bool(value):
            current_run += 1
            maximum_run = max(
                maximum_run,
                current_run
            )
        else:
            current_run = 0

    return int(maximum_run)


def calculate_trailing_true_run(boolean_values):
    """
    Calculate the number of consecutive True values
    at the end of the sequence.
    """

    trailing_run = 0

    for value in reversed(boolean_values):

        if bool(value):
            trailing_run += 1
        else:
            break

    return int(trailing_run)


# ------------------------------------------------------------
# 5. Build one demand-behaviour record per product
# ------------------------------------------------------------

product_profile_records = []

for (
    canonical_product_id,
    product_group
) in demand_analysis_panel_df.groupby(
    "CanonicalProductID",
    sort=False
):

    product_group = (
        product_group
        .sort_values(
            "OperatingDaySequence",
            kind="stable"
        )
        .reset_index(drop=True)
    )

    canonical_product_name = (
        product_group[
            "CanonicalProductName"
        ].iloc[0]
    )

    total_demand_values = (
        product_group["TotalDemand"]
    )

    positive_demand_group = (
        product_group.loc[
            total_demand_values > 0
        ]
    )

    history_operating_days = int(
        len(product_group)
    )

    positive_demand_days = int(
        len(positive_demand_group)
    )

    zero_demand_days = int(
        history_operating_days
        - positive_demand_days
    )

    positive_sequences = (
        positive_demand_group[
            "OperatingDaySequence"
        ]
        .to_numpy()
    )

    positive_intervals = np.diff(
        positive_sequences
    )

    zero_demand_flags = (
        total_demand_values.eq(0)
        .to_numpy()
    )

    mean_positive_demand = (
        positive_demand_group[
            "TotalDemand"
        ].mean()
    )

    median_positive_demand = (
        positive_demand_group[
            "TotalDemand"
        ].median()
    )

    if positive_demand_days >= 2:

        positive_demand_standard_deviation = (
            positive_demand_group[
                "TotalDemand"
            ].std(ddof=1)
        )

        cv_squared_positive_demand = (
            (
                positive_demand_standard_deviation
                / mean_positive_demand
            ) ** 2
        )

        mean_positive_interval = float(
            positive_intervals.mean()
        )

        median_positive_interval = float(
            np.median(positive_intervals)
        )

        maximum_positive_interval = int(
            positive_intervals.max()
        )

    else:

        positive_demand_standard_deviation = np.nan
        cv_squared_positive_demand = np.nan
        mean_positive_interval = np.nan
        median_positive_interval = np.nan
        maximum_positive_interval = np.nan

    average_demand_interval = (
        history_operating_days
        / positive_demand_days
    )

    longest_zero_demand_run = (
        calculate_longest_true_run(
            zero_demand_flags
        )
    )

    trailing_zero_demand_run = (
        calculate_trailing_true_run(
            zero_demand_flags
        )
    )

    first_positive_demand_date = (
        positive_demand_group["Date"].min()
    )

    last_positive_demand_date = (
        positive_demand_group["Date"].max()
    )

    last_positive_operating_sequence = int(
        positive_demand_group[
            "OperatingDaySequence"
        ].max()
    )

    operating_days_since_last_positive_demand = (
        FINAL_OPERATING_SEQUENCE
        - last_positive_operating_sequence
    )

    product_profile_records.append({
        "CanonicalProductID":
            canonical_product_id,

        "CanonicalProductName":
            canonical_product_name,

        "ProductFirstObservedDate":
            product_group[
                "ProductFirstObservedDate"
            ].iloc[0],

        "PanelFinalDate":
            product_group["Date"].max(),

        "HistoryOperatingDays":
            history_operating_days,

        "PositiveDemandDays":
            positive_demand_days,

        "ZeroDemandDays":
            zero_demand_days,

        "PositiveDemandRate":
            (
                positive_demand_days
                / history_operating_days
            ),

        "ZeroDemandRate":
            (
                zero_demand_days
                / history_operating_days
            ),

        "NormalDemandUnits":
            int(
                product_group[
                    "NormalDemand"
                ].sum()
            ),

        "BulkDemandUnits":
            int(
                product_group[
                    "BulkDemand"
                ].sum()
            ),

        "TotalDemandUnits":
            int(
                total_demand_values.sum()
            ),

        "MeanDemandAllOperatingDays":
            float(
                total_demand_values.mean()
            ),

        "MedianDemandAllOperatingDays":
            float(
                total_demand_values.median()
            ),

        "MeanPositiveDemand":
            float(mean_positive_demand),

        "MedianPositiveDemand":
            float(median_positive_demand),

        "StandardDeviationPositiveDemand":
            (
                float(
                    positive_demand_standard_deviation
                )
                if pd.notna(
                    positive_demand_standard_deviation
                )
                else np.nan
            ),

        "CVSquaredPositiveDemand":
            (
                float(
                    cv_squared_positive_demand
                )
                if pd.notna(
                    cv_squared_positive_demand
                )
                else np.nan
            ),

        "AverageDemandInterval_ADI":
            float(average_demand_interval),

        "MeanIntervalBetweenPositiveDemand":
            mean_positive_interval,

        "MedianIntervalBetweenPositiveDemand":
            median_positive_interval,

        "MaximumIntervalBetweenPositiveDemand":
            maximum_positive_interval,

        "LongestZeroDemandRun":
            longest_zero_demand_run,

        "TrailingZeroDemandRun":
            trailing_zero_demand_run,

        "FirstPositiveDemandDate":
            first_positive_demand_date,

        "LastObservedPositiveDemandDate":
            last_positive_demand_date,

        "OperatingDaysSinceLastPositiveDemand":
            int(
                operating_days_since_last_positive_demand
            ),

        "SoldOnFinalOperatingDate":
            bool(
                operating_days_since_last_positive_demand
                == 0
            ),

        "RetainForForecasting":
            True
    })


product_demand_profile_df = pd.DataFrame(
    product_profile_records
)

assert len(product_demand_profile_df) == 227

assert product_demand_profile_df[
    "CanonicalProductID"
].nunique() == 227


# ------------------------------------------------------------
# 6. Validate profile calculations
# ------------------------------------------------------------

history_day_mismatches = int(
    (
        product_demand_profile_df[
            "PositiveDemandDays"
        ]
        + product_demand_profile_df[
            "ZeroDemandDays"
        ]
        != product_demand_profile_df[
            "HistoryOperatingDays"
        ]
    ).sum()
)

demand_component_mismatches = int(
    (
        product_demand_profile_df[
            "NormalDemandUnits"
        ]
        + product_demand_profile_df[
            "BulkDemandUnits"
        ]
        != product_demand_profile_df[
            "TotalDemandUnits"
        ]
    ).sum()
)

trailing_run_mismatches = int(
    (
        product_demand_profile_df[
            "TrailingZeroDemandRun"
        ]
        != product_demand_profile_df[
            "OperatingDaysSinceLastPositiveDemand"
        ]
    ).sum()
)

first_positive_date_mismatches = int(
    (
        product_demand_profile_df[
            "FirstPositiveDemandDate"
        ]
        != product_demand_profile_df[
            "ProductFirstObservedDate"
        ]
    ).sum()
)

products_not_continued_to_final_date = int(
    (
        product_demand_profile_df[
            "PanelFinalDate"
        ]
        != FINAL_OPERATING_DATE
    ).sum()
)

products_with_no_positive_demand = int(
    (
        product_demand_profile_df[
            "PositiveDemandDays"
        ] == 0
    ).sum()
)

assert history_day_mismatches == 0
assert demand_component_mismatches == 0
assert trailing_run_mismatches == 0
assert first_positive_date_mismatches == 0
assert products_not_continued_to_final_date == 0
assert products_with_no_positive_demand == 0

assert product_demand_profile_df[
    "NormalDemandUnits"
].sum() == 114_186

assert product_demand_profile_df[
    "BulkDemandUnits"
].sum() == 1_972

assert product_demand_profile_df[
    "TotalDemandUnits"
].sum() == 116_158


# ------------------------------------------------------------
# 7. Apply ADI-CV² demand-pattern classification
# ------------------------------------------------------------

ADI_THRESHOLD = 1.32
CV_SQUARED_THRESHOLD = 0.49

classification_conditions = [
    (
        product_demand_profile_df[
            "PositiveDemandDays"
        ] < 2
    ),
    (
        product_demand_profile_df[
            "AverageDemandInterval_ADI"
        ] < ADI_THRESHOLD
    )
    & (
        product_demand_profile_df[
            "CVSquaredPositiveDemand"
        ] < CV_SQUARED_THRESHOLD
    ),
    (
        product_demand_profile_df[
            "AverageDemandInterval_ADI"
        ] >= ADI_THRESHOLD
    )
    & (
        product_demand_profile_df[
            "CVSquaredPositiveDemand"
        ] < CV_SQUARED_THRESHOLD
    ),
    (
        product_demand_profile_df[
            "AverageDemandInterval_ADI"
        ] < ADI_THRESHOLD
    )
    & (
        product_demand_profile_df[
            "CVSquaredPositiveDemand"
        ] >= CV_SQUARED_THRESHOLD
    ),
    (
        product_demand_profile_df[
            "AverageDemandInterval_ADI"
        ] >= ADI_THRESHOLD
    )
    & (
        product_demand_profile_df[
            "CVSquaredPositiveDemand"
        ] >= CV_SQUARED_THRESHOLD
    )
]

classification_labels = [
    "SINGLE_POSITIVE_DEMAND_DAY",
    "SMOOTH",
    "INTERMITTENT",
    "ERRATIC",
    "LUMPY"
]

product_demand_profile_df[
    "DemandPatternClass"
] = np.select(
    classification_conditions,
    classification_labels,
    default="UNCLASSIFIED"
)

unclassified_products = int(
    (
        product_demand_profile_df[
            "DemandPatternClass"
        ]
        == "UNCLASSIFIED"
    ).sum()
)

assert unclassified_products == 0


# ------------------------------------------------------------
# 8. Add history and sparsity bands
# ------------------------------------------------------------

product_demand_profile_df[
    "HistoryLengthBand"
] = pd.cut(
    product_demand_profile_df[
        "HistoryOperatingDays"
    ],
    bins=[
        0,
        29,
        59,
        119,
        179,
        np.inf
    ],
    labels=[
        "LT_30_DAYS",
        "30_TO_59_DAYS",
        "60_TO_119_DAYS",
        "120_TO_179_DAYS",
        "180_PLUS_DAYS"
    ],
    include_lowest=True
)

product_demand_profile_df[
    "PositiveDemandEvidenceBand"
] = pd.cut(
    product_demand_profile_df[
        "PositiveDemandDays"
    ],
    bins=[
        0,
        1,
        4,
        14,
        29,
        np.inf
    ],
    labels=[
        "1_DAY",
        "2_TO_4_DAYS",
        "5_TO_14_DAYS",
        "15_TO_29_DAYS",
        "30_PLUS_DAYS"
    ],
    include_lowest=True
)

product_demand_profile_df[
    "ZeroDemandRateBand"
] = pd.cut(
    product_demand_profile_df[
        "ZeroDemandRate"
    ],
    bins=[
        -0.0001,
        0.25,
        0.50,
        0.75,
        0.90,
        1.00
    ],
    labels=[
        "LE_25_PERCENT",
        "25_TO_50_PERCENT",
        "50_TO_75_PERCENT",
        "75_TO_90_PERCENT",
        "GT_90_PERCENT"
    ],
    include_lowest=True
)


# ------------------------------------------------------------
# 9. Add diagnostic forecasting-treatment groups
# ------------------------------------------------------------

forecast_treatment_mapping = {
    "SMOOTH":
        "REGULAR_DEMAND_METHODS",

    "ERRATIC":
        "REGULAR_DEMAND_METHODS",

    "INTERMITTENT":
        "INTERMITTENT_DEMAND_METHODS",

    "LUMPY":
        "INTERMITTENT_DEMAND_METHODS",

    "SINGLE_POSITIVE_DEMAND_DAY":
        "LIMITED_POSITIVE_DEMAND_EVIDENCE"
}

product_demand_profile_df[
    "DiagnosticForecastTreatment"
] = (
    product_demand_profile_df[
        "DemandPatternClass"
    ]
    .map(forecast_treatment_mapping)
)

assert product_demand_profile_df[
    "DiagnosticForecastTreatment"
].isna().sum() == 0


# ------------------------------------------------------------
# 10. Sort the completed profile
# ------------------------------------------------------------

product_demand_profile_df = (
    product_demand_profile_df
    .sort_values(
        [
            "DemandPatternClass",
            "ZeroDemandRate",
            "CanonicalProductID"
        ],
        ascending=[
            True,
            False,
            True
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Print Cell 15 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 4 — PRODUCT DEMAND PROFILE CREATED")
print("=" * 75)

print()
print("Panel confirmed:")
print(
    f"Rows analysed: "
    f"{len(demand_analysis_panel_df):,}"
)
print(
    "Products analysed:",
    product_demand_profile_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    demand_analysis_panel_df[
        "Date"
    ].nunique()
)
print(
    "Total demand units:",
    f"{product_demand_profile_df['TotalDemandUnits'].sum():,}"
)

print()
print("Profile validation mismatches:")
print(
    "History-day mismatches:",
    history_day_mismatches
)
print(
    "Demand-component mismatches:",
    demand_component_mismatches
)
print(
    "Trailing-zero-run mismatches:",
    trailing_run_mismatches
)
print(
    "First-positive-date mismatches:",
    first_positive_date_mismatches
)
print(
    "Products not continued to final date:",
    products_not_continued_to_final_date
)

print()
print("Demand-pattern classification:")
display(
    product_demand_profile_df[
        "DemandPatternClass"
    ]
    .value_counts()
    .rename_axis("DemandPatternClass")
    .reset_index(name="ProductCount")
)

print()
print("Cell 15 completed successfully.")

STEP 1, PART 4 — PRODUCT DEMAND PROFILE CREATED

Panel confirmed:
Rows analysed: 43,774
Products analysed: 227
Operating dates: 245
Total demand units: 116,158

Profile validation mismatches:
History-day mismatches: 0
Demand-component mismatches: 0
Trailing-zero-run mismatches: 0
First-positive-date mismatches: 0
Products not continued to final date: 0

Demand-pattern classification:


,DemandPatternClass,ProductCount
0,INTERMITTENT,126
1,LUMPY,43
2,SMOOTH,35
3,ERRATIC,16
4,SINGLE_POSITIVE_DEMAND_DAY,7



Cell 15 completed successfully.


In [27]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 4
# Cell 16: Create demand-history, sparsity and treatment summaries
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 15 object exists
# ------------------------------------------------------------

if "product_demand_profile_df" not in globals():
    raise NameError(
        "product_demand_profile_df is missing. "
        "Run Cell 15 before Cell 16."
    )


# ------------------------------------------------------------
# 2. Demand-pattern summary
# ------------------------------------------------------------

demand_pattern_summary_df = (
    product_demand_profile_df
    .groupby(
        "DemandPatternClass",
        observed=False,
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        TotalDemandUnits=(
            "TotalDemandUnits",
            "sum"
        ),
        MedianHistoryOperatingDays=(
            "HistoryOperatingDays",
            "median"
        ),
        MedianPositiveDemandDays=(
            "PositiveDemandDays",
            "median"
        ),
        MedianZeroDemandRate=(
            "ZeroDemandRate",
            "median"
        ),
        MedianADI=(
            "AverageDemandInterval_ADI",
            "median"
        ),
        MedianCVSquared=(
            "CVSquaredPositiveDemand",
            "median"
        )
    )
)

demand_pattern_summary_df[
    "ProductPercentage"
] = (
    demand_pattern_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)

demand_pattern_summary_df[
    "DemandUnitPercentage"
] = (
    demand_pattern_summary_df[
        "TotalDemandUnits"
    ]
    / 116_158
    * 100
).round(2)

demand_pattern_summary_df[
    "MedianZeroDemandRate"
] = (
    demand_pattern_summary_df[
        "MedianZeroDemandRate"
    ]
    .round(4)
)

demand_pattern_summary_df[
    "MedianADI"
] = (
    demand_pattern_summary_df[
        "MedianADI"
    ]
    .round(4)
)

demand_pattern_summary_df[
    "MedianCVSquared"
] = (
    demand_pattern_summary_df[
        "MedianCVSquared"
    ]
    .round(4)
)


# ------------------------------------------------------------
# 3. History-length summary
# ------------------------------------------------------------

history_length_summary_df = (
    product_demand_profile_df[
        "HistoryLengthBand"
    ]
    .value_counts(sort=False)
    .rename_axis("HistoryLengthBand")
    .reset_index(name="ProductCount")
)

history_length_summary_df[
    "ProductPercentage"
] = (
    history_length_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)


# ------------------------------------------------------------
# 4. Positive-demand evidence summary
# ------------------------------------------------------------

positive_evidence_summary_df = (
    product_demand_profile_df[
        "PositiveDemandEvidenceBand"
    ]
    .value_counts(sort=False)
    .rename_axis(
        "PositiveDemandEvidenceBand"
    )
    .reset_index(name="ProductCount")
)

positive_evidence_summary_df[
    "ProductPercentage"
] = (
    positive_evidence_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)


# ------------------------------------------------------------
# 5. Zero-demand-rate summary
# ------------------------------------------------------------

zero_demand_rate_summary_df = (
    product_demand_profile_df[
        "ZeroDemandRateBand"
    ]
    .value_counts(sort=False)
    .rename_axis("ZeroDemandRateBand")
    .reset_index(name="ProductCount")
)

zero_demand_rate_summary_df[
    "ProductPercentage"
] = (
    zero_demand_rate_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)


# ------------------------------------------------------------
# 6. Forecast-treatment summary
# ------------------------------------------------------------

forecast_treatment_summary_df = (
    product_demand_profile_df
    .groupby(
        "DiagnosticForecastTreatment",
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        TotalDemandUnits=(
            "TotalDemandUnits",
            "sum"
        )
    )
)

forecast_treatment_summary_df[
    "ProductPercentage"
] = (
    forecast_treatment_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)

forecast_treatment_summary_df[
    "DemandUnitPercentage"
] = (
    forecast_treatment_summary_df[
        "TotalDemandUnits"
    ]
    / 116_158
    * 100
).round(2)


# ------------------------------------------------------------
# 7. High-sparsity audit
# ------------------------------------------------------------

high_sparsity_product_audit_df = (
    product_demand_profile_df.loc[
        product_demand_profile_df[
            "ZeroDemandRate"
        ] > 0.90
    ]
    .copy()
    .sort_values(
        [
            "ZeroDemandRate",
            "PositiveDemandDays",
            "CanonicalProductID"
        ],
        ascending=[
            False,
            True,
            True
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

high_sparsity_product_audit_df[
    "AuditReason"
] = (
    "ZERO_DEMAND_RATE_GREATER_THAN_90_PERCENT"
)


# ------------------------------------------------------------
# 8. Limited positive-demand evidence audit
# ------------------------------------------------------------

limited_demand_evidence_audit_df = (
    product_demand_profile_df.loc[
        (
            product_demand_profile_df[
                "PositiveDemandDays"
            ] < 5
        )
        |
        (
            product_demand_profile_df[
                "HistoryOperatingDays"
            ] < 30
        )
    ]
    .copy()
    .sort_values(
        [
            "PositiveDemandDays",
            "HistoryOperatingDays",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

limited_demand_evidence_audit_df[
    "AuditReason"
] = np.select(
    [
        (
            limited_demand_evidence_audit_df[
                "PositiveDemandDays"
            ] < 5
        )
        &
        (
            limited_demand_evidence_audit_df[
                "HistoryOperatingDays"
            ] < 30
        ),
        (
            limited_demand_evidence_audit_df[
                "PositiveDemandDays"
            ] < 5
        ),
        (
            limited_demand_evidence_audit_df[
                "HistoryOperatingDays"
            ] < 30
        )
    ],
    [
        "LIMITED_HISTORY_AND_FEWER_THAN_5_POSITIVE_DAYS",
        "FEWER_THAN_5_POSITIVE_DEMAND_DAYS",
        "FEWER_THAN_30_OPERATING_DAYS_HISTORY"
    ],
    default="REVIEW"
)


# ------------------------------------------------------------
# 9. Validate all summaries
# ------------------------------------------------------------

# Recalculate cohort sizes directly from the completed profile.
# This avoids relying on manually entered hardcoded counts.

expected_high_sparsity_count = int(
    (
        product_demand_profile_df[
            "ZeroDemandRate"
        ] > 0.90
    ).sum()
)

expected_limited_evidence_count = int(
    (
        (
            product_demand_profile_df[
                "PositiveDemandDays"
            ] < 5
        )
        |
        (
            product_demand_profile_df[
                "HistoryOperatingDays"
            ] < 30
        )
    ).sum()
)

assert demand_pattern_summary_df[
    "ProductCount"
].sum() == 227

assert history_length_summary_df[
    "ProductCount"
].sum() == 227

assert positive_evidence_summary_df[
    "ProductCount"
].sum() == 227

assert zero_demand_rate_summary_df[
    "ProductCount"
].sum() == 227

assert forecast_treatment_summary_df[
    "ProductCount"
].sum() == 227

assert demand_pattern_summary_df[
    "TotalDemandUnits"
].sum() == 116_158

assert forecast_treatment_summary_df[
    "TotalDemandUnits"
].sum() == 116_158

assert len(
    high_sparsity_product_audit_df
) == expected_high_sparsity_count, (
    "High-sparsity audit count does not match "
    "the product-profile condition."
)

assert len(
    limited_demand_evidence_audit_df
) == expected_limited_evidence_count, (
    "Limited-evidence audit count does not match "
    "the product-profile condition."
)

assert expected_high_sparsity_count == 65

assert expected_limited_evidence_count == 22

assert product_demand_profile_df[
    "RetainForForecasting"
].all()

print("Summary validation passed.")
print(
    "Expected high-sparsity products:",
    expected_high_sparsity_count
)
print(
    "Expected limited-evidence products:",
    expected_limited_evidence_count
)

# ------------------------------------------------------------
# 10. Print summaries
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 4 — DEMAND DIAGNOSTIC SUMMARIES CREATED")
print("=" * 75)

print()
print("Demand-pattern summary:")
display(demand_pattern_summary_df)

print()
print("History-length summary:")
display(history_length_summary_df)

print()
print("Positive-demand evidence summary:")
display(positive_evidence_summary_df)

print()
print("Zero-demand-rate summary:")
display(zero_demand_rate_summary_df)

print()
print("Diagnostic forecasting-treatment summary:")
display(forecast_treatment_summary_df)

print()
print("Audit cohort sizes:")
print(
    "Products with more than 90% zero-demand dates:",
    len(high_sparsity_product_audit_df)
)
print(
    "Products with limited history or fewer than "
    "5 positive-demand days:",
    len(limited_demand_evidence_audit_df)
)

print()
print(
    "All 227 products remain retained for forecasting."
)

print()
print("Cell 16 completed successfully.")

Summary validation passed.
Expected high-sparsity products: 65
Expected limited-evidence products: 22
STEP 1, PART 4 — DEMAND DIAGNOSTIC SUMMARIES CREATED

Demand-pattern summary:


,DemandPatternClass,ProductCount,TotalDemandUnits,MedianHistoryOperatingDays,MedianPositiveDemandDays,MedianZeroDemandRate,MedianADI,MedianCVSquared,ProductPercentage,DemandUnitPercentage
0,ERRATIC,16,25209,245.0,212.5,0.0909,1.1000,0.8313,7.05,21.70
1,INTERMITTENT,126,18360,243.0,29.0,0.8525,6.7778,0.2665,55.51,15.81
2,LUMPY,43,20668,244.0,71.0,0.6653,2.9878,0.6088,18.94,17.79
3,SINGLE_POSITIVE_DEMAND_DAY,7,7,120.0,1.0,0.9917,120.0000,NaN,3.08,0.01
4,SMOOTH,35,51914,110.0,104.0,0.0727,1.0784,0.2947,15.42,44.69



History-length summary:


,HistoryLengthBand,ProductCount,ProductPercentage
0,LT_30_DAYS,3,1.32
1,30_TO_59_DAYS,1,0.44
2,60_TO_119_DAYS,68,29.96
3,120_TO_179_DAYS,5,2.20
4,180_PLUS_DAYS,150,66.08



Positive-demand evidence summary:


...


Zero-demand-rate summary:


,ZeroDemandRateBand,ProductCount,ProductPercentage
0,LE_25_PERCENT,51,22.47
1,25_TO_50_PERCENT,19,8.37
2,50_TO_75_PERCENT,52,22.91
3,75_TO_90_PERCENT,40,17.62
4,GT_90_PERCENT,65,28.63



Diagnostic forecasting-treatment summary:


,DiagnosticForecastTreatment,ProductCount,TotalDemandUnits,ProductPercentage,DemandUnitPercentage
0,INTERMITTENT_DEMAND_METHODS,169,39028,74.45,33.60
1,LIMITED_POSITIVE_DEMAND_EVIDENCE,7,7,3.08,0.01
2,REGULAR_DEMAND_METHODS,51,77123,22.47,66.39



Audit cohort sizes:
Products with more than 90% zero-demand dates: 65
Products with limited history or fewer than 5 positive-demand days: 22

All 227 products remain retained for forecasting.

Cell 16 completed successfully.


In [28]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 4
# Corrected Cell 17:
# Save Part 4 outputs and update the project handoff
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Confirm all required Part 4 objects exist
# ------------------------------------------------------------

required_part_4_objects = [
    "product_demand_profile_df",
    "demand_pattern_summary_df",
    "history_length_summary_df",
    "positive_evidence_summary_df",
    "zero_demand_rate_summary_df",
    "forecast_treatment_summary_df",
    "high_sparsity_product_audit_df",
    "limited_demand_evidence_audit_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_part_4_objects = [
    object_name
    for object_name in required_part_4_objects
    if object_name not in globals()
]

if missing_part_4_objects:
    raise NameError(
        "The following Part 4 objects are missing:\n"
        f"{missing_part_4_objects}\n\n"
        "Run Cells 15 and the corrected Cell 16 before "
        "running Cell 17."
    )


# ------------------------------------------------------------
# 2. Recalculate expected diagnostic cohort sizes
# ------------------------------------------------------------

expected_high_sparsity_count = int(
    (
        product_demand_profile_df[
            "ZeroDemandRate"
        ] > 0.90
    ).sum()
)

expected_limited_evidence_count = int(
    (
        (
            product_demand_profile_df[
                "PositiveDemandDays"
            ] < 5
        )
        |
        (
            product_demand_profile_df[
                "HistoryOperatingDays"
            ] < 30
        )
    ).sum()
)

expected_high_sparsity_ids = set(
    product_demand_profile_df.loc[
        product_demand_profile_df[
            "ZeroDemandRate"
        ] > 0.90,
        "CanonicalProductID"
    ].astype(str)
)

expected_limited_evidence_ids = set(
    product_demand_profile_df.loc[
        (
            product_demand_profile_df[
                "PositiveDemandDays"
            ] < 5
        )
        |
        (
            product_demand_profile_df[
                "HistoryOperatingDays"
            ] < 30
        ),
        "CanonicalProductID"
    ].astype(str)
)

assert expected_high_sparsity_count == len(
    high_sparsity_product_audit_df
), (
    "The high-sparsity audit does not match the condition "
    "ZeroDemandRate > 0.90."
)

assert expected_limited_evidence_count == len(
    limited_demand_evidence_audit_df
), (
    "The limited-evidence audit does not match the condition "
    "PositiveDemandDays < 5 or HistoryOperatingDays < 30."
)


# ------------------------------------------------------------
# 3. Validate the in-memory Part 4 results before saving
# ------------------------------------------------------------

assert len(product_demand_profile_df) == 227

assert product_demand_profile_df[
    "CanonicalProductID"
].nunique() == 227

assert product_demand_profile_df[
    "TotalDemandUnits"
].sum() == 116_158

assert demand_pattern_summary_df[
    "ProductCount"
].sum() == 227

assert demand_pattern_summary_df[
    "TotalDemandUnits"
].sum() == 116_158

assert history_length_summary_df[
    "ProductCount"
].sum() == 227

assert positive_evidence_summary_df[
    "ProductCount"
].sum() == 227

assert zero_demand_rate_summary_df[
    "ProductCount"
].sum() == 227

assert forecast_treatment_summary_df[
    "ProductCount"
].sum() == 227

assert forecast_treatment_summary_df[
    "TotalDemandUnits"
].sum() == 116_158

assert product_demand_profile_df[
    "RetainForForecasting"
].astype(bool).all()

assert set(
    high_sparsity_product_audit_df[
        "CanonicalProductID"
    ].astype(str)
) == expected_high_sparsity_ids

assert set(
    limited_demand_evidence_audit_df[
        "CanonicalProductID"
    ].astype(str)
) == expected_limited_evidence_ids


# ------------------------------------------------------------
# 4. Define Part 4 output paths
# ------------------------------------------------------------

PRODUCT_DEMAND_PROFILE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_product_demand_behaviour_profile.csv"
)

DEMAND_PATTERN_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_demand_pattern_summary.csv"
)

HISTORY_LENGTH_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_history_length_summary.csv"
)

POSITIVE_EVIDENCE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_positive_demand_evidence_summary.csv"
)

ZERO_DEMAND_RATE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_zero_demand_rate_summary.csv"
)

FORECAST_TREATMENT_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_forecast_treatment_summary.csv"
)

HIGH_SPARSITY_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_high_sparsity_product_audit.csv"
)

LIMITED_EVIDENCE_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "04_limited_demand_evidence_audit.csv"
)


# ------------------------------------------------------------
# 5. Save all Part 4 outputs
# ------------------------------------------------------------

product_demand_profile_df.to_csv(
    PRODUCT_DEMAND_PROFILE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

demand_pattern_summary_df.to_csv(
    DEMAND_PATTERN_SUMMARY_OUTPUT,
    index=False
)

history_length_summary_df.to_csv(
    HISTORY_LENGTH_SUMMARY_OUTPUT,
    index=False
)

positive_evidence_summary_df.to_csv(
    POSITIVE_EVIDENCE_SUMMARY_OUTPUT,
    index=False
)

zero_demand_rate_summary_df.to_csv(
    ZERO_DEMAND_RATE_SUMMARY_OUTPUT,
    index=False
)

forecast_treatment_summary_df.to_csv(
    FORECAST_TREATMENT_SUMMARY_OUTPUT,
    index=False
)

high_sparsity_product_audit_df.to_csv(
    HIGH_SPARSITY_AUDIT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

limited_demand_evidence_audit_df.to_csv(
    LIMITED_EVIDENCE_AUDIT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)


# ------------------------------------------------------------
# 6. Confirm every output file exists
# ------------------------------------------------------------

part_4_output_paths = [
    PRODUCT_DEMAND_PROFILE_OUTPUT,
    DEMAND_PATTERN_SUMMARY_OUTPUT,
    HISTORY_LENGTH_SUMMARY_OUTPUT,
    POSITIVE_EVIDENCE_SUMMARY_OUTPUT,
    ZERO_DEMAND_RATE_SUMMARY_OUTPUT,
    FORECAST_TREATMENT_SUMMARY_OUTPUT,
    HIGH_SPARSITY_AUDIT_OUTPUT,
    LIMITED_EVIDENCE_AUDIT_OUTPUT
]

missing_saved_files = [
    file_path
    for file_path in part_4_output_paths
    if not file_path.exists()
]

assert not missing_saved_files, (
    "The following Part 4 files were not saved:\n"
    f"{missing_saved_files}"
)


# ------------------------------------------------------------
# 7. Reload all saved outputs
# ------------------------------------------------------------

saved_product_profile = pd.read_csv(
    PRODUCT_DEMAND_PROFILE_OUTPUT,
    low_memory=False
)

saved_pattern_summary = pd.read_csv(
    DEMAND_PATTERN_SUMMARY_OUTPUT,
    low_memory=False
)

saved_history_summary = pd.read_csv(
    HISTORY_LENGTH_SUMMARY_OUTPUT,
    low_memory=False
)

saved_positive_summary = pd.read_csv(
    POSITIVE_EVIDENCE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_zero_rate_summary = pd.read_csv(
    ZERO_DEMAND_RATE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_treatment_summary = pd.read_csv(
    FORECAST_TREATMENT_SUMMARY_OUTPUT,
    low_memory=False
)

saved_high_sparsity = pd.read_csv(
    HIGH_SPARSITY_AUDIT_OUTPUT,
    low_memory=False
)

saved_limited_evidence = pd.read_csv(
    LIMITED_EVIDENCE_AUDIT_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 8. Validate the reloaded outputs
# ------------------------------------------------------------

assert len(saved_product_profile) == 227

assert saved_product_profile[
    "CanonicalProductID"
].nunique() == 227

assert saved_product_profile[
    "TotalDemandUnits"
].sum() == 116_158

assert saved_pattern_summary[
    "ProductCount"
].sum() == 227

assert saved_pattern_summary[
    "TotalDemandUnits"
].sum() == 116_158

assert saved_history_summary[
    "ProductCount"
].sum() == 227

assert saved_positive_summary[
    "ProductCount"
].sum() == 227

assert saved_zero_rate_summary[
    "ProductCount"
].sum() == 227

assert saved_treatment_summary[
    "ProductCount"
].sum() == 227

assert saved_treatment_summary[
    "TotalDemandUnits"
].sum() == 116_158

assert len(
    saved_high_sparsity
) == expected_high_sparsity_count

assert len(
    saved_limited_evidence
) == expected_limited_evidence_count

assert set(
    saved_high_sparsity[
        "CanonicalProductID"
    ].astype(str)
) == expected_high_sparsity_ids

assert set(
    saved_limited_evidence[
        "CanonicalProductID"
    ].astype(str)
) == expected_limited_evidence_ids


# ------------------------------------------------------------
# 9. Validate the retained-for-forecasting field after reload
# ------------------------------------------------------------

saved_retain_values = (
    saved_product_profile[
        "RetainForForecasting"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

saved_retain_boolean = saved_retain_values.map({
    "true": True,
    "false": False,
    "1": True,
    "0": False
})

assert saved_retain_boolean.isna().sum() == 0, (
    "Unrecognised RetainForForecasting values were found "
    "after reloading the product profile."
)

products_retained_for_forecasting = int(
    saved_retain_boolean.sum()
)

assert products_retained_for_forecasting == 227


# ------------------------------------------------------------
# 10. Extract verified demand-pattern counts
# ------------------------------------------------------------

pattern_count_lookup = (
    saved_product_profile[
        "DemandPatternClass"
    ]
    .value_counts()
    .to_dict()
)

smooth_product_count = int(
    pattern_count_lookup.get(
        "SMOOTH",
        0
    )
)

intermittent_product_count = int(
    pattern_count_lookup.get(
        "INTERMITTENT",
        0
    )
)

erratic_product_count = int(
    pattern_count_lookup.get(
        "ERRATIC",
        0
    )
)

lumpy_product_count = int(
    pattern_count_lookup.get(
        "LUMPY",
        0
    )
)

single_positive_day_product_count = int(
    pattern_count_lookup.get(
        "SINGLE_POSITIVE_DEMAND_DAY",
        0
    )
)

assert (
    smooth_product_count
    + intermittent_product_count
    + erratic_product_count
    + lumpy_product_count
    + single_positive_day_product_count
    == 227
)


# ------------------------------------------------------------
# 11. Extract verified treatment-group counts
# ------------------------------------------------------------

treatment_count_lookup = (
    saved_product_profile[
        "DiagnosticForecastTreatment"
    ]
    .value_counts()
    .to_dict()
)

regular_method_product_count = int(
    treatment_count_lookup.get(
        "REGULAR_DEMAND_METHODS",
        0
    )
)

intermittent_method_product_count = int(
    treatment_count_lookup.get(
        "INTERMITTENT_DEMAND_METHODS",
        0
    )
)

limited_evidence_treatment_count = int(
    treatment_count_lookup.get(
        "LIMITED_POSITIVE_DEMAND_EVIDENCE",
        0
    )
)

assert (
    regular_method_product_count
    + intermittent_method_product_count
    + limited_evidence_treatment_count
    == 227
)


# ------------------------------------------------------------
# 12. Build Markdown summary tables
# ------------------------------------------------------------

pattern_markdown_rows = "\n".join(
    (
        f"- `{row.DemandPatternClass}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%), "
        f"{int(row.TotalDemandUnits):,} demand units "
        f"({row.DemandUnitPercentage:.2f}%)"
    )
    for row in demand_pattern_summary_df.itertuples()
)

history_markdown_rows = "\n".join(
    (
        f"- `{row.HistoryLengthBand}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%)"
    )
    for row in history_length_summary_df.itertuples()
)

positive_evidence_markdown_rows = "\n".join(
    (
        f"- `{row.PositiveDemandEvidenceBand}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%)"
    )
    for row in positive_evidence_summary_df.itertuples()
)

treatment_markdown_rows = "\n".join(
    (
        f"- `{row.DiagnosticForecastTreatment}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%), "
        f"{int(row.TotalDemandUnits):,} demand units "
        f"({row.DemandUnitPercentage:.2f}%)"
    )
    for row in forecast_treatment_summary_df.itertuples()
)


# ------------------------------------------------------------
# 13. Update the Markdown handoff
# ------------------------------------------------------------

part_4_summary = f"""
**Status:** Completed and validated

### Purpose

Demand history, sparsity and intermittent-demand behaviour were
analysed for all 227 continuing canonical products using the
corrected 43,774-row temporal panel.

No products were removed, labelled inactive or classified as
discontinued.

### Demand-pattern method

The diagnostic classification uses:

- Average Demand Interval threshold: `ADI = 1.32`
- Squared coefficient of variation threshold: `CV² = 0.49`

Products with only one positive-demand day were recorded separately
because a meaningful positive-demand variance cannot be calculated.

### Demand-pattern results

- Smooth products: {smooth_product_count}
- Intermittent products: {intermittent_product_count}
- Erratic products: {erratic_product_count}
- Lumpy products: {lumpy_product_count}
- Products with one positive-demand day: {single_positive_day_product_count}

{pattern_markdown_rows}

### Diagnostic forecasting-treatment groups

- Regular-demand methods: {regular_method_product_count} products
- Intermittent-demand methods: {intermittent_method_product_count} products
- Limited positive-demand evidence: {limited_evidence_treatment_count} products

{treatment_markdown_rows}

These are diagnostic groupings only. Final forecasting-method
selection will be based on time-based validation and comparative
forecast performance.

### History-length distribution

{history_markdown_rows}

### Positive-demand evidence distribution

{positive_evidence_markdown_rows}

### Sparsity and evidence audits

- Products with more than 90% zero-demand operating dates: {expected_high_sparsity_count}
- Products with fewer than 5 positive-demand days or fewer than 30 operating days of history: {expected_limited_evidence_count}
- Products retained for forecasting: {products_retained_for_forecasting}

The audit groups identify products requiring careful validation.
They do not remove products from the forecasting project.

### Demand preservation

- Total historical demand units represented: {int(saved_product_profile["TotalDemandUnits"].sum()):,}

### Saved Part 4 outputs

- `04_product_demand_behaviour_profile.csv`
- `04_demand_pattern_summary.csv`
- `04_history_length_summary.csv`
- `04_positive_demand_evidence_summary.csv`
- `04_zero_demand_rate_summary.csv`
- `04_forecast_treatment_summary.csv`
- `04_high_sparsity_product_audit.csv`
- `04_limited_demand_evidence_audit.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_4",
    section_title=(
        "Forecasting Preparation — Step 1, Part 4"
    ),
    section_body=part_4_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 14. Print final Part 4 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 1, PART 4 COMPLETED"
)
print("=" * 75)

print()
print("Products analysed:")
print(
    f"Total products: "
    f"{len(saved_product_profile):,}"
)
print(
    "Products retained for forecasting:",
    products_retained_for_forecasting
)

print()
print("Demand-pattern classification:")
print(f"Smooth: {smooth_product_count}")
print(
    f"Intermittent: "
    f"{intermittent_product_count}"
)
print(f"Erratic: {erratic_product_count}")
print(f"Lumpy: {lumpy_product_count}")
print(
    "Single positive-demand day:",
    single_positive_day_product_count
)

print()
print("Diagnostic forecasting-treatment groups:")
print(
    "Regular-demand methods:",
    regular_method_product_count
)
print(
    "Intermittent-demand methods:",
    intermittent_method_product_count
)
print(
    "Limited positive-demand evidence:",
    limited_evidence_treatment_count
)

print()
print("Diagnostic audit cohorts:")
print(
    "More than 90% zero-demand dates:",
    len(saved_high_sparsity)
)
print(
    "Limited history or positive-demand evidence:",
    len(saved_limited_evidence)
)

print()
print("Demand preserved:")
print(
    "Total demand units represented:",
    f"{saved_product_profile['TotalDemandUnits'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {PRODUCT_DEMAND_PROFILE_OUTPUT}")
print(f"2. {DEMAND_PATTERN_SUMMARY_OUTPUT}")
print(f"3. {HISTORY_LENGTH_SUMMARY_OUTPUT}")
print(f"4. {POSITIVE_EVIDENCE_SUMMARY_OUTPUT}")
print(f"5. {ZERO_DEMAND_RATE_SUMMARY_OUTPUT}")
print(f"6. {FORECAST_TREATMENT_SUMMARY_OUTPUT}")
print(f"7. {HIGH_SPARSITY_AUDIT_OUTPUT}")
print(f"8. {LIMITED_EVIDENCE_AUDIT_OUTPUT}")
print(f"9. {HANDOFF_FILE}")

print()
print(
    "All Step 1, Part 4 validation checks passed."
)

FORECASTING PREPARATION — STEP 1, PART 4 COMPLETED

Products analysed:
Total products: 227
Products retained for forecasting: 227

Demand-pattern classification:
Smooth: 35
Intermittent: 126
Erratic: 16
Lumpy: 43
Single positive-demand day: 7

Diagnostic forecasting-treatment groups:
Regular-demand methods: 51
Intermittent-demand methods: 169
Limited positive-demand evidence: 7

Diagnostic audit cohorts:
More than 90% zero-demand dates: 65
Limited history or positive-demand evidence: 22

Demand preserved:
Total demand units represented: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/04_product_demand_behaviour_profile.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/04_demand_pattern_summary.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/04_history_length_summary.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/04_positive_demand_evidence_summary

In [29]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 5
# Cell 18: Define forecasting scope, minimum-history rules,
# and evaluation readiness
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load the completed Part 4 product profile
# ------------------------------------------------------------

PART_4_PROFILE_FILE = (
    FORECAST_PREPARATION_DIR
    / "04_product_demand_behaviour_profile.csv"
)

if "product_demand_profile_df" in globals():

    eligibility_source_df = (
        product_demand_profile_df.copy()
    )

elif PART_4_PROFILE_FILE.exists():

    eligibility_source_df = pd.read_csv(
        PART_4_PROFILE_FILE,
        low_memory=False
    )

else:

    raise FileNotFoundError(
        "The completed Part 4 product profile could not be found.\n"
        f"Expected file:\n{PART_4_PROFILE_FILE}"
    )


# ------------------------------------------------------------
# 2. Validate the source profile
# ------------------------------------------------------------

required_eligibility_columns = [
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "PanelFinalDate",
    "HistoryOperatingDays",
    "PositiveDemandDays",
    "ZeroDemandDays",
    "ZeroDemandRate",
    "TotalDemandUnits",
    "DemandPatternClass",
    "DiagnosticForecastTreatment",
    "RetainForForecasting"
]

missing_eligibility_columns = [
    column
    for column in required_eligibility_columns
    if column not in eligibility_source_df.columns
]

if missing_eligibility_columns:
    raise KeyError(
        "The following required eligibility columns are missing:\n"
        f"{missing_eligibility_columns}"
    )

eligibility_source_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    eligibility_source_df[
        "ProductFirstObservedDate"
    ],
    errors="coerce"
)

eligibility_source_df[
    "PanelFinalDate"
] = pd.to_datetime(
    eligibility_source_df[
        "PanelFinalDate"
    ],
    errors="coerce"
)

numeric_eligibility_columns = [
    "HistoryOperatingDays",
    "PositiveDemandDays",
    "ZeroDemandDays",
    "ZeroDemandRate",
    "TotalDemandUnits"
]

for column in numeric_eligibility_columns:

    eligibility_source_df[column] = pd.to_numeric(
        eligibility_source_df[column],
        errors="coerce"
    )

assert len(eligibility_source_df) == 227

assert eligibility_source_df[
    "CanonicalProductID"
].nunique() == 227

assert eligibility_source_df[
    required_eligibility_columns
].isna().sum().sum() == 0

assert eligibility_source_df[
    "TotalDemandUnits"
].sum() == 116_158


# ------------------------------------------------------------
# 3. Define the provisional evaluation policy
# ------------------------------------------------------------

EVALUATION_HORIZON_OPERATING_DAYS = 20

MINIMUM_TRAINING_OPERATING_DAYS = 60

ROLLING_ORIGIN_FOLDS = 3

MINIMUM_HISTORY_MULTI_FOLD = (
    MINIMUM_TRAINING_OPERATING_DAYS
    + (
        EVALUATION_HORIZON_OPERATING_DAYS
        * ROLLING_ORIGIN_FOLDS
    )
)

MINIMUM_HISTORY_SINGLE_HOLDOUT = (
    MINIMUM_TRAINING_OPERATING_DAYS
    + EVALUATION_HORIZON_OPERATING_DAYS
)

MINIMUM_HISTORY_LIMITED_HOLDOUT = 30

MINIMUM_POSITIVE_DAYS_MULTI_FOLD = 10

MINIMUM_POSITIVE_DAYS_SINGLE_HOLDOUT = 5

MINIMUM_POSITIVE_DAYS_LIMITED_HOLDOUT = 2


forecasting_policy_df = pd.DataFrame({
    "PolicyParameter": [
        "EvaluationHorizonOperatingDays",
        "MinimumTrainingOperatingDays",
        "RollingOriginFolds",
        "MinimumHistoryMultiFold",
        "MinimumHistorySingleHoldout",
        "MinimumHistoryLimitedHoldout",
        "MinimumPositiveDaysMultiFold",
        "MinimumPositiveDaysSingleHoldout",
        "MinimumPositiveDaysLimitedHoldout"
    ],
    "Value": [
        EVALUATION_HORIZON_OPERATING_DAYS,
        MINIMUM_TRAINING_OPERATING_DAYS,
        ROLLING_ORIGIN_FOLDS,
        MINIMUM_HISTORY_MULTI_FOLD,
        MINIMUM_HISTORY_SINGLE_HOLDOUT,
        MINIMUM_HISTORY_LIMITED_HOLDOUT,
        MINIMUM_POSITIVE_DAYS_MULTI_FOLD,
        MINIMUM_POSITIVE_DAYS_SINGLE_HOLDOUT,
        MINIMUM_POSITIVE_DAYS_LIMITED_HOLDOUT
    ],
    "Purpose": [
        "Number of future operating dates evaluated per holdout.",
        "Minimum training observations before a standard holdout.",
        "Number of expanding-window validation folds.",
        "History required for three full validation folds.",
        "History required for one standard validation holdout.",
        "Minimum history for a cautious limited holdout.",
        "Minimum positive-demand dates for multi-fold evaluation.",
        "Minimum positive-demand dates for one standard holdout.",
        "Minimum positive-demand dates for limited evaluation."
    ]
})

assert MINIMUM_HISTORY_MULTI_FOLD == 120
assert MINIMUM_HISTORY_SINGLE_HOLDOUT == 80


# ------------------------------------------------------------
# 4. Create the official product eligibility register
# ------------------------------------------------------------

forecasting_eligibility_df = (
    eligibility_source_df.copy()
)

# Every product remains inside the forecasting project.
forecasting_eligibility_df[
    "ForecastInProjectScope"
] = True

forecasting_eligibility_df[
    "ProductContinuationAssumption"
] = (
    "CONTINUING_PRODUCT"
)

forecasting_eligibility_df[
    "EvaluationCohort"
] = np.select(
    [
        (
            forecasting_eligibility_df[
                "HistoryOperatingDays"
            ] >= MINIMUM_HISTORY_MULTI_FOLD
        )
        &
        (
            forecasting_eligibility_df[
                "PositiveDemandDays"
            ] >= MINIMUM_POSITIVE_DAYS_MULTI_FOLD
        ),

        (
            forecasting_eligibility_df[
                "HistoryOperatingDays"
            ] >= MINIMUM_HISTORY_SINGLE_HOLDOUT
        )
        &
        (
            forecasting_eligibility_df[
                "PositiveDemandDays"
            ] >= MINIMUM_POSITIVE_DAYS_SINGLE_HOLDOUT
        ),

        (
            forecasting_eligibility_df[
                "HistoryOperatingDays"
            ] >= MINIMUM_HISTORY_LIMITED_HOLDOUT
        )
        &
        (
            forecasting_eligibility_df[
                "PositiveDemandDays"
            ] >= MINIMUM_POSITIVE_DAYS_LIMITED_HOLDOUT
        )
    ],
    [
        "MULTI_FOLD_BACKTEST_READY",
        "SINGLE_HOLDOUT_BACKTEST_READY",
        "LIMITED_HOLDOUT_ONLY"
    ],
    default="MINIMAL_EVIDENCE_FORECAST_ONLY"
)


# ------------------------------------------------------------
# 5. Add evaluation priority and eligibility flags
# ------------------------------------------------------------

cohort_priority_mapping = {
    "MULTI_FOLD_BACKTEST_READY": 1,
    "SINGLE_HOLDOUT_BACKTEST_READY": 2,
    "LIMITED_HOLDOUT_ONLY": 3,
    "MINIMAL_EVIDENCE_FORECAST_ONLY": 4
}

forecasting_eligibility_df[
    "EvaluationCohortPriority"
] = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ]
    .map(cohort_priority_mapping)
)

forecasting_eligibility_df[
    "StandardComparativeBacktestEligible"
] = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ].isin([
        "MULTI_FOLD_BACKTEST_READY",
        "SINGLE_HOLDOUT_BACKTEST_READY"
    ])
)

forecasting_eligibility_df[
    "AnyHistoricalHoldoutEligible"
] = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ] != "MINIMAL_EVIDENCE_FORECAST_ONLY"
)

forecasting_eligibility_df[
    "RollingOriginEligible"
] = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ] == "MULTI_FOLD_BACKTEST_READY"
)


# ------------------------------------------------------------
# 6. Define the evaluation action for each cohort
# ------------------------------------------------------------

evaluation_action_mapping = {
    "MULTI_FOLD_BACKTEST_READY":
        "THREE_FOLD_EXPANDING_WINDOW_BACKTEST",

    "SINGLE_HOLDOUT_BACKTEST_READY":
        "SINGLE_TIME_BASED_HOLDOUT_BACKTEST",

    "LIMITED_HOLDOUT_ONLY":
        "LIMITED_TIME_HOLDOUT_WITH_CAUTION",

    "MINIMAL_EVIDENCE_FORECAST_ONLY":
        "POOLED_MODEL_AND_CONSERVATIVE_BASELINE"
}

forecasting_eligibility_df[
    "EvaluationAction"
] = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ]
    .map(evaluation_action_mapping)
)


# ------------------------------------------------------------
# 7. Define provisional modelling-family recommendations
# ------------------------------------------------------------

forecasting_eligibility_df[
    "RecommendedMethodFamily"
] = np.select(
    [
        (
            forecasting_eligibility_df[
                "EvaluationCohort"
            ]
            == "MINIMAL_EVIDENCE_FORECAST_ONLY"
        ),

        (
            forecasting_eligibility_df[
                "DiagnosticForecastTreatment"
            ]
            == "INTERMITTENT_DEMAND_METHODS"
        ),

        (
            forecasting_eligibility_df[
                "DiagnosticForecastTreatment"
            ]
            == "REGULAR_DEMAND_METHODS"
        )
    ],
    [
        (
            "POOLED_GLOBAL_MODEL_AND_"
            "CONSERVATIVE_NAIVE_BASELINES"
        ),
        (
            "CROSTON_FAMILY_ZERO_AWARE_"
            "BASELINES_AND_MACHINE_LEARNING"
        ),
        (
            "SEASONAL_NAIVE_MOVING_AVERAGE_"
            "AND_MACHINE_LEARNING"
        )
    ],
    default=(
        "POOLED_GLOBAL_MODEL_AND_"
        "CONSERVATIVE_BASELINES"
    )
)


# ------------------------------------------------------------
# 8. Define metric policy
# ------------------------------------------------------------

forecasting_eligibility_df[
    "ZeroSafeMetricsRequired"
] = True

forecasting_eligibility_df[
    "MAPEPrimaryMetricAllowed"
] = False

forecasting_eligibility_df[
    "EvaluationMetricPolicy"
] = (
    "USE_MAE_RMSE_WAPE_AND_SCALED_ZERO_SAFE_METRICS"
)


# ------------------------------------------------------------
# 9. Add a readable reason for each cohort
# ------------------------------------------------------------

def build_evaluation_reason(row):

    if (
        row["EvaluationCohort"]
        == "MULTI_FOLD_BACKTEST_READY"
    ):
        return (
            "At least 120 operating days and at least "
            "10 positive-demand days."
        )

    if (
        row["EvaluationCohort"]
        == "SINGLE_HOLDOUT_BACKTEST_READY"
    ):
        return (
            "At least 80 operating days and at least "
            "5 positive-demand days, but not enough evidence "
            "for three full validation folds."
        )

    if (
        row["EvaluationCohort"]
        == "LIMITED_HOLDOUT_ONLY"
    ):
        return (
            "At least 30 operating days and at least "
            "2 positive-demand days, but insufficient evidence "
            "for a standard comparative backtest."
        )

    return (
        "Too little historical or positive-demand evidence "
        "for an independent holdout; product remains forecast "
        "using pooled/global information and conservative "
        "baselines."
    )


forecasting_eligibility_df[
    "EvaluationCohortReason"
] = forecasting_eligibility_df.apply(
    build_evaluation_reason,
    axis=1
)


# ------------------------------------------------------------
# 10. Validate the completed eligibility register
# ------------------------------------------------------------

assert len(forecasting_eligibility_df) == 227

assert forecasting_eligibility_df[
    "CanonicalProductID"
].nunique() == 227

assert forecasting_eligibility_df[
    "ForecastInProjectScope"
].all()

assert (
    forecasting_eligibility_df[
        "ProductContinuationAssumption"
    ]
    == "CONTINUING_PRODUCT"
).all()

assert forecasting_eligibility_df[
    "EvaluationCohort"
].isna().sum() == 0

assert forecasting_eligibility_df[
    "EvaluationCohortPriority"
].isna().sum() == 0

assert forecasting_eligibility_df[
    "EvaluationAction"
].isna().sum() == 0

assert forecasting_eligibility_df[
    "RecommendedMethodFamily"
].isna().sum() == 0

assert forecasting_eligibility_df[
    "EvaluationCohortReason"
].isna().sum() == 0

assert forecasting_eligibility_df[
    "TotalDemandUnits"
].sum() == 116_158


# ------------------------------------------------------------
# 11. Sort the eligibility register
# ------------------------------------------------------------

forecasting_eligibility_df = (
    forecasting_eligibility_df
    .sort_values(
        [
            "EvaluationCohortPriority",
            "DemandPatternClass",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Print Cell 18 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 5 — FORECASTING ELIGIBILITY REGISTER CREATED")
print("=" * 75)

print()
print("Policy:")
print(
    "Evaluation horizon:",
    EVALUATION_HORIZON_OPERATING_DAYS,
    "operating days"
)
print(
    "Minimum training history:",
    MINIMUM_TRAINING_OPERATING_DAYS,
    "operating days"
)
print(
    "Minimum multi-fold history:",
    MINIMUM_HISTORY_MULTI_FOLD,
    "operating days"
)
print(
    "Minimum single-holdout history:",
    MINIMUM_HISTORY_SINGLE_HOLDOUT,
    "operating days"
)

print()
print("Forecasting scope:")
print(
    "Products in project scope:",
    int(
        forecasting_eligibility_df[
            "ForecastInProjectScope"
        ].sum()
    )
)
print(
    "Products classified as discontinued:",
    0
)

print()
print("Evaluation cohorts:")
display(
    forecasting_eligibility_df[
        "EvaluationCohort"
    ]
    .value_counts()
    .rename_axis("EvaluationCohort")
    .reset_index(name="ProductCount")
)

print()
print("Cell 18 completed successfully.")

STEP 1, PART 5 — FORECASTING ELIGIBILITY REGISTER CREATED

Policy:
Evaluation horizon: 20 operating days
Minimum training history: 60 operating days
Minimum multi-fold history: 120 operating days
Minimum single-holdout history: 80 operating days

Forecasting scope:
Products in project scope: 227
Products classified as discontinued: 0

Evaluation cohorts:


,EvaluationCohort,ProductCount
0,MULTI_FOLD_BACKTEST_READY,127
1,SINGLE_HOLDOUT_BACKTEST_READY,77
2,LIMITED_HOLDOUT_ONLY,14
3,MINIMAL_EVIDENCE_FORECAST_ONLY,9



Cell 18 completed successfully.


In [30]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 5
# Cell 19: Create evaluation-cohort summaries and audit tables
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 18 objects exist
# ------------------------------------------------------------

required_cell_18_objects = [
    "forecasting_eligibility_df",
    "forecasting_policy_df"
]

missing_cell_18_objects = [
    object_name
    for object_name in required_cell_18_objects
    if object_name not in globals()
]

if missing_cell_18_objects:
    raise NameError(
        "The following Cell 18 objects are missing:\n"
        f"{missing_cell_18_objects}\n\n"
        "Run Cell 18 before running Cell 19."
    )


# ------------------------------------------------------------
# 2. Evaluation-cohort summary
# ------------------------------------------------------------

evaluation_cohort_summary_df = (
    forecasting_eligibility_df
    .groupby(
        [
            "EvaluationCohortPriority",
            "EvaluationCohort"
        ],
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        TotalDemandUnits=(
            "TotalDemandUnits",
            "sum"
        ),
        MedianHistoryOperatingDays=(
            "HistoryOperatingDays",
            "median"
        ),
        MedianPositiveDemandDays=(
            "PositiveDemandDays",
            "median"
        ),
        MedianZeroDemandRate=(
            "ZeroDemandRate",
            "median"
        )
    )
    .sort_values(
        "EvaluationCohortPriority"
    )
    .reset_index(drop=True)
)

evaluation_cohort_summary_df[
    "ProductPercentage"
] = (
    evaluation_cohort_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)

evaluation_cohort_summary_df[
    "DemandUnitPercentage"
] = (
    evaluation_cohort_summary_df[
        "TotalDemandUnits"
    ]
    / 116_158
    * 100
).round(2)

evaluation_cohort_summary_df[
    "MedianZeroDemandRate"
] = (
    evaluation_cohort_summary_df[
        "MedianZeroDemandRate"
    ]
    .round(4)
)


# ------------------------------------------------------------
# 3. Demand pattern by evaluation cohort
# ------------------------------------------------------------

evaluation_by_demand_pattern_df = pd.crosstab(
    forecasting_eligibility_df[
        "DemandPatternClass"
    ],
    forecasting_eligibility_df[
        "EvaluationCohort"
    ],
    margins=True,
    margins_name="TOTAL"
).reset_index()


# ------------------------------------------------------------
# 4. Diagnostic treatment by evaluation cohort
# ------------------------------------------------------------

evaluation_by_treatment_df = pd.crosstab(
    forecasting_eligibility_df[
        "DiagnosticForecastTreatment"
    ],
    forecasting_eligibility_df[
        "EvaluationCohort"
    ],
    margins=True,
    margins_name="TOTAL"
).reset_index()


# ------------------------------------------------------------
# 5. Recommended method-family summary
# ------------------------------------------------------------

recommended_method_summary_df = (
    forecasting_eligibility_df
    .groupby(
        "RecommendedMethodFamily",
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        TotalDemandUnits=(
            "TotalDemandUnits",
            "sum"
        )
    )
)

recommended_method_summary_df[
    "ProductPercentage"
] = (
    recommended_method_summary_df[
        "ProductCount"
    ]
    / 227
    * 100
).round(2)

recommended_method_summary_df[
    "DemandUnitPercentage"
] = (
    recommended_method_summary_df[
        "TotalDemandUnits"
    ]
    / 116_158
    * 100
).round(2)


# ------------------------------------------------------------
# 6. Create product-level audit cohorts
# ------------------------------------------------------------

standard_backtest_register_df = (
    forecasting_eligibility_df.loc[
        forecasting_eligibility_df[
            "StandardComparativeBacktestEligible"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

limited_holdout_audit_df = (
    forecasting_eligibility_df.loc[
        forecasting_eligibility_df[
            "EvaluationCohort"
        ]
        == "LIMITED_HOLDOUT_ONLY"
    ]
    .copy()
    .reset_index(drop=True)
)

minimal_evidence_forecast_audit_df = (
    forecasting_eligibility_df.loc[
        forecasting_eligibility_df[
            "EvaluationCohort"
        ]
        == "MINIMAL_EVIDENCE_FORECAST_ONLY"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Validate cohort partitioning
# ------------------------------------------------------------

evaluation_cohort_counts = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ]
    .value_counts()
)

assert evaluation_cohort_counts.sum() == 227

assert evaluation_cohort_summary_df[
    "ProductCount"
].sum() == 227

assert evaluation_cohort_summary_df[
    "TotalDemandUnits"
].sum() == 116_158

assert recommended_method_summary_df[
    "ProductCount"
].sum() == 227

assert recommended_method_summary_df[
    "TotalDemandUnits"
].sum() == 116_158

assert (
    len(standard_backtest_register_df)
    + len(limited_holdout_audit_df)
    + len(minimal_evidence_forecast_audit_df)
    == 227
)

assert set(
    standard_backtest_register_df[
        "CanonicalProductID"
    ]
).isdisjoint(
    set(
        limited_holdout_audit_df[
            "CanonicalProductID"
        ]
    )
)

assert set(
    standard_backtest_register_df[
        "CanonicalProductID"
    ]
).isdisjoint(
    set(
        minimal_evidence_forecast_audit_df[
            "CanonicalProductID"
        ]
    )
)

assert set(
    limited_holdout_audit_df[
        "CanonicalProductID"
    ]
).isdisjoint(
    set(
        minimal_evidence_forecast_audit_df[
            "CanonicalProductID"
        ]
    )
)

assert forecasting_eligibility_df[
    "ForecastInProjectScope"
].all()


# ------------------------------------------------------------
# 8. Print Cell 19 summaries
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 5 — EVALUATION COHORT SUMMARIES CREATED")
print("=" * 75)

print()
print("Evaluation-cohort summary:")
display(evaluation_cohort_summary_df)

print()
print("Demand pattern by evaluation cohort:")
display(evaluation_by_demand_pattern_df)

print()
print("Diagnostic treatment by evaluation cohort:")
display(evaluation_by_treatment_df)

print()
print("Recommended method-family summary:")
display(recommended_method_summary_df)

print()
print("Product-level cohort sizes:")
print(
    "Standard comparative backtest:",
    len(standard_backtest_register_df)
)
print(
    "Limited holdout only:",
    len(limited_holdout_audit_df)
)
print(
    "Minimal evidence forecast only:",
    len(minimal_evidence_forecast_audit_df)
)

print()
print(
    "Total products retained in forecasting scope:",
    int(
        forecasting_eligibility_df[
            "ForecastInProjectScope"
        ].sum()
    )
)

print()
print("Cell 19 completed successfully.")

STEP 1, PART 5 — EVALUATION COHORT SUMMARIES CREATED

Evaluation-cohort summary:


,EvaluationCohortPriority,EvaluationCohort,ProductCount,TotalDemandUnits,MedianHistoryOperatingDays,MedianPositiveDemandDays,MedianZeroDemandRate,ProductPercentage,DemandUnitPercentage
0,1,MULTI_FOLD_BACKTEST_READY,127,69256,245.0,70.0,0.7143,55.95,59.62
1,2,SINGLE_HOLDOUT_BACKTEST_READY,77,46068,110.0,58.0,0.3761,33.92,39.66
2,3,LIMITED_HOLDOUT_ONLY,14,769,159.5,3.0,0.9845,6.17,0.66
3,4,MINIMAL_EVIDENCE_FORECAST_ONLY,9,65,101.0,1.0,0.9901,3.96,0.06



Demand pattern by evaluation cohort:


EvaluationCohort,DemandPatternClass,LIMITED_HOLDOUT_ONLY,MINIMAL_EVIDENCE_FORECAST_ONLY,MULTI_FOLD_BACKTEST_READY,SINGLE_HOLDOUT_BACKTEST_READY,TOTAL
0,ERRATIC,0,0,10,6,16
1,INTERMITTENT,14,0,84,28,126
2,LUMPY,0,1,28,14,43
3,SINGLE_POSITIVE_DEMAND_DAY,0,7,0,0,7
4,SMOOTH,0,1,5,29,35
5,TOTAL,14,9,127,77,227



Diagnostic treatment by evaluation cohort:


EvaluationCohort,DiagnosticForecastTreatment,LIMITED_HOLDOUT_ONLY,MINIMAL_EVIDENCE_FORECAST_ONLY,MULTI_FOLD_BACKTEST_READY,SINGLE_HOLDOUT_BACKTEST_READY,TOTAL
0,INTERMITTENT_DEMAND_METHODS,14,1,112,42,169
1,LIMITED_POSITIVE_DEMAND_EVIDENCE,0,7,0,0,7
2,REGULAR_DEMAND_METHODS,0,1,15,35,51
3,TOTAL,14,9,127,77,227



Recommended method-family summary:


,RecommendedMethodFamily,ProductCount,TotalDemandUnits,ProductPercentage,DemandUnitPercentage
0,CROSTON_FAMILY_ZERO_AWARE_BASELINES_AND_MACHIN...,168,39015,74.01,33.59
1,POOLED_GLOBAL_MODEL_AND_CONSERVATIVE_NAIVE_BAS...,9,65,3.96,0.06
2,SEASONAL_NAIVE_MOVING_AVERAGE_AND_MACHINE_LEAR...,50,77078,22.03,66.36



Product-level cohort sizes:
Standard comparative backtest: 204
Limited holdout only: 14
Minimal evidence forecast only: 9

Total products retained in forecasting scope: 227

Cell 19 completed successfully.


In [31]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 5
# Cell 20: Save eligibility outputs and update project handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 5 objects exist
# ------------------------------------------------------------

required_part_5_objects = [
    "forecasting_eligibility_df",
    "forecasting_policy_df",
    "evaluation_cohort_summary_df",
    "evaluation_by_demand_pattern_df",
    "evaluation_by_treatment_df",
    "recommended_method_summary_df",
    "standard_backtest_register_df",
    "limited_holdout_audit_df",
    "minimal_evidence_forecast_audit_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_part_5_objects = [
    object_name
    for object_name in required_part_5_objects
    if object_name not in globals()
]

if missing_part_5_objects:
    raise NameError(
        "The following Part 5 objects are missing:\n"
        f"{missing_part_5_objects}\n\n"
        "Run Cells 18 and 19 before running Cell 20."
    )


# ------------------------------------------------------------
# 2. Define Part 5 output paths
# ------------------------------------------------------------

FORECASTING_ELIGIBILITY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_forecasting_eligibility_register.csv"
)

FORECASTING_POLICY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_forecasting_policy_parameters.csv"
)

EVALUATION_COHORT_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_evaluation_cohort_summary.csv"
)

EVALUATION_PATTERN_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_evaluation_cohort_by_demand_pattern.csv"
)

EVALUATION_TREATMENT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_evaluation_cohort_by_treatment.csv"
)

RECOMMENDED_METHOD_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_recommended_method_family_summary.csv"
)

STANDARD_BACKTEST_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_standard_backtest_product_register.csv"
)

LIMITED_HOLDOUT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_limited_holdout_product_audit.csv"
)

MINIMAL_EVIDENCE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "05_minimal_evidence_forecast_audit.csv"
)


# ------------------------------------------------------------
# 3. Save all Part 5 outputs
# ------------------------------------------------------------

forecasting_eligibility_df.to_csv(
    FORECASTING_ELIGIBILITY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

forecasting_policy_df.to_csv(
    FORECASTING_POLICY_OUTPUT,
    index=False
)

evaluation_cohort_summary_df.to_csv(
    EVALUATION_COHORT_SUMMARY_OUTPUT,
    index=False
)

evaluation_by_demand_pattern_df.to_csv(
    EVALUATION_PATTERN_OUTPUT,
    index=False
)

evaluation_by_treatment_df.to_csv(
    EVALUATION_TREATMENT_OUTPUT,
    index=False
)

recommended_method_summary_df.to_csv(
    RECOMMENDED_METHOD_OUTPUT,
    index=False
)

standard_backtest_register_df.to_csv(
    STANDARD_BACKTEST_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

limited_holdout_audit_df.to_csv(
    LIMITED_HOLDOUT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

minimal_evidence_forecast_audit_df.to_csv(
    MINIMAL_EVIDENCE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)


# ------------------------------------------------------------
# 4. Reload and validate saved outputs
# ------------------------------------------------------------

saved_eligibility = pd.read_csv(
    FORECASTING_ELIGIBILITY_OUTPUT,
    low_memory=False
)

saved_policy = pd.read_csv(
    FORECASTING_POLICY_OUTPUT,
    low_memory=False
)

saved_cohort_summary = pd.read_csv(
    EVALUATION_COHORT_SUMMARY_OUTPUT,
    low_memory=False
)

saved_pattern_cross_tab = pd.read_csv(
    EVALUATION_PATTERN_OUTPUT,
    low_memory=False
)

saved_treatment_cross_tab = pd.read_csv(
    EVALUATION_TREATMENT_OUTPUT,
    low_memory=False
)

saved_method_summary = pd.read_csv(
    RECOMMENDED_METHOD_OUTPUT,
    low_memory=False
)

saved_standard_backtest = pd.read_csv(
    STANDARD_BACKTEST_OUTPUT,
    low_memory=False
)

saved_limited_holdout = pd.read_csv(
    LIMITED_HOLDOUT_OUTPUT,
    low_memory=False
)

saved_minimal_evidence = pd.read_csv(
    MINIMAL_EVIDENCE_OUTPUT,
    low_memory=False
)

assert len(saved_eligibility) == 227

assert saved_eligibility[
    "CanonicalProductID"
].nunique() == 227

assert saved_eligibility[
    "TotalDemandUnits"
].sum() == 116_158

assert len(saved_policy) == 9

assert saved_cohort_summary[
    "ProductCount"
].sum() == 227

assert saved_cohort_summary[
    "TotalDemandUnits"
].sum() == 116_158

assert saved_method_summary[
    "ProductCount"
].sum() == 227

assert (
    len(saved_standard_backtest)
    + len(saved_limited_holdout)
    + len(saved_minimal_evidence)
    == 227
)

saved_scope_values = (
    saved_eligibility[
        "ForecastInProjectScope"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
)

assert saved_scope_values.isna().sum() == 0

assert int(saved_scope_values.sum()) == 227


# ------------------------------------------------------------
# 5. Extract verified cohort counts
# ------------------------------------------------------------

cohort_count_lookup = (
    forecasting_eligibility_df[
        "EvaluationCohort"
    ]
    .value_counts()
    .to_dict()
)

multi_fold_product_count = int(
    cohort_count_lookup.get(
        "MULTI_FOLD_BACKTEST_READY",
        0
    )
)

single_holdout_product_count = int(
    cohort_count_lookup.get(
        "SINGLE_HOLDOUT_BACKTEST_READY",
        0
    )
)

limited_holdout_product_count = int(
    cohort_count_lookup.get(
        "LIMITED_HOLDOUT_ONLY",
        0
    )
)

minimal_evidence_product_count = int(
    cohort_count_lookup.get(
        "MINIMAL_EVIDENCE_FORECAST_ONLY",
        0
    )
)

assert (
    multi_fold_product_count
    + single_holdout_product_count
    + limited_holdout_product_count
    + minimal_evidence_product_count
    == 227
)


# ------------------------------------------------------------
# 6. Build Markdown summary rows
# ------------------------------------------------------------

cohort_markdown_rows = "\n".join(
    (
        f"- `{row.EvaluationCohort}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%), "
        f"{int(row.TotalDemandUnits):,} demand units "
        f"({row.DemandUnitPercentage:.2f}%)"
    )
    for row in evaluation_cohort_summary_df.itertuples()
)

method_markdown_rows = "\n".join(
    (
        f"- `{row.RecommendedMethodFamily}`: "
        f"{int(row.ProductCount)} products "
        f"({row.ProductPercentage:.2f}%)"
    )
    for row in recommended_method_summary_df.itertuples()
)


# ------------------------------------------------------------
# 7. Update the Markdown handoff
# ------------------------------------------------------------

part_5_summary = f"""
**Status:** Completed and validated

### Forecasting scope

All 227 canonical products remain in the forecasting project and
are treated as continuing products.

The evaluation cohorts control how much independent historical
testing is possible. They do not remove products from forecasting.

### Evaluation policy

- Evaluation horizon: {EVALUATION_HORIZON_OPERATING_DAYS} operating days
- Minimum training history: {MINIMUM_TRAINING_OPERATING_DAYS} operating days
- Rolling-origin folds: {ROLLING_ORIGIN_FOLDS}
- Minimum multi-fold history: {MINIMUM_HISTORY_MULTI_FOLD} operating days
- Minimum single-holdout history: {MINIMUM_HISTORY_SINGLE_HOLDOUT} operating days
- Minimum limited-holdout history: {MINIMUM_HISTORY_LIMITED_HOLDOUT} operating days

### Evaluation cohorts

- Multi-fold backtest ready: {multi_fold_product_count}
- Single-holdout backtest ready: {single_holdout_product_count}
- Limited-holdout only: {limited_holdout_product_count}
- Minimal-evidence forecast only: {minimal_evidence_product_count}

{cohort_markdown_rows}

### Evaluation interpretation

- Multi-fold products receive three expanding-window backtests.
- Single-holdout products receive one time-based holdout.
- Limited-holdout products receive cautious evaluation with fewer observations.
- Minimal-evidence products remain forecast using pooled/global models and conservative baselines.

### Recommended method families

{method_markdown_rows}

### Metric policy

All products require zero-safe evaluation metrics because the
dataset contains substantial zero demand.

`MAPE` will not be used as the primary metric. Comparative
evaluation will use metrics such as MAE, RMSE, WAPE and appropriate
scaled zero-safe metrics.

### Saved Part 5 outputs

- `05_forecasting_eligibility_register.csv`
- `05_forecasting_policy_parameters.csv`
- `05_evaluation_cohort_summary.csv`
- `05_evaluation_cohort_by_demand_pattern.csv`
- `05_evaluation_cohort_by_treatment.csv`
- `05_recommended_method_family_summary.csv`
- `05_standard_backtest_product_register.csv`
- `05_limited_holdout_product_audit.csv`
- `05_minimal_evidence_forecast_audit.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_5",
    section_title=(
        "Forecasting Preparation — Step 1, Part 5"
    ),
    section_body=part_5_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 8. Print final Part 5 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 1, PART 5 COMPLETED"
)
print("=" * 75)

print()
print("Forecasting scope:")
print("Products retained: 227")
print("Products removed: 0")
print("Products labelled discontinued: 0")

print()
print("Evaluation cohorts:")
print(
    "Multi-fold backtest ready:",
    multi_fold_product_count
)
print(
    "Single-holdout backtest ready:",
    single_holdout_product_count
)
print(
    "Limited-holdout only:",
    limited_holdout_product_count
)
print(
    "Minimal-evidence forecast only:",
    minimal_evidence_product_count
)

print()
print("Evaluation policy:")
print(
    "Evaluation horizon:",
    EVALUATION_HORIZON_OPERATING_DAYS,
    "operating days"
)
print(
    "Minimum training history:",
    MINIMUM_TRAINING_OPERATING_DAYS,
    "operating days"
)
print(
    "MAPE used as primary metric:",
    False
)

print()
print("Demand preserved:")
print(
    "Total demand units represented:",
    f"{saved_eligibility['TotalDemandUnits'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {FORECASTING_ELIGIBILITY_OUTPUT}")
print(f"2. {FORECASTING_POLICY_OUTPUT}")
print(f"3. {EVALUATION_COHORT_SUMMARY_OUTPUT}")
print(f"4. {EVALUATION_PATTERN_OUTPUT}")
print(f"5. {EVALUATION_TREATMENT_OUTPUT}")
print(f"6. {RECOMMENDED_METHOD_OUTPUT}")
print(f"7. {STANDARD_BACKTEST_OUTPUT}")
print(f"8. {LIMITED_HOLDOUT_OUTPUT}")
print(f"9. {MINIMAL_EVIDENCE_OUTPUT}")
print(f"10. {HANDOFF_FILE}")

print()
print(
    "All Step 1, Part 5 validation checks passed."
)

FORECASTING PREPARATION — STEP 1, PART 5 COMPLETED

Forecasting scope:
Products retained: 227
Products removed: 0
Products labelled discontinued: 0

Evaluation cohorts:
Multi-fold backtest ready: 127
Single-holdout backtest ready: 77
Limited-holdout only: 14
Minimal-evidence forecast only: 9

Evaluation policy:
Evaluation horizon: 20 operating days
Minimum training history: 60 operating days
MAPE used as primary metric: False

Demand preserved:
Total demand units represented: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/05_forecasting_eligibility_register.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/05_forecasting_policy_parameters.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/05_evaluation_cohort_summary.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/05_evaluation_cohort_by_demand_pattern.csv
5. /Users/ryansmac/Desktop/Meng Project/eden

In [32]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 6
# Cell 21: Build the leakage-safe Step 1 modelling base
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the earlier forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define official input files
# ------------------------------------------------------------

CORRECTED_TEMPORAL_PANEL_FILE = (
    FORECAST_PREPARATION_DIR
    / "03_temporal_panel_validated.csv"
)

COLUMN_ROLE_AUDIT_FILE = (
    FORECAST_PREPARATION_DIR
    / "02_column_role_audit.csv"
)

PRODUCT_DEMAND_PROFILE_FILE = (
    FORECAST_PREPARATION_DIR
    / "04_product_demand_behaviour_profile.csv"
)

FORECASTING_ELIGIBILITY_FILE = (
    FORECAST_PREPARATION_DIR
    / "05_forecasting_eligibility_register.csv"
)

required_input_files = [
    CORRECTED_TEMPORAL_PANEL_FILE,
    COLUMN_ROLE_AUDIT_FILE,
    PRODUCT_DEMAND_PROFILE_FILE,
    FORECASTING_ELIGIBILITY_FILE
]

missing_input_files = [
    file_path
    for file_path in required_input_files
    if not file_path.exists()
]

if missing_input_files:
    raise FileNotFoundError(
        "The following required input files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load the official inputs
# ------------------------------------------------------------

step1_source_panel_df = pd.read_csv(
    CORRECTED_TEMPORAL_PANEL_FILE,
    low_memory=False
)

step1_column_role_df = pd.read_csv(
    COLUMN_ROLE_AUDIT_FILE,
    low_memory=False
)

step1_product_profile_df = pd.read_csv(
    PRODUCT_DEMAND_PROFILE_FILE,
    low_memory=False
)

step1_eligibility_register_df = pd.read_csv(
    FORECASTING_ELIGIBILITY_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Parse and validate important dates
# ------------------------------------------------------------

step1_source_panel_df["Date"] = pd.to_datetime(
    step1_source_panel_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

step1_source_panel_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    step1_source_panel_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

assert step1_source_panel_df["Date"].isna().sum() == 0

assert step1_source_panel_df[
    "ProductFirstObservedDate"
].isna().sum() == 0


# ------------------------------------------------------------
# 5. Validate the corrected source panel
# ------------------------------------------------------------

assert step1_source_panel_df.shape == (
    43_774,
    38
), (
    "The corrected temporal panel dimensions are unexpected.\n"
    f"Found: {step1_source_panel_df.shape}"
)

assert step1_source_panel_df[
    "CanonicalProductID"
].nunique() == 227

assert step1_source_panel_df[
    "Date"
].nunique() == 245

assert step1_source_panel_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step1_source_panel_df[
    "TotalDemand"
].sum() == 116_158

assert len(step1_column_role_df) == 38

assert step1_column_role_df[
    "Column"
].nunique() == 38

assert len(step1_product_profile_df) == 227

assert step1_product_profile_df[
    "CanonicalProductID"
].nunique() == 227

assert len(step1_eligibility_register_df) == 227

assert step1_eligibility_register_df[
    "CanonicalProductID"
].nunique() == 227


# ------------------------------------------------------------
# 6. Define which original columns enter the Step 1 base
# ------------------------------------------------------------

included_model_use_values = {
    "RETAIN",
    "SUPPORT_ONLY",
    "CANDIDATE",
    "CANDIDATE_CONDITIONAL",
    "TARGET"
}

included_column_set = set(
    step1_column_role_df.loc[
        step1_column_role_df[
            "ModelUse"
        ].isin(included_model_use_values),
        "Column"
    ]
)

# Preserve the original dataset column order.
step1_modelling_columns = [
    column
    for column in step1_source_panel_df.columns
    if column in included_column_set
]

step1_excluded_columns = [
    column
    for column in step1_source_panel_df.columns
    if column not in included_column_set
]


# ------------------------------------------------------------
# 7. Confirm the expected excluded columns
# ------------------------------------------------------------

expected_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

expected_high_cardinality_exclusions = {
    "SourcePLUCodes",
    "SourcePLUNames"
}

expected_redundant_exclusions = {
    "MonthName",
    "DayOfWeek"
}

expected_version_exclusions = {
    "DailyPanelVersion",
    "ProductMetadataVersion"
}

expected_all_exclusions = (
    expected_leakage_columns
    | expected_high_cardinality_exclusions
    | expected_redundant_exclusions
    | expected_version_exclusions
)

assert set(step1_excluded_columns) == expected_all_exclusions, (
    "The Step 1 excluded-column list differs from the "
    "approved column-role register.\n\n"
    f"Expected exclusions:\n{sorted(expected_all_exclusions)}\n\n"
    f"Actual exclusions:\n{sorted(step1_excluded_columns)}"
)

assert expected_leakage_columns.isdisjoint(
    step1_modelling_columns
)


# ------------------------------------------------------------
# 8. Create the leakage-safe modelling base
# ------------------------------------------------------------

step1_modelling_base_df = (
    step1_source_panel_df[
        step1_modelling_columns
    ]
    .copy()
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Validate the modelling base structure
# ------------------------------------------------------------

required_step1_base_columns = {
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "TotalDemand"
}

assert required_step1_base_columns.issubset(
    step1_modelling_base_df.columns
)

assert step1_modelling_base_df.shape == (
    43_774,
    27
), (
    "Expected a 43,774-row, 27-column modelling base.\n"
    f"Found: {step1_modelling_base_df.shape}"
)

assert step1_modelling_base_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step1_modelling_base_df[
    "CanonicalProductID"
].nunique() == 227

assert step1_modelling_base_df[
    "Date"
].nunique() == 245

assert step1_modelling_base_df[
    "TotalDemand"
].isna().sum() == 0

assert (
    step1_modelling_base_df[
        "TotalDemand"
    ] < 0
).sum() == 0

assert step1_modelling_base_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 10. Count approved column types
# ------------------------------------------------------------

direct_candidate_count = int(
    step1_column_role_df[
        "ModelUse"
    ].eq("CANDIDATE").sum()
)

conditional_candidate_count = int(
    step1_column_role_df[
        "ModelUse"
    ].eq("CANDIDATE_CONDITIONAL").sum()
)

key_column_count = int(
    step1_column_role_df[
        "ModelUse"
    ].eq("RETAIN").sum()
)

support_column_count = int(
    step1_column_role_df[
        "ModelUse"
    ].eq("SUPPORT_ONLY").sum()
)

target_column_count = int(
    step1_column_role_df[
        "ModelUse"
    ].eq("TARGET").sum()
)

assert direct_candidate_count == 15
assert conditional_candidate_count == 7
assert key_column_count == 2
assert support_column_count == 2
assert target_column_count == 1

assert (
    direct_candidate_count
    + conditional_candidate_count
    + key_column_count
    + support_column_count
    + target_column_count
    == 27
)


# ------------------------------------------------------------
# 11. Print Cell 21 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 6 — LEAKAGE-SAFE MODELLING BASE CREATED")
print("=" * 75)

print()
print("Corrected source panel:")
print(f"Rows: {len(step1_source_panel_df):,}")
print(f"Columns: {step1_source_panel_df.shape[1]}")

print()
print("Step 1 modelling base:")
print(f"Rows: {len(step1_modelling_base_df):,}")
print(f"Columns: {step1_modelling_base_df.shape[1]}")
print(
    "Canonical products:",
    step1_modelling_base_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step1_modelling_base_df[
        "Date"
    ].nunique()
)
print(
    "Total demand units:",
    f"{step1_modelling_base_df['TotalDemand'].sum():,}"
)

print()
print("Approved column composition:")
print(
    "Direct candidate features:",
    direct_candidate_count
)
print(
    "Conditional candidate features:",
    conditional_candidate_count
)
print("Key columns:", key_column_count)
print("Support-only columns:", support_column_count)
print("Target columns:", target_column_count)

print()
print("Excluded columns:")
for column in step1_excluded_columns:
    print(f"- {column}")

print()
print("Cell 21 completed successfully.")

STEP 1, PART 6 — LEAKAGE-SAFE MODELLING BASE CREATED

Corrected source panel:
Rows: 43,774
Columns: 38

Step 1 modelling base:
Rows: 43,774
Columns: 27
Canonical products: 227
Operating dates: 245
Total demand units: 116,158

Approved column composition:
Direct candidate features: 15
Conditional candidate features: 7
Key columns: 2
Support-only columns: 2
Target columns: 1

Excluded columns:
- SourcePLUCodes
- SourcePLUNames
- MonthName
- DayOfWeek
- NormalDemand
- BulkDemand
- IsObservedProductDate
- IsZeroDemandRow
- DemandRecordSource
- DailyPanelVersion
- ProductMetadataVersion

Cell 21 completed successfully.


In [33]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 6
# Cell 22: Create the feature contract and product-level
# modelling strategy register
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 21 objects exist
# ------------------------------------------------------------

required_cell_21_objects = [
    "step1_modelling_base_df",
    "step1_column_role_df",
    "step1_product_profile_df",
    "step1_eligibility_register_df",
    "step1_excluded_columns"
]

missing_cell_21_objects = [
    object_name
    for object_name in required_cell_21_objects
    if object_name not in globals()
]

if missing_cell_21_objects:
    raise NameError(
        "The following Cell 21 objects are missing:\n"
        f"{missing_cell_21_objects}\n\n"
        "Run Cell 21 before running Cell 22."
    )


# ------------------------------------------------------------
# 2. Build the Step 1 feature contract
# ------------------------------------------------------------

step1_feature_contract_df = (
    step1_column_role_df.loc[
        step1_column_role_df[
            "Column"
        ].isin(
            step1_modelling_base_df.columns
        )
    ]
    .copy()
)

# Preserve modelling-base column order.
column_order_lookup = {
    column: position
    for position, column in enumerate(
        step1_modelling_base_df.columns
    )
}

step1_feature_contract_df[
    "ColumnOrder"
] = (
    step1_feature_contract_df[
        "Column"
    ]
    .map(column_order_lookup)
)

step1_feature_contract_df = (
    step1_feature_contract_df
    .sort_values("ColumnOrder")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Add data type and missingness information
# ------------------------------------------------------------

data_type_lookup = {
    column: str(
        step1_modelling_base_df[
            column
        ].dtype
    )
    for column in step1_modelling_base_df.columns
}

missing_count_lookup = {
    column: int(
        step1_modelling_base_df[
            column
        ].isna().sum()
    )
    for column in step1_modelling_base_df.columns
}

missing_percentage_lookup = {
    column: round(
        (
            step1_modelling_base_df[
                column
            ].isna().sum()
            / len(step1_modelling_base_df)
            * 100
        ),
        2
    )
    for column in step1_modelling_base_df.columns
}

unique_value_lookup = {
    column: int(
        step1_modelling_base_df[
            column
        ].nunique(dropna=True)
    )
    for column in step1_modelling_base_df.columns
}

step1_feature_contract_df[
    "DataType"
] = (
    step1_feature_contract_df[
        "Column"
    ]
    .map(data_type_lookup)
)

step1_feature_contract_df[
    "MissingCount"
] = (
    step1_feature_contract_df[
        "Column"
    ]
    .map(missing_count_lookup)
)

step1_feature_contract_df[
    "MissingPercentage"
] = (
    step1_feature_contract_df[
        "Column"
    ]
    .map(missing_percentage_lookup)
)

step1_feature_contract_df[
    "UniqueNonNullValues"
] = (
    step1_feature_contract_df[
        "Column"
    ]
    .map(unique_value_lookup)
)


# ------------------------------------------------------------
# 4. Define how each included column may be used
# ------------------------------------------------------------

step1_feature_contract_df[
    "DirectModelInputAllowed"
] = (
    step1_feature_contract_df[
        "ModelUse"
    ].isin([
        "CANDIDATE",
        "CANDIDATE_CONDITIONAL"
    ])
)

step1_feature_contract_df[
    "UsageStage"
] = (
    step1_feature_contract_df[
        "ModelUse"
    ]
    .map({
        "RETAIN":
            "IDENTIFIER_OR_TIME_KEY",

        "SUPPORT_ONLY":
            "SUPPORT_AND_AUDIT_ONLY",

        "CANDIDATE":
            "DIRECT_FEATURE_CANDIDATE",

        "CANDIDATE_CONDITIONAL":
            "CONDITIONAL_FEATURE_EXPERIMENT",

        "TARGET":
            "FORECAST_TARGET"
    })
)

step1_feature_contract_df[
    "PredictionTimeAvailability"
] = np.where(
    step1_feature_contract_df[
        "ModelUse"
    ].eq("TARGET"),
    "UNKNOWN_UNTIL_OUTCOME",
    "AVAILABLE_OR_DERIVABLE_BEFORE_FORECAST"
)

categorical_candidate_mask = (
    step1_feature_contract_df[
        "DirectModelInputAllowed"
    ]
    &
    step1_feature_contract_df[
        "DataType"
    ].isin([
        "object",
        "string"
    ])
)

step1_feature_contract_df[
    "RequiresEncodingBeforeModel"
] = (
    categorical_candidate_mask
)

step1_feature_contract_df[
    "MissingValueHandlingDecision"
] = np.select(
    [
        (
            step1_feature_contract_df[
                "ModelUse"
            ]
            == "CANDIDATE_CONDITIONAL"
        )
        &
        (
            step1_feature_contract_df[
                "MissingCount"
            ] > 0
        ),

        (
            step1_feature_contract_df[
                "MissingCount"
            ] == 0
        )
    ],
    [
        (
            "RETAIN_MISSING_AS_NOT_APPLICABLE_"
            "UNTIL_FEATURE_EXPERIMENT"
        ),
        "NO_MISSING_VALUE_ACTION_REQUIRED"
    ],
    default="REVIEW_BEFORE_FEATURE_ENGINEERING"
)

step1_feature_contract_df[
    "LeakageStatus"
] = "NO_CONFIRMED_SAME_DAY_TARGET_LEAKAGE"


# ------------------------------------------------------------
# 5. Validate the completed feature contract
# ------------------------------------------------------------

assert len(step1_feature_contract_df) == 27

assert step1_feature_contract_df[
    "Column"
].nunique() == 27

assert step1_feature_contract_df[
    "DataType"
].isna().sum() == 0

assert step1_feature_contract_df[
    "UsageStage"
].isna().sum() == 0

assert step1_feature_contract_df[
    "PredictionTimeAvailability"
].isna().sum() == 0

assert int(
    step1_feature_contract_df[
        "DirectModelInputAllowed"
    ].sum()
) == 22

assert (
    step1_feature_contract_df.loc[
        step1_feature_contract_df[
            "ModelUse"
        ].eq("TARGET"),
        "Column"
    ].tolist()
    == ["TotalDemand"]
)


# ------------------------------------------------------------
# 6. Select product-profile fields
# ------------------------------------------------------------

product_profile_strategy_columns = [
    "CanonicalProductID",
    "CanonicalProductName",
    "HistoryOperatingDays",
    "PositiveDemandDays",
    "ZeroDemandDays",
    "PositiveDemandRate",
    "ZeroDemandRate",
    "TotalDemandUnits",
    "AverageDemandInterval_ADI",
    "CVSquaredPositiveDemand",
    "LongestZeroDemandRun",
    "TrailingZeroDemandRun",
    "OperatingDaysSinceLastPositiveDemand",
    "DemandPatternClass",
    "DiagnosticForecastTreatment",
    "RetainForForecasting"
]

missing_profile_strategy_columns = [
    column
    for column in product_profile_strategy_columns
    if column not in step1_product_profile_df.columns
]

if missing_profile_strategy_columns:
    raise KeyError(
        "The following product-profile columns are missing:\n"
        f"{missing_profile_strategy_columns}"
    )

product_strategy_base_df = (
    step1_product_profile_df[
        product_profile_strategy_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 7. Select eligibility and evaluation fields
# ------------------------------------------------------------

eligibility_strategy_columns = [
    "CanonicalProductID",
    "ForecastInProjectScope",
    "ProductContinuationAssumption",
    "EvaluationCohort",
    "EvaluationCohortPriority",
    "StandardComparativeBacktestEligible",
    "AnyHistoricalHoldoutEligible",
    "RollingOriginEligible",
    "EvaluationAction",
    "RecommendedMethodFamily",
    "ZeroSafeMetricsRequired",
    "MAPEPrimaryMetricAllowed",
    "EvaluationMetricPolicy",
    "EvaluationCohortReason"
]

missing_eligibility_strategy_columns = [
    column
    for column in eligibility_strategy_columns
    if column not in step1_eligibility_register_df.columns
]

if missing_eligibility_strategy_columns:
    raise KeyError(
        "The following eligibility columns are missing:\n"
        f"{missing_eligibility_strategy_columns}"
    )


# ------------------------------------------------------------
# 8. Create the product modelling strategy register
# ------------------------------------------------------------

product_modelling_strategy_df = (
    product_strategy_base_df
    .merge(
        step1_eligibility_register_df[
            eligibility_strategy_columns
        ],
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        [
            "EvaluationCohortPriority",
            "DemandPatternClass",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Validate the product strategy register
# ------------------------------------------------------------

assert len(product_modelling_strategy_df) == 227

assert product_modelling_strategy_df[
    "CanonicalProductID"
].nunique() == 227

assert product_modelling_strategy_df[
    "ForecastInProjectScope"
].astype(bool).all()

assert product_modelling_strategy_df[
    "RetainForForecasting"
].astype(bool).all()

assert (
    product_modelling_strategy_df[
        "ProductContinuationAssumption"
    ]
    == "CONTINUING_PRODUCT"
).all()

assert product_modelling_strategy_df[
    "TotalDemandUnits"
].sum() == 116_158

strategy_cohort_counts = (
    product_modelling_strategy_df[
        "EvaluationCohort"
    ]
    .value_counts()
    .to_dict()
)

assert strategy_cohort_counts.get(
    "MULTI_FOLD_BACKTEST_READY",
    0
) == 127

assert strategy_cohort_counts.get(
    "SINGLE_HOLDOUT_BACKTEST_READY",
    0
) == 77

assert strategy_cohort_counts.get(
    "LIMITED_HOLDOUT_ONLY",
    0
) == 14

assert strategy_cohort_counts.get(
    "MINIMAL_EVIDENCE_FORECAST_ONLY",
    0
) == 9


# ------------------------------------------------------------
# 10. Create the Step 1 validation summary
# ------------------------------------------------------------

step1_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "SourcePanelRows",
        "SourcePanelColumns",
        "Step1ModellingBaseRows",
        "Step1ModellingBaseColumns",
        "CanonicalProducts",
        "OperatingDates",
        "DuplicateProductDateRows",
        "TotalDemandUnits",
        "DirectCandidateFeatures",
        "ConditionalCandidateFeatures",
        "DirectModelInputCandidates",
        "LeakageColumnsIncluded",
        "ExcludedColumns",
        "ProductStrategyRegisterRows",
        "ProductsRetained",
        "ProductsRemoved",
        "ProductsLabelledDiscontinued"
    ],
    "Value": [
        len(step1_source_panel_df),
        step1_source_panel_df.shape[1],
        len(step1_modelling_base_df),
        step1_modelling_base_df.shape[1],
        step1_modelling_base_df[
            "CanonicalProductID"
        ].nunique(),
        step1_modelling_base_df[
            "Date"
        ].nunique(),
        int(
            step1_modelling_base_df[
                [
                    "Date",
                    "CanonicalProductID"
                ]
            ].duplicated().sum()
        ),
        int(
            step1_modelling_base_df[
                "TotalDemand"
            ].sum()
        ),
        direct_candidate_count,
        conditional_candidate_count,
        int(
            step1_feature_contract_df[
                "DirectModelInputAllowed"
            ].sum()
        ),
        len(
            expected_leakage_columns.intersection(
                step1_modelling_base_df.columns
            )
        ),
        len(step1_excluded_columns),
        len(product_modelling_strategy_df),
        int(
            product_modelling_strategy_df[
                "ForecastInProjectScope"
            ].astype(bool).sum()
        ),
        0,
        0
    ]
})


# ------------------------------------------------------------
# 11. Final Cell 22 validations
# ------------------------------------------------------------

summary_lookup = dict(
    zip(
        step1_validation_summary_df[
            "ValidationMetric"
        ],
        step1_validation_summary_df[
            "Value"
        ]
    )
)

assert summary_lookup[
    "Step1ModellingBaseRows"
] == 43_774

assert summary_lookup[
    "Step1ModellingBaseColumns"
] == 27

assert summary_lookup[
    "CanonicalProducts"
] == 227

assert summary_lookup[
    "OperatingDates"
] == 245

assert summary_lookup[
    "DuplicateProductDateRows"
] == 0

assert summary_lookup[
    "TotalDemandUnits"
] == 116_158

assert summary_lookup[
    "DirectModelInputCandidates"
] == 22

assert summary_lookup[
    "LeakageColumnsIncluded"
] == 0


# ------------------------------------------------------------
# 12. Print Cell 22 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1, PART 6 — DATA CONTRACT AND STRATEGY CREATED")
print("=" * 75)

print()
print("Feature contract:")
print(
    "Columns documented:",
    len(step1_feature_contract_df)
)
print(
    "Direct model-input candidates:",
    int(
        step1_feature_contract_df[
            "DirectModelInputAllowed"
        ].sum()
    )
)
print(
    "Same-day leakage columns included:",
    summary_lookup[
        "LeakageColumnsIncluded"
    ]
)

print()
print("Product modelling strategy:")
print(
    "Products documented:",
    len(product_modelling_strategy_df)
)
print(
    "Products retained:",
    int(
        product_modelling_strategy_df[
            "ForecastInProjectScope"
        ].astype(bool).sum()
    )
)

print()
print("Evaluation cohorts:")
display(
    product_modelling_strategy_df[
        "EvaluationCohort"
    ]
    .value_counts()
    .rename_axis("EvaluationCohort")
    .reset_index(name="ProductCount")
)

print()
print("Step 1 validation summary:")
display(step1_validation_summary_df)

print()
print("Cell 22 completed successfully.")

STEP 1, PART 6 — DATA CONTRACT AND STRATEGY CREATED

Feature contract:
Columns documented: 27
Direct model-input candidates: 22
Same-day leakage columns included: 0

Product modelling strategy:
Products documented: 227
Products retained: 227

Evaluation cohorts:


,EvaluationCohort,ProductCount
0,MULTI_FOLD_BACKTEST_READY,127
1,SINGLE_HOLDOUT_BACKTEST_READY,77
2,LIMITED_HOLDOUT_ONLY,14
3,MINIMAL_EVIDENCE_FORECAST_ONLY,9



Step 1 validation summary:


,ValidationMetric,Value
0,SourcePanelRows,43774
1,SourcePanelColumns,38
2,Step1ModellingBaseRows,43774
3,Step1ModellingBaseColumns,27
4,CanonicalProducts,227
5,OperatingDates,245
6,DuplicateProductDateRows,0
7,TotalDemandUnits,116158
8,DirectCandidateFeatures,15
9,ConditionalCandidateFeatures,7



Cell 22 completed successfully.


In [34]:
# ============================================================
# FORECASTING PREPARATION
# STEP 1 — PART 6
# Cell 23: Save final Step 1 outputs and update the handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 6 objects exist
# ------------------------------------------------------------

required_part_6_objects = [
    "step1_modelling_base_df",
    "step1_feature_contract_df",
    "product_modelling_strategy_df",
    "step1_validation_summary_df",
    "FORECAST_PREPARATION_DIR",
    "HANDOFF_FILE",
    "upsert_markdown_section"
]

missing_part_6_objects = [
    object_name
    for object_name in required_part_6_objects
    if object_name not in globals()
]

if missing_part_6_objects:
    raise NameError(
        "The following Part 6 objects are missing:\n"
        f"{missing_part_6_objects}\n\n"
        "Run Cells 21 and 22 before running Cell 23."
    )


# ------------------------------------------------------------
# 2. Define Part 6 output paths
# ------------------------------------------------------------

STEP1_MODELLING_BASE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "06_step1_leakage_safe_modelling_base.csv"
)

STEP1_FEATURE_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "06_step1_feature_contract.csv"
)

PRODUCT_MODELLING_STRATEGY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "06_product_modelling_strategy_register.csv"
)

STEP1_VALIDATION_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "06_step1_validation_summary.csv"
)


# ------------------------------------------------------------
# 3. Save all Part 6 outputs
# ------------------------------------------------------------

step1_modelling_base_df.to_csv(
    STEP1_MODELLING_BASE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step1_feature_contract_df.to_csv(
    STEP1_FEATURE_CONTRACT_OUTPUT,
    index=False
)

product_modelling_strategy_df.to_csv(
    PRODUCT_MODELLING_STRATEGY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step1_validation_summary_df.to_csv(
    STEP1_VALIDATION_SUMMARY_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 4. Confirm output files exist
# ------------------------------------------------------------

part_6_output_files = [
    STEP1_MODELLING_BASE_OUTPUT,
    STEP1_FEATURE_CONTRACT_OUTPUT,
    PRODUCT_MODELLING_STRATEGY_OUTPUT,
    STEP1_VALIDATION_SUMMARY_OUTPUT
]

missing_part_6_files = [
    file_path
    for file_path in part_6_output_files
    if not file_path.exists()
]

assert not missing_part_6_files, (
    "The following Part 6 files were not saved:\n"
    f"{missing_part_6_files}"
)


# ------------------------------------------------------------
# 5. Reload and validate saved outputs
# ------------------------------------------------------------

saved_step1_modelling_base = pd.read_csv(
    STEP1_MODELLING_BASE_OUTPUT,
    low_memory=False
)

saved_step1_feature_contract = pd.read_csv(
    STEP1_FEATURE_CONTRACT_OUTPUT,
    low_memory=False
)

saved_product_strategy = pd.read_csv(
    PRODUCT_MODELLING_STRATEGY_OUTPUT,
    low_memory=False
)

saved_step1_validation = pd.read_csv(
    STEP1_VALIDATION_SUMMARY_OUTPUT,
    low_memory=False
)

assert saved_step1_modelling_base.shape == (
    43_774,
    27
)

assert saved_step1_modelling_base[
    "CanonicalProductID"
].nunique() == 227

assert saved_step1_modelling_base[
    "Date"
].nunique() == 245

assert saved_step1_modelling_base[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_step1_modelling_base[
    "TotalDemand"
].sum() == 116_158

assert expected_leakage_columns.isdisjoint(
    saved_step1_modelling_base.columns
)

assert len(saved_step1_feature_contract) == 27

assert saved_step1_feature_contract[
    "Column"
].nunique() == 27

assert len(saved_product_strategy) == 227

assert saved_product_strategy[
    "CanonicalProductID"
].nunique() == 227

assert saved_product_strategy[
    "TotalDemandUnits"
].sum() == 116_158

assert len(saved_step1_validation) == 17


# ------------------------------------------------------------
# 6. Extract final verified values
# ------------------------------------------------------------

saved_model_input_count = int(
    saved_step1_feature_contract[
        "DirectModelInputAllowed"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
    .sum()
)

assert saved_model_input_count == 22

saved_cohort_lookup = (
    saved_product_strategy[
        "EvaluationCohort"
    ]
    .value_counts()
    .to_dict()
)

multi_fold_count = int(
    saved_cohort_lookup.get(
        "MULTI_FOLD_BACKTEST_READY",
        0
    )
)

single_holdout_count = int(
    saved_cohort_lookup.get(
        "SINGLE_HOLDOUT_BACKTEST_READY",
        0
    )
)

limited_holdout_count = int(
    saved_cohort_lookup.get(
        "LIMITED_HOLDOUT_ONLY",
        0
    )
)

minimal_evidence_count = int(
    saved_cohort_lookup.get(
        "MINIMAL_EVIDENCE_FORECAST_ONLY",
        0
    )
)

assert multi_fold_count == 127
assert single_holdout_count == 77
assert limited_holdout_count == 14
assert minimal_evidence_count == 9


# ------------------------------------------------------------
# 7. Build the Step 1 Markdown completion section
# ------------------------------------------------------------

excluded_columns_markdown = "\n".join(
    f"- `{column}`"
    for column in step1_excluded_columns
)

part_6_summary = f"""
**Status:** Completed and validated

### Purpose

Part 6 froze the final data contract for Forecasting Preparation
Step 1 and created a leakage-safe modelling base.

### Step 1 modelling base

- Rows: {len(saved_step1_modelling_base):,}
- Columns: {saved_step1_modelling_base.shape[1]}
- Canonical products: {saved_step1_modelling_base["CanonicalProductID"].nunique()}
- Operating dates: {saved_step1_modelling_base["Date"].nunique()}
- Duplicate product-date rows: 0
- Total demand units: {int(saved_step1_modelling_base["TotalDemand"].sum()):,}

### Approved column composition

- Direct feature candidates: {direct_candidate_count}
- Conditional feature candidates: {conditional_candidate_count}
- Total potential model-input candidates: {saved_model_input_count}
- Key columns: {key_column_count}
- Support-only columns: {support_column_count}
- Target columns: {target_column_count}

### Excluded from the modelling base

{excluded_columns_markdown}

The same-day demand components and outcome-derived fields were
excluded to prevent target leakage.

### Product modelling strategy register

All 227 products remain separate `CanonicalProductID` forecast
series.

- Multi-fold backtest ready: {multi_fold_count}
- Single-holdout backtest ready: {single_holdout_count}
- Limited-holdout only: {limited_holdout_count}
- Minimal-evidence forecast only: {minimal_evidence_count}
- Products removed: 0
- Products labelled discontinued: 0

Demand-pattern and evaluation-cohort fields are stored in the
product-level strategy register. They are not automatically used
as row-level predictive features because they were calculated from
the complete historical period.

### Saved Part 6 outputs

- `06_step1_leakage_safe_modelling_base.csv`
- `06_step1_feature_contract.csv`
- `06_product_modelling_strategy_register.csv`
- `06_step1_validation_summary.csv`

### Step 1 completion

Forecasting Preparation Step 1 is complete.

Lag features, rolling statistics and time-based training features
have not yet been created. Those will be handled in the next
forecasting-preparation step using only past demand information.
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_1_part_6",
    section_title=(
        "Forecasting Preparation — Step 1, Part 6"
    ),
    section_body=part_6_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 8. Print final Part 6 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 1, PART 6 COMPLETED"
)
print("=" * 75)

print()
print("Leakage-safe Step 1 modelling base:")
print(
    f"Rows saved: "
    f"{len(saved_step1_modelling_base):,}"
)
print(
    "Columns saved:",
    saved_step1_modelling_base.shape[1]
)
print(
    "Canonical products:",
    saved_step1_modelling_base[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_step1_modelling_base[
        "Date"
    ].nunique()
)
print(
    "Duplicate product-date rows:",
    saved_step1_modelling_base[
        [
            "Date",
            "CanonicalProductID"
        ]
    ].duplicated().sum()
)

print()
print("Leakage protection:")
print(
    "Same-day leakage columns included:",
    len(
        expected_leakage_columns.intersection(
            saved_step1_modelling_base.columns
        )
    )
)
print(
    "Potential model-input candidates:",
    saved_model_input_count
)

print()
print("Product forecasting scope:")
print("Products retained: 227")
print("Products removed: 0")
print("Products labelled discontinued: 0")

print()
print("Evaluation cohorts:")
print(
    "Multi-fold backtest ready:",
    multi_fold_count
)
print(
    "Single-holdout backtest ready:",
    single_holdout_count
)
print(
    "Limited-holdout only:",
    limited_holdout_count
)
print(
    "Minimal-evidence forecast only:",
    minimal_evidence_count
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_step1_modelling_base['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP1_MODELLING_BASE_OUTPUT}")
print(f"2. {STEP1_FEATURE_CONTRACT_OUTPUT}")
print(f"3. {PRODUCT_MODELLING_STRATEGY_OUTPUT}")
print(f"4. {STEP1_VALIDATION_SUMMARY_OUTPUT}")
print(f"5. {HANDOFF_FILE}")

print()
print(
    "All Step 1, Part 6 validation checks passed."
)
print(
    "FORECASTING PREPARATION STEP 1 IS COMPLETE."
)

FORECASTING PREPARATION — STEP 1, PART 6 COMPLETED

Leakage-safe Step 1 modelling base:
Rows saved: 43,774
Columns saved: 27
Canonical products: 227
Operating dates: 245
Duplicate product-date rows: 0

Leakage protection:
Same-day leakage columns included: 0
Potential model-input candidates: 22

Product forecasting scope:
Products retained: 227
Products removed: 0
Products labelled discontinued: 0

Evaluation cohorts:
Multi-fold backtest ready: 127
Single-holdout backtest ready: 77
Limited-holdout only: 14
Minimal-evidence forecast only: 9

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/06_step1_leakage_safe_modelling_base.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/06_step1_feature_contract.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/06_product_modelling_strategy_register.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/

In [35]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 1
# Cell 24: Load and validate the historical feature source panel
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation folder exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the earlier forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define official Step 1 input files
# ------------------------------------------------------------

STEP1_MODELLING_BASE_FILE = (
    FORECAST_PREPARATION_DIR
    / "06_step1_leakage_safe_modelling_base.csv"
)

STEP1_FEATURE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "06_step1_feature_contract.csv"
)

PRODUCT_MODELLING_STRATEGY_FILE = (
    FORECAST_PREPARATION_DIR
    / "06_product_modelling_strategy_register.csv"
)

required_step2_input_files = [
    STEP1_MODELLING_BASE_FILE,
    STEP1_FEATURE_CONTRACT_FILE,
    PRODUCT_MODELLING_STRATEGY_FILE
]

missing_step2_input_files = [
    file_path
    for file_path in required_step2_input_files
    if not file_path.exists()
]

if missing_step2_input_files:
    raise FileNotFoundError(
        "The following required Step 1 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_step2_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load the official Step 1 outputs
# ------------------------------------------------------------

step2_feature_source_df = pd.read_csv(
    STEP1_MODELLING_BASE_FILE,
    low_memory=False
)

step2_source_contract_df = pd.read_csv(
    STEP1_FEATURE_CONTRACT_FILE,
    low_memory=False
)

step2_product_strategy_df = pd.read_csv(
    PRODUCT_MODELLING_STRATEGY_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Parse the date columns
# ------------------------------------------------------------

date_columns = [
    "Date",
    "ProductFirstObservedDate"
]

for column in date_columns:

    step2_feature_source_df[column] = pd.to_datetime(
        step2_feature_source_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step2_feature_source_df[
    date_columns
].isna().sum().sum() == 0, (
    "Invalid or missing date values were found."
)


# ------------------------------------------------------------
# 5. Convert important numeric columns
# ------------------------------------------------------------

required_numeric_columns = [
    "ProductAgeOperatingDays",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "DaysSincePreviousOperatingDate",
    "TotalDemand"
]

for column in required_numeric_columns:

    step2_feature_source_df[column] = pd.to_numeric(
        step2_feature_source_df[column],
        errors="coerce"
    )

assert step2_feature_source_df[
    required_numeric_columns
].isna().sum().sum() == 0, (
    "Missing or non-numeric values were found in required "
    "numeric columns."
)


# ------------------------------------------------------------
# 6. Standardise product identifiers
# ------------------------------------------------------------

step2_feature_source_df[
    "CanonicalProductID"
] = (
    step2_feature_source_df[
        "CanonicalProductID"
    ]
    .astype("string")
    .str.strip()
)

step2_feature_source_df[
    "CanonicalProductName"
] = (
    step2_feature_source_df[
        "CanonicalProductName"
    ]
    .astype("string")
    .str.strip()
)

assert step2_feature_source_df[
    "CanonicalProductID"
].isna().sum() == 0

assert step2_feature_source_df[
    "CanonicalProductName"
].isna().sum() == 0


# ------------------------------------------------------------
# 7. Validate the Step 1 modelling base
# ------------------------------------------------------------

assert step2_feature_source_df.shape == (
    43_774,
    27
), (
    "Unexpected Step 1 modelling-base dimensions.\n"
    f"Found: {step2_feature_source_df.shape}"
)

assert step2_feature_source_df[
    "CanonicalProductID"
].nunique() == 227

assert step2_feature_source_df[
    "Date"
].nunique() == 245

assert step2_feature_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_feature_source_df[
    "TotalDemand"
].isna().sum() == 0

assert (
    step2_feature_source_df[
        "TotalDemand"
    ] < 0
).sum() == 0

assert step2_feature_source_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 8. Confirm leakage columns remain absent
# ------------------------------------------------------------

same_day_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

leakage_columns_found = sorted(
    same_day_leakage_columns.intersection(
        step2_feature_source_df.columns
    )
)

assert len(leakage_columns_found) == 0, (
    "Same-day target-leakage columns were found:\n"
    f"{leakage_columns_found}"
)


# ------------------------------------------------------------
# 9. Sort chronologically within each product
# ------------------------------------------------------------

step2_feature_source_df = (
    step2_feature_source_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Validate chronological order and series continuity
# ------------------------------------------------------------

step2_feature_source_df[
    "_PreviousOperatingSequence"
] = (
    step2_feature_source_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .shift(1)
)

step2_feature_source_df[
    "_OperatingSequenceDifference"
] = (
    step2_feature_source_df[
        "OperatingDaySequence"
    ]
    - step2_feature_source_df[
        "_PreviousOperatingSequence"
    ]
)

internal_gap_mask = (
    step2_feature_source_df[
        "_PreviousOperatingSequence"
    ].notna()
    &
    (
        step2_feature_source_df[
            "_OperatingSequenceDifference"
        ] != 1
    )
)

internal_gap_rows = int(
    internal_gap_mask.sum()
)

products_with_internal_gaps = int(
    step2_feature_source_df.loc[
        internal_gap_mask,
        "CanonicalProductID"
    ].nunique()
)

date_difference = (
    step2_feature_source_df
    .groupby("CanonicalProductID")[
        "Date"
    ]
    .diff()
)

non_increasing_date_rows = int(
    (
        date_difference.dt.days <= 0
    ).sum()
)

assert internal_gap_rows == 0
assert products_with_internal_gaps == 0
assert non_increasing_date_rows == 0


# ------------------------------------------------------------
# 11. Validate ProductAgeOperatingDays
# ------------------------------------------------------------

expected_product_age = (
    step2_feature_source_df
    .groupby("CanonicalProductID")
    .cumcount()
)

product_age_mismatches = int(
    (
        step2_feature_source_df[
            "ProductAgeOperatingDays"
        ].astype(int)
        != expected_product_age
    ).sum()
)

assert product_age_mismatches == 0, (
    "ProductAgeOperatingDays does not match the chronological "
    "row position for one or more products."
)


# ------------------------------------------------------------
# 12. Confirm every product continues to the final date
# ------------------------------------------------------------

final_operating_date = (
    step2_feature_source_df[
        "Date"
    ].max()
)

product_final_dates = (
    step2_feature_source_df
    .groupby("CanonicalProductID")[
        "Date"
    ]
    .max()
)

products_not_continued_to_final_date = int(
    (
        product_final_dates
        != final_operating_date
    ).sum()
)

assert final_operating_date == pd.Timestamp(
    "2026-03-30"
)

assert products_not_continued_to_final_date == 0


# ------------------------------------------------------------
# 13. Remove temporary validation columns
# ------------------------------------------------------------

step2_feature_source_df = (
    step2_feature_source_df
    .drop(
        columns=[
            "_PreviousOperatingSequence",
            "_OperatingSequenceDifference"
        ]
    )
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert step2_feature_source_df.shape == (
    43_774,
    27
)


# ------------------------------------------------------------
# 14. Print Cell 24 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 2, PART 1 — FEATURE SOURCE PANEL VALIDATED")
print("=" * 75)

print()
print("Source panel:")
print(f"Rows: {len(step2_feature_source_df):,}")
print(f"Columns: {step2_feature_source_df.shape[1]}")
print(
    "Canonical products:",
    step2_feature_source_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step2_feature_source_df[
        "Date"
    ].nunique()
)
print(
    "Date range:",
    step2_feature_source_df[
        "Date"
    ].min().date(),
    "to",
    step2_feature_source_df[
        "Date"
    ].max().date()
)

print()
print("Temporal validation:")
print(
    "Duplicate product-date rows:",
    step2_feature_source_df[
        [
            "Date",
            "CanonicalProductID"
        ]
    ].duplicated().sum()
)
print(
    "Internal product-series gaps:",
    internal_gap_rows
)
print(
    "Products with internal gaps:",
    products_with_internal_gaps
)
print(
    "Non-increasing product dates:",
    non_increasing_date_rows
)
print(
    "Product-age mismatches:",
    product_age_mismatches
)
print(
    "Products not continued to final date:",
    products_not_continued_to_final_date
)

print()
print("Leakage validation:")
print(
    "Same-day leakage columns found:",
    len(leakage_columns_found)
)
print(
    "Total demand units:",
    f"{step2_feature_source_df['TotalDemand'].sum():,}"
)

print()
print("Cell 24 completed successfully.")

STEP 2, PART 1 — FEATURE SOURCE PANEL VALIDATED

Source panel:
Rows: 43,774
Columns: 27
Canonical products: 227
Operating dates: 245
Date range: 2025-04-01 to 2026-03-30

Temporal validation:
Duplicate product-date rows: 0
Internal product-series gaps: 0
Products with internal gaps: 0
Non-increasing product dates: 0
Product-age mismatches: 0
Products not continued to final date: 0

Leakage validation:
Same-day leakage columns found: 0
Total demand units: 116,158

Cell 24 completed successfully.


In [36]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 1
# Cell 25: Freeze the feature-engineering contract and plan
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 24 objects exist
# ------------------------------------------------------------

required_cell_24_objects = [
    "step2_feature_source_df",
    "step2_source_contract_df",
    "step2_product_strategy_df"
]

missing_cell_24_objects = [
    object_name
    for object_name in required_cell_24_objects
    if object_name not in globals()
]

if missing_cell_24_objects:
    raise NameError(
        "The following Cell 24 objects are missing:\n"
        f"{missing_cell_24_objects}\n\n"
        "Run Cell 24 before running Cell 25."
    )


# ------------------------------------------------------------
# 2. Define the Step 2 column groups
# ------------------------------------------------------------

STEP2_GROUP_KEY = "CanonicalProductID"
STEP2_TIME_KEY = "Date"
STEP2_TIME_INDEX = "OperatingDaySequence"
STEP2_TARGET = "TotalDemand"

identifier_columns = [
    "Date",
    "CanonicalProductID"
]

support_columns = [
    "CanonicalProductName",
    "ProductFirstObservedDate"
]

known_ahead_time_features = [
    "ProductAgeOperatingDays",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay"
]

product_metadata_features = [
    "SourcePLUCount",
    "SourceGroupCodes",
    "SourceGroupNames",
    "IsMultiPLUCanonicalProduct"
]

conditional_product_metadata_features = [
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration"
]

current_date_feature_candidates = (
    known_ahead_time_features
    + product_metadata_features
    + conditional_product_metadata_features
)

assert len(known_ahead_time_features) == 12
assert len(product_metadata_features) == 4
assert len(conditional_product_metadata_features) == 6
assert len(current_date_feature_candidates) == 22


# ------------------------------------------------------------
# 3. Confirm all planned columns exist
# ------------------------------------------------------------

planned_columns = (
    identifier_columns
    + support_columns
    + current_date_feature_candidates
    + [STEP2_TARGET]
)

missing_planned_columns = [
    column
    for column in planned_columns
    if column not in step2_feature_source_df.columns
]

assert not missing_planned_columns, (
    "The following planned Step 2 columns are missing:\n"
    f"{missing_planned_columns}"
)

assert len(set(planned_columns)) == 27


# ------------------------------------------------------------
# 4. Create the Step 2 feature contract
# ------------------------------------------------------------

step2_feature_contract_df = pd.DataFrame({
    "Column": step2_feature_source_df.columns
})

step2_feature_contract_df[
    "Step2Role"
] = "UNCLASSIFIED"

step2_feature_contract_df.loc[
    step2_feature_contract_df[
        "Column"
    ].isin(identifier_columns),
    "Step2Role"
] = "IDENTIFIER_OR_TIME_KEY"

step2_feature_contract_df.loc[
    step2_feature_contract_df[
        "Column"
    ].isin(support_columns),
    "Step2Role"
] = "SUPPORT_AND_AUDIT_ONLY"

step2_feature_contract_df.loc[
    step2_feature_contract_df[
        "Column"
    ].isin(known_ahead_time_features),
    "Step2Role"
] = "KNOWN_AHEAD_TIME_FEATURE"

step2_feature_contract_df.loc[
    step2_feature_contract_df[
        "Column"
    ].isin(product_metadata_features),
    "Step2Role"
] = "PRODUCT_METADATA_FEATURE"

step2_feature_contract_df.loc[
    step2_feature_contract_df[
        "Column"
    ].isin(
        conditional_product_metadata_features
    ),
    "Step2Role"
] = "CONDITIONAL_PRODUCT_METADATA_FEATURE"

step2_feature_contract_df.loc[
    step2_feature_contract_df[
        "Column"
    ].eq(STEP2_TARGET),
    "Step2Role"
] = "TARGET_AND_HISTORICAL_FEATURE_SOURCE"


# ------------------------------------------------------------
# 5. Define prediction-time usage rules
# ------------------------------------------------------------

step2_feature_contract_df[
    "AllowedAsCurrentRowModelInput"
] = (
    step2_feature_contract_df[
        "Column"
    ].isin(current_date_feature_candidates)
)

step2_feature_contract_df[
    "AllowedAsIdentifier"
] = (
    step2_feature_contract_df[
        "Column"
    ].isin(identifier_columns)
)

step2_feature_contract_df[
    "AllowedAsSupportColumn"
] = (
    step2_feature_contract_df[
        "Column"
    ].isin(support_columns)
)

step2_feature_contract_df[
    "AllowedAsForecastTarget"
] = (
    step2_feature_contract_df[
        "Column"
    ].eq(STEP2_TARGET)
)

step2_feature_contract_df[
    "AllowedToGeneratePastDemandFeatures"
] = (
    step2_feature_contract_df[
        "Column"
    ].eq(STEP2_TARGET)
)

step2_feature_contract_df[
    "ShiftRequiredBeforeFeatureUse"
] = (
    step2_feature_contract_df[
        "Column"
    ].eq(STEP2_TARGET)
)

step2_feature_contract_df[
    "SameRowValueAllowedAsFeature"
] = (
    step2_feature_contract_df[
        "Column"
    ].isin(current_date_feature_candidates)
)


# ------------------------------------------------------------
# 6. Add explicit leakage rules
# ------------------------------------------------------------

step2_feature_contract_df[
    "LeakageRule"
] = np.select(
    [
        step2_feature_contract_df[
            "Column"
        ].eq(STEP2_TARGET),

        step2_feature_contract_df[
            "Column"
        ].isin(current_date_feature_candidates),

        step2_feature_contract_df[
            "Column"
        ].isin(identifier_columns),

        step2_feature_contract_df[
            "Column"
        ].isin(support_columns)
    ],
    [
        (
            "CURRENT_ROW_TOTALDEMAND_PROHIBITED_AS_FEATURE; "
            "USE_ONLY_AFTER_GROUPED_SHIFT_OF_AT_LEAST_1"
        ),
        (
            "CURRENT_ROW_VALUE_ALLOWED_BECAUSE_AVAILABLE_OR_"
            "DERIVABLE_BEFORE_FORECAST"
        ),
        "USE_AS_KEY_ONLY_NOT_AS_RAW_NUMERIC_FEATURE",
        "RETAIN_FOR_AUDIT_OR_DISPLAY_NOT_DIRECT_MODEL_INPUT"
    ],
    default="REVIEW"
)


# ------------------------------------------------------------
# 7. Add data-quality information
# ------------------------------------------------------------

step2_feature_contract_df[
    "DataType"
] = step2_feature_contract_df[
    "Column"
].map(
    {
        column: str(
            step2_feature_source_df[
                column
            ].dtype
        )
        for column in step2_feature_source_df.columns
    }
)

step2_feature_contract_df[
    "MissingCount"
] = step2_feature_contract_df[
    "Column"
].map(
    {
        column: int(
            step2_feature_source_df[
                column
            ].isna().sum()
        )
        for column in step2_feature_source_df.columns
    }
)

step2_feature_contract_df[
    "MissingPercentage"
] = step2_feature_contract_df[
    "Column"
].map(
    {
        column: round(
            (
                step2_feature_source_df[
                    column
                ].isna().sum()
                / len(step2_feature_source_df)
                * 100
            ),
            2
        )
        for column in step2_feature_source_df.columns
    }
)

step2_feature_contract_df[
    "UniqueNonNullValues"
] = step2_feature_contract_df[
    "Column"
].map(
    {
        column: int(
            step2_feature_source_df[
                column
            ].nunique(dropna=True)
        )
        for column in step2_feature_source_df.columns
    }
)


# ------------------------------------------------------------
# 8. Validate the Step 2 contract
# ------------------------------------------------------------

assert len(step2_feature_contract_df) == 27

assert step2_feature_contract_df[
    "Column"
].nunique() == 27

assert (
    step2_feature_contract_df[
        "Step2Role"
    ] == "UNCLASSIFIED"
).sum() == 0

assert int(
    step2_feature_contract_df[
        "AllowedAsCurrentRowModelInput"
    ].sum()
) == 22

assert int(
    step2_feature_contract_df[
        "AllowedToGeneratePastDemandFeatures"
    ].sum()
) == 1

assert int(
    step2_feature_contract_df[
        "ShiftRequiredBeforeFeatureUse"
    ].sum()
) == 1

assert (
    step2_feature_contract_df.loc[
        step2_feature_contract_df[
            "ShiftRequiredBeforeFeatureUse"
        ],
        "Column"
    ].tolist()
    == ["TotalDemand"]
)


# ------------------------------------------------------------
# 9. Define the historical feature-engineering plan
# ------------------------------------------------------------

step2_historical_feature_plan_df = pd.DataFrame([
    {
        "FeatureFamily":
            "DEMAND_LAG",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "1,2,3,5,10,20",
        "ShiftRule":
            "GROUP_BY_PRODUCT_SHIFT_N",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Previous observed product demand at selected "
            "operating-day lags."
    },
    {
        "FeatureFamily":
            "ROLLING_MEAN",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "3,5,10,20",
        "ShiftRule":
            "SHIFT_1_THEN_ROLL",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Past-only average demand over recent operating days."
    },
    {
        "FeatureFamily":
            "ROLLING_STANDARD_DEVIATION",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "3,5,10,20",
        "ShiftRule":
            "SHIFT_1_THEN_ROLL",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Past-only short-term demand variability."
    },
    {
        "FeatureFamily":
            "ROLLING_MEDIAN",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "3,5,10,20",
        "ShiftRule":
            "SHIFT_1_THEN_ROLL",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Past-only robust central demand estimate."
    },
    {
        "FeatureFamily":
            "ROLLING_SUM",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "3,5,10,20",
        "ShiftRule":
            "SHIFT_1_THEN_ROLL",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Past-only cumulative recent demand."
    },
    {
        "FeatureFamily":
            "ROLLING_ZERO_DEMAND_RATE",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "5,10,20",
        "ShiftRule":
            "SHIFT_1_THEN_COMPARE_ZERO_AND_ROLL",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Recent proportion of operating dates with zero demand."
    },
    {
        "FeatureFamily":
            "ROLLING_POSITIVE_DEMAND_COUNT",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "5,10,20",
        "ShiftRule":
            "SHIFT_1_THEN_COMPARE_POSITIVE_AND_ROLL",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Number of recent operating dates with positive demand."
    },
    {
        "FeatureFamily":
            "DAYS_SINCE_PREVIOUS_POSITIVE_DEMAND",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "PAST_ONLY",
        "ShiftRule":
            "USE_ONLY_POSITIVE_DEMAND_BEFORE_CURRENT_ROW",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Demand recency before the forecast date."
    },
    {
        "FeatureFamily":
            "EXPANDING_PAST_MEAN",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "MIN_PERIODS_1",
        "ShiftRule":
            "SHIFT_1_THEN_EXPANDING",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Long-run historical average using only prior rows."
    },
    {
        "FeatureFamily":
            "EXPANDING_PAST_POSITIVE_RATE",
        "SourceColumn":
            "TotalDemand",
        "Parameters":
            "MIN_PERIODS_1",
        "ShiftRule":
            "SHIFT_1_THEN_COMPARE_POSITIVE_AND_EXPANDING",
        "UsesCurrentRowTarget":
            False,
        "Purpose":
            "Long-run historical probability of positive demand."
    }
])


# ------------------------------------------------------------
# 10. Validate the feature plan
# ------------------------------------------------------------

assert len(step2_historical_feature_plan_df) == 10

assert step2_historical_feature_plan_df[
    "FeatureFamily"
].nunique() == 10

assert (
    step2_historical_feature_plan_df[
        "SourceColumn"
    ] == "TotalDemand"
).all()

assert not step2_historical_feature_plan_df[
    "UsesCurrentRowTarget"
].any()

assert (
    step2_historical_feature_plan_df[
        "ShiftRule"
    ]
    .str.contains(
        "SHIFT|PAST|BEFORE|GROUP",
        regex=True
    )
    .all()
)


# ------------------------------------------------------------
# 11. Create Part 1 validation summary
# ------------------------------------------------------------

step2_part1_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "FeatureSourceRows",
        "FeatureSourceColumns",
        "CanonicalProducts",
        "OperatingDates",
        "DuplicateProductDateRows",
        "InternalProductSeriesGaps",
        "ProductsNotContinuedToFinalDate",
        "ProductAgeMismatches",
        "SameDayLeakageColumnsPresent",
        "CurrentDateFeatureCandidates",
        "HistoricalDemandSourceColumns",
        "PlannedHistoricalFeatureFamilies",
        "TotalDemandUnits"
    ],
    "Value": [
        len(step2_feature_source_df),
        step2_feature_source_df.shape[1],
        step2_feature_source_df[
            "CanonicalProductID"
        ].nunique(),
        step2_feature_source_df[
            "Date"
        ].nunique(),
        int(
            step2_feature_source_df[
                [
                    "Date",
                    "CanonicalProductID"
                ]
            ].duplicated().sum()
        ),
        internal_gap_rows,
        products_not_continued_to_final_date,
        product_age_mismatches,
        len(leakage_columns_found),
        len(current_date_feature_candidates),
        int(
            step2_feature_contract_df[
                "AllowedToGeneratePastDemandFeatures"
            ].sum()
        ),
        len(step2_historical_feature_plan_df),
        int(
            step2_feature_source_df[
                "TotalDemand"
            ].sum()
        )
    ]
})

summary_lookup = dict(
    zip(
        step2_part1_validation_summary_df[
            "ValidationMetric"
        ],
        step2_part1_validation_summary_df[
            "Value"
        ]
    )
)

assert summary_lookup["FeatureSourceRows"] == 43_774
assert summary_lookup["FeatureSourceColumns"] == 27
assert summary_lookup["CanonicalProducts"] == 227
assert summary_lookup["OperatingDates"] == 245
assert summary_lookup["DuplicateProductDateRows"] == 0
assert summary_lookup["InternalProductSeriesGaps"] == 0
assert summary_lookup["SameDayLeakageColumnsPresent"] == 0
assert summary_lookup["CurrentDateFeatureCandidates"] == 22
assert summary_lookup["HistoricalDemandSourceColumns"] == 1
assert summary_lookup["PlannedHistoricalFeatureFamilies"] == 10
assert summary_lookup["TotalDemandUnits"] == 116_158


# ------------------------------------------------------------
# 12. Print Cell 25 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 2, PART 1 — FEATURE CONTRACT AND PLAN FROZEN")
print("=" * 75)

print()
print("Step 2 data contract:")
print(
    "Columns documented:",
    len(step2_feature_contract_df)
)
print(
    "Current-date feature candidates:",
    len(current_date_feature_candidates)
)
print(
    "Historical demand source columns:",
    int(
        step2_feature_contract_df[
            "AllowedToGeneratePastDemandFeatures"
        ].sum()
    )
)
print(
    "Columns requiring shift before feature use:",
    int(
        step2_feature_contract_df[
            "ShiftRequiredBeforeFeatureUse"
        ].sum()
    )
)

print()
print("Historical feature plan:")
display(step2_historical_feature_plan_df)

print()
print("Part 1 validation summary:")
display(step2_part1_validation_summary_df)

print()
print("Cell 25 completed successfully.")

STEP 2, PART 1 — FEATURE CONTRACT AND PLAN FROZEN

Step 2 data contract:
Columns documented: 27
Current-date feature candidates: 22
Historical demand source columns: 1
Columns requiring shift before feature use: 1

Historical feature plan:


,FeatureFamily,SourceColumn,Parameters,ShiftRule,UsesCurrentRowTarget,Purpose
0,DEMAND_LAG,TotalDemand,"1,2,3,5,10,20",GROUP_BY_PRODUCT_SHIFT_N,False,Previous observed product demand at selected o...
1,ROLLING_MEAN,TotalDemand,"3,5,10,20",SHIFT_1_THEN_ROLL,False,Past-only average demand over recent operating...
2,ROLLING_STANDARD_DEVIATION,TotalDemand,"3,5,10,20",SHIFT_1_THEN_ROLL,False,Past-only short-term demand variability.
3,ROLLING_MEDIAN,TotalDemand,"3,5,10,20",SHIFT_1_THEN_ROLL,False,Past-only robust central demand estimate.
4,ROLLING_SUM,TotalDemand,"3,5,10,20",SHIFT_1_THEN_ROLL,False,Past-only cumulative recent demand.
5,ROLLING_ZERO_DEMAND_RATE,TotalDemand,"5,10,20",SHIFT_1_THEN_COMPARE_ZERO_AND_ROLL,False,Recent proportion of operating dates with zero...
6,ROLLING_POSITIVE_DEMAND_COUNT,TotalDemand,"5,10,20",SHIFT_1_THEN_COMPARE_POSITIVE_AND_ROLL,False,Number of recent operating dates with positive...
7,DAYS_SINCE_PREVIOUS_POSITIVE_DEMAND,TotalDemand,PAST_ONLY,USE_ONLY_POSITIVE_DEMAND_BEFORE_CURRENT_ROW,False,Demand recency before the forecast date.
8,EXPANDING_PAST_MEAN,TotalDemand,MIN_PERIODS_1,SHIFT_1_THEN_EXPANDING,False,Long-run historical average using only prior r...
9,EXPANDING_PAST_POSITIVE_RATE,TotalDemand,MIN_PERIODS_1,SHIFT_1_THEN_COMPARE_POSITIVE_AND_EXPANDING,False,Long-run historical probability of positive de...



Part 1 validation summary:


,ValidationMetric,Value
0,FeatureSourceRows,43774
1,FeatureSourceColumns,27
2,CanonicalProducts,227
3,OperatingDates,245
4,DuplicateProductDateRows,0
5,InternalProductSeriesGaps,0
6,ProductsNotContinuedToFinalDate,0
7,ProductAgeMismatches,0
8,SameDayLeakageColumnsPresent,0
9,CurrentDateFeatureCandidates,22



Cell 25 completed successfully.


In [37]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 1
# Cell 26: Save controlled outputs and update the handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 1 objects exist
# ------------------------------------------------------------

required_step2_part1_objects = [
    "step2_feature_source_df",
    "step2_feature_contract_df",
    "step2_historical_feature_plan_df",
    "step2_part1_validation_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step2_part1_objects = [
    object_name
    for object_name in required_step2_part1_objects
    if object_name not in globals()
]

if missing_step2_part1_objects:
    raise NameError(
        "The following Step 2 Part 1 objects are missing:\n"
        f"{missing_step2_part1_objects}\n\n"
        "Run Cells 24 and 25 before running Cell 26."
    )


# ------------------------------------------------------------
# 2. Restore the handoff helper if the kernel was restarted
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):
        """
        Add or replace a controlled Markdown section.
        """

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Step 2 Part 1 output paths
# ------------------------------------------------------------

STEP2_FEATURE_SOURCE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "07_step2_feature_engineering_source_panel.csv"
)

STEP2_FEATURE_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "07_step2_feature_engineering_contract.csv"
)

STEP2_HISTORICAL_FEATURE_PLAN_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "07_step2_historical_feature_plan.csv"
)

STEP2_PART1_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "07_step2_part1_validation_summary.csv"
)


# ------------------------------------------------------------
# 4. Save the controlled outputs
# ------------------------------------------------------------

step2_feature_source_df.to_csv(
    STEP2_FEATURE_SOURCE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step2_feature_contract_df.to_csv(
    STEP2_FEATURE_CONTRACT_OUTPUT,
    index=False
)

step2_historical_feature_plan_df.to_csv(
    STEP2_HISTORICAL_FEATURE_PLAN_OUTPUT,
    index=False
)

step2_part1_validation_summary_df.to_csv(
    STEP2_PART1_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 5. Reload and validate saved files
# ------------------------------------------------------------

saved_step2_source = pd.read_csv(
    STEP2_FEATURE_SOURCE_OUTPUT,
    low_memory=False
)

saved_step2_contract = pd.read_csv(
    STEP2_FEATURE_CONTRACT_OUTPUT,
    low_memory=False
)

saved_step2_plan = pd.read_csv(
    STEP2_HISTORICAL_FEATURE_PLAN_OUTPUT,
    low_memory=False
)

saved_step2_validation = pd.read_csv(
    STEP2_PART1_VALIDATION_OUTPUT,
    low_memory=False
)

assert saved_step2_source.shape == (
    43_774,
    27
)

assert saved_step2_source[
    "CanonicalProductID"
].nunique() == 227

assert saved_step2_source[
    "Date"
].nunique() == 245

assert saved_step2_source[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_step2_source[
    "TotalDemand"
].sum() == 116_158

assert same_day_leakage_columns.isdisjoint(
    saved_step2_source.columns
)

assert len(saved_step2_contract) == 27

assert saved_step2_contract[
    "Column"
].nunique() == 27

assert len(saved_step2_plan) == 10

assert len(saved_step2_validation) == 13


# ------------------------------------------------------------
# 6. Update the Markdown handoff
# ------------------------------------------------------------

planned_feature_markdown = "\n".join(
    (
        f"- `{row.FeatureFamily}`: "
        f"{row.Parameters}; "
        f"rule `{row.ShiftRule}`"
    )
    for row in (
        step2_historical_feature_plan_df
        .itertuples()
    )
)

step2_part1_summary = f"""
**Status:** Completed and validated

### Purpose

Step 2 Part 1 prepared the controlled source panel for historical
feature engineering. No lag or rolling-demand values were created
in this part.

### Feature source panel

- Rows: {len(step2_feature_source_df):,}
- Columns: {step2_feature_source_df.shape[1]}
- Canonical products: {step2_feature_source_df["CanonicalProductID"].nunique()}
- Operating dates: {step2_feature_source_df["Date"].nunique()}
- Duplicate product-date rows: 0
- Internal product-series gaps: {internal_gap_rows}
- Products not continued to final date: {products_not_continued_to_final_date}
- Total demand units: {int(step2_feature_source_df["TotalDemand"].sum()):,}

### Prediction-time data contract

- Current-date feature candidates: {len(current_date_feature_candidates)}
- Historical demand source columns: 1
- Target and historical source: `TotalDemand`
- Same-day `TotalDemand` allowed as a feature: no
- Minimum required shift before historical feature use: 1 row
- Same-day leakage columns present: 0

### Historical feature families planned

{planned_feature_markdown}

All demand lags, rolling statistics, expanding statistics and
recency features must be grouped by `CanonicalProductID` and must
use only rows before the current forecast date.

### Saved Step 2 Part 1 outputs

- `07_step2_feature_engineering_source_panel.csv`
- `07_step2_feature_engineering_contract.csv`
- `07_step2_historical_feature_plan.csv`
- `07_step2_part1_validation_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_2_part_1",
    section_title=(
        "Forecasting Preparation — Step 2, Part 1"
    ),
    section_body=step2_part1_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 7. Print final completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 2, PART 1 COMPLETED"
)
print("=" * 75)

print()
print("Feature-engineering source panel:")
print(
    f"Rows saved: "
    f"{len(saved_step2_source):,}"
)
print(
    "Columns saved:",
    saved_step2_source.shape[1]
)
print(
    "Canonical products:",
    saved_step2_source[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_step2_source[
        "Date"
    ].nunique()
)

print()
print("Temporal validation:")
print(
    "Duplicate product-date rows:",
    saved_step2_source[
        [
            "Date",
            "CanonicalProductID"
        ]
    ].duplicated().sum()
)
print(
    "Internal product-series gaps:",
    internal_gap_rows
)
print(
    "Products not continued to final date:",
    products_not_continued_to_final_date
)

print()
print("Leakage protection:")
print(
    "Same-day leakage columns present:",
    len(
        same_day_leakage_columns.intersection(
            saved_step2_source.columns
        )
    )
)
print(
    "Current-date feature candidates:",
    len(current_date_feature_candidates)
)
print(
    "Historical feature families planned:",
    len(saved_step2_plan)
)
print(
    "Current-row TotalDemand allowed as feature:",
    False
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_step2_source['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP2_FEATURE_SOURCE_OUTPUT}")
print(f"2. {STEP2_FEATURE_CONTRACT_OUTPUT}")
print(f"3. {STEP2_HISTORICAL_FEATURE_PLAN_OUTPUT}")
print(f"4. {STEP2_PART1_VALIDATION_OUTPUT}")
print(f"5. {HANDOFF_FILE}")

print()
print(
    "All Step 2, Part 1 validation checks passed."
)

FORECASTING PREPARATION — STEP 2, PART 1 COMPLETED

Feature-engineering source panel:
Rows saved: 43,774
Columns saved: 27
Canonical products: 227
Operating dates: 245

Temporal validation:
Duplicate product-date rows: 0
Internal product-series gaps: 0
Products not continued to final date: 0

Leakage protection:
Same-day leakage columns present: 0
Current-date feature candidates: 22
Historical feature families planned: 10
Current-row TotalDemand allowed as feature: False

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/07_step2_feature_engineering_source_panel.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/07_step2_feature_engineering_contract.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/07_step2_historical_feature_plan.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/07_step2_part1_validation_summary.csv


In [38]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 2
# Cell 27: Generate leakage-safe product demand lag features
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the completed Step 2 Part 1 source panel
# ------------------------------------------------------------

STEP2_FEATURE_SOURCE_FILE = (
    FORECAST_PREPARATION_DIR
    / "07_step2_feature_engineering_source_panel.csv"
)

if "step2_feature_source_df" in globals():

    lag_source_df = step2_feature_source_df.copy()

elif STEP2_FEATURE_SOURCE_FILE.exists():

    lag_source_df = pd.read_csv(
        STEP2_FEATURE_SOURCE_FILE,
        low_memory=False
    )

else:

    raise FileNotFoundError(
        "The Step 2 Part 1 feature source panel could not "
        "be found.\n"
        f"Expected file:\n{STEP2_FEATURE_SOURCE_FILE}"
    )


# ------------------------------------------------------------
# 2. Parse and validate important columns
# ------------------------------------------------------------

lag_source_df["Date"] = pd.to_datetime(
    lag_source_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

lag_source_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    lag_source_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

numeric_columns = [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]

for column in numeric_columns:

    lag_source_df[column] = pd.to_numeric(
        lag_source_df[column],
        errors="coerce"
    )

assert lag_source_df["Date"].isna().sum() == 0

assert lag_source_df[
    "ProductFirstObservedDate"
].isna().sum() == 0

assert lag_source_df[
    numeric_columns
].isna().sum().sum() == 0

assert lag_source_df.shape == (
    43_774,
    27
)

assert lag_source_df[
    "CanonicalProductID"
].nunique() == 227

assert lag_source_df[
    "Date"
].nunique() == 245

assert lag_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert lag_source_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 3. Sort strictly by product and operating-day sequence
# ------------------------------------------------------------

lag_source_df = (
    lag_source_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Validate continuity before generating lag features
# ------------------------------------------------------------

sequence_difference = (
    lag_source_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .diff()
)

internal_gap_rows = int(
    (
        sequence_difference.notna()
        & sequence_difference.ne(1)
    ).sum()
)

assert internal_gap_rows == 0, (
    "Internal product-series gaps were found. "
    "Lag generation has been stopped."
)

expected_product_age = (
    lag_source_df
    .groupby("CanonicalProductID")
    .cumcount()
)

product_age_mismatches = int(
    (
        lag_source_df[
            "ProductAgeOperatingDays"
        ].astype(int)
        != expected_product_age
    ).sum()
)

assert product_age_mismatches == 0


# ------------------------------------------------------------
# 5. Define the approved lag periods
# ------------------------------------------------------------

DEMAND_LAG_PERIODS = [
    1,
    2,
    3,
    5,
    10,
    20
]

lag_feature_columns = [
    f"TotalDemandLag_{lag_period}"
    for lag_period in DEMAND_LAG_PERIODS
]


# ------------------------------------------------------------
# 6. Generate each lag within CanonicalProductID
# ------------------------------------------------------------

step2_lag_feature_df = lag_source_df.copy()

product_demand_group = (
    step2_lag_feature_df
    .groupby(
        "CanonicalProductID",
        sort=False
    )["TotalDemand"]
)

for lag_period in DEMAND_LAG_PERIODS:

    feature_name = (
        f"TotalDemandLag_{lag_period}"
    )

    step2_lag_feature_df[
        feature_name
    ] = product_demand_group.shift(
        lag_period
    )


# ------------------------------------------------------------
# 7. Validate final structure
# ------------------------------------------------------------

assert step2_lag_feature_df.shape == (
    43_774,
    33
), (
    "Expected 27 source columns plus 6 lag columns.\n"
    f"Found shape: {step2_lag_feature_df.shape}"
)

assert step2_lag_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_lag_feature_df[
    "TotalDemand"
].sum() == 116_158

assert all(
    column in step2_lag_feature_df.columns
    for column in lag_feature_columns
)


# ------------------------------------------------------------
# 8. Confirm legitimate missing-value locations
# ------------------------------------------------------------

lag_missing_validation_records = []

for lag_period in DEMAND_LAG_PERIODS:

    feature_name = (
        f"TotalDemandLag_{lag_period}"
    )

    expected_missing_mask = (
        step2_lag_feature_df[
            "ProductAgeOperatingDays"
        ] < lag_period
    )

    actual_missing_mask = (
        step2_lag_feature_df[
            feature_name
        ].isna()
    )

    unexpected_missing_rows = int(
        (
            actual_missing_mask
            & ~expected_missing_mask
        ).sum()
    )

    prematurely_populated_rows = int(
        (
            ~actual_missing_mask
            & expected_missing_mask
        ).sum()
    )

    expected_missing_rows = int(
        expected_missing_mask.sum()
    )

    actual_missing_rows = int(
        actual_missing_mask.sum()
    )

    assert unexpected_missing_rows == 0, (
        f"{feature_name} contains missing values after "
        "sufficient history should be available."
    )

    assert prematurely_populated_rows == 0, (
        f"{feature_name} contains values before the required "
        "historical lag exists."
    )

    assert (
        actual_missing_rows
        == expected_missing_rows
    )

    lag_missing_validation_records.append({
        "FeatureName": feature_name,
        "LagOperatingDays": lag_period,
        "ExpectedMissingRows":
            expected_missing_rows,
        "ActualMissingRows":
            actual_missing_rows,
        "UnexpectedMissingRows":
            unexpected_missing_rows,
        "PrematurelyPopulatedRows":
            prematurely_populated_rows,
        "PopulatedRows":
            (
                len(step2_lag_feature_df)
                - actual_missing_rows
            )
    })


lag_missing_validation_df = pd.DataFrame(
    lag_missing_validation_records
)


# ------------------------------------------------------------
# 9. Create the lag-feature contract
# ------------------------------------------------------------

lag_feature_contract_df = pd.DataFrame({
    "FeatureName": lag_feature_columns,
    "SourceColumn": "TotalDemand",
    "GroupKey": "CanonicalProductID",
    "TimeIndex": "OperatingDaySequence",
    "LagOperatingDays": DEMAND_LAG_PERIODS,
    "GenerationRule": [
        (
            "GROUP_BY_CANONICAL_PRODUCT_ID_"
            f"SHIFT_{lag_period}"
        )
        for lag_period in DEMAND_LAG_PERIODS
    ],
    "UsesCurrentRowTarget": False,
    "PredictionTimeAvailable": True,
    "EarlyHistoryMissingMeaning": (
        "INSUFFICIENT_PRIOR_PRODUCT_HISTORY"
    ),
    "MissingValueFilledInPart2": False
})


# ------------------------------------------------------------
# 10. Print Cell 27 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 2, PART 2 — DEMAND LAG FEATURES CREATED")
print("=" * 75)

print()
print("Lag-feature dataset:")
print(
    f"Rows: "
    f"{len(step2_lag_feature_df):,}"
)
print(
    "Columns:",
    step2_lag_feature_df.shape[1]
)
print(
    "Canonical products:",
    step2_lag_feature_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step2_lag_feature_df[
        "Date"
    ].nunique()
)

print()
print("Lag features created:")
for feature_name in lag_feature_columns:
    print(f"- {feature_name}")

print()
print("Missing-value validation:")
display(lag_missing_validation_df)

print()
print("Cell 27 completed successfully.")

STEP 2, PART 2 — DEMAND LAG FEATURES CREATED

Lag-feature dataset:
Rows: 43,774
Columns: 33
Canonical products: 227
Operating dates: 245

Lag features created:
- TotalDemandLag_1
- TotalDemandLag_2
- TotalDemandLag_3
- TotalDemandLag_5
- TotalDemandLag_10
- TotalDemandLag_20

Missing-value validation:


,FeatureName,LagOperatingDays,ExpectedMissingRows,ActualMissingRows,UnexpectedMissingRows,PrematurelyPopulatedRows,PopulatedRows
0,TotalDemandLag_1,1,227,227,0,0,43547
1,TotalDemandLag_2,2,454,454,0,0,43320
2,TotalDemandLag_3,3,681,681,0,0,43093
3,TotalDemandLag_5,5,1135,1135,0,0,42639
4,TotalDemandLag_10,10,2270,2270,0,0,41504
5,TotalDemandLag_20,20,4540,4540,0,0,39234



Cell 27 completed successfully.


In [39]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 2
# Cell 28: Independently validate lag values and leakage safety
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 27 objects exist
# ------------------------------------------------------------

required_cell_27_objects = [
    "step2_lag_feature_df",
    "lag_feature_contract_df",
    "lag_missing_validation_df",
    "DEMAND_LAG_PERIODS",
    "lag_feature_columns"
]

missing_cell_27_objects = [
    object_name
    for object_name in required_cell_27_objects
    if object_name not in globals()
]

if missing_cell_27_objects:
    raise NameError(
        "The following Cell 27 objects are missing:\n"
        f"{missing_cell_27_objects}\n\n"
        "Run Cell 27 before running Cell 28."
    )


# ------------------------------------------------------------
# 2. Create an independent demand lookup
# ------------------------------------------------------------

historical_demand_lookup_df = (
    step2_lag_feature_df[
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "TotalDemand"
        ]
    ]
    .copy()
    .rename(
        columns={
            "OperatingDaySequence":
                "SourceOperatingDaySequence",
            "TotalDemand":
                "ExpectedHistoricalDemand"
        }
    )
)

assert historical_demand_lookup_df[
    [
        "CanonicalProductID",
        "SourceOperatingDaySequence"
    ]
].duplicated().sum() == 0


# ------------------------------------------------------------
# 3. Independently check every lag feature
# ------------------------------------------------------------

lag_value_validation_records = []

for lag_period in DEMAND_LAG_PERIODS:

    feature_name = (
        f"TotalDemandLag_{lag_period}"
    )

    validation_df = (
        step2_lag_feature_df[
            [
                "CanonicalProductID",
                "OperatingDaySequence",
                "ProductAgeOperatingDays",
                feature_name
            ]
        ]
        .copy()
    )

    validation_df[
        "SourceOperatingDaySequence"
    ] = (
        validation_df[
            "OperatingDaySequence"
        ]
        - lag_period
    )

    validation_df = (
        validation_df
        .merge(
            historical_demand_lookup_df,
            on=[
                "CanonicalProductID",
                "SourceOperatingDaySequence"
            ],
            how="left",
            validate="many_to_one"
        )
    )

    both_missing_mask = (
        validation_df[
            feature_name
        ].isna()
        &
        validation_df[
            "ExpectedHistoricalDemand"
        ].isna()
    )

    both_present_equal_mask = (
        validation_df[
            feature_name
        ].notna()
        &
        validation_df[
            "ExpectedHistoricalDemand"
        ].notna()
        &
        np.isclose(
            validation_df[
                feature_name
            ],
            validation_df[
                "ExpectedHistoricalDemand"
            ],
            equal_nan=False
        )
    )

    correct_value_mask = (
        both_missing_mask
        | both_present_equal_mask
    )

    mismatch_rows = int(
        (~correct_value_mask).sum()
    )

    future_or_same_row_source_rows = int(
        (
            validation_df[
                "SourceOperatingDaySequence"
            ]
            >= validation_df[
                "OperatingDaySequence"
            ]
        ).sum()
    )

    available_rows = int(
        validation_df[
            feature_name
        ].notna().sum()
    )

    missing_rows = int(
        validation_df[
            feature_name
        ].isna().sum()
    )

    earliest_available_product_age = (
        validation_df.loc[
            validation_df[
                feature_name
            ].notna(),
            "ProductAgeOperatingDays"
        ].min()
    )

    assert mismatch_rows == 0, (
        f"{feature_name} does not match the independently "
        "located historical demand values."
    )

    assert future_or_same_row_source_rows == 0, (
        f"{feature_name} uses a same-row or future sequence."
    )

    assert int(
        earliest_available_product_age
    ) == lag_period

    lag_value_validation_records.append({
        "FeatureName": feature_name,
        "LagOperatingDays": lag_period,
        "AvailableRows": available_rows,
        "MissingRows": missing_rows,
        "MismatchRows": mismatch_rows,
        "SameOrFutureSourceRows":
            future_or_same_row_source_rows,
        "EarliestAvailableProductAge":
            int(
                earliest_available_product_age
            ),
        "ValidationStatus": "PASSED"
    })


lag_value_validation_df = pd.DataFrame(
    lag_value_validation_records
)


# ------------------------------------------------------------
# 4. Create row-level lag readiness indicators for audit
# ------------------------------------------------------------

step2_lag_feature_df[
    "AvailableLagFeatureCount"
] = (
    step2_lag_feature_df[
        lag_feature_columns
    ]
    .notna()
    .sum(axis=1)
    .astype(int)
)

step2_lag_feature_df[
    "AllApprovedDemandLagsAvailable"
] = (
    step2_lag_feature_df[
        lag_feature_columns
    ]
    .notna()
    .all(axis=1)
)

all_lags_available_rows = int(
    step2_lag_feature_df[
        "AllApprovedDemandLagsAvailable"
    ].sum()
)

partial_or_no_lag_rows = int(
    (
        ~step2_lag_feature_df[
            "AllApprovedDemandLagsAvailable"
        ]
    ).sum()
)

assert (
    all_lags_available_rows
    + partial_or_no_lag_rows
    == len(step2_lag_feature_df)
)


# ------------------------------------------------------------
# 5. Create lag-readiness summary
# ------------------------------------------------------------

lag_readiness_summary_df = (
    step2_lag_feature_df[
        "AvailableLagFeatureCount"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "AvailableLagFeatureCount"
    )
    .reset_index(name="RowCount")
)

lag_readiness_summary_df[
    "RowPercentage"
] = (
    lag_readiness_summary_df[
        "RowCount"
    ]
    / len(step2_lag_feature_df)
    * 100
).round(2)


# ------------------------------------------------------------
# 6. Validate dataset integrity after audit fields
# ------------------------------------------------------------

assert step2_lag_feature_df.shape == (
    43_774,
    35
), (
    "Expected 27 source columns, 6 lag columns and "
    "2 lag-readiness audit columns."
)

assert step2_lag_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_lag_feature_df[
    "TotalDemand"
].sum() == 116_158

assert lag_value_validation_df[
    "MismatchRows"
].sum() == 0

assert lag_value_validation_df[
    "SameOrFutureSourceRows"
].sum() == 0


# ------------------------------------------------------------
# 7. Print Cell 28 results
# ------------------------------------------------------------

print("=" * 75)
print("STEP 2, PART 2 — DEMAND LAG VALIDATION PASSED")
print("=" * 75)

print()
print("Independent lag-value validation:")
display(lag_value_validation_df)

print()
print("Lag-readiness summary:")
display(lag_readiness_summary_df)

print()
print("Overall readiness:")
print(
    "Rows with all six approved lags:",
    f"{all_lags_available_rows:,}"
)
print(
    "Rows with partial or no lag history:",
    f"{partial_or_no_lag_rows:,}"
)

print()
print("Leakage validation:")
print(
    "Lag-value mismatches:",
    int(
        lag_value_validation_df[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Same-row or future-source uses:",
    int(
        lag_value_validation_df[
            "SameOrFutureSourceRows"
        ].sum()
    )
)
print(
    "Current-row TotalDemand directly copied:",
    False
)

print()
print("Cell 28 completed successfully.")

STEP 2, PART 2 — DEMAND LAG VALIDATION PASSED

Independent lag-value validation:


,FeatureName,LagOperatingDays,AvailableRows,MissingRows,MismatchRows,SameOrFutureSourceRows,EarliestAvailableProductAge,ValidationStatus
0,TotalDemandLag_1,1,43547,227,0,0,1,PASSED
1,TotalDemandLag_2,2,43320,454,0,0,2,PASSED
2,TotalDemandLag_3,3,43093,681,0,0,3,PASSED
3,TotalDemandLag_5,5,42639,1135,0,0,5,PASSED
4,TotalDemandLag_10,10,41504,2270,0,0,10,PASSED
5,TotalDemandLag_20,20,39234,4540,0,0,20,PASSED



Lag-readiness summary:


,AvailableLagFeatureCount,RowCount,RowPercentage
0,0,227,0.52
1,1,227,0.52
2,2,227,0.52
3,3,454,1.04
4,4,1135,2.59
5,5,2270,5.19
6,6,39234,89.63



Overall readiness:
Rows with all six approved lags: 39,234
Rows with partial or no lag history: 4,540

Leakage validation:
Lag-value mismatches: 0
Same-row or future-source uses: 0
Current-row TotalDemand directly copied: False

Cell 28 completed successfully.


In [40]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 2
# Cell 29: Save lag-feature outputs and update the handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 2 objects exist
# ------------------------------------------------------------

required_step2_part2_objects = [
    "step2_lag_feature_df",
    "lag_feature_contract_df",
    "lag_missing_validation_df",
    "lag_value_validation_df",
    "lag_readiness_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step2_part2_objects = [
    object_name
    for object_name in required_step2_part2_objects
    if object_name not in globals()
]

if missing_step2_part2_objects:
    raise NameError(
        "The following Step 2 Part 2 objects are missing:\n"
        f"{missing_step2_part2_objects}\n\n"
        "Run Cells 27 and 28 before running Cell 29."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Step 2 Part 2 output paths
# ------------------------------------------------------------

STEP2_LAG_FEATURE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_features.csv"
)

STEP2_LAG_FEATURE_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_feature_contract.csv"
)

STEP2_LAG_MISSING_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_missing_audit.csv"
)

STEP2_LAG_VALUE_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_value_validation.csv"
)

STEP2_LAG_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_readiness_summary.csv"
)


# ------------------------------------------------------------
# 4. Save all Part 2 outputs
# ------------------------------------------------------------

step2_lag_feature_df.to_csv(
    STEP2_LAG_FEATURE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

lag_feature_contract_df.to_csv(
    STEP2_LAG_FEATURE_CONTRACT_OUTPUT,
    index=False
)

lag_missing_validation_df.to_csv(
    STEP2_LAG_MISSING_AUDIT_OUTPUT,
    index=False
)

lag_value_validation_df.to_csv(
    STEP2_LAG_VALUE_VALIDATION_OUTPUT,
    index=False
)

lag_readiness_summary_df.to_csv(
    STEP2_LAG_READINESS_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 5. Reload and validate saved outputs
# ------------------------------------------------------------

saved_lag_feature_df = pd.read_csv(
    STEP2_LAG_FEATURE_OUTPUT,
    low_memory=False
)

saved_lag_contract = pd.read_csv(
    STEP2_LAG_FEATURE_CONTRACT_OUTPUT,
    low_memory=False
)

saved_lag_missing_audit = pd.read_csv(
    STEP2_LAG_MISSING_AUDIT_OUTPUT,
    low_memory=False
)

saved_lag_value_validation = pd.read_csv(
    STEP2_LAG_VALUE_VALIDATION_OUTPUT,
    low_memory=False
)

saved_lag_readiness = pd.read_csv(
    STEP2_LAG_READINESS_OUTPUT,
    low_memory=False
)

assert saved_lag_feature_df.shape == (
    43_774,
    35
)

assert saved_lag_feature_df[
    "CanonicalProductID"
].nunique() == 227

assert saved_lag_feature_df[
    "Date"
].nunique() == 245

assert saved_lag_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_lag_feature_df[
    "TotalDemand"
].sum() == 116_158

assert len(saved_lag_contract) == 6
assert len(saved_lag_missing_audit) == 6
assert len(saved_lag_value_validation) == 6

assert saved_lag_value_validation[
    "MismatchRows"
].sum() == 0

assert saved_lag_value_validation[
    "SameOrFutureSourceRows"
].sum() == 0

assert saved_lag_readiness[
    "RowCount"
].sum() == 43_774


# ------------------------------------------------------------
# 6. Confirm early lag missingness was preserved
# ------------------------------------------------------------

for feature_name in lag_feature_columns:

    assert (
        saved_lag_feature_df[
            feature_name
        ].isna().sum()
        > 0
    ), (
        f"{feature_name} has no missing early-history values. "
        "The lag may have been incorrectly filled."
    )


# ------------------------------------------------------------
# 7. Build Markdown lag summary
# ------------------------------------------------------------

lag_markdown_rows = "\n".join(
    (
        f"- `{row.FeatureName}`: "
        f"{int(row.LagOperatingDays)} operating days; "
        f"{int(row.AvailableRows):,} populated rows; "
        f"{int(row.MissingRows):,} legitimate early-history "
        f"missing rows"
    )
    for row in lag_value_validation_df.itertuples()
)


# ------------------------------------------------------------
# 8. Update the Markdown handoff
# ------------------------------------------------------------

step2_part2_summary = f"""
**Status:** Completed and validated

### Purpose

Step 2 Part 2 created direct historical demand lags for every
canonical product. All lags were generated within
`CanonicalProductID` and ordered using `OperatingDaySequence`.

### Lag features created

{lag_markdown_rows}

### Leakage protection

- Source column: `TotalDemand`
- Product grouping key: `CanonicalProductID`
- Time index: `OperatingDaySequence`
- Minimum shift: 1 operating row
- Current-row target used directly: no
- Same-row or future-source uses: 0
- Independent lag-value mismatches: 0

### Dataset structure

- Rows: {len(saved_lag_feature_df):,}
- Columns: {saved_lag_feature_df.shape[1]}
- Canonical products: {saved_lag_feature_df["CanonicalProductID"].nunique()}
- Operating dates: {saved_lag_feature_df["Date"].nunique()}
- Duplicate product-date rows: 0
- Total demand units: {int(saved_lag_feature_df["TotalDemand"].sum()):,}

### Missing lag values

Early product-history rows contain legitimate missing lag values
where the required previous operating dates do not exist. These
values were not filled in Part 2.

- Rows with all six lag features available: {all_lags_available_rows:,}
- Rows with partial or no lag history: {partial_or_no_lag_rows:,}

### Saved Step 2 Part 2 outputs

- `08_step2_demand_lag_features.csv`
- `08_step2_demand_lag_feature_contract.csv`
- `08_step2_demand_lag_missing_audit.csv`
- `08_step2_demand_lag_value_validation.csv`
- `08_step2_demand_lag_readiness_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_2_part_2",
    section_title=(
        "Forecasting Preparation — Step 2, Part 2"
    ),
    section_body=step2_part2_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 9. Print final completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 2, PART 2 COMPLETED"
)
print("=" * 75)

print()
print("Lag-feature dataset:")
print(
    f"Rows saved: "
    f"{len(saved_lag_feature_df):,}"
)
print(
    "Columns saved:",
    saved_lag_feature_df.shape[1]
)
print(
    "Canonical products:",
    saved_lag_feature_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_lag_feature_df[
        "Date"
    ].nunique()
)

print()
print("Demand lag features:")
for feature_name in lag_feature_columns:
    print(f"- {feature_name}")

print()
print("Leakage validation:")
print(
    "Lag-value mismatches:",
    int(
        saved_lag_value_validation[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Same-row or future-source uses:",
    int(
        saved_lag_value_validation[
            "SameOrFutureSourceRows"
        ].sum()
    )
)
print(
    "Current-row TotalDemand used directly:",
    False
)

print()
print("Feature readiness:")
print(
    "Rows with all six demand lags:",
    f"{all_lags_available_rows:,}"
)
print(
    "Rows with partial or no lag history:",
    f"{partial_or_no_lag_rows:,}"
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_lag_feature_df['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP2_LAG_FEATURE_OUTPUT}")
print(f"2. {STEP2_LAG_FEATURE_CONTRACT_OUTPUT}")
print(f"3. {STEP2_LAG_MISSING_AUDIT_OUTPUT}")
print(f"4. {STEP2_LAG_VALUE_VALIDATION_OUTPUT}")
print(f"5. {STEP2_LAG_READINESS_OUTPUT}")
print(f"6. {HANDOFF_FILE}")

print()
print(
    "All Step 2, Part 2 validation checks passed."
)

FORECASTING PREPARATION — STEP 2, PART 2 COMPLETED

Lag-feature dataset:
Rows saved: 43,774
Columns saved: 35
Canonical products: 227
Operating dates: 245

Demand lag features:
- TotalDemandLag_1
- TotalDemandLag_2
- TotalDemandLag_3
- TotalDemandLag_5
- TotalDemandLag_10
- TotalDemandLag_20

Leakage validation:
Lag-value mismatches: 0
Same-row or future-source uses: 0
Current-row TotalDemand used directly: False

Feature readiness:
Rows with all six demand lags: 39,234
Rows with partial or no lag history: 4,540

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/08_step2_demand_lag_features.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/08_step2_demand_lag_feature_contract.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/08_step2_demand_lag_missing_audit.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/08_step2_

In [41]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 3
# Cell 30: Generate leakage-safe rolling demand features
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the completed Step 2 Part 2 dataset
# ------------------------------------------------------------

STEP2_LAG_FEATURE_FILE = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_features.csv"
)

if "step2_lag_feature_df" in globals():

    rolling_source_df = (
        step2_lag_feature_df.copy()
    )

elif STEP2_LAG_FEATURE_FILE.exists():

    rolling_source_df = pd.read_csv(
        STEP2_LAG_FEATURE_FILE,
        low_memory=False
    )

else:

    raise FileNotFoundError(
        "The Step 2 Part 2 lag-feature dataset could "
        "not be found.\n"
        f"Expected file:\n{STEP2_LAG_FEATURE_FILE}"
    )


# ------------------------------------------------------------
# 2. Parse and validate important columns
# ------------------------------------------------------------

rolling_source_df["Date"] = pd.to_datetime(
    rolling_source_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

rolling_source_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    rolling_source_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

required_numeric_columns = [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]

for column in required_numeric_columns:

    rolling_source_df[column] = pd.to_numeric(
        rolling_source_df[column],
        errors="coerce"
    )

assert rolling_source_df["Date"].isna().sum() == 0

assert rolling_source_df[
    "ProductFirstObservedDate"
].isna().sum() == 0

assert rolling_source_df[
    required_numeric_columns
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 3. Confirm the completed Part 2 structure
# ------------------------------------------------------------

approved_lag_columns = [
    "TotalDemandLag_1",
    "TotalDemandLag_2",
    "TotalDemandLag_3",
    "TotalDemandLag_5",
    "TotalDemandLag_10",
    "TotalDemandLag_20"
]

missing_lag_columns = [
    column
    for column in approved_lag_columns
    if column not in rolling_source_df.columns
]

assert not missing_lag_columns, (
    "The following approved lag columns are missing:\n"
    f"{missing_lag_columns}"
)

assert rolling_source_df.shape == (
    43_774,
    35
), (
    "Unexpected Step 2 Part 2 dataset dimensions.\n"
    f"Found: {rolling_source_df.shape}"
)

assert rolling_source_df[
    "CanonicalProductID"
].nunique() == 227

assert rolling_source_df[
    "Date"
].nunique() == 245

assert rolling_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert rolling_source_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 4. Sort strictly within each product series
# ------------------------------------------------------------

rolling_source_df = (
    rolling_source_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Validate continuity before rolling calculations
# ------------------------------------------------------------

sequence_difference = (
    rolling_source_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .diff()
)

internal_gap_rows = int(
    (
        sequence_difference.notna()
        & sequence_difference.ne(1)
    ).sum()
)

expected_product_age = (
    rolling_source_df
    .groupby("CanonicalProductID")
    .cumcount()
)

product_age_mismatches = int(
    (
        rolling_source_df[
            "ProductAgeOperatingDays"
        ].astype(int)
        != expected_product_age
    ).sum()
)

assert internal_gap_rows == 0, (
    "Internal product-series gaps were found. "
    "Rolling-feature generation was stopped."
)

assert product_age_mismatches == 0, (
    "ProductAgeOperatingDays does not match the "
    "chronological product-row position."
)


# ------------------------------------------------------------
# 6. Define approved windows and feature names
# ------------------------------------------------------------

ROLLING_DEMAND_WINDOWS = [
    3,
    5,
    10,
    20
]

ROLLING_MIN_PERIODS = 1

rolling_feature_columns = []

for window in ROLLING_DEMAND_WINDOWS:

    rolling_feature_columns.extend([
        f"PastDemandRollingMean_{window}",
        f"PastDemandRollingMedian_{window}",
        f"PastDemandRollingStd_{window}",
        f"PastDemandRollingSum_{window}"
    ])

assert len(rolling_feature_columns) == 16
assert len(set(rolling_feature_columns)) == 16


# ------------------------------------------------------------
# 7. Generate all rolling features
# ------------------------------------------------------------

step2_rolling_feature_df = (
    rolling_source_df.copy()
)

product_demand_group = (
    step2_rolling_feature_df
    .groupby(
        "CanonicalProductID",
        sort=False
    )["TotalDemand"]
)

for window in ROLLING_DEMAND_WINDOWS:

    # Mean of up to the previous N operating-day demands
    step2_rolling_feature_df[
        f"PastDemandRollingMean_{window}"
    ] = product_demand_group.transform(
        lambda series, current_window=window:
        series
        .shift(1)
        .rolling(
            window=current_window,
            min_periods=ROLLING_MIN_PERIODS
        )
        .mean()
    )

    # Median of up to the previous N operating-day demands
    step2_rolling_feature_df[
        f"PastDemandRollingMedian_{window}"
    ] = product_demand_group.transform(
        lambda series, current_window=window:
        series
        .shift(1)
        .rolling(
            window=current_window,
            min_periods=ROLLING_MIN_PERIODS
        )
        .median()
    )

    # Population standard deviation of prior demand values.
    # ddof=0 allows one previous observation to produce 0.
    step2_rolling_feature_df[
        f"PastDemandRollingStd_{window}"
    ] = product_demand_group.transform(
        lambda series, current_window=window:
        series
        .shift(1)
        .rolling(
            window=current_window,
            min_periods=ROLLING_MIN_PERIODS
        )
        .std(ddof=0)
    )

    # Sum of up to the previous N operating-day demands
    step2_rolling_feature_df[
        f"PastDemandRollingSum_{window}"
    ] = product_demand_group.transform(
        lambda series, current_window=window:
        series
        .shift(1)
        .rolling(
            window=current_window,
            min_periods=ROLLING_MIN_PERIODS
        )
        .sum()
    )


# ------------------------------------------------------------
# 8. Confirm the completed structure
# ------------------------------------------------------------

assert step2_rolling_feature_df.shape == (
    43_774,
    51
), (
    "Expected 35 Part 2 columns plus 16 rolling features.\n"
    f"Found: {step2_rolling_feature_df.shape}"
)

assert all(
    column in step2_rolling_feature_df.columns
    for column in rolling_feature_columns
)

assert step2_rolling_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_rolling_feature_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 9. Validate legitimate rolling-feature missingness
# ------------------------------------------------------------

rolling_missing_validation_records = []

first_product_row_mask = (
    step2_rolling_feature_df[
        "ProductAgeOperatingDays"
    ].eq(0)
)

expected_missing_rows = int(
    first_product_row_mask.sum()
)

assert expected_missing_rows == 227

for feature_name in rolling_feature_columns:

    actual_missing_mask = (
        step2_rolling_feature_df[
            feature_name
        ].isna()
    )

    unexpected_missing_rows = int(
        (
            actual_missing_mask
            & ~first_product_row_mask
        ).sum()
    )

    prematurely_populated_rows = int(
        (
            ~actual_missing_mask
            & first_product_row_mask
        ).sum()
    )

    actual_missing_rows = int(
        actual_missing_mask.sum()
    )

    assert unexpected_missing_rows == 0, (
        f"{feature_name} contains missing values after "
        "at least one prior operating-day observation exists."
    )

    assert prematurely_populated_rows == 0, (
        f"{feature_name} is populated on a product's first "
        "row even though no prior demand exists."
    )

    assert actual_missing_rows == expected_missing_rows

    rolling_missing_validation_records.append({
        "FeatureName": feature_name,
        "ExpectedMissingRows": expected_missing_rows,
        "ActualMissingRows": actual_missing_rows,
        "UnexpectedMissingRows":
            unexpected_missing_rows,
        "PrematurelyPopulatedRows":
            prematurely_populated_rows,
        "PopulatedRows": (
            len(step2_rolling_feature_df)
            - actual_missing_rows
        )
    })


rolling_missing_validation_df = pd.DataFrame(
    rolling_missing_validation_records
)


# ------------------------------------------------------------
# 10. Create the rolling-feature contract
# ------------------------------------------------------------

rolling_contract_records = []

for window in ROLLING_DEMAND_WINDOWS:

    statistic_definitions = [
        (
            "MEAN",
            f"PastDemandRollingMean_{window}",
            "PAST_ONLY_ROLLING_MEAN"
        ),
        (
            "MEDIAN",
            f"PastDemandRollingMedian_{window}",
            "PAST_ONLY_ROLLING_MEDIAN"
        ),
        (
            "STANDARD_DEVIATION",
            f"PastDemandRollingStd_{window}",
            "PAST_ONLY_ROLLING_POPULATION_STD_DDOF_0"
        ),
        (
            "SUM",
            f"PastDemandRollingSum_{window}",
            "PAST_ONLY_ROLLING_SUM"
        )
    ]

    for (
        statistic_name,
        feature_name,
        generation_rule
    ) in statistic_definitions:

        rolling_contract_records.append({
            "FeatureName": feature_name,
            "FeatureStatistic": statistic_name,
            "SourceColumn": "TotalDemand",
            "GroupKey": "CanonicalProductID",
            "TimeIndex": "OperatingDaySequence",
            "ShiftBeforeRolling": 1,
            "WindowOperatingDays": window,
            "MinimumPeriods": ROLLING_MIN_PERIODS,
            "GenerationRule": generation_rule,
            "UsesCurrentRowTarget": False,
            "UsesFutureTarget": False,
            "FirstProductRowMissing": True,
            "MissingValueFilledInPart3": False
        })


rolling_feature_contract_df = pd.DataFrame(
    rolling_contract_records
)

assert len(rolling_feature_contract_df) == 16

assert rolling_feature_contract_df[
    "FeatureName"
].nunique() == 16

assert not rolling_feature_contract_df[
    "UsesCurrentRowTarget"
].any()

assert not rolling_feature_contract_df[
    "UsesFutureTarget"
].any()


# ------------------------------------------------------------
# 11. Print Cell 30 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 3 — "
    "PAST-ONLY ROLLING DEMAND FEATURES CREATED"
)
print("=" * 75)

print()
print("Rolling-feature dataset:")
print(
    f"Rows: "
    f"{len(step2_rolling_feature_df):,}"
)
print(
    "Columns:",
    step2_rolling_feature_df.shape[1]
)
print(
    "Canonical products:",
    step2_rolling_feature_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step2_rolling_feature_df[
        "Date"
    ].nunique()
)

print()
print("Rolling features created:")
for feature_name in rolling_feature_columns:
    print(f"- {feature_name}")

print()
print("Missing-value validation:")
display(rolling_missing_validation_df)

print()
print("Cell 30 completed successfully.")

STEP 2, PART 3 — PAST-ONLY ROLLING DEMAND FEATURES CREATED

Rolling-feature dataset:
Rows: 43,774
Columns: 51
Canonical products: 227
Operating dates: 245

Rolling features created:
- PastDemandRollingMean_3
- PastDemandRollingMedian_3
- PastDemandRollingStd_3
- PastDemandRollingSum_3
- PastDemandRollingMean_5
- PastDemandRollingMedian_5
- PastDemandRollingStd_5
- PastDemandRollingSum_5
- PastDemandRollingMean_10
- PastDemandRollingMedian_10
- PastDemandRollingStd_10
- PastDemandRollingSum_10
- PastDemandRollingMean_20
- PastDemandRollingMedian_20
- PastDemandRollingStd_20
- PastDemandRollingSum_20

Missing-value validation:


,FeatureName,ExpectedMissingRows,ActualMissingRows,UnexpectedMissingRows,PrematurelyPopulatedRows,PopulatedRows
0,PastDemandRollingMean_3,227,227,0,0,43547
1,PastDemandRollingMedian_3,227,227,0,0,43547
2,PastDemandRollingStd_3,227,227,0,0,43547
3,PastDemandRollingSum_3,227,227,0,0,43547
4,PastDemandRollingMean_5,227,227,0,0,43547
5,PastDemandRollingMedian_5,227,227,0,0,43547
6,PastDemandRollingStd_5,227,227,0,0,43547
7,PastDemandRollingSum_5,227,227,0,0,43547
8,PastDemandRollingMean_10,227,227,0,0,43547
9,PastDemandRollingMedian_10,227,227,0,0,43547



Cell 30 completed successfully.


In [42]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 3
# Cell 31: Independently validate rolling values and leakage
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 30 objects exist
# ------------------------------------------------------------

required_cell_30_objects = [
    "step2_rolling_feature_df",
    "rolling_feature_contract_df",
    "rolling_missing_validation_df",
    "ROLLING_DEMAND_WINDOWS",
    "rolling_feature_columns"
]

missing_cell_30_objects = [
    object_name
    for object_name in required_cell_30_objects
    if object_name not in globals()
]

if missing_cell_30_objects:
    raise NameError(
        "The following Cell 30 objects are missing:\n"
        f"{missing_cell_30_objects}\n\n"
        "Run Cell 30 before running Cell 31."
    )


# ------------------------------------------------------------
# 2. Prepare a strictly ordered validation copy
# ------------------------------------------------------------

rolling_validation_df = (
    step2_rolling_feature_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

row_count = len(rolling_validation_df)

total_demand_array = (
    rolling_validation_df[
        "TotalDemand"
    ]
    .astype(float)
    .to_numpy()
)

product_age_array = (
    rolling_validation_df[
        "ProductAgeOperatingDays"
    ]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 3. Create independent expected-value arrays
# ------------------------------------------------------------

rolling_value_validation_records = []

for window in ROLLING_DEMAND_WINDOWS:

    expected_mean = np.full(
        row_count,
        np.nan,
        dtype=float
    )

    expected_median = np.full(
        row_count,
        np.nan,
        dtype=float
    )

    expected_std = np.full(
        row_count,
        np.nan,
        dtype=float
    )

    expected_sum = np.full(
        row_count,
        np.nan,
        dtype=float
    )

    # Explicitly inspect only rows before the current row.
    for (
        canonical_product_id,
        product_index
    ) in rolling_validation_df.groupby(
        "CanonicalProductID",
        sort=False
    ).groups.items():

        product_index = np.asarray(
            list(product_index),
            dtype=int
        )

        product_demand = (
            total_demand_array[
                product_index
            ]
        )

        for local_position, global_index in enumerate(
            product_index
        ):

            history_start = max(
                0,
                local_position - window
            )

            # The slice ends before local_position.
            # Therefore, the current row is excluded.
            prior_demand = product_demand[
                history_start:local_position
            ]

            if prior_demand.size == 0:
                continue

            expected_mean[
                global_index
            ] = float(
                np.mean(prior_demand)
            )

            expected_median[
                global_index
            ] = float(
                np.median(prior_demand)
            )

            expected_std[
                global_index
            ] = float(
                np.std(
                    prior_demand,
                    ddof=0
                )
            )

            expected_sum[
                global_index
            ] = float(
                np.sum(prior_demand)
            )


    feature_expectations = [
        (
            "MEAN",
            f"PastDemandRollingMean_{window}",
            expected_mean
        ),
        (
            "MEDIAN",
            f"PastDemandRollingMedian_{window}",
            expected_median
        ),
        (
            "STANDARD_DEVIATION",
            f"PastDemandRollingStd_{window}",
            expected_std
        ),
        (
            "SUM",
            f"PastDemandRollingSum_{window}",
            expected_sum
        )
    ]

    for (
        statistic_name,
        feature_name,
        expected_values
    ) in feature_expectations:

        actual_values = (
            rolling_validation_df[
                feature_name
            ]
            .astype(float)
            .to_numpy()
        )

        value_match_mask = np.isclose(
            actual_values,
            expected_values,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True
        )

        mismatch_rows = int(
            (~value_match_mask).sum()
        )

        missing_rows = int(
            np.isnan(actual_values).sum()
        )

        compared_rows = int(
            (~np.isnan(actual_values)).sum()
        )

        prematurely_populated_rows = int(
            (
                ~np.isnan(actual_values)
                & (product_age_array == 0)
            ).sum()
        )

        available_age_values = (
            product_age_array[
                ~np.isnan(actual_values)
            ]
        )

        earliest_available_product_age = int(
            available_age_values.min()
        )

        assert mismatch_rows == 0, (
            f"{feature_name} differs from the independently "
            "calculated past-only values."
        )

        assert prematurely_populated_rows == 0, (
            f"{feature_name} is populated before any historical "
            "demand exists."
        )

        assert earliest_available_product_age == 1

        rolling_value_validation_records.append({
            "FeatureName": feature_name,
            "FeatureStatistic": statistic_name,
            "WindowOperatingDays": window,
            "ComparedRows": compared_rows,
            "MissingRows": missing_rows,
            "MismatchRows": mismatch_rows,
            "PrematurelyPopulatedRows":
                prematurely_populated_rows,
            "EarliestAvailableProductAge":
                earliest_available_product_age,
            "CurrentRowExcluded": True,
            "ValidationStatus": "PASSED"
        })


rolling_value_validation_df = pd.DataFrame(
    rolling_value_validation_records
)


# ------------------------------------------------------------
# 4. Validate mean-sum-window consistency
# ------------------------------------------------------------

mean_sum_consistency_records = []

for window in ROLLING_DEMAND_WINDOWS:

    mean_column = (
        f"PastDemandRollingMean_{window}"
    )

    sum_column = (
        f"PastDemandRollingSum_{window}"
    )

    available_history_count = np.minimum(
        rolling_validation_df[
            "ProductAgeOperatingDays"
        ].astype(int),
        window
    )

    expected_sum_from_mean = (
        rolling_validation_df[
            mean_column
        ]
        * available_history_count
    )

    comparison_mask = (
        rolling_validation_df[
            mean_column
        ].notna()
        &
        rolling_validation_df[
            sum_column
        ].notna()
    )

    consistency_mismatches = int(
        (
            ~np.isclose(
                rolling_validation_df.loc[
                    comparison_mask,
                    sum_column
                ],
                expected_sum_from_mean.loc[
                    comparison_mask
                ],
                rtol=1e-10,
                atol=1e-12
            )
        ).sum()
    )

    assert consistency_mismatches == 0

    mean_sum_consistency_records.append({
        "WindowOperatingDays": window,
        "ComparedRows": int(
            comparison_mask.sum()
        ),
        "MeanSumConsistencyMismatches":
            consistency_mismatches,
        "ValidationStatus": "PASSED"
    })


rolling_mean_sum_consistency_df = pd.DataFrame(
    mean_sum_consistency_records
)


# ------------------------------------------------------------
# 5. Create rolling-feature readiness summary
# ------------------------------------------------------------

available_rolling_feature_count = (
    rolling_validation_df[
        rolling_feature_columns
    ]
    .notna()
    .sum(axis=1)
    .astype(int)
)

rolling_readiness_summary_df = (
    available_rolling_feature_count
    .value_counts()
    .sort_index()
    .rename_axis(
        "AvailableRollingFeatureCount"
    )
    .reset_index(name="RowCount")
)

rolling_readiness_summary_df[
    "RowPercentage"
] = (
    rolling_readiness_summary_df[
        "RowCount"
    ]
    / len(rolling_validation_df)
    * 100
).round(2)

rows_with_all_rolling_features = int(
    (
        available_rolling_feature_count
        == len(rolling_feature_columns)
    ).sum()
)

rows_without_rolling_history = int(
    (
        available_rolling_feature_count
        == 0
    ).sum()
)

rows_with_partial_rolling_features = int(
    (
        available_rolling_feature_count
        .between(
            1,
            len(rolling_feature_columns) - 1
        )
    ).sum()
)

assert rows_without_rolling_history == 227

assert rows_with_partial_rolling_features == 0

assert rows_with_all_rolling_features == (
    43_774 - 227
)

assert (
    rows_with_all_rolling_features
    + rows_without_rolling_history
    + rows_with_partial_rolling_features
    == 43_774
)


# ------------------------------------------------------------
# 6. Final integrity and leakage assertions
# ------------------------------------------------------------

assert len(rolling_value_validation_df) == 16

assert rolling_value_validation_df[
    "MismatchRows"
].sum() == 0

assert rolling_value_validation_df[
    "PrematurelyPopulatedRows"
].sum() == 0

assert rolling_value_validation_df[
    "CurrentRowExcluded"
].all()

assert rolling_mean_sum_consistency_df[
    "MeanSumConsistencyMismatches"
].sum() == 0

assert rolling_validation_df.shape == (
    43_774,
    51
)

assert rolling_validation_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert rolling_validation_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 7. Print Cell 31 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 3 — "
    "ROLLING FEATURE VALIDATION PASSED"
)
print("=" * 75)

print()
print("Independent rolling-value validation:")
display(rolling_value_validation_df)

print()
print("Mean-sum consistency:")
display(rolling_mean_sum_consistency_df)

print()
print("Rolling-feature readiness:")
display(rolling_readiness_summary_df)

print()
print("Overall readiness:")
print(
    "Rows with all 16 rolling features:",
    f"{rows_with_all_rolling_features:,}"
)
print(
    "Rows with no prior rolling history:",
    f"{rows_without_rolling_history:,}"
)
print(
    "Rows with partial rolling features:",
    f"{rows_with_partial_rolling_features:,}"
)

print()
print("Leakage validation:")
print(
    "Rolling-value mismatches:",
    int(
        rolling_value_validation_df[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Prematurely populated rows:",
    int(
        rolling_value_validation_df[
            "PrematurelyPopulatedRows"
        ].sum()
    )
)
print(
    "Current-row TotalDemand included:",
    False
)

print()
print("Cell 31 completed successfully.")

STEP 2, PART 3 — ROLLING FEATURE VALIDATION PASSED

Independent rolling-value validation:


,FeatureName,FeatureStatistic,WindowOperatingDays,ComparedRows,MissingRows,MismatchRows,PrematurelyPopulatedRows,EarliestAvailableProductAge,CurrentRowExcluded,ValidationStatus
0,PastDemandRollingMean_3,MEAN,3,43547,227,0,0,1,True,PASSED
1,PastDemandRollingMedian_3,MEDIAN,3,43547,227,0,0,1,True,PASSED
2,PastDemandRollingStd_3,STANDARD_DEVIATION,3,43547,227,0,0,1,True,PASSED
3,PastDemandRollingSum_3,SUM,3,43547,227,0,0,1,True,PASSED
4,PastDemandRollingMean_5,MEAN,5,43547,227,0,0,1,True,PASSED
5,PastDemandRollingMedian_5,MEDIAN,5,43547,227,0,0,1,True,PASSED
6,PastDemandRollingStd_5,STANDARD_DEVIATION,5,43547,227,0,0,1,True,PASSED
7,PastDemandRollingSum_5,SUM,5,43547,227,0,0,1,True,PASSED
8,PastDemandRollingMean_10,MEAN,10,43547,227,0,0,1,True,PASSED
9,PastDemandRollingMedian_10,MEDIAN,10,43547,227,0,0,1,True,PASSED



Mean-sum consistency:


,WindowOperatingDays,ComparedRows,MeanSumConsistencyMismatches,ValidationStatus
0,3,43547,0,PASSED
1,5,43547,0,PASSED
2,10,43547,0,PASSED
3,20,43547,0,PASSED



Rolling-feature readiness:


,AvailableRollingFeatureCount,RowCount,RowPercentage
0,0,227,0.52
1,16,43547,99.48



Overall readiness:
Rows with all 16 rolling features: 43,547
Rows with no prior rolling history: 227
Rows with partial rolling features: 0

Leakage validation:
Rolling-value mismatches: 0
Prematurely populated rows: 0
Current-row TotalDemand included: False

Cell 31 completed successfully.


In [43]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 3
# Cell 32: Save rolling-feature outputs and update the handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 3 objects exist
# ------------------------------------------------------------

required_step2_part3_objects = [
    "step2_rolling_feature_df",
    "rolling_feature_contract_df",
    "rolling_missing_validation_df",
    "rolling_value_validation_df",
    "rolling_mean_sum_consistency_df",
    "rolling_readiness_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step2_part3_objects = [
    object_name
    for object_name in required_step2_part3_objects
    if object_name not in globals()
]

if missing_step2_part3_objects:
    raise NameError(
        "The following Step 2 Part 3 objects are missing:\n"
        f"{missing_step2_part3_objects}\n\n"
        "Run Cells 30 and 31 before running Cell 32."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Step 2 Part 3 output paths
# ------------------------------------------------------------

STEP2_ROLLING_FEATURE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_features.csv"
)

STEP2_ROLLING_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_feature_contract.csv"
)

STEP2_ROLLING_MISSING_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_missing_audit.csv"
)

STEP2_ROLLING_VALUE_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_value_validation.csv"
)

STEP2_ROLLING_MEAN_SUM_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_mean_sum_audit.csv"
)

STEP2_ROLLING_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_readiness_summary.csv"
)


# ------------------------------------------------------------
# 4. Prepare the official output ordering
# ------------------------------------------------------------

rolling_feature_output_df = (
    step2_rolling_feature_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Save all Part 3 outputs
# ------------------------------------------------------------

rolling_feature_output_df.to_csv(
    STEP2_ROLLING_FEATURE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

rolling_feature_contract_df.to_csv(
    STEP2_ROLLING_CONTRACT_OUTPUT,
    index=False
)

rolling_missing_validation_df.to_csv(
    STEP2_ROLLING_MISSING_AUDIT_OUTPUT,
    index=False
)

rolling_value_validation_df.to_csv(
    STEP2_ROLLING_VALUE_VALIDATION_OUTPUT,
    index=False
)

rolling_mean_sum_consistency_df.to_csv(
    STEP2_ROLLING_MEAN_SUM_AUDIT_OUTPUT,
    index=False
)

rolling_readiness_summary_df.to_csv(
    STEP2_ROLLING_READINESS_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 6. Reload and validate saved outputs
# ------------------------------------------------------------

saved_rolling_feature_df = pd.read_csv(
    STEP2_ROLLING_FEATURE_OUTPUT,
    low_memory=False
)

saved_rolling_contract = pd.read_csv(
    STEP2_ROLLING_CONTRACT_OUTPUT,
    low_memory=False
)

saved_rolling_missing_audit = pd.read_csv(
    STEP2_ROLLING_MISSING_AUDIT_OUTPUT,
    low_memory=False
)

saved_rolling_value_validation = pd.read_csv(
    STEP2_ROLLING_VALUE_VALIDATION_OUTPUT,
    low_memory=False
)

saved_rolling_mean_sum_audit = pd.read_csv(
    STEP2_ROLLING_MEAN_SUM_AUDIT_OUTPUT,
    low_memory=False
)

saved_rolling_readiness = pd.read_csv(
    STEP2_ROLLING_READINESS_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 7. Validate saved data and audits
# ------------------------------------------------------------

assert saved_rolling_feature_df.shape == (
    43_774,
    51
)

assert saved_rolling_feature_df[
    "CanonicalProductID"
].nunique() == 227

assert saved_rolling_feature_df[
    "Date"
].nunique() == 245

assert saved_rolling_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_rolling_feature_df[
    "TotalDemand"
].sum() == 116_158

assert len(saved_rolling_contract) == 16

assert saved_rolling_contract[
    "FeatureName"
].nunique() == 16

assert len(saved_rolling_missing_audit) == 16

assert saved_rolling_missing_audit[
    "UnexpectedMissingRows"
].sum() == 0

assert saved_rolling_missing_audit[
    "PrematurelyPopulatedRows"
].sum() == 0

assert len(saved_rolling_value_validation) == 16

assert saved_rolling_value_validation[
    "MismatchRows"
].sum() == 0

assert saved_rolling_value_validation[
    "PrematurelyPopulatedRows"
].sum() == 0

assert saved_rolling_mean_sum_audit[
    "MeanSumConsistencyMismatches"
].sum() == 0

assert saved_rolling_readiness[
    "RowCount"
].sum() == 43_774


# ------------------------------------------------------------
# 8. Confirm first-row missingness was not filled
# ------------------------------------------------------------

for feature_name in rolling_feature_columns:

    assert (
        saved_rolling_feature_df[
            feature_name
        ].isna().sum()
        == 227
    ), (
        f"{feature_name} should contain exactly 227 "
        "first-product-row missing values."
    )


# ------------------------------------------------------------
# 9. Build Markdown feature summary
# ------------------------------------------------------------

rolling_markdown_rows = "\n".join(
    (
        f"- `{row.FeatureName}`: "
        f"{int(row.WindowOperatingDays)} prior operating days; "
        f"{int(row.ComparedRows):,} populated rows; "
        f"{int(row.MissingRows):,} legitimate missing rows; "
        f"{int(row.MismatchRows)} mismatches"
    )
    for row in (
        rolling_value_validation_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 10. Update the Markdown handoff
# ------------------------------------------------------------

step2_part3_summary = f"""
**Status:** Completed and validated

### Purpose

Step 2 Part 3 created past-only rolling demand statistics for
every canonical product.

Before every rolling calculation, `TotalDemand` was shifted by one
row within `CanonicalProductID`. Therefore, the current forecast
date's demand was excluded.

### Rolling calculation policy

- Source column: `TotalDemand`
- Product grouping key: `CanonicalProductID`
- Time index: `OperatingDaySequence`
- Shift before rolling: 1 operating row
- Windows: 3, 5, 10 and 20 operating days
- Minimum periods: 1
- Standard-deviation convention: population standard deviation,
  `ddof=0`
- Current-row target used: no
- Future target used: no

### Rolling features created

{rolling_markdown_rows}

### Validation

- Independent rolling-value mismatches: {int(rolling_value_validation_df["MismatchRows"].sum())}
- Prematurely populated rows: {int(rolling_value_validation_df["PrematurelyPopulatedRows"].sum())}
- Mean-sum consistency mismatches: {int(rolling_mean_sum_consistency_df["MeanSumConsistencyMismatches"].sum())}
- Rows with all 16 rolling features: {rows_with_all_rolling_features:,}
- Rows with no previous demand history: {rows_without_rolling_history:,}
- Rows with partial rolling features: {rows_with_partial_rolling_features:,}

### Dataset structure

- Rows: {len(saved_rolling_feature_df):,}
- Columns: {saved_rolling_feature_df.shape[1]}
- Canonical products: {saved_rolling_feature_df["CanonicalProductID"].nunique()}
- Operating dates: {saved_rolling_feature_df["Date"].nunique()}
- Duplicate product-date rows: 0
- Total demand units: {int(saved_rolling_feature_df["TotalDemand"].sum()):,}

### Missing-value interpretation

The first row of each product contains no historical demand.
Consequently, each rolling feature contains exactly 227 legitimate
missing values. These values were not filled in Part 3.

### Saved Step 2 Part 3 outputs

- `09_step2_demand_rolling_features.csv`
- `09_step2_demand_rolling_feature_contract.csv`
- `09_step2_demand_rolling_missing_audit.csv`
- `09_step2_demand_rolling_value_validation.csv`
- `09_step2_demand_rolling_mean_sum_audit.csv`
- `09_step2_demand_rolling_readiness_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_2_part_3",
    section_title=(
        "Forecasting Preparation — Step 2, Part 3"
    ),
    section_body=step2_part3_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 11. Print final Part 3 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 2, PART 3 COMPLETED"
)
print("=" * 75)

print()
print("Rolling-feature dataset:")
print(
    f"Rows saved: "
    f"{len(saved_rolling_feature_df):,}"
)
print(
    "Columns saved:",
    saved_rolling_feature_df.shape[1]
)
print(
    "Canonical products:",
    saved_rolling_feature_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_rolling_feature_df[
        "Date"
    ].nunique()
)

print()
print("Rolling features created:")
print("Rolling means: 4")
print("Rolling medians: 4")
print("Rolling standard deviations: 4")
print("Rolling sums: 4")
print("Total rolling features: 16")

print()
print("Leakage and value validation:")
print(
    "Rolling-value mismatches:",
    int(
        saved_rolling_value_validation[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Prematurely populated rows:",
    int(
        saved_rolling_value_validation[
            "PrematurelyPopulatedRows"
        ].sum()
    )
)
print(
    "Mean-sum consistency mismatches:",
    int(
        saved_rolling_mean_sum_audit[
            "MeanSumConsistencyMismatches"
        ].sum()
    )
)
print(
    "Current-row TotalDemand included:",
    False
)

print()
print("Feature readiness:")
print(
    "Rows with all 16 rolling features:",
    f"{rows_with_all_rolling_features:,}"
)
print(
    "Rows with no prior rolling history:",
    f"{rows_without_rolling_history:,}"
)
print(
    "Rows with partial rolling features:",
    f"{rows_with_partial_rolling_features:,}"
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_rolling_feature_df['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP2_ROLLING_FEATURE_OUTPUT}")
print(f"2. {STEP2_ROLLING_CONTRACT_OUTPUT}")
print(f"3. {STEP2_ROLLING_MISSING_AUDIT_OUTPUT}")
print(f"4. {STEP2_ROLLING_VALUE_VALIDATION_OUTPUT}")
print(f"5. {STEP2_ROLLING_MEAN_SUM_AUDIT_OUTPUT}")
print(f"6. {STEP2_ROLLING_READINESS_OUTPUT}")
print(f"7. {HANDOFF_FILE}")

print()
print(
    "All Step 2, Part 3 validation checks passed."
)

FORECASTING PREPARATION — STEP 2, PART 3 COMPLETED

Rolling-feature dataset:
Rows saved: 43,774
Columns saved: 51
Canonical products: 227
Operating dates: 245

Rolling features created:
Rolling means: 4
Rolling medians: 4
Rolling standard deviations: 4
Rolling sums: 4
Total rolling features: 16

Leakage and value validation:
Rolling-value mismatches: 0
Prematurely populated rows: 0
Mean-sum consistency mismatches: 0
Current-row TotalDemand included: False

Feature readiness:
Rows with all 16 rolling features: 43,547
Rows with no prior rolling history: 227
Rows with partial rolling features: 0

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/09_step2_demand_rolling_features.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/09_step2_demand_rolling_feature_contract.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/09_step2_demand_rolling_missing_

In [44]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 4
# Cell 33: Generate past-only demand occurrence and
# positive-demand recency features
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the completed Step 2 Part 3 dataset
# ------------------------------------------------------------

STEP2_ROLLING_FEATURE_FILE = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_features.csv"
)

if "step2_rolling_feature_df" in globals():

    occurrence_source_df = (
        step2_rolling_feature_df.copy()
    )

elif STEP2_ROLLING_FEATURE_FILE.exists():

    occurrence_source_df = pd.read_csv(
        STEP2_ROLLING_FEATURE_FILE,
        low_memory=False
    )

else:

    raise FileNotFoundError(
        "The Step 2 Part 3 rolling-feature dataset could "
        "not be found.\n"
        f"Expected file:\n{STEP2_ROLLING_FEATURE_FILE}"
    )


# ------------------------------------------------------------
# 2. Parse and validate important columns
# ------------------------------------------------------------

occurrence_source_df["Date"] = pd.to_datetime(
    occurrence_source_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

occurrence_source_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    occurrence_source_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

required_numeric_columns = [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]

for column in required_numeric_columns:

    occurrence_source_df[column] = pd.to_numeric(
        occurrence_source_df[column],
        errors="coerce"
    )

assert occurrence_source_df["Date"].isna().sum() == 0

assert occurrence_source_df[
    "ProductFirstObservedDate"
].isna().sum() == 0

assert occurrence_source_df[
    required_numeric_columns
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 3. Confirm the completed Part 3 structure
# ------------------------------------------------------------

assert occurrence_source_df.shape == (
    43_774,
    51
), (
    "Unexpected Step 2 Part 3 dataset dimensions.\n"
    f"Found: {occurrence_source_df.shape}"
)

assert occurrence_source_df[
    "CanonicalProductID"
].nunique() == 227

assert occurrence_source_df[
    "Date"
].nunique() == 245

assert occurrence_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert occurrence_source_df[
    "TotalDemand"
].sum() == 116_158

assert (
    occurrence_source_df["TotalDemand"] < 0
).sum() == 0


# ------------------------------------------------------------
# 4. Sort strictly within each product
# ------------------------------------------------------------

occurrence_source_df = (
    occurrence_source_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Validate temporal continuity
# ------------------------------------------------------------

sequence_difference = (
    occurrence_source_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .diff()
)

internal_gap_rows = int(
    (
        sequence_difference.notna()
        & sequence_difference.ne(1)
    ).sum()
)

expected_product_age = (
    occurrence_source_df
    .groupby("CanonicalProductID")
    .cumcount()
)

product_age_mismatches = int(
    (
        occurrence_source_df[
            "ProductAgeOperatingDays"
        ].astype(int)
        != expected_product_age
    ).sum()
)

assert internal_gap_rows == 0

assert product_age_mismatches == 0


# ------------------------------------------------------------
# 6. Confirm each product begins on a positive-demand date
# ------------------------------------------------------------

first_product_rows_df = (
    occurrence_source_df
    .groupby(
        "CanonicalProductID",
        sort=False
    )
    .head(1)
)

products_whose_first_row_is_not_positive = int(
    (
        first_product_rows_df[
            "TotalDemand"
        ] <= 0
    ).sum()
)

assert len(first_product_rows_df) == 227

assert products_whose_first_row_is_not_positive == 0, (
    "At least one product does not begin on its first "
    "positive-demand date."
)


# ------------------------------------------------------------
# 7. Define approved occurrence windows
# ------------------------------------------------------------

OCCURRENCE_WINDOWS = [
    5,
    10,
    20
]

OCCURRENCE_MIN_PERIODS = 1

zero_rate_feature_columns = [
    f"PastZeroDemandRate_{window}"
    for window in OCCURRENCE_WINDOWS
]

positive_count_feature_columns = [
    f"PastPositiveDemandCount_{window}"
    for window in OCCURRENCE_WINDOWS
]

recency_feature_column = (
    "OperatingDaysSincePreviousPositiveDemand"
)

occurrence_recency_feature_columns = (
    zero_rate_feature_columns
    + positive_count_feature_columns
    + [recency_feature_column]
)

assert len(occurrence_recency_feature_columns) == 7
assert len(set(occurrence_recency_feature_columns)) == 7


# ------------------------------------------------------------
# 8. Create the one-row-shifted demand source
# ------------------------------------------------------------

step2_occurrence_feature_df = (
    occurrence_source_df.copy()
)

product_group_key = (
    step2_occurrence_feature_df[
        "CanonicalProductID"
    ]
)

previous_demand = (
    step2_occurrence_feature_df
    .groupby(
        "CanonicalProductID",
        sort=False
    )["TotalDemand"]
    .shift(1)
)

previous_zero_indicator = (
    previous_demand.eq(0).astype(float)
)

previous_zero_indicator.loc[
    previous_demand.isna()
] = np.nan

previous_positive_indicator = (
    previous_demand.gt(0).astype(float)
)

previous_positive_indicator.loc[
    previous_demand.isna()
] = np.nan


# ------------------------------------------------------------
# 9. Create rolling zero-demand rates
# ------------------------------------------------------------

for window in OCCURRENCE_WINDOWS:

    feature_name = (
        f"PastZeroDemandRate_{window}"
    )

    step2_occurrence_feature_df[
        feature_name
    ] = (
        previous_zero_indicator
        .groupby(
            product_group_key,
            sort=False
        )
        .transform(
            lambda series, current_window=window:
            series
            .rolling(
                window=current_window,
                min_periods=OCCURRENCE_MIN_PERIODS
            )
            .mean()
        )
    )


# ------------------------------------------------------------
# 10. Create rolling positive-demand counts
# ------------------------------------------------------------

for window in OCCURRENCE_WINDOWS:

    feature_name = (
        f"PastPositiveDemandCount_{window}"
    )

    step2_occurrence_feature_df[
        feature_name
    ] = (
        previous_positive_indicator
        .groupby(
            product_group_key,
            sort=False
        )
        .transform(
            lambda series, current_window=window:
            series
            .rolling(
                window=current_window,
                min_periods=OCCURRENCE_MIN_PERIODS
            )
            .sum()
        )
    )


# ------------------------------------------------------------
# 11. Create days since the previous positive-demand date
# ------------------------------------------------------------

positive_sequence_marker = (
    step2_occurrence_feature_df[
        "OperatingDaySequence"
    ]
    .where(
        step2_occurrence_feature_df[
            "TotalDemand"
        ] > 0
    )
)

previous_positive_sequence = (
    positive_sequence_marker
    .groupby(
        product_group_key,
        sort=False
    )
    .transform(
        lambda series:
        series.ffill().shift(1)
    )
)

step2_occurrence_feature_df[
    recency_feature_column
] = (
    step2_occurrence_feature_df[
        "OperatingDaySequence"
    ]
    - previous_positive_sequence
)


# ------------------------------------------------------------
# 12. Validate the completed structure
# ------------------------------------------------------------

assert step2_occurrence_feature_df.shape == (
    43_774,
    58
), (
    "Expected 51 Part 3 columns plus 7 new features.\n"
    f"Found: {step2_occurrence_feature_df.shape}"
)

assert all(
    column in step2_occurrence_feature_df.columns
    for column in occurrence_recency_feature_columns
)

assert step2_occurrence_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_occurrence_feature_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 13. Validate legitimate missingness
# ------------------------------------------------------------

first_product_row_mask = (
    step2_occurrence_feature_df[
        "ProductAgeOperatingDays"
    ].eq(0)
)

expected_missing_rows = int(
    first_product_row_mask.sum()
)

assert expected_missing_rows == 227

occurrence_missing_validation_records = []

for feature_name in occurrence_recency_feature_columns:

    actual_missing_mask = (
        step2_occurrence_feature_df[
            feature_name
        ].isna()
    )

    unexpected_missing_rows = int(
        (
            actual_missing_mask
            & ~first_product_row_mask
        ).sum()
    )

    prematurely_populated_rows = int(
        (
            ~actual_missing_mask
            & first_product_row_mask
        ).sum()
    )

    actual_missing_rows = int(
        actual_missing_mask.sum()
    )

    assert unexpected_missing_rows == 0, (
        f"{feature_name} contains unexpected missing values."
    )

    assert prematurely_populated_rows == 0, (
        f"{feature_name} is populated on a product's "
        "first row."
    )

    assert actual_missing_rows == expected_missing_rows

    occurrence_missing_validation_records.append({
        "FeatureName": feature_name,
        "ExpectedMissingRows": expected_missing_rows,
        "ActualMissingRows": actual_missing_rows,
        "UnexpectedMissingRows":
            unexpected_missing_rows,
        "PrematurelyPopulatedRows":
            prematurely_populated_rows,
        "PopulatedRows": (
            len(step2_occurrence_feature_df)
            - actual_missing_rows
        )
    })


occurrence_missing_validation_df = pd.DataFrame(
    occurrence_missing_validation_records
)


# ------------------------------------------------------------
# 14. Create the feature contract
# ------------------------------------------------------------

occurrence_contract_records = []

for window in OCCURRENCE_WINDOWS:

    occurrence_contract_records.append({
        "FeatureName":
            f"PastZeroDemandRate_{window}",
        "FeatureFamily":
            "ROLLING_ZERO_DEMAND_RATE",
        "SourceColumn":
            "TotalDemand",
        "GroupKey":
            "CanonicalProductID",
        "TimeIndex":
            "OperatingDaySequence",
        "ShiftBeforeCalculation":
            1,
        "WindowOperatingDays":
            window,
        "MinimumPeriods":
            OCCURRENCE_MIN_PERIODS,
        "GenerationRule":
            "SHIFT_1_COMPARE_ZERO_THEN_ROLLING_MEAN",
        "UsesCurrentRowTarget":
            False,
        "UsesFutureTarget":
            False,
        "FirstProductRowMissing":
            True
    })

    occurrence_contract_records.append({
        "FeatureName":
            f"PastPositiveDemandCount_{window}",
        "FeatureFamily":
            "ROLLING_POSITIVE_DEMAND_COUNT",
        "SourceColumn":
            "TotalDemand",
        "GroupKey":
            "CanonicalProductID",
        "TimeIndex":
            "OperatingDaySequence",
        "ShiftBeforeCalculation":
            1,
        "WindowOperatingDays":
            window,
        "MinimumPeriods":
            OCCURRENCE_MIN_PERIODS,
        "GenerationRule":
            "SHIFT_1_COMPARE_POSITIVE_THEN_ROLLING_SUM",
        "UsesCurrentRowTarget":
            False,
        "UsesFutureTarget":
            False,
        "FirstProductRowMissing":
            True
    })


occurrence_contract_records.append({
    "FeatureName":
        recency_feature_column,
    "FeatureFamily":
        "DAYS_SINCE_PREVIOUS_POSITIVE_DEMAND",
    "SourceColumn":
        "TotalDemand",
    "GroupKey":
        "CanonicalProductID",
    "TimeIndex":
        "OperatingDaySequence",
    "ShiftBeforeCalculation":
        1,
    "WindowOperatingDays":
        np.nan,
    "MinimumPeriods":
        1,
    "GenerationRule":
        (
            "CURRENT_SEQUENCE_MINUS_MOST_RECENT_"
            "POSITIVE_SEQUENCE_BEFORE_CURRENT_ROW"
        ),
    "UsesCurrentRowTarget":
        False,
    "UsesFutureTarget":
        False,
    "FirstProductRowMissing":
        True
})


occurrence_feature_contract_df = pd.DataFrame(
    occurrence_contract_records
)

assert len(occurrence_feature_contract_df) == 7

assert occurrence_feature_contract_df[
    "FeatureName"
].nunique() == 7

assert not occurrence_feature_contract_df[
    "UsesCurrentRowTarget"
].any()

assert not occurrence_feature_contract_df[
    "UsesFutureTarget"
].any()


# ------------------------------------------------------------
# 15. Print Cell 33 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 4 — "
    "OCCURRENCE AND RECENCY FEATURES CREATED"
)
print("=" * 75)

print()
print("Feature dataset:")
print(
    f"Rows: "
    f"{len(step2_occurrence_feature_df):,}"
)
print(
    "Columns:",
    step2_occurrence_feature_df.shape[1]
)
print(
    "Canonical products:",
    step2_occurrence_feature_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step2_occurrence_feature_df[
        "Date"
    ].nunique()
)

print()
print("Features created:")
for feature_name in occurrence_recency_feature_columns:
    print(f"- {feature_name}")

print()
print("Missing-value validation:")
display(occurrence_missing_validation_df)

print()
print("Cell 33 completed successfully.")

STEP 2, PART 4 — OCCURRENCE AND RECENCY FEATURES CREATED

Feature dataset:
Rows: 43,774
Columns: 58
Canonical products: 227
Operating dates: 245

Features created:
- PastZeroDemandRate_5
- PastZeroDemandRate_10
- PastZeroDemandRate_20
- PastPositiveDemandCount_5
- PastPositiveDemandCount_10
- PastPositiveDemandCount_20
- OperatingDaysSincePreviousPositiveDemand

Missing-value validation:


,FeatureName,ExpectedMissingRows,ActualMissingRows,UnexpectedMissingRows,PrematurelyPopulatedRows,PopulatedRows
0,PastZeroDemandRate_5,227,227,0,0,43547
1,PastZeroDemandRate_10,227,227,0,0,43547
2,PastZeroDemandRate_20,227,227,0,0,43547
3,PastPositiveDemandCount_5,227,227,0,0,43547
4,PastPositiveDemandCount_10,227,227,0,0,43547
5,PastPositiveDemandCount_20,227,227,0,0,43547
6,OperatingDaysSincePreviousPositiveDemand,227,227,0,0,43547



Cell 33 completed successfully.


In [45]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 4
# Cell 34: Independently validate occurrence and recency values
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 33 objects exist
# ------------------------------------------------------------

required_cell_33_objects = [
    "step2_occurrence_feature_df",
    "occurrence_feature_contract_df",
    "occurrence_missing_validation_df",
    "OCCURRENCE_WINDOWS",
    "occurrence_recency_feature_columns"
]

missing_cell_33_objects = [
    object_name
    for object_name in required_cell_33_objects
    if object_name not in globals()
]

if missing_cell_33_objects:
    raise NameError(
        "The following Cell 33 objects are missing:\n"
        f"{missing_cell_33_objects}\n\n"
        "Run Cell 33 before running Cell 34."
    )


# ------------------------------------------------------------
# 2. Create a strictly ordered validation copy
# ------------------------------------------------------------

occurrence_validation_df = (
    step2_occurrence_feature_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

row_count = len(occurrence_validation_df)

demand_array = (
    occurrence_validation_df[
        "TotalDemand"
    ]
    .astype(float)
    .to_numpy()
)

sequence_array = (
    occurrence_validation_df[
        "OperatingDaySequence"
    ]
    .astype(int)
    .to_numpy()
)

product_age_array = (
    occurrence_validation_df[
        "ProductAgeOperatingDays"
    ]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 3. Prepare independent expected-value arrays
# ------------------------------------------------------------

expected_value_arrays = {}

for window in OCCURRENCE_WINDOWS:

    expected_value_arrays[
        f"PastZeroDemandRate_{window}"
    ] = np.full(
        row_count,
        np.nan,
        dtype=float
    )

    expected_value_arrays[
        f"PastPositiveDemandCount_{window}"
    ] = np.full(
        row_count,
        np.nan,
        dtype=float
    )


expected_value_arrays[
    recency_feature_column
] = np.full(
    row_count,
    np.nan,
    dtype=float
)


# ------------------------------------------------------------
# 4. Explicitly calculate values from prior rows only
# ------------------------------------------------------------

for (
    canonical_product_id,
    product_index
) in occurrence_validation_df.groupby(
    "CanonicalProductID",
    sort=False
).groups.items():

    product_index = np.asarray(
        list(product_index),
        dtype=int
    )

    product_demand = (
        demand_array[product_index]
    )

    product_sequence = (
        sequence_array[product_index]
    )

    previous_positive_sequence_value = None

    for local_position, global_index in enumerate(
        product_index
    ):

        # Recency is calculated before inspecting the
        # current row's demand.
        if previous_positive_sequence_value is not None:

            expected_value_arrays[
                recency_feature_column
            ][global_index] = (
                product_sequence[local_position]
                - previous_positive_sequence_value
            )

        for window in OCCURRENCE_WINDOWS:

            history_start = max(
                0,
                local_position - window
            )

            prior_demand = product_demand[
                history_start:local_position
            ]

            if prior_demand.size == 0:
                continue

            expected_value_arrays[
                f"PastZeroDemandRate_{window}"
            ][global_index] = float(
                np.mean(
                    prior_demand == 0
                )
            )

            expected_value_arrays[
                f"PastPositiveDemandCount_{window}"
            ][global_index] = float(
                np.sum(
                    prior_demand > 0
                )
            )

        # Update the previous positive sequence only after
        # all current-row feature calculations are complete.
        if product_demand[local_position] > 0:

            previous_positive_sequence_value = (
                product_sequence[local_position]
            )


# ------------------------------------------------------------
# 5. Compare actual and expected values
# ------------------------------------------------------------

occurrence_value_validation_records = []

for feature_name in occurrence_recency_feature_columns:

    actual_values = (
        occurrence_validation_df[
            feature_name
        ]
        .astype(float)
        .to_numpy()
    )

    expected_values = (
        expected_value_arrays[
            feature_name
        ]
    )

    value_match_mask = np.isclose(
        actual_values,
        expected_values,
        rtol=1e-10,
        atol=1e-12,
        equal_nan=True
    )

    mismatch_rows = int(
        (~value_match_mask).sum()
    )

    missing_rows = int(
        np.isnan(actual_values).sum()
    )

    compared_rows = int(
        (~np.isnan(actual_values)).sum()
    )

    prematurely_populated_rows = int(
        (
            ~np.isnan(actual_values)
            & (product_age_array == 0)
        ).sum()
    )

    available_product_ages = (
        product_age_array[
            ~np.isnan(actual_values)
        ]
    )

    earliest_available_product_age = int(
        available_product_ages.min()
    )

    assert mismatch_rows == 0, (
        f"{feature_name} differs from independently "
        "calculated past-only values."
    )

    assert prematurely_populated_rows == 0

    assert earliest_available_product_age == 1

    occurrence_value_validation_records.append({
        "FeatureName": feature_name,
        "ComparedRows": compared_rows,
        "MissingRows": missing_rows,
        "MismatchRows": mismatch_rows,
        "PrematurelyPopulatedRows":
            prematurely_populated_rows,
        "EarliestAvailableProductAge":
            earliest_available_product_age,
        "CurrentRowExcluded": True,
        "ValidationStatus": "PASSED"
    })


occurrence_value_validation_df = pd.DataFrame(
    occurrence_value_validation_records
)


# ------------------------------------------------------------
# 6. Validate zero-rate and positive-count consistency
# ------------------------------------------------------------

occurrence_consistency_records = []

for window in OCCURRENCE_WINDOWS:

    zero_rate_column = (
        f"PastZeroDemandRate_{window}"
    )

    positive_count_column = (
        f"PastPositiveDemandCount_{window}"
    )

    available_history_count = np.minimum(
        occurrence_validation_df[
            "ProductAgeOperatingDays"
        ].astype(int),
        window
    )

    implied_zero_count = (
        occurrence_validation_df[
            zero_rate_column
        ]
        * available_history_count
    )

    comparison_mask = (
        occurrence_validation_df[
            zero_rate_column
        ].notna()
        &
        occurrence_validation_df[
            positive_count_column
        ].notna()
    )

    consistency_mismatches = int(
        (
            ~np.isclose(
                (
                    implied_zero_count.loc[
                        comparison_mask
                    ]
                    + occurrence_validation_df.loc[
                        comparison_mask,
                        positive_count_column
                    ]
                ),
                available_history_count.loc[
                    comparison_mask
                ],
                rtol=1e-10,
                atol=1e-12
            )
        ).sum()
    )

    assert consistency_mismatches == 0

    occurrence_consistency_records.append({
        "WindowOperatingDays": window,
        "ComparedRows": int(
            comparison_mask.sum()
        ),
        "ZeroCountPlusPositiveCountMismatches":
            consistency_mismatches,
        "ValidationStatus": "PASSED"
    })


occurrence_consistency_df = pd.DataFrame(
    occurrence_consistency_records
)


# ------------------------------------------------------------
# 7. Validate feature ranges
# ------------------------------------------------------------

for window in OCCURRENCE_WINDOWS:

    zero_rate_column = (
        f"PastZeroDemandRate_{window}"
    )

    positive_count_column = (
        f"PastPositiveDemandCount_{window}"
    )

    assert (
        occurrence_validation_df[
            zero_rate_column
        ]
        .dropna()
        .between(0, 1)
        .all()
    )

    assert (
        occurrence_validation_df[
            positive_count_column
        ]
        .dropna()
        .between(0, window)
        .all()
    )


assert (
    occurrence_validation_df[
        recency_feature_column
    ]
    .dropna()
    .ge(1)
    .all()
)


# ------------------------------------------------------------
# 8. Create feature-readiness summary
# ------------------------------------------------------------

available_occurrence_feature_count = (
    occurrence_validation_df[
        occurrence_recency_feature_columns
    ]
    .notna()
    .sum(axis=1)
    .astype(int)
)

occurrence_readiness_summary_df = (
    available_occurrence_feature_count
    .value_counts()
    .sort_index()
    .rename_axis(
        "AvailableOccurrenceRecencyFeatureCount"
    )
    .reset_index(name="RowCount")
)

occurrence_readiness_summary_df[
    "RowPercentage"
] = (
    occurrence_readiness_summary_df[
        "RowCount"
    ]
    / len(occurrence_validation_df)
    * 100
).round(2)

rows_with_all_occurrence_features = int(
    (
        available_occurrence_feature_count
        == len(
            occurrence_recency_feature_columns
        )
    ).sum()
)

rows_without_occurrence_history = int(
    (
        available_occurrence_feature_count
        == 0
    ).sum()
)

rows_with_partial_occurrence_features = int(
    (
        available_occurrence_feature_count
        .between(
            1,
            len(
                occurrence_recency_feature_columns
            ) - 1
        )
    ).sum()
)

assert rows_without_occurrence_history == 227

assert rows_with_partial_occurrence_features == 0

assert rows_with_all_occurrence_features == (
    43_774 - 227
)


# ------------------------------------------------------------
# 9. Final integrity and leakage assertions
# ------------------------------------------------------------

assert len(occurrence_value_validation_df) == 7

assert occurrence_value_validation_df[
    "MismatchRows"
].sum() == 0

assert occurrence_value_validation_df[
    "PrematurelyPopulatedRows"
].sum() == 0

assert occurrence_value_validation_df[
    "CurrentRowExcluded"
].all()

assert occurrence_consistency_df[
    "ZeroCountPlusPositiveCountMismatches"
].sum() == 0

assert occurrence_validation_df.shape == (
    43_774,
    58
)

assert occurrence_validation_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert occurrence_validation_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 10. Print Cell 34 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 4 — "
    "OCCURRENCE AND RECENCY VALIDATION PASSED"
)
print("=" * 75)

print()
print("Independent feature validation:")
display(occurrence_value_validation_df)

print()
print("Occurrence consistency:")
display(occurrence_consistency_df)

print()
print("Feature readiness:")
display(occurrence_readiness_summary_df)

print()
print("Overall readiness:")
print(
    "Rows with all seven features:",
    f"{rows_with_all_occurrence_features:,}"
)
print(
    "Rows with no prior occurrence history:",
    f"{rows_without_occurrence_history:,}"
)
print(
    "Rows with partial features:",
    f"{rows_with_partial_occurrence_features:,}"
)

print()
print("Leakage validation:")
print(
    "Feature-value mismatches:",
    int(
        occurrence_value_validation_df[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Prematurely populated rows:",
    int(
        occurrence_value_validation_df[
            "PrematurelyPopulatedRows"
        ].sum()
    )
)
print(
    "Current-row TotalDemand included:",
    False
)

print()
print("Cell 34 completed successfully.")

STEP 2, PART 4 — OCCURRENCE AND RECENCY VALIDATION PASSED

Independent feature validation:


,FeatureName,ComparedRows,MissingRows,MismatchRows,PrematurelyPopulatedRows,EarliestAvailableProductAge,CurrentRowExcluded,ValidationStatus
0,PastZeroDemandRate_5,43547,227,0,0,1,True,PASSED
1,PastZeroDemandRate_10,43547,227,0,0,1,True,PASSED
2,PastZeroDemandRate_20,43547,227,0,0,1,True,PASSED
3,PastPositiveDemandCount_5,43547,227,0,0,1,True,PASSED
4,PastPositiveDemandCount_10,43547,227,0,0,1,True,PASSED
5,PastPositiveDemandCount_20,43547,227,0,0,1,True,PASSED
6,OperatingDaysSincePreviousPositiveDemand,43547,227,0,0,1,True,PASSED



Occurrence consistency:


,WindowOperatingDays,ComparedRows,ZeroCountPlusPositiveCountMismatches,ValidationStatus
0,5,43547,0,PASSED
1,10,43547,0,PASSED
2,20,43547,0,PASSED



Feature readiness:


,AvailableOccurrenceRecencyFeatureCount,RowCount,RowPercentage
0,0,227,0.52
1,7,43547,99.48



Overall readiness:
Rows with all seven features: 43,547
Rows with no prior occurrence history: 227
Rows with partial features: 0

Leakage validation:
Feature-value mismatches: 0
Prematurely populated rows: 0
Current-row TotalDemand included: False

Cell 34 completed successfully.


In [46]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 4
# Cell 35: Save occurrence and recency outputs and update handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 4 objects exist
# ------------------------------------------------------------

required_step2_part4_objects = [
    "step2_occurrence_feature_df",
    "occurrence_feature_contract_df",
    "occurrence_missing_validation_df",
    "occurrence_value_validation_df",
    "occurrence_consistency_df",
    "occurrence_readiness_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step2_part4_objects = [
    object_name
    for object_name in required_step2_part4_objects
    if object_name not in globals()
]

if missing_step2_part4_objects:
    raise NameError(
        "The following Step 2 Part 4 objects are missing:\n"
        f"{missing_step2_part4_objects}\n\n"
        "Run Cells 33 and 34 before running Cell 35."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Part 4 output paths
# ------------------------------------------------------------

STEP2_OCCURRENCE_FEATURE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_features.csv"
)

STEP2_OCCURRENCE_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_contract.csv"
)

STEP2_OCCURRENCE_MISSING_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_missing_audit.csv"
)

STEP2_OCCURRENCE_VALUE_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_value_validation.csv"
)

STEP2_OCCURRENCE_CONSISTENCY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_consistency_audit.csv"
)

STEP2_OCCURRENCE_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_readiness_summary.csv"
)


# ------------------------------------------------------------
# 4. Prepare official output ordering
# ------------------------------------------------------------

occurrence_feature_output_df = (
    step2_occurrence_feature_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Save all Part 4 outputs
# ------------------------------------------------------------

occurrence_feature_output_df.to_csv(
    STEP2_OCCURRENCE_FEATURE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

occurrence_feature_contract_df.to_csv(
    STEP2_OCCURRENCE_CONTRACT_OUTPUT,
    index=False
)

occurrence_missing_validation_df.to_csv(
    STEP2_OCCURRENCE_MISSING_AUDIT_OUTPUT,
    index=False
)

occurrence_value_validation_df.to_csv(
    STEP2_OCCURRENCE_VALUE_VALIDATION_OUTPUT,
    index=False
)

occurrence_consistency_df.to_csv(
    STEP2_OCCURRENCE_CONSISTENCY_OUTPUT,
    index=False
)

occurrence_readiness_summary_df.to_csv(
    STEP2_OCCURRENCE_READINESS_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 6. Reload saved outputs
# ------------------------------------------------------------

saved_occurrence_features = pd.read_csv(
    STEP2_OCCURRENCE_FEATURE_OUTPUT,
    low_memory=False
)

saved_occurrence_contract = pd.read_csv(
    STEP2_OCCURRENCE_CONTRACT_OUTPUT,
    low_memory=False
)

saved_occurrence_missing = pd.read_csv(
    STEP2_OCCURRENCE_MISSING_AUDIT_OUTPUT,
    low_memory=False
)

saved_occurrence_validation = pd.read_csv(
    STEP2_OCCURRENCE_VALUE_VALIDATION_OUTPUT,
    low_memory=False
)

saved_occurrence_consistency = pd.read_csv(
    STEP2_OCCURRENCE_CONSISTENCY_OUTPUT,
    low_memory=False
)

saved_occurrence_readiness = pd.read_csv(
    STEP2_OCCURRENCE_READINESS_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 7. Validate saved outputs
# ------------------------------------------------------------

assert saved_occurrence_features.shape == (
    43_774,
    58
)

assert saved_occurrence_features[
    "CanonicalProductID"
].nunique() == 227

assert saved_occurrence_features[
    "Date"
].nunique() == 245

assert saved_occurrence_features[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_occurrence_features[
    "TotalDemand"
].sum() == 116_158

assert len(saved_occurrence_contract) == 7

assert saved_occurrence_contract[
    "FeatureName"
].nunique() == 7

assert len(saved_occurrence_missing) == 7

assert saved_occurrence_missing[
    "UnexpectedMissingRows"
].sum() == 0

assert saved_occurrence_missing[
    "PrematurelyPopulatedRows"
].sum() == 0

assert len(saved_occurrence_validation) == 7

assert saved_occurrence_validation[
    "MismatchRows"
].sum() == 0

assert saved_occurrence_validation[
    "PrematurelyPopulatedRows"
].sum() == 0

assert saved_occurrence_consistency[
    "ZeroCountPlusPositiveCountMismatches"
].sum() == 0

assert saved_occurrence_readiness[
    "RowCount"
].sum() == 43_774


# ------------------------------------------------------------
# 8. Confirm legitimate first-row missingness
# ------------------------------------------------------------

for feature_name in occurrence_recency_feature_columns:

    assert (
        saved_occurrence_features[
            feature_name
        ].isna().sum()
        == 227
    ), (
        f"{feature_name} should contain exactly 227 "
        "first-product-row missing values."
    )


# ------------------------------------------------------------
# 9. Build Markdown summary
# ------------------------------------------------------------

occurrence_markdown_rows = "\n".join(
    (
        f"- `{row.FeatureName}`: "
        f"{int(row.ComparedRows):,} populated rows; "
        f"{int(row.MissingRows):,} legitimate missing rows; "
        f"{int(row.MismatchRows)} mismatches"
    )
    for row in (
        occurrence_value_validation_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 10. Update the Markdown handoff
# ------------------------------------------------------------

step2_part4_summary = f"""
**Status:** Completed and validated

### Purpose

Step 2 Part 4 created past-only demand-occurrence and
positive-demand recency features for every canonical product.

### Features created

{occurrence_markdown_rows}

### Calculation policy

- Source column: `TotalDemand`
- Product grouping key: `CanonicalProductID`
- Time index: `OperatingDaySequence`
- Rolling windows: 5, 10 and 20 operating days
- Minimum rolling periods: 1
- Current-row target used: no
- Future target used: no

`OperatingDaysSincePreviousPositiveDemand` measures the operating
day distance from the current date to the most recent positive
demand before the current row.

### Validation

- Independent feature-value mismatches: {int(occurrence_value_validation_df["MismatchRows"].sum())}
- Prematurely populated rows: {int(occurrence_value_validation_df["PrematurelyPopulatedRows"].sum())}
- Zero-count and positive-count consistency mismatches: {int(occurrence_consistency_df["ZeroCountPlusPositiveCountMismatches"].sum())}
- Rows with all seven features: {rows_with_all_occurrence_features:,}
- Rows with no previous occurrence history: {rows_without_occurrence_history:,}
- Rows with partial features: {rows_with_partial_occurrence_features:,}

### Dataset structure

- Rows: {len(saved_occurrence_features):,}
- Columns: {saved_occurrence_features.shape[1]}
- Canonical products: {saved_occurrence_features["CanonicalProductID"].nunique()}
- Operating dates: {saved_occurrence_features["Date"].nunique()}
- Duplicate product-date rows: 0
- Total demand units: {int(saved_occurrence_features["TotalDemand"].sum()):,}

### Missing-value interpretation

Each product's first row has no prior demand history. All seven
features therefore contain exactly 227 legitimate missing values.
These values were not filled in Part 4.

### Saved Step 2 Part 4 outputs

- `10_step2_demand_occurrence_recency_features.csv`
- `10_step2_demand_occurrence_recency_contract.csv`
- `10_step2_demand_occurrence_recency_missing_audit.csv`
- `10_step2_demand_occurrence_recency_value_validation.csv`
- `10_step2_demand_occurrence_consistency_audit.csv`
- `10_step2_demand_occurrence_recency_readiness_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_2_part_4",
    section_title=(
        "Forecasting Preparation — Step 2, Part 4"
    ),
    section_body=step2_part4_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 11. Print final completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 2, PART 4 COMPLETED"
)
print("=" * 75)

print()
print("Occurrence and recency dataset:")
print(
    f"Rows saved: "
    f"{len(saved_occurrence_features):,}"
)
print(
    "Columns saved:",
    saved_occurrence_features.shape[1]
)
print(
    "Canonical products:",
    saved_occurrence_features[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_occurrence_features[
        "Date"
    ].nunique()
)

print()
print("Features created:")
print("Past zero-demand rates: 3")
print("Past positive-demand counts: 3")
print("Positive-demand recency features: 1")
print("Total new features: 7")

print()
print("Leakage and value validation:")
print(
    "Feature-value mismatches:",
    int(
        saved_occurrence_validation[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Prematurely populated rows:",
    int(
        saved_occurrence_validation[
            "PrematurelyPopulatedRows"
        ].sum()
    )
)
print(
    "Occurrence consistency mismatches:",
    int(
        saved_occurrence_consistency[
            "ZeroCountPlusPositiveCountMismatches"
        ].sum()
    )
)
print(
    "Current-row TotalDemand included:",
    False
)

print()
print("Feature readiness:")
print(
    "Rows with all seven features:",
    f"{rows_with_all_occurrence_features:,}"
)
print(
    "Rows with no prior occurrence history:",
    f"{rows_without_occurrence_history:,}"
)
print(
    "Rows with partial features:",
    f"{rows_with_partial_occurrence_features:,}"
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_occurrence_features['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP2_OCCURRENCE_FEATURE_OUTPUT}")
print(f"2. {STEP2_OCCURRENCE_CONTRACT_OUTPUT}")
print(f"3. {STEP2_OCCURRENCE_MISSING_AUDIT_OUTPUT}")
print(f"4. {STEP2_OCCURRENCE_VALUE_VALIDATION_OUTPUT}")
print(f"5. {STEP2_OCCURRENCE_CONSISTENCY_OUTPUT}")
print(f"6. {STEP2_OCCURRENCE_READINESS_OUTPUT}")
print(f"7. {HANDOFF_FILE}")

print()
print(
    "All Step 2, Part 4 validation checks passed."
)

FORECASTING PREPARATION — STEP 2, PART 4 COMPLETED

Occurrence and recency dataset:
Rows saved: 43,774
Columns saved: 58
Canonical products: 227
Operating dates: 245

Features created:
Past zero-demand rates: 3
Past positive-demand counts: 3
Positive-demand recency features: 1
Total new features: 7

Leakage and value validation:
Feature-value mismatches: 0
Prematurely populated rows: 0
Occurrence consistency mismatches: 0
Current-row TotalDemand included: False

Feature readiness:
Rows with all seven features: 43,547
Rows with no prior occurrence history: 227
Rows with partial features: 0

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/10_step2_demand_occurrence_recency_features.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/10_step2_demand_occurrence_recency_contract.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/10_step2_demand_occurr

In [47]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 5
# Cell 36: Generate past-only expanding historical features
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the completed Step 2 Part 4 dataset
# ------------------------------------------------------------

STEP2_OCCURRENCE_FEATURE_FILE = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_features.csv"
)

if "step2_occurrence_feature_df" in globals():

    expanding_source_df = (
        step2_occurrence_feature_df.copy()
    )

elif STEP2_OCCURRENCE_FEATURE_FILE.exists():

    expanding_source_df = pd.read_csv(
        STEP2_OCCURRENCE_FEATURE_FILE,
        low_memory=False
    )

else:

    raise FileNotFoundError(
        "The Step 2 Part 4 occurrence-feature dataset "
        "could not be found.\n"
        f"Expected file:\n{STEP2_OCCURRENCE_FEATURE_FILE}"
    )


# ------------------------------------------------------------
# 2. Parse and validate important columns
# ------------------------------------------------------------

expanding_source_df["Date"] = pd.to_datetime(
    expanding_source_df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

expanding_source_df[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    expanding_source_df[
        "ProductFirstObservedDate"
    ],
    format="%Y-%m-%d",
    errors="coerce"
)

required_numeric_columns = [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]

for column in required_numeric_columns:

    expanding_source_df[column] = pd.to_numeric(
        expanding_source_df[column],
        errors="coerce"
    )

assert expanding_source_df["Date"].isna().sum() == 0

assert expanding_source_df[
    "ProductFirstObservedDate"
].isna().sum() == 0

assert expanding_source_df[
    required_numeric_columns
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 3. Confirm the completed Part 4 structure
# ------------------------------------------------------------

assert expanding_source_df.shape == (
    43_774,
    58
), (
    "Unexpected Step 2 Part 4 dataset dimensions.\n"
    f"Found: {expanding_source_df.shape}"
)

assert expanding_source_df[
    "CanonicalProductID"
].nunique() == 227

assert expanding_source_df[
    "Date"
].nunique() == 245

assert expanding_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert expanding_source_df[
    "TotalDemand"
].sum() == 116_158

assert (
    expanding_source_df[
        "TotalDemand"
    ] < 0
).sum() == 0


# ------------------------------------------------------------
# 4. Sort strictly within each product series
# ------------------------------------------------------------

expanding_source_df = (
    expanding_source_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence",
            "Date"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Validate temporal continuity
# ------------------------------------------------------------

sequence_difference = (
    expanding_source_df
    .groupby("CanonicalProductID")[
        "OperatingDaySequence"
    ]
    .diff()
)

internal_gap_rows = int(
    (
        sequence_difference.notna()
        & sequence_difference.ne(1)
    ).sum()
)

expected_product_age = (
    expanding_source_df
    .groupby("CanonicalProductID")
    .cumcount()
)

product_age_mismatches = int(
    (
        expanding_source_df[
            "ProductAgeOperatingDays"
        ].astype(int)
        != expected_product_age
    ).sum()
)

assert internal_gap_rows == 0, (
    "Internal product-series gaps were found."
)

assert product_age_mismatches == 0, (
    "ProductAgeOperatingDays does not match the "
    "chronological product-row position."
)


# ------------------------------------------------------------
# 6. Define expanding feature names
# ------------------------------------------------------------

EXPANDING_MIN_PERIODS = 1

expanding_feature_columns = [
    "ExpandingPastMeanDemand",
    "ExpandingPastPositiveDemandRate"
]

assert len(expanding_feature_columns) == 2


# ------------------------------------------------------------
# 7. Create the shifted historical demand source
# ------------------------------------------------------------

step2_expanding_feature_df = (
    expanding_source_df.copy()
)

product_group_key = (
    step2_expanding_feature_df[
        "CanonicalProductID"
    ]
)

previous_demand = (
    step2_expanding_feature_df
    .groupby(
        "CanonicalProductID",
        sort=False
    )["TotalDemand"]
    .shift(1)
)

previous_positive_indicator = (
    previous_demand.gt(0).astype(float)
)

# The first row of every product has no previous demand.
previous_positive_indicator.loc[
    previous_demand.isna()
] = np.nan


# ------------------------------------------------------------
# 8. Create expanding past mean demand
# ------------------------------------------------------------

step2_expanding_feature_df[
    "ExpandingPastMeanDemand"
] = (
    previous_demand
    .groupby(
        product_group_key,
        sort=False
    )
    .transform(
        lambda series:
        series
        .expanding(
            min_periods=EXPANDING_MIN_PERIODS
        )
        .mean()
    )
)


# ------------------------------------------------------------
# 9. Create expanding past positive-demand rate
# ------------------------------------------------------------

step2_expanding_feature_df[
    "ExpandingPastPositiveDemandRate"
] = (
    previous_positive_indicator
    .groupby(
        product_group_key,
        sort=False
    )
    .transform(
        lambda series:
        series
        .expanding(
            min_periods=EXPANDING_MIN_PERIODS
        )
        .mean()
    )
)


# ------------------------------------------------------------
# 10. Validate the completed structure
# ------------------------------------------------------------

assert step2_expanding_feature_df.shape == (
    43_774,
    60
), (
    "Expected 58 Part 4 columns plus 2 expanding features.\n"
    f"Found: {step2_expanding_feature_df.shape}"
)

assert all(
    column in step2_expanding_feature_df.columns
    for column in expanding_feature_columns
)

assert step2_expanding_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_expanding_feature_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 11. Validate legitimate missingness
# ------------------------------------------------------------

first_product_row_mask = (
    step2_expanding_feature_df[
        "ProductAgeOperatingDays"
    ].eq(0)
)

expected_missing_rows = int(
    first_product_row_mask.sum()
)

assert expected_missing_rows == 227

expanding_missing_validation_records = []

for feature_name in expanding_feature_columns:

    actual_missing_mask = (
        step2_expanding_feature_df[
            feature_name
        ].isna()
    )

    unexpected_missing_rows = int(
        (
            actual_missing_mask
            & ~first_product_row_mask
        ).sum()
    )

    prematurely_populated_rows = int(
        (
            ~actual_missing_mask
            & first_product_row_mask
        ).sum()
    )

    actual_missing_rows = int(
        actual_missing_mask.sum()
    )

    assert unexpected_missing_rows == 0, (
        f"{feature_name} contains unexpected missing values."
    )

    assert prematurely_populated_rows == 0, (
        f"{feature_name} is populated on a product's "
        "first row."
    )

    assert actual_missing_rows == expected_missing_rows

    expanding_missing_validation_records.append({
        "FeatureName": feature_name,
        "ExpectedMissingRows": expected_missing_rows,
        "ActualMissingRows": actual_missing_rows,
        "UnexpectedMissingRows":
            unexpected_missing_rows,
        "PrematurelyPopulatedRows":
            prematurely_populated_rows,
        "PopulatedRows": (
            len(step2_expanding_feature_df)
            - actual_missing_rows
        )
    })


expanding_missing_validation_df = pd.DataFrame(
    expanding_missing_validation_records
)


# ------------------------------------------------------------
# 12. Create the expanding-feature contract
# ------------------------------------------------------------

expanding_feature_contract_df = pd.DataFrame([
    {
        "FeatureName":
            "ExpandingPastMeanDemand",
        "FeatureFamily":
            "EXPANDING_PAST_MEAN",
        "SourceColumn":
            "TotalDemand",
        "GroupKey":
            "CanonicalProductID",
        "TimeIndex":
            "OperatingDaySequence",
        "ShiftBeforeCalculation":
            1,
        "MinimumPeriods":
            EXPANDING_MIN_PERIODS,
        "GenerationRule":
            "SHIFT_1_THEN_EXPANDING_MEAN",
        "UsesCurrentRowTarget":
            False,
        "UsesFutureTarget":
            False,
        "FirstProductRowMissing":
            True,
        "MissingValueFilledInPart5":
            False
    },
    {
        "FeatureName":
            "ExpandingPastPositiveDemandRate",
        "FeatureFamily":
            "EXPANDING_PAST_POSITIVE_RATE",
        "SourceColumn":
            "TotalDemand",
        "GroupKey":
            "CanonicalProductID",
        "TimeIndex":
            "OperatingDaySequence",
        "ShiftBeforeCalculation":
            1,
        "MinimumPeriods":
            EXPANDING_MIN_PERIODS,
        "GenerationRule":
            (
                "SHIFT_1_COMPARE_POSITIVE_"
                "THEN_EXPANDING_MEAN"
            ),
        "UsesCurrentRowTarget":
            False,
        "UsesFutureTarget":
            False,
        "FirstProductRowMissing":
            True,
        "MissingValueFilledInPart5":
            False
    }
])

assert len(expanding_feature_contract_df) == 2

assert expanding_feature_contract_df[
    "FeatureName"
].nunique() == 2

assert not expanding_feature_contract_df[
    "UsesCurrentRowTarget"
].any()

assert not expanding_feature_contract_df[
    "UsesFutureTarget"
].any()


# ------------------------------------------------------------
# 13. Print Cell 36 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 5 — "
    "PAST-ONLY EXPANDING FEATURES CREATED"
)
print("=" * 75)

print()
print("Expanding-feature dataset:")
print(
    f"Rows: "
    f"{len(step2_expanding_feature_df):,}"
)
print(
    "Columns:",
    step2_expanding_feature_df.shape[1]
)
print(
    "Canonical products:",
    step2_expanding_feature_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step2_expanding_feature_df[
        "Date"
    ].nunique()
)

print()
print("Features created:")
for feature_name in expanding_feature_columns:
    print(f"- {feature_name}")

print()
print("Missing-value validation:")
display(expanding_missing_validation_df)

print()
print("Cell 36 completed successfully.")

STEP 2, PART 5 — PAST-ONLY EXPANDING FEATURES CREATED

Expanding-feature dataset:
Rows: 43,774
Columns: 60
Canonical products: 227
Operating dates: 245

Features created:
- ExpandingPastMeanDemand
- ExpandingPastPositiveDemandRate

Missing-value validation:


,FeatureName,ExpectedMissingRows,ActualMissingRows,UnexpectedMissingRows,PrematurelyPopulatedRows,PopulatedRows
0,ExpandingPastMeanDemand,227,227,0,0,43547
1,ExpandingPastPositiveDemandRate,227,227,0,0,43547



Cell 36 completed successfully.


In [48]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 5
# Cell 37: Independently validate expanding feature values
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 36 objects exist
# ------------------------------------------------------------

required_cell_36_objects = [
    "step2_expanding_feature_df",
    "expanding_feature_contract_df",
    "expanding_missing_validation_df",
    "expanding_feature_columns"
]

missing_cell_36_objects = [
    object_name
    for object_name in required_cell_36_objects
    if object_name not in globals()
]

if missing_cell_36_objects:
    raise NameError(
        "The following Cell 36 objects are missing:\n"
        f"{missing_cell_36_objects}\n\n"
        "Run Cell 36 before running Cell 37."
    )


# ------------------------------------------------------------
# 2. Create a strictly ordered validation copy
# ------------------------------------------------------------

expanding_validation_df = (
    step2_expanding_feature_df
    .sort_values(
        [
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

row_count = len(expanding_validation_df)

demand_array = (
    expanding_validation_df[
        "TotalDemand"
    ]
    .astype(float)
    .to_numpy()
)

product_age_array = (
    expanding_validation_df[
        "ProductAgeOperatingDays"
    ]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 3. Create independent expected-value arrays
# ------------------------------------------------------------

expected_expanding_mean = np.full(
    row_count,
    np.nan,
    dtype=float
)

expected_expanding_positive_rate = np.full(
    row_count,
    np.nan,
    dtype=float
)


# ------------------------------------------------------------
# 4. Explicitly calculate values from prior rows only
# ------------------------------------------------------------

for (
    canonical_product_id,
    product_index
) in expanding_validation_df.groupby(
    "CanonicalProductID",
    sort=False
).groups.items():

    product_index = np.asarray(
        list(product_index),
        dtype=int
    )

    product_demand = (
        demand_array[
            product_index
        ]
    )

    for local_position, global_index in enumerate(
        product_index
    ):

        prior_demand = product_demand[
            :local_position
        ]

        if prior_demand.size == 0:
            continue

        expected_expanding_mean[
            global_index
        ] = float(
            np.mean(prior_demand)
        )

        expected_expanding_positive_rate[
            global_index
        ] = float(
            np.mean(prior_demand > 0)
        )


# ------------------------------------------------------------
# 5. Compare actual and expected feature values
# ------------------------------------------------------------

feature_expectations = [
    (
        "ExpandingPastMeanDemand",
        expected_expanding_mean
    ),
    (
        "ExpandingPastPositiveDemandRate",
        expected_expanding_positive_rate
    )
]

expanding_value_validation_records = []

for (
    feature_name,
    expected_values
) in feature_expectations:

    actual_values = (
        expanding_validation_df[
            feature_name
        ]
        .astype(float)
        .to_numpy()
    )

    value_match_mask = np.isclose(
        actual_values,
        expected_values,
        rtol=1e-10,
        atol=1e-12,
        equal_nan=True
    )

    mismatch_rows = int(
        (~value_match_mask).sum()
    )

    missing_rows = int(
        np.isnan(actual_values).sum()
    )

    compared_rows = int(
        (~np.isnan(actual_values)).sum()
    )

    prematurely_populated_rows = int(
        (
            ~np.isnan(actual_values)
            & (product_age_array == 0)
        ).sum()
    )

    available_product_ages = (
        product_age_array[
            ~np.isnan(actual_values)
        ]
    )

    earliest_available_product_age = int(
        available_product_ages.min()
    )

    assert mismatch_rows == 0, (
        f"{feature_name} differs from independently "
        "calculated past-only values."
    )

    assert prematurely_populated_rows == 0

    assert earliest_available_product_age == 1

    expanding_value_validation_records.append({
        "FeatureName": feature_name,
        "ComparedRows": compared_rows,
        "MissingRows": missing_rows,
        "MismatchRows": mismatch_rows,
        "PrematurelyPopulatedRows":
            prematurely_populated_rows,
        "EarliestAvailableProductAge":
            earliest_available_product_age,
        "CurrentRowExcluded": True,
        "ValidationStatus": "PASSED"
    })


expanding_value_validation_df = pd.DataFrame(
    expanding_value_validation_records
)


# ------------------------------------------------------------
# 6. Validate mean against cumulative prior demand
# ------------------------------------------------------------

product_group = (
    expanding_validation_df[
        "CanonicalProductID"
    ]
)

prior_cumulative_demand = (
    expanding_validation_df
    .groupby(
        "CanonicalProductID",
        sort=False
    )["TotalDemand"]
    .cumsum()
    - expanding_validation_df[
        "TotalDemand"
    ]
)

prior_positive_count = (
    expanding_validation_df[
        "TotalDemand"
    ]
    .gt(0)
    .astype(int)
    .groupby(
        product_group,
        sort=False
    )
    .cumsum()
    - expanding_validation_df[
        "TotalDemand"
    ].gt(0).astype(int)
)

available_history_count = (
    expanding_validation_df[
        "ProductAgeOperatingDays"
    ].astype(int)
)

history_available_mask = (
    available_history_count > 0
)

implied_prior_demand_sum = (
    expanding_validation_df[
        "ExpandingPastMeanDemand"
    ]
    * available_history_count
)

implied_prior_positive_count = (
    expanding_validation_df[
        "ExpandingPastPositiveDemandRate"
    ]
    * available_history_count
)

mean_cumulative_sum_mismatches = int(
    (
        ~np.isclose(
            implied_prior_demand_sum.loc[
                history_available_mask
            ],
            prior_cumulative_demand.loc[
                history_available_mask
            ],
            rtol=1e-10,
            atol=1e-12
        )
    ).sum()
)

positive_rate_count_mismatches = int(
    (
        ~np.isclose(
            implied_prior_positive_count.loc[
                history_available_mask
            ],
            prior_positive_count.loc[
                history_available_mask
            ],
            rtol=1e-10,
            atol=1e-12
        )
    ).sum()
)

assert mean_cumulative_sum_mismatches == 0

assert positive_rate_count_mismatches == 0


expanding_consistency_df = pd.DataFrame([
    {
        "ConsistencyCheck":
            "MEAN_TIMES_HISTORY_EQUALS_PRIOR_DEMAND_SUM",
        "ComparedRows":
            int(history_available_mask.sum()),
        "MismatchRows":
            mean_cumulative_sum_mismatches,
        "ValidationStatus":
            "PASSED"
    },
    {
        "ConsistencyCheck":
            "POSITIVE_RATE_TIMES_HISTORY_EQUALS_PRIOR_POSITIVE_COUNT",
        "ComparedRows":
            int(history_available_mask.sum()),
        "MismatchRows":
            positive_rate_count_mismatches,
        "ValidationStatus":
            "PASSED"
    }
])


# ------------------------------------------------------------
# 7. Validate feature ranges
# ------------------------------------------------------------

assert (
    expanding_validation_df[
        "ExpandingPastMeanDemand"
    ]
    .dropna()
    .ge(0)
    .all()
)

assert (
    expanding_validation_df[
        "ExpandingPastPositiveDemandRate"
    ]
    .dropna()
    .between(0, 1)
    .all()
)


# ------------------------------------------------------------
# 8. Create expanding-feature readiness summary
# ------------------------------------------------------------

available_expanding_feature_count = (
    expanding_validation_df[
        expanding_feature_columns
    ]
    .notna()
    .sum(axis=1)
    .astype(int)
)

expanding_readiness_summary_df = (
    available_expanding_feature_count
    .value_counts()
    .sort_index()
    .rename_axis(
        "AvailableExpandingFeatureCount"
    )
    .reset_index(name="RowCount")
)

expanding_readiness_summary_df[
    "RowPercentage"
] = (
    expanding_readiness_summary_df[
        "RowCount"
    ]
    / len(expanding_validation_df)
    * 100
).round(2)

rows_with_all_expanding_features = int(
    (
        available_expanding_feature_count
        == len(expanding_feature_columns)
    ).sum()
)

rows_without_expanding_history = int(
    (
        available_expanding_feature_count
        == 0
    ).sum()
)

rows_with_partial_expanding_features = int(
    (
        available_expanding_feature_count
        .between(
            1,
            len(expanding_feature_columns) - 1
        )
    ).sum()
)

assert rows_without_expanding_history == 227

assert rows_with_partial_expanding_features == 0

assert rows_with_all_expanding_features == (
    43_774 - 227
)


# ------------------------------------------------------------
# 9. Final integrity and leakage assertions
# ------------------------------------------------------------

assert len(expanding_value_validation_df) == 2

assert expanding_value_validation_df[
    "MismatchRows"
].sum() == 0

assert expanding_value_validation_df[
    "PrematurelyPopulatedRows"
].sum() == 0

assert expanding_value_validation_df[
    "CurrentRowExcluded"
].all()

assert expanding_consistency_df[
    "MismatchRows"
].sum() == 0

assert expanding_validation_df.shape == (
    43_774,
    60
)

assert expanding_validation_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert expanding_validation_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 10. Print Cell 37 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 5 — "
    "EXPANDING FEATURE VALIDATION PASSED"
)
print("=" * 75)

print()
print("Independent feature validation:")
display(expanding_value_validation_df)

print()
print("Expanding-feature consistency:")
display(expanding_consistency_df)

print()
print("Feature readiness:")
display(expanding_readiness_summary_df)

print()
print("Overall readiness:")
print(
    "Rows with both expanding features:",
    f"{rows_with_all_expanding_features:,}"
)
print(
    "Rows with no prior expanding history:",
    f"{rows_without_expanding_history:,}"
)
print(
    "Rows with partial expanding features:",
    f"{rows_with_partial_expanding_features:,}"
)

print()
print("Leakage validation:")
print(
    "Feature-value mismatches:",
    int(
        expanding_value_validation_df[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Consistency mismatches:",
    int(
        expanding_consistency_df[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Prematurely populated rows:",
    int(
        expanding_value_validation_df[
            "PrematurelyPopulatedRows"
        ].sum()
    )
)
print(
    "Current-row TotalDemand included:",
    False
)

print()
print("Cell 37 completed successfully.")

STEP 2, PART 5 — EXPANDING FEATURE VALIDATION PASSED

Independent feature validation:


,FeatureName,ComparedRows,MissingRows,MismatchRows,PrematurelyPopulatedRows,EarliestAvailableProductAge,CurrentRowExcluded,ValidationStatus
0,ExpandingPastMeanDemand,43547,227,0,0,1,True,PASSED
1,ExpandingPastPositiveDemandRate,43547,227,0,0,1,True,PASSED



Expanding-feature consistency:


,ConsistencyCheck,ComparedRows,MismatchRows,ValidationStatus
0,MEAN_TIMES_HISTORY_EQUALS_PRIOR_DEMAND_SUM,43547,0,PASSED
1,POSITIVE_RATE_TIMES_HISTORY_EQUALS_PRIOR_POSIT...,43547,0,PASSED



Feature readiness:


,AvailableExpandingFeatureCount,RowCount,RowPercentage
0,0,227,0.52
1,2,43547,99.48



Overall readiness:
Rows with both expanding features: 43,547
Rows with no prior expanding history: 227
Rows with partial expanding features: 0

Leakage validation:
Feature-value mismatches: 0
Consistency mismatches: 0
Prematurely populated rows: 0
Current-row TotalDemand included: False

Cell 37 completed successfully.


In [49]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 5
# Cell 38: Save expanding-feature outputs and update handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 5 objects exist
# ------------------------------------------------------------

required_step2_part5_objects = [
    "step2_expanding_feature_df",
    "expanding_feature_contract_df",
    "expanding_missing_validation_df",
    "expanding_value_validation_df",
    "expanding_consistency_df",
    "expanding_readiness_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step2_part5_objects = [
    object_name
    for object_name in required_step2_part5_objects
    if object_name not in globals()
]

if missing_step2_part5_objects:
    raise NameError(
        "The following Step 2 Part 5 objects are missing:\n"
        f"{missing_step2_part5_objects}\n\n"
        "Run Cells 36 and 37 before running Cell 38."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Part 5 output paths
# ------------------------------------------------------------

STEP2_EXPANDING_FEATURE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_features.csv"
)

STEP2_EXPANDING_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_feature_contract.csv"
)

STEP2_EXPANDING_MISSING_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_missing_audit.csv"
)

STEP2_EXPANDING_VALUE_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_value_validation.csv"
)

STEP2_EXPANDING_CONSISTENCY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_consistency_audit.csv"
)

STEP2_EXPANDING_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_readiness_summary.csv"
)


# ------------------------------------------------------------
# 4. Prepare official output ordering
# ------------------------------------------------------------

expanding_feature_output_df = (
    step2_expanding_feature_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Save all Part 5 outputs
# ------------------------------------------------------------

expanding_feature_output_df.to_csv(
    STEP2_EXPANDING_FEATURE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

expanding_feature_contract_df.to_csv(
    STEP2_EXPANDING_CONTRACT_OUTPUT,
    index=False
)

expanding_missing_validation_df.to_csv(
    STEP2_EXPANDING_MISSING_AUDIT_OUTPUT,
    index=False
)

expanding_value_validation_df.to_csv(
    STEP2_EXPANDING_VALUE_VALIDATION_OUTPUT,
    index=False
)

expanding_consistency_df.to_csv(
    STEP2_EXPANDING_CONSISTENCY_OUTPUT,
    index=False
)

expanding_readiness_summary_df.to_csv(
    STEP2_EXPANDING_READINESS_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 6. Reload saved outputs
# ------------------------------------------------------------

saved_expanding_features = pd.read_csv(
    STEP2_EXPANDING_FEATURE_OUTPUT,
    low_memory=False
)

saved_expanding_contract = pd.read_csv(
    STEP2_EXPANDING_CONTRACT_OUTPUT,
    low_memory=False
)

saved_expanding_missing = pd.read_csv(
    STEP2_EXPANDING_MISSING_AUDIT_OUTPUT,
    low_memory=False
)

saved_expanding_validation = pd.read_csv(
    STEP2_EXPANDING_VALUE_VALIDATION_OUTPUT,
    low_memory=False
)

saved_expanding_consistency = pd.read_csv(
    STEP2_EXPANDING_CONSISTENCY_OUTPUT,
    low_memory=False
)

saved_expanding_readiness = pd.read_csv(
    STEP2_EXPANDING_READINESS_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 7. Validate saved outputs
# ------------------------------------------------------------

assert saved_expanding_features.shape == (
    43_774,
    60
)

assert saved_expanding_features[
    "CanonicalProductID"
].nunique() == 227

assert saved_expanding_features[
    "Date"
].nunique() == 245

assert saved_expanding_features[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_expanding_features[
    "TotalDemand"
].sum() == 116_158

assert len(saved_expanding_contract) == 2

assert saved_expanding_contract[
    "FeatureName"
].nunique() == 2

assert len(saved_expanding_missing) == 2

assert saved_expanding_missing[
    "UnexpectedMissingRows"
].sum() == 0

assert saved_expanding_missing[
    "PrematurelyPopulatedRows"
].sum() == 0

assert len(saved_expanding_validation) == 2

assert saved_expanding_validation[
    "MismatchRows"
].sum() == 0

assert saved_expanding_validation[
    "PrematurelyPopulatedRows"
].sum() == 0

assert saved_expanding_consistency[
    "MismatchRows"
].sum() == 0

assert saved_expanding_readiness[
    "RowCount"
].sum() == 43_774


# ------------------------------------------------------------
# 8. Confirm legitimate first-row missingness
# ------------------------------------------------------------

for feature_name in expanding_feature_columns:

    assert (
        saved_expanding_features[
            feature_name
        ].isna().sum()
        == 227
    ), (
        f"{feature_name} should contain exactly 227 "
        "first-product-row missing values."
    )


# ------------------------------------------------------------
# 9. Build Markdown feature summary
# ------------------------------------------------------------

expanding_markdown_rows = "\n".join(
    (
        f"- `{row.FeatureName}`: "
        f"{int(row.ComparedRows):,} populated rows; "
        f"{int(row.MissingRows):,} legitimate missing rows; "
        f"{int(row.MismatchRows)} mismatches"
    )
    for row in (
        expanding_value_validation_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 10. Update the Markdown handoff
# ------------------------------------------------------------

step2_part5_summary = f"""
**Status:** Completed and validated

### Purpose

Step 2 Part 5 created long-term expanding historical features
using every prior operating-day observation available for the
same canonical product.

### Features created

{expanding_markdown_rows}

### Calculation policy

- Source column: `TotalDemand`
- Product grouping key: `CanonicalProductID`
- Time index: `OperatingDaySequence`
- Shift before expanding calculation: 1 operating row
- Minimum periods: 1
- Current-row target used: no
- Future target used: no

`ExpandingPastMeanDemand` represents the average demand over every
previous operating date for the same product.

`ExpandingPastPositiveDemandRate` represents the proportion of
all previous operating dates on which the product had positive
demand.

### Validation

- Independent feature-value mismatches: {int(expanding_value_validation_df["MismatchRows"].sum())}
- Consistency mismatches: {int(expanding_consistency_df["MismatchRows"].sum())}
- Prematurely populated rows: {int(expanding_value_validation_df["PrematurelyPopulatedRows"].sum())}
- Rows with both expanding features: {rows_with_all_expanding_features:,}
- Rows with no previous expanding history: {rows_without_expanding_history:,}
- Rows with partial features: {rows_with_partial_expanding_features:,}

### Dataset structure

- Rows: {len(saved_expanding_features):,}
- Columns: {saved_expanding_features.shape[1]}
- Canonical products: {saved_expanding_features["CanonicalProductID"].nunique()}
- Operating dates: {saved_expanding_features["Date"].nunique()}
- Duplicate product-date rows: 0
- Total demand units: {int(saved_expanding_features["TotalDemand"].sum()):,}

### Missing-value interpretation

Each product's first row has no earlier demand history. Both
expanding features therefore contain exactly 227 legitimate
missing values. These values were not filled in Part 5.

### Saved Step 2 Part 5 outputs

- `11_step2_expanding_history_features.csv`
- `11_step2_expanding_history_feature_contract.csv`
- `11_step2_expanding_history_missing_audit.csv`
- `11_step2_expanding_history_value_validation.csv`
- `11_step2_expanding_history_consistency_audit.csv`
- `11_step2_expanding_history_readiness_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_2_part_5",
    section_title=(
        "Forecasting Preparation — Step 2, Part 5"
    ),
    section_body=step2_part5_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 11. Print final Part 5 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 2, PART 5 COMPLETED"
)
print("=" * 75)

print()
print("Expanding-feature dataset:")
print(
    f"Rows saved: "
    f"{len(saved_expanding_features):,}"
)
print(
    "Columns saved:",
    saved_expanding_features.shape[1]
)
print(
    "Canonical products:",
    saved_expanding_features[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_expanding_features[
        "Date"
    ].nunique()
)

print()
print("Features created:")
print("- ExpandingPastMeanDemand")
print("- ExpandingPastPositiveDemandRate")

print()
print("Leakage and value validation:")
print(
    "Feature-value mismatches:",
    int(
        saved_expanding_validation[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Consistency mismatches:",
    int(
        saved_expanding_consistency[
            "MismatchRows"
        ].sum()
    )
)
print(
    "Prematurely populated rows:",
    int(
        saved_expanding_validation[
            "PrematurelyPopulatedRows"
        ].sum()
    )
)
print(
    "Current-row TotalDemand included:",
    False
)

print()
print("Feature readiness:")
print(
    "Rows with both expanding features:",
    f"{rows_with_all_expanding_features:,}"
)
print(
    "Rows with no prior expanding history:",
    f"{rows_without_expanding_history:,}"
)
print(
    "Rows with partial expanding features:",
    f"{rows_with_partial_expanding_features:,}"
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_expanding_features['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP2_EXPANDING_FEATURE_OUTPUT}")
print(f"2. {STEP2_EXPANDING_CONTRACT_OUTPUT}")
print(f"3. {STEP2_EXPANDING_MISSING_AUDIT_OUTPUT}")
print(f"4. {STEP2_EXPANDING_VALUE_VALIDATION_OUTPUT}")
print(f"5. {STEP2_EXPANDING_CONSISTENCY_OUTPUT}")
print(f"6. {STEP2_EXPANDING_READINESS_OUTPUT}")
print(f"7. {HANDOFF_FILE}")

print()
print(
    "All Step 2, Part 5 validation checks passed."
)

FORECASTING PREPARATION — STEP 2, PART 5 COMPLETED

Expanding-feature dataset:
Rows saved: 43,774
Columns saved: 60
Canonical products: 227
Operating dates: 245

Features created:
- ExpandingPastMeanDemand
- ExpandingPastPositiveDemandRate

Leakage and value validation:
Feature-value mismatches: 0
Consistency mismatches: 0
Prematurely populated rows: 0
Current-row TotalDemand included: False

Feature readiness:
Rows with both expanding features: 43,547
Rows with no prior expanding history: 227
Rows with partial expanding features: 0

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/11_step2_expanding_history_features.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/11_step2_expanding_history_feature_contract.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/11_step2_expanding_history_missing_audit.csv
4. /Users/ryansmac/Desktop/Meng Project/ed

In [50]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 6
# Cell 39: Consolidate all historical features and create
# the final Step 2 model feature view
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the earlier forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define the official Step 2 input files
# ------------------------------------------------------------

STEP2_EXPANDING_FEATURE_FILE = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_features.csv"
)

STEP2_ORIGINAL_SOURCE_FILE = (
    FORECAST_PREPARATION_DIR
    / "07_step2_feature_engineering_source_panel.csv"
)

required_step2_part6_input_files = [
    STEP2_EXPANDING_FEATURE_FILE,
    STEP2_ORIGINAL_SOURCE_FILE
]

missing_step2_part6_input_files = [
    file_path
    for file_path in required_step2_part6_input_files
    if not file_path.exists()
]

if missing_step2_part6_input_files:
    raise FileNotFoundError(
        "The following required Step 2 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_step2_part6_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load the completed feature panel and original source
# ------------------------------------------------------------

step2_historical_full_df = pd.read_csv(
    STEP2_EXPANDING_FEATURE_FILE,
    low_memory=False
)

step2_original_source_df = pd.read_csv(
    STEP2_ORIGINAL_SOURCE_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Parse date columns consistently
# ------------------------------------------------------------

date_columns = [
    "Date",
    "ProductFirstObservedDate"
]

for dataframe in [
    step2_historical_full_df,
    step2_original_source_df
]:

    for column in date_columns:

        dataframe[column] = pd.to_datetime(
            dataframe[column],
            format="%Y-%m-%d",
            errors="coerce"
        )

        assert dataframe[column].isna().sum() == 0


# ------------------------------------------------------------
# 5. Validate the completed Step 2 panel
# ------------------------------------------------------------

assert step2_historical_full_df.shape == (
    43_774,
    60
), (
    "Unexpected completed Step 2 panel dimensions.\n"
    f"Found: {step2_historical_full_df.shape}"
)

assert step2_original_source_df.shape == (
    43_774,
    27
)

assert step2_historical_full_df[
    "CanonicalProductID"
].nunique() == 227

assert step2_historical_full_df[
    "Date"
].nunique() == 245

assert step2_historical_full_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_historical_full_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 6. Define all approved historical feature groups
# ------------------------------------------------------------

demand_lag_feature_columns = [
    "TotalDemandLag_1",
    "TotalDemandLag_2",
    "TotalDemandLag_3",
    "TotalDemandLag_5",
    "TotalDemandLag_10",
    "TotalDemandLag_20"
]

rolling_feature_columns = []

for window in [
    3,
    5,
    10,
    20
]:

    rolling_feature_columns.extend([
        f"PastDemandRollingMean_{window}",
        f"PastDemandRollingMedian_{window}",
        f"PastDemandRollingStd_{window}",
        f"PastDemandRollingSum_{window}"
    ])


occurrence_recency_feature_columns = [
    "PastZeroDemandRate_5",
    "PastZeroDemandRate_10",
    "PastZeroDemandRate_20",
    "PastPositiveDemandCount_5",
    "PastPositiveDemandCount_10",
    "PastPositiveDemandCount_20",
    "OperatingDaysSincePreviousPositiveDemand"
]

expanding_feature_columns = [
    "ExpandingPastMeanDemand",
    "ExpandingPastPositiveDemandRate"
]

all_historical_feature_columns = (
    demand_lag_feature_columns
    + rolling_feature_columns
    + occurrence_recency_feature_columns
    + expanding_feature_columns
)

assert len(demand_lag_feature_columns) == 6
assert len(rolling_feature_columns) == 16
assert len(occurrence_recency_feature_columns) == 7
assert len(expanding_feature_columns) == 2

assert len(all_historical_feature_columns) == 31
assert len(set(all_historical_feature_columns)) == 31


# ------------------------------------------------------------
# 7. Confirm every historical feature exists
# ------------------------------------------------------------

missing_historical_feature_columns = [
    column
    for column in all_historical_feature_columns
    if column not in step2_historical_full_df.columns
]

assert not missing_historical_feature_columns, (
    "The following historical features are missing:\n"
    f"{missing_historical_feature_columns}"
)


# ------------------------------------------------------------
# 8. Identify audit-only columns
# ------------------------------------------------------------

step2_audit_only_columns = [
    "AvailableLagFeatureCount",
    "AllApprovedDemandLagsAvailable"
]

missing_audit_columns = [
    column
    for column in step2_audit_only_columns
    if column not in step2_historical_full_df.columns
]

assert not missing_audit_columns, (
    "The following expected lag-readiness audit columns "
    "are missing:\n"
    f"{missing_audit_columns}"
)


# ------------------------------------------------------------
# 9. Create the clean model feature view
# ------------------------------------------------------------

step2_model_feature_columns = [
    column
    for column in step2_historical_full_df.columns
    if column not in step2_audit_only_columns
]

step2_historical_model_df = (
    step2_historical_full_df[
        step2_model_feature_columns
    ]
    .copy()
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert step2_historical_model_df.shape == (
    43_774,
    58
), (
    "Expected a 43,774-row, 58-column model feature view.\n"
    f"Found: {step2_historical_model_df.shape}"
)

assert all(
    column not in step2_historical_model_df.columns
    for column in step2_audit_only_columns
)


# ------------------------------------------------------------
# 10. Confirm the original 27 columns were preserved unchanged
# ------------------------------------------------------------

original_source_columns = (
    step2_original_source_df.columns.tolist()
)

assert all(
    column in step2_historical_model_df.columns
    for column in original_source_columns
)

comparison_sort_columns = [
    "CanonicalProductID",
    "OperatingDaySequence",
    "Date"
]

original_source_sorted = (
    step2_original_source_df
    .sort_values(
        comparison_sort_columns,
        kind="stable"
    )
    .reset_index(drop=True)
)

final_source_columns_sorted = (
    step2_historical_model_df[
        original_source_columns
    ]
    .sort_values(
        comparison_sort_columns,
        kind="stable"
    )
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    original_source_sorted,
    final_source_columns_sorted,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 11. Confirm leakage columns remain absent
# ------------------------------------------------------------

same_day_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

same_day_leakage_columns_found = sorted(
    same_day_leakage_columns.intersection(
        step2_historical_model_df.columns
    )
)

assert len(
    same_day_leakage_columns_found
) == 0


# ------------------------------------------------------------
# 12. Build historical-feature missingness audit
# ------------------------------------------------------------

def count_missing_after_first_available(
    dataframe,
    feature_name
):
    """
    Count missing values that occur after a feature first becomes
    available within a product series.
    """

    unexpected_missing_count = 0

    for (
        canonical_product_id,
        product_group
    ) in dataframe.groupby(
        "CanonicalProductID",
        sort=False
    ):

        product_group = (
            product_group
            .sort_values(
                "OperatingDaySequence",
                kind="stable"
            )
        )

        available_values = (
            product_group[
                feature_name
            ]
            .notna()
            .to_numpy()
        )

        if available_values.any():

            first_available_position = int(
                np.argmax(
                    available_values
                )
            )

            unexpected_missing_count += int(
                (
                    ~available_values[
                        first_available_position:
                    ]
                ).sum()
            )

    return unexpected_missing_count


historical_missingness_records = []

for feature_name in all_historical_feature_columns:

    feature_series = (
        step2_historical_model_df[
            feature_name
        ]
    )

    missing_rows = int(
        feature_series.isna().sum()
    )

    populated_rows = int(
        feature_series.notna().sum()
    )

    available_product_ages = (
        step2_historical_model_df.loc[
            feature_series.notna(),
            "ProductAgeOperatingDays"
        ]
    )

    if len(available_product_ages) > 0:

        first_available_product_age = int(
            available_product_ages.min()
        )

    else:

        first_available_product_age = np.nan

    unexpected_missing_after_available = (
        count_missing_after_first_available(
            step2_historical_model_df,
            feature_name
        )
    )

    assert unexpected_missing_after_available == 0, (
        f"{feature_name} contains missing values after it "
        "became available within a product series."
    )

    historical_missingness_records.append({
        "FeatureName":
            feature_name,

        "FeatureGroup":
            (
                "DEMAND_LAG"
                if feature_name
                in demand_lag_feature_columns

                else "ROLLING_STATISTIC"
                if feature_name
                in rolling_feature_columns

                else "OCCURRENCE_OR_RECENCY"
                if feature_name
                in occurrence_recency_feature_columns

                else "EXPANDING_HISTORY"
            ),

        "MissingRows":
            missing_rows,

        "PopulatedRows":
            populated_rows,

        "MissingPercentage":
            round(
                missing_rows
                / len(
                    step2_historical_model_df
                )
                * 100,
                4
            ),

        "FirstAvailableProductAge":
            first_available_product_age,

        "UnexpectedMissingAfterFirstAvailable":
            unexpected_missing_after_available,

        "EarlyHistoryMissingExpected":
            True,

        "MissingValuesFilled":
            False
    })


historical_feature_missingness_df = pd.DataFrame(
    historical_missingness_records
)

assert len(
    historical_feature_missingness_df
) == 31

assert historical_feature_missingness_df[
    "UnexpectedMissingAfterFirstAvailable"
].sum() == 0

assert not historical_feature_missingness_df[
    "MissingValuesFilled"
].any()


# ------------------------------------------------------------
# 13. Create row-level historical-feature readiness summary
# ------------------------------------------------------------

historical_feature_available_count = (
    step2_historical_model_df[
        all_historical_feature_columns
    ]
    .notna()
    .sum(axis=1)
    .astype(int)
)

all_historical_features_available_mask = (
    historical_feature_available_count
    == len(
        all_historical_feature_columns
    )
)

no_historical_features_available_mask = (
    historical_feature_available_count
    == 0
)

partial_historical_features_available_mask = (
    historical_feature_available_count
    .between(
        1,
        len(
            all_historical_feature_columns
        ) - 1
    )
)

rows_with_all_historical_features = int(
    all_historical_features_available_mask.sum()
)

rows_with_no_historical_features = int(
    no_historical_features_available_mask.sum()
)

rows_with_partial_historical_features = int(
    partial_historical_features_available_mask.sum()
)

assert (
    rows_with_all_historical_features
    + rows_with_no_historical_features
    + rows_with_partial_historical_features
    == 43_774
)


historical_feature_readiness_summary_df = (
    historical_feature_available_count
    .value_counts()
    .sort_index()
    .rename_axis(
        "AvailableHistoricalFeatureCount"
    )
    .reset_index(name="RowCount")
)

historical_feature_readiness_summary_df[
    "RowPercentage"
] = (
    historical_feature_readiness_summary_df[
        "RowCount"
    ]
    / len(step2_historical_model_df)
    * 100
).round(2)


# ------------------------------------------------------------
# 14. Validate readiness against lag-20 availability
# ------------------------------------------------------------

def parse_boolean_series(series):
    """
    Convert common CSV boolean representations to Boolean.
    """

    if pd.api.types.is_bool_dtype(series):

        return series.astype(bool)

    normalised_values = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    converted_values = normalised_values.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if converted_values.isna().sum() > 0:

        invalid_values = (
            series.loc[
                converted_values.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unrecognised Boolean values were found:\n"
            f"{invalid_values}"
        )

    return converted_values.astype(bool)


approved_lag_readiness = parse_boolean_series(
    step2_historical_full_df[
        "AllApprovedDemandLagsAvailable"
    ]
)

lag_readiness_mismatches = int(
    (
        approved_lag_readiness
        != all_historical_features_available_mask
    ).sum()
)

assert lag_readiness_mismatches == 0, (
    "Complete historical-feature readiness does not match "
    "the approved lag-readiness indicator."
)


# ------------------------------------------------------------
# 15. Final structure and demand assertions
# ------------------------------------------------------------

assert step2_historical_model_df[
    "CanonicalProductID"
].nunique() == 227

assert step2_historical_model_df[
    "Date"
].nunique() == 245

assert step2_historical_model_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step2_historical_model_df[
    "TotalDemand"
].sum() == 116_158

assert len(
    all_historical_feature_columns
) == 31


# ------------------------------------------------------------
# 16. Print Cell 39 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 6 — "
    "HISTORICAL FEATURES CONSOLIDATED"
)
print("=" * 75)

print()
print("Full Step 2 audit panel:")
print(
    f"Rows: "
    f"{len(step2_historical_full_df):,}"
)
print(
    "Columns:",
    step2_historical_full_df.shape[1]
)

print()
print("Step 2 model feature view:")
print(
    f"Rows: "
    f"{len(step2_historical_model_df):,}"
)
print(
    "Columns:",
    step2_historical_model_df.shape[1]
)
print(
    "Canonical products:",
    step2_historical_model_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step2_historical_model_df[
        "Date"
    ].nunique()
)

print()
print("Historical feature groups:")
print(
    "Demand lag features:",
    len(demand_lag_feature_columns)
)
print(
    "Rolling statistic features:",
    len(rolling_feature_columns)
)
print(
    "Occurrence and recency features:",
    len(
        occurrence_recency_feature_columns
    )
)
print(
    "Expanding history features:",
    len(expanding_feature_columns)
)
print(
    "Total historical features:",
    len(all_historical_feature_columns)
)

print()
print("Historical feature readiness:")
print(
    "Rows with all 31 features:",
    f"{rows_with_all_historical_features:,}"
)
print(
    "Rows with partial features:",
    f"{rows_with_partial_historical_features:,}"
)
print(
    "Rows with no historical features:",
    f"{rows_with_no_historical_features:,}"
)

print()
print("Validation:")
print(
    "Unexpected missing values after availability:",
    int(
        historical_feature_missingness_df[
            "UnexpectedMissingAfterFirstAvailable"
        ].sum()
    )
)
print(
    "Lag-readiness mismatches:",
    lag_readiness_mismatches
)
print(
    "Same-day leakage columns found:",
    len(
        same_day_leakage_columns_found
    )
)
print(
    "Total demand units:",
    f"{step2_historical_model_df['TotalDemand'].sum():,}"
)

print()
print("Cell 39 completed successfully.")

STEP 2, PART 6 — HISTORICAL FEATURES CONSOLIDATED

Full Step 2 audit panel:
Rows: 43,774
Columns: 60

Step 2 model feature view:
Rows: 43,774
Columns: 58
Canonical products: 227
Operating dates: 245

Historical feature groups:
Demand lag features: 6
Rolling statistic features: 16
Occurrence and recency features: 7
Expanding history features: 2
Total historical features: 31

Historical feature readiness:
Rows with all 31 features: 39,234
Rows with partial features: 4,313
Rows with no historical features: 227

Validation:
Unexpected missing values after availability: 0
Lag-readiness mismatches: 0
Same-day leakage columns found: 0
Total demand units: 116,158

Cell 39 completed successfully.


In [51]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 6
# Cell 40: Create the combined Step 2 feature contract
# and final validation summary
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 39 objects exist
# ------------------------------------------------------------

required_cell_39_objects = [
    "step2_historical_full_df",
    "step2_historical_model_df",
    "all_historical_feature_columns",
    "historical_feature_missingness_df",
    "historical_feature_readiness_summary_df",
    "demand_lag_feature_columns",
    "rolling_feature_columns",
    "occurrence_recency_feature_columns",
    "expanding_feature_columns"
]

missing_cell_39_objects = [
    object_name
    for object_name in required_cell_39_objects
    if object_name not in globals()
]

if missing_cell_39_objects:
    raise NameError(
        "The following Cell 39 objects are missing:\n"
        f"{missing_cell_39_objects}\n\n"
        "Run Cell 39 before running Cell 40."
    )


# ------------------------------------------------------------
# 2. Define all feature-contract input files
# ------------------------------------------------------------

STEP2_BASE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "07_step2_feature_engineering_contract.csv"
)

STEP2_LAG_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "08_step2_demand_lag_feature_contract.csv"
)

STEP2_ROLLING_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "09_step2_demand_rolling_feature_contract.csv"
)

STEP2_OCCURRENCE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "10_step2_demand_occurrence_recency_contract.csv"
)

STEP2_EXPANDING_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "11_step2_expanding_history_feature_contract.csv"
)

contract_input_files = [
    STEP2_BASE_CONTRACT_FILE,
    STEP2_LAG_CONTRACT_FILE,
    STEP2_ROLLING_CONTRACT_FILE,
    STEP2_OCCURRENCE_CONTRACT_FILE,
    STEP2_EXPANDING_CONTRACT_FILE
]

missing_contract_files = [
    file_path
    for file_path in contract_input_files
    if not file_path.exists()
]

if missing_contract_files:
    raise FileNotFoundError(
        "The following feature-contract files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_contract_files
        )
    )


# ------------------------------------------------------------
# 3. Load all existing contracts
# ------------------------------------------------------------

base_contract_source_df = pd.read_csv(
    STEP2_BASE_CONTRACT_FILE,
    low_memory=False
)

lag_contract_source_df = pd.read_csv(
    STEP2_LAG_CONTRACT_FILE,
    low_memory=False
)

rolling_contract_source_df = pd.read_csv(
    STEP2_ROLLING_CONTRACT_FILE,
    low_memory=False
)

occurrence_contract_source_df = pd.read_csv(
    STEP2_OCCURRENCE_CONTRACT_FILE,
    low_memory=False
)

expanding_contract_source_df = pd.read_csv(
    STEP2_EXPANDING_CONTRACT_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Validate source contract sizes
# ------------------------------------------------------------

assert len(base_contract_source_df) == 27
assert len(lag_contract_source_df) == 6
assert len(rolling_contract_source_df) == 16
assert len(occurrence_contract_source_df) == 7
assert len(expanding_contract_source_df) == 2


# ------------------------------------------------------------
# 5. Parse Boolean fields from the base contract
# ------------------------------------------------------------

base_direct_input_allowed = parse_boolean_series(
    base_contract_source_df[
        "AllowedAsCurrentRowModelInput"
    ]
)

base_identifier_flag = parse_boolean_series(
    base_contract_source_df[
        "AllowedAsIdentifier"
    ]
)

base_support_flag = parse_boolean_series(
    base_contract_source_df[
        "AllowedAsSupportColumn"
    ]
)

base_target_flag = parse_boolean_series(
    base_contract_source_df[
        "AllowedAsForecastTarget"
    ]
)


# ------------------------------------------------------------
# 6. Standardise the base/current-date feature contract
# ------------------------------------------------------------

base_combined_contract_df = pd.DataFrame({
    "Column":
        base_contract_source_df["Column"],

    "ContractSection":
        "BASE_AND_CURRENT_DATE",

    "FeatureFamily":
        base_contract_source_df["Step2Role"],

    "SourceColumn":
        base_contract_source_df["Column"],

    "GroupKey":
        np.nan,

    "TimeIndex":
        np.nan,

    "LagOperatingDays":
        np.nan,

    "WindowOperatingDays":
        np.nan,

    "MinimumShiftOperatingRows":
        np.where(
            base_direct_input_allowed,
            0,
            np.nan
        ),

    "MinimumPeriods":
        np.nan,

    "GenerationRule":
        base_contract_source_df["LeakageRule"],

    "DirectModelInputAllowed":
        base_direct_input_allowed,

    "IsHistoricalDemandFeature":
        False,

    "IsIdentifier":
        base_identifier_flag,

    "IsSupportColumn":
        base_support_flag,

    "IsForecastTarget":
        base_target_flag,

    "UsesCurrentRowTarget":
        False,

    "UsesFutureTarget":
        False,

    "DataType":
        base_contract_source_df["DataType"],

    "MissingCount":
        base_contract_source_df["MissingCount"],

    "MissingPercentage":
        base_contract_source_df[
            "MissingPercentage"
        ],

    "MissingValuePolicy":
        np.select(
            [
                (
                    base_contract_source_df[
                        "Step2Role"
                    ]
                    == (
                        "CONDITIONAL_PRODUCT_"
                        "METADATA_FEATURE"
                    )
                )
                &
                (
                    base_contract_source_df[
                        "MissingCount"
                    ] > 0
                ),

                (
                    base_contract_source_df[
                        "MissingCount"
                    ] == 0
                )
            ],
            [
                (
                    "PRESERVE_NOT_APPLICABLE_NA_"
                    "UNTIL_MODEL_PIPELINE"
                ),
                "NO_MISSING_VALUE_ACTION_REQUIRED"
            ],
            default="REVIEW_BEFORE_MODEL_PIPELINE"
        ),

    "PredictionTimeStatus":
        np.where(
            base_target_flag,
            "OUTCOME_NOT_AVAILABLE_AT_PREDICTION_TIME",
            (
                "AVAILABLE_OR_DERIVABLE_"
                "BEFORE_PREDICTION_TIME"
            )
        )
})


# ------------------------------------------------------------
# 7. Helper for historical feature metadata
# ------------------------------------------------------------

def get_feature_missing_metadata(
    feature_name
):
    """
    Return missingness information for one historical feature.
    """

    matching_record = (
        historical_feature_missingness_df.loc[
            historical_feature_missingness_df[
                "FeatureName"
            ] == feature_name
        ]
    )

    assert len(matching_record) == 1

    record = matching_record.iloc[0]

    return (
        int(record["MissingRows"]),
        float(record["MissingPercentage"])
    )


def get_feature_dtype(
    feature_name
):
    return str(
        step2_historical_model_df[
            feature_name
        ].dtype
    )


# ------------------------------------------------------------
# 8. Standardise demand-lag feature contracts
# ------------------------------------------------------------

historical_contract_records = []

for row in lag_contract_source_df.itertuples():

    missing_count, missing_percentage = (
        get_feature_missing_metadata(
            row.FeatureName
        )
    )

    historical_contract_records.append({
        "Column":
            row.FeatureName,

        "ContractSection":
            "HISTORICAL_DEMAND",

        "FeatureFamily":
            "DEMAND_LAG",

        "SourceColumn":
            "TotalDemand",

        "GroupKey":
            "CanonicalProductID",

        "TimeIndex":
            "OperatingDaySequence",

        "LagOperatingDays":
            int(row.LagOperatingDays),

        "WindowOperatingDays":
            np.nan,

        "MinimumShiftOperatingRows":
            int(row.LagOperatingDays),

        "MinimumPeriods":
            np.nan,

        "GenerationRule":
            row.GenerationRule,

        "DirectModelInputAllowed":
            True,

        "IsHistoricalDemandFeature":
            True,

        "IsIdentifier":
            False,

        "IsSupportColumn":
            False,

        "IsForecastTarget":
            False,

        "UsesCurrentRowTarget":
            False,

        "UsesFutureTarget":
            False,

        "DataType":
            get_feature_dtype(
                row.FeatureName
            ),

        "MissingCount":
            missing_count,

        "MissingPercentage":
            missing_percentage,

        "MissingValuePolicy":
            (
                "PRESERVE_EARLY_HISTORY_NA_"
                "UNTIL_MODEL_PIPELINE"
            ),

        "PredictionTimeStatus":
            (
                "AVAILABLE_FROM_PRIOR_PRODUCT_"
                "DEMAND_HISTORY"
            )
    })


# ------------------------------------------------------------
# 9. Standardise rolling feature contracts
# ------------------------------------------------------------

for row in rolling_contract_source_df.itertuples():

    missing_count, missing_percentage = (
        get_feature_missing_metadata(
            row.FeatureName
        )
    )

    historical_contract_records.append({
        "Column":
            row.FeatureName,

        "ContractSection":
            "HISTORICAL_DEMAND",

        "FeatureFamily":
            row.FeatureStatistic,

        "SourceColumn":
            "TotalDemand",

        "GroupKey":
            "CanonicalProductID",

        "TimeIndex":
            "OperatingDaySequence",

        "LagOperatingDays":
            np.nan,

        "WindowOperatingDays":
            int(row.WindowOperatingDays),

        "MinimumShiftOperatingRows":
            int(row.ShiftBeforeRolling),

        "MinimumPeriods":
            int(row.MinimumPeriods),

        "GenerationRule":
            row.GenerationRule,

        "DirectModelInputAllowed":
            True,

        "IsHistoricalDemandFeature":
            True,

        "IsIdentifier":
            False,

        "IsSupportColumn":
            False,

        "IsForecastTarget":
            False,

        "UsesCurrentRowTarget":
            False,

        "UsesFutureTarget":
            False,

        "DataType":
            get_feature_dtype(
                row.FeatureName
            ),

        "MissingCount":
            missing_count,

        "MissingPercentage":
            missing_percentage,

        "MissingValuePolicy":
            (
                "PRESERVE_EARLY_HISTORY_NA_"
                "UNTIL_MODEL_PIPELINE"
            ),

        "PredictionTimeStatus":
            (
                "AVAILABLE_FROM_PRIOR_PRODUCT_"
                "DEMAND_HISTORY"
            )
    })


# ------------------------------------------------------------
# 10. Standardise occurrence/recency contracts
# ------------------------------------------------------------

for row in occurrence_contract_source_df.itertuples():

    missing_count, missing_percentage = (
        get_feature_missing_metadata(
            row.FeatureName
        )
    )

    historical_contract_records.append({
        "Column":
            row.FeatureName,

        "ContractSection":
            "HISTORICAL_DEMAND",

        "FeatureFamily":
            row.FeatureFamily,

        "SourceColumn":
            "TotalDemand",

        "GroupKey":
            "CanonicalProductID",

        "TimeIndex":
            "OperatingDaySequence",

        "LagOperatingDays":
            np.nan,

        "WindowOperatingDays":
            (
                row.WindowOperatingDays
                if pd.notna(
                    row.WindowOperatingDays
                )
                else np.nan
            ),

        "MinimumShiftOperatingRows":
            int(
                row.ShiftBeforeCalculation
            ),

        "MinimumPeriods":
            int(row.MinimumPeriods),

        "GenerationRule":
            row.GenerationRule,

        "DirectModelInputAllowed":
            True,

        "IsHistoricalDemandFeature":
            True,

        "IsIdentifier":
            False,

        "IsSupportColumn":
            False,

        "IsForecastTarget":
            False,

        "UsesCurrentRowTarget":
            False,

        "UsesFutureTarget":
            False,

        "DataType":
            get_feature_dtype(
                row.FeatureName
            ),

        "MissingCount":
            missing_count,

        "MissingPercentage":
            missing_percentage,

        "MissingValuePolicy":
            (
                "PRESERVE_EARLY_HISTORY_NA_"
                "UNTIL_MODEL_PIPELINE"
            ),

        "PredictionTimeStatus":
            (
                "AVAILABLE_FROM_PRIOR_PRODUCT_"
                "DEMAND_HISTORY"
            )
    })


# ------------------------------------------------------------
# 11. Standardise expanding feature contracts
# ------------------------------------------------------------

for row in expanding_contract_source_df.itertuples():

    missing_count, missing_percentage = (
        get_feature_missing_metadata(
            row.FeatureName
        )
    )

    historical_contract_records.append({
        "Column":
            row.FeatureName,

        "ContractSection":
            "HISTORICAL_DEMAND",

        "FeatureFamily":
            row.FeatureFamily,

        "SourceColumn":
            "TotalDemand",

        "GroupKey":
            "CanonicalProductID",

        "TimeIndex":
            "OperatingDaySequence",

        "LagOperatingDays":
            np.nan,

        "WindowOperatingDays":
            np.nan,

        "MinimumShiftOperatingRows":
            int(
                row.ShiftBeforeCalculation
            ),

        "MinimumPeriods":
            int(row.MinimumPeriods),

        "GenerationRule":
            row.GenerationRule,

        "DirectModelInputAllowed":
            True,

        "IsHistoricalDemandFeature":
            True,

        "IsIdentifier":
            False,

        "IsSupportColumn":
            False,

        "IsForecastTarget":
            False,

        "UsesCurrentRowTarget":
            False,

        "UsesFutureTarget":
            False,

        "DataType":
            get_feature_dtype(
                row.FeatureName
            ),

        "MissingCount":
            missing_count,

        "MissingPercentage":
            missing_percentage,

        "MissingValuePolicy":
            (
                "PRESERVE_EARLY_HISTORY_NA_"
                "UNTIL_MODEL_PIPELINE"
            ),

        "PredictionTimeStatus":
            (
                "AVAILABLE_FROM_PRIOR_PRODUCT_"
                "DEMAND_HISTORY"
            )
    })


historical_combined_contract_df = pd.DataFrame(
    historical_contract_records
)

assert len(
    historical_combined_contract_df
) == 31

assert historical_combined_contract_df[
    "Column"
].nunique() == 31


# ------------------------------------------------------------
# 12. Combine base and historical contracts
# ------------------------------------------------------------

step2_combined_feature_contract_df = pd.concat(
    [
        base_combined_contract_df,
        historical_combined_contract_df
    ],
    ignore_index=True
)

model_column_order_lookup = {
    column: position
    for position, column in enumerate(
        step2_historical_model_df.columns
    )
}

step2_combined_feature_contract_df[
    "ColumnOrder"
] = (
    step2_combined_feature_contract_df[
        "Column"
    ]
    .map(model_column_order_lookup)
)

assert step2_combined_feature_contract_df[
    "ColumnOrder"
].isna().sum() == 0

step2_combined_feature_contract_df = (
    step2_combined_feature_contract_df
    .sort_values("ColumnOrder")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. Validate the combined contract
# ------------------------------------------------------------

assert len(
    step2_combined_feature_contract_df
) == 58

assert step2_combined_feature_contract_df[
    "Column"
].nunique() == 58

assert set(
    step2_combined_feature_contract_df[
        "Column"
    ]
) == set(
    step2_historical_model_df.columns
)

assert not set(
    step2_audit_only_columns
).intersection(
    step2_combined_feature_contract_df[
        "Column"
    ]
)

direct_model_input_count = int(
    step2_combined_feature_contract_df[
        "DirectModelInputAllowed"
    ].sum()
)

historical_model_input_count = int(
    step2_combined_feature_contract_df[
        "IsHistoricalDemandFeature"
    ].sum()
)

identifier_count = int(
    step2_combined_feature_contract_df[
        "IsIdentifier"
    ].sum()
)

support_column_count = int(
    step2_combined_feature_contract_df[
        "IsSupportColumn"
    ].sum()
)

target_count = int(
    step2_combined_feature_contract_df[
        "IsForecastTarget"
    ].sum()
)

assert direct_model_input_count == 53
assert historical_model_input_count == 31
assert identifier_count == 2
assert support_column_count == 2
assert target_count == 1

assert not step2_combined_feature_contract_df.loc[
    step2_combined_feature_contract_df[
        "IsHistoricalDemandFeature"
    ],
    "UsesCurrentRowTarget"
].any()

assert not step2_combined_feature_contract_df.loc[
    step2_combined_feature_contract_df[
        "IsHistoricalDemandFeature"
    ],
    "UsesFutureTarget"
].any()


# ------------------------------------------------------------
# 14. Create feature-family summary
# ------------------------------------------------------------

step2_feature_family_summary_df = (
    step2_combined_feature_contract_df
    .groupby(
        [
            "ContractSection",
            "FeatureFamily"
        ],
        dropna=False,
        as_index=False
    )
    .agg(
        ColumnCount=(
            "Column",
            "nunique"
        ),
        DirectModelInputCount=(
            "DirectModelInputAllowed",
            "sum"
        ),
        TotalMissingCells=(
            "MissingCount",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 15. Create final Step 2 validation summary
# ------------------------------------------------------------

step2_final_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "FullAuditPanelRows",
        "FullAuditPanelColumns",
        "ModelFeatureViewRows",
        "ModelFeatureViewColumns",
        "CanonicalProducts",
        "OperatingDates",
        "DuplicateProductDateRows",
        "BaseCurrentDateFeatureCandidates",
        "HistoricalDemandFeatures",
        "TotalDirectModelInputCandidates",
        "IdentifierColumns",
        "SupportColumns",
        "TargetColumns",
        "AuditColumnsExcludedFromModelView",
        "SameDayLeakageColumnsPresent",
        "HistoricalCurrentRowTargetUses",
        "HistoricalFutureTargetUses",
        "UnexpectedHistoricalMissingAfterAvailability",
        "RowsWithAllHistoricalFeatures",
        "RowsWithPartialHistoricalFeatures",
        "RowsWithNoHistoricalFeatures",
        "TotalDemandUnits"
    ],
    "Value": [
        len(step2_historical_full_df),
        step2_historical_full_df.shape[1],
        len(step2_historical_model_df),
        step2_historical_model_df.shape[1],
        step2_historical_model_df[
            "CanonicalProductID"
        ].nunique(),
        step2_historical_model_df[
            "Date"
        ].nunique(),
        int(
            step2_historical_model_df[
                [
                    "Date",
                    "CanonicalProductID"
                ]
            ].duplicated().sum()
        ),
        (
            direct_model_input_count
            - historical_model_input_count
        ),
        historical_model_input_count,
        direct_model_input_count,
        identifier_count,
        support_column_count,
        target_count,
        len(step2_audit_only_columns),
        len(
            same_day_leakage_columns_found
        ),
        int(
            step2_combined_feature_contract_df.loc[
                step2_combined_feature_contract_df[
                    "IsHistoricalDemandFeature"
                ],
                "UsesCurrentRowTarget"
            ].sum()
        ),
        int(
            step2_combined_feature_contract_df.loc[
                step2_combined_feature_contract_df[
                    "IsHistoricalDemandFeature"
                ],
                "UsesFutureTarget"
            ].sum()
        ),
        int(
            historical_feature_missingness_df[
                "UnexpectedMissingAfterFirstAvailable"
            ].sum()
        ),
        rows_with_all_historical_features,
        rows_with_partial_historical_features,
        rows_with_no_historical_features,
        int(
            step2_historical_model_df[
                "TotalDemand"
            ].sum()
        )
    ]
})


# ------------------------------------------------------------
# 16. Validate final summary
# ------------------------------------------------------------

step2_final_summary_lookup = dict(
    zip(
        step2_final_validation_summary_df[
            "ValidationMetric"
        ],
        step2_final_validation_summary_df[
            "Value"
        ]
    )
)

assert step2_final_summary_lookup[
    "FullAuditPanelRows"
] == 43_774

assert step2_final_summary_lookup[
    "FullAuditPanelColumns"
] == 60

assert step2_final_summary_lookup[
    "ModelFeatureViewRows"
] == 43_774

assert step2_final_summary_lookup[
    "ModelFeatureViewColumns"
] == 58

assert step2_final_summary_lookup[
    "BaseCurrentDateFeatureCandidates"
] == 22

assert step2_final_summary_lookup[
    "HistoricalDemandFeatures"
] == 31

assert step2_final_summary_lookup[
    "TotalDirectModelInputCandidates"
] == 53

assert step2_final_summary_lookup[
    "SameDayLeakageColumnsPresent"
] == 0

assert step2_final_summary_lookup[
    "HistoricalCurrentRowTargetUses"
] == 0

assert step2_final_summary_lookup[
    "HistoricalFutureTargetUses"
] == 0

assert step2_final_summary_lookup[
    "UnexpectedHistoricalMissingAfterAvailability"
] == 0

assert step2_final_summary_lookup[
    "TotalDemandUnits"
] == 116_158


# ------------------------------------------------------------
# 17. Print Cell 40 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 2, PART 6 — "
    "COMBINED FEATURE CONTRACT CREATED"
)
print("=" * 75)

print()
print("Combined feature contract:")
print(
    "Columns documented:",
    len(
        step2_combined_feature_contract_df
    )
)
print(
    "Direct model-input candidates:",
    direct_model_input_count
)
print(
    "Current-date candidates:",
    (
        direct_model_input_count
        - historical_model_input_count
    )
)
print(
    "Historical demand candidates:",
    historical_model_input_count
)
print(
    "Identifier columns:",
    identifier_count
)
print(
    "Support columns:",
    support_column_count
)
print(
    "Target columns:",
    target_count
)

print()
print("Feature-family summary:")
display(step2_feature_family_summary_df)

print()
print("Final Step 2 validation summary:")
display(step2_final_validation_summary_df)

print()
print("Cell 40 completed successfully.")

STEP 2, PART 6 — COMBINED FEATURE CONTRACT CREATED

Combined feature contract:
Columns documented: 58
Direct model-input candidates: 53
Current-date candidates: 22
Historical demand candidates: 31
Identifier columns: 2
Support columns: 2
Target columns: 1

Feature-family summary:


,ContractSection,FeatureFamily,ColumnCount,DirectModelInputCount,TotalMissingCells
0,BASE_AND_CURRENT_DATE,CONDITIONAL_PRODUCT_METADATA_FEATURE,6,6,249366
1,BASE_AND_CURRENT_DATE,IDENTIFIER_OR_TIME_KEY,2,0,0
2,BASE_AND_CURRENT_DATE,KNOWN_AHEAD_TIME_FEATURE,12,12,0
3,BASE_AND_CURRENT_DATE,PRODUCT_METADATA_FEATURE,4,4,0
4,BASE_AND_CURRENT_DATE,SUPPORT_AND_AUDIT_ONLY,2,0,0
5,BASE_AND_CURRENT_DATE,TARGET_AND_HISTORICAL_FEATURE_SOURCE,1,0,0
6,HISTORICAL_DEMAND,DAYS_SINCE_PREVIOUS_POSITIVE_DEMAND,1,1,227
7,HISTORICAL_DEMAND,DEMAND_LAG,6,6,9307
8,HISTORICAL_DEMAND,EXPANDING_PAST_MEAN,1,1,227
9,HISTORICAL_DEMAND,EXPANDING_PAST_POSITIVE_RATE,1,1,227



Final Step 2 validation summary:


,ValidationMetric,Value
0,FullAuditPanelRows,43774
1,FullAuditPanelColumns,60
2,ModelFeatureViewRows,43774
3,ModelFeatureViewColumns,58
4,CanonicalProducts,227
5,OperatingDates,245
6,DuplicateProductDateRows,0
7,BaseCurrentDateFeatureCandidates,22
8,HistoricalDemandFeatures,31
9,TotalDirectModelInputCandidates,53



Cell 40 completed successfully.


In [52]:
# ============================================================
# FORECASTING PREPARATION
# STEP 2 — PART 6
# Cell 41: Save final Step 2 outputs, update the handoff,
# and complete Forecasting Preparation Step 2
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 6 objects exist
# ------------------------------------------------------------

required_step2_part6_objects = [
    "step2_historical_full_df",
    "step2_historical_model_df",
    "step2_combined_feature_contract_df",
    "historical_feature_missingness_df",
    "historical_feature_readiness_summary_df",
    "step2_feature_family_summary_df",
    "step2_final_validation_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step2_part6_objects = [
    object_name
    for object_name in required_step2_part6_objects
    if object_name not in globals()
]

if missing_step2_part6_objects:
    raise NameError(
        "The following Step 2 Part 6 objects are missing:\n"
        f"{missing_step2_part6_objects}\n\n"
        "Run Cells 39 and 40 before running Cell 41."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define final Step 2 output paths
# ------------------------------------------------------------

STEP2_FULL_AUDIT_PANEL_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_panel_full.csv"
)

STEP2_MODEL_FEATURE_VIEW_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_model_view.csv"
)

STEP2_COMBINED_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_combined_feature_contract.csv"
)

STEP2_MISSINGNESS_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_missingness_audit.csv"
)

STEP2_READINESS_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_readiness_summary.csv"
)

STEP2_FEATURE_FAMILY_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_feature_family_summary.csv"
)

STEP2_FINAL_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "12_step2_final_validation_summary.csv"
)


# ------------------------------------------------------------
# 4. Prepare official row ordering
# ------------------------------------------------------------

step2_full_audit_output_df = (
    step2_historical_full_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

step2_model_feature_output_df = (
    step2_historical_model_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Save all final Step 2 outputs
# ------------------------------------------------------------

step2_full_audit_output_df.to_csv(
    STEP2_FULL_AUDIT_PANEL_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step2_model_feature_output_df.to_csv(
    STEP2_MODEL_FEATURE_VIEW_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step2_combined_feature_contract_df.to_csv(
    STEP2_COMBINED_CONTRACT_OUTPUT,
    index=False
)

historical_feature_missingness_df.to_csv(
    STEP2_MISSINGNESS_AUDIT_OUTPUT,
    index=False
)

historical_feature_readiness_summary_df.to_csv(
    STEP2_READINESS_SUMMARY_OUTPUT,
    index=False
)

step2_feature_family_summary_df.to_csv(
    STEP2_FEATURE_FAMILY_SUMMARY_OUTPUT,
    index=False
)

step2_final_validation_summary_df.to_csv(
    STEP2_FINAL_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 6. Reload saved outputs
# ------------------------------------------------------------

saved_step2_full_panel = pd.read_csv(
    STEP2_FULL_AUDIT_PANEL_OUTPUT,
    low_memory=False
)

saved_step2_model_view = pd.read_csv(
    STEP2_MODEL_FEATURE_VIEW_OUTPUT,
    low_memory=False
)

saved_step2_contract = pd.read_csv(
    STEP2_COMBINED_CONTRACT_OUTPUT,
    low_memory=False
)

saved_step2_missingness = pd.read_csv(
    STEP2_MISSINGNESS_AUDIT_OUTPUT,
    low_memory=False
)

saved_step2_readiness = pd.read_csv(
    STEP2_READINESS_SUMMARY_OUTPUT,
    low_memory=False
)

saved_step2_family_summary = pd.read_csv(
    STEP2_FEATURE_FAMILY_SUMMARY_OUTPUT,
    low_memory=False
)

saved_step2_validation = pd.read_csv(
    STEP2_FINAL_VALIDATION_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 7. Validate saved final outputs
# ------------------------------------------------------------

assert saved_step2_full_panel.shape == (
    43_774,
    60
)

assert saved_step2_model_view.shape == (
    43_774,
    58
)

assert saved_step2_model_view[
    "CanonicalProductID"
].nunique() == 227

assert saved_step2_model_view[
    "Date"
].nunique() == 245

assert saved_step2_model_view[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert saved_step2_model_view[
    "TotalDemand"
].sum() == 116_158

assert all(
    column not in saved_step2_model_view.columns
    for column in step2_audit_only_columns
)

assert same_day_leakage_columns.isdisjoint(
    saved_step2_model_view.columns
)

assert len(saved_step2_contract) == 58

assert saved_step2_contract[
    "Column"
].nunique() == 58

assert len(saved_step2_missingness) == 31

assert saved_step2_missingness[
    "UnexpectedMissingAfterFirstAvailable"
].sum() == 0

assert saved_step2_readiness[
    "RowCount"
].sum() == 43_774

assert len(saved_step2_validation) == 22


# ------------------------------------------------------------
# 8. Parse and validate saved contract Boolean fields
# ------------------------------------------------------------

saved_direct_input_allowed = (
    parse_boolean_series(
        saved_step2_contract[
            "DirectModelInputAllowed"
        ]
    )
)

saved_historical_feature_flag = (
    parse_boolean_series(
        saved_step2_contract[
            "IsHistoricalDemandFeature"
        ]
    )
)

saved_current_target_use = (
    parse_boolean_series(
        saved_step2_contract[
            "UsesCurrentRowTarget"
        ]
    )
)

saved_future_target_use = (
    parse_boolean_series(
        saved_step2_contract[
            "UsesFutureTarget"
        ]
    )
)

assert int(
    saved_direct_input_allowed.sum()
) == 53

assert int(
    saved_historical_feature_flag.sum()
) == 31

assert int(
    saved_current_target_use.loc[
        saved_historical_feature_flag
    ].sum()
) == 0

assert int(
    saved_future_target_use.loc[
        saved_historical_feature_flag
    ].sum()
) == 0


# ------------------------------------------------------------
# 9. Build Markdown historical feature summary
# ------------------------------------------------------------

historical_group_markdown = "\n".join([
    (
        f"- Demand lag features: "
        f"{len(demand_lag_feature_columns)}"
    ),
    (
        f"- Rolling statistic features: "
        f"{len(rolling_feature_columns)}"
    ),
    (
        f"- Occurrence and recency features: "
        f"{len(occurrence_recency_feature_columns)}"
    ),
    (
        f"- Expanding history features: "
        f"{len(expanding_feature_columns)}"
    ),
    (
        f"- Total historical demand features: "
        f"{len(all_historical_feature_columns)}"
    )
])


# ------------------------------------------------------------
# 10. Update the Markdown handoff
# ------------------------------------------------------------

step2_part6_summary = f"""
**Status:** Completed and validated

### Purpose

Step 2 Part 6 consolidated every historical demand feature,
created the final Step 2 model feature view and froze the combined
Step 2 feature contract.

### Historical feature composition

{historical_group_markdown}

### Final Step 2 datasets

#### Full audit panel

- Rows: {len(saved_step2_full_panel):,}
- Columns: {saved_step2_full_panel.shape[1]}
- Includes the two lag-readiness audit fields

#### Model feature view

- Rows: {len(saved_step2_model_view):,}
- Columns: {saved_step2_model_view.shape[1]}
- Canonical products: {saved_step2_model_view["CanonicalProductID"].nunique()}
- Operating dates: {saved_step2_model_view["Date"].nunique()}
- Duplicate product-date rows: 0
- Total demand units: {int(saved_step2_model_view["TotalDemand"].sum()):,}

### Model-input contract

- Current-date feature candidates: 22
- Historical demand feature candidates: 31
- Total direct model-input candidates: 53
- Identifier columns: 2
- Support-only columns: 2
- Forecast target columns: 1

The audit fields `AvailableLagFeatureCount` and
`AllApprovedDemandLagsAvailable` are preserved in the full panel
but excluded from the model feature view.

### Leakage protection

- Same-day leakage columns present: 0
- Historical features using current-row demand: 0
- Historical features using future demand: 0
- Unexpected historical missing values after feature availability: 0

### Early-history readiness

- Rows with all 31 historical features: {rows_with_all_historical_features:,}
- Rows with partial historical features: {rows_with_partial_historical_features:,}
- Rows with no historical features: {rows_with_no_historical_features:,}

Early-history missing values remain unchanged. They have not been
replaced with zero or another imputed value. Missing-value
treatment will be handled inside the later modelling pipeline.

### Saved Step 2 Part 6 outputs

- `12_step2_historical_feature_panel_full.csv`
- `12_step2_historical_feature_model_view.csv`
- `12_step2_combined_feature_contract.csv`
- `12_step2_historical_feature_missingness_audit.csv`
- `12_step2_historical_feature_readiness_summary.csv`
- `12_step2_feature_family_summary.csv`
- `12_step2_final_validation_summary.csv`

### Step 2 completion

Forecasting Preparation Step 2 is complete.

All planned historical feature families were generated using only
information available before the current forecast row.
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_2_part_6",
    section_title=(
        "Forecasting Preparation — Step 2, Part 6"
    ),
    section_body=step2_part6_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 11. Print final Step 2 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 2, PART 6 COMPLETED"
)
print("=" * 75)

print()
print("Full Step 2 audit panel:")
print(
    f"Rows saved: "
    f"{len(saved_step2_full_panel):,}"
)
print(
    "Columns saved:",
    saved_step2_full_panel.shape[1]
)

print()
print("Step 2 model feature view:")
print(
    f"Rows saved: "
    f"{len(saved_step2_model_view):,}"
)
print(
    "Columns saved:",
    saved_step2_model_view.shape[1]
)
print(
    "Canonical products:",
    saved_step2_model_view[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_step2_model_view[
        "Date"
    ].nunique()
)

print()
print("Feature contract:")
print(
    "Current-date feature candidates:",
    22
)
print(
    "Historical feature candidates:",
    int(
        saved_historical_feature_flag.sum()
    )
)
print(
    "Total direct model-input candidates:",
    int(
        saved_direct_input_allowed.sum()
    )
)

print()
print("Leakage validation:")
print(
    "Same-day leakage columns present:",
    len(
        same_day_leakage_columns.intersection(
            saved_step2_model_view.columns
        )
    )
)
print(
    "Historical current-row target uses:",
    int(
        saved_current_target_use.loc[
            saved_historical_feature_flag
        ].sum()
    )
)
print(
    "Historical future-target uses:",
    int(
        saved_future_target_use.loc[
            saved_historical_feature_flag
        ].sum()
    )
)

print()
print("Historical feature readiness:")
print(
    "Rows with all 31 historical features:",
    f"{rows_with_all_historical_features:,}"
)
print(
    "Rows with partial historical features:",
    f"{rows_with_partial_historical_features:,}"
)
print(
    "Rows with no historical features:",
    f"{rows_with_no_historical_features:,}"
)

print()
print("Demand preserved:")
print(
    "Total demand units:",
    f"{saved_step2_model_view['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP2_FULL_AUDIT_PANEL_OUTPUT}")
print(f"2. {STEP2_MODEL_FEATURE_VIEW_OUTPUT}")
print(f"3. {STEP2_COMBINED_CONTRACT_OUTPUT}")
print(f"4. {STEP2_MISSINGNESS_AUDIT_OUTPUT}")
print(f"5. {STEP2_READINESS_SUMMARY_OUTPUT}")
print(f"6. {STEP2_FEATURE_FAMILY_SUMMARY_OUTPUT}")
print(f"7. {STEP2_FINAL_VALIDATION_OUTPUT}")
print(f"8. {HANDOFF_FILE}")

print()
print(
    "All Step 2, Part 6 validation checks passed."
)
print(
    "FORECASTING PREPARATION STEP 2 IS COMPLETE."
)

FORECASTING PREPARATION — STEP 2, PART 6 COMPLETED

Full Step 2 audit panel:
Rows saved: 43,774
Columns saved: 60

Step 2 model feature view:
Rows saved: 43,774
Columns saved: 58
Canonical products: 227
Operating dates: 245

Feature contract:
Current-date feature candidates: 22
Historical feature candidates: 31
Total direct model-input candidates: 53

Leakage validation:
Same-day leakage columns present: 0
Historical current-row target uses: 0
Historical future-target uses: 0

Historical feature readiness:
Rows with all 31 historical features: 39,234
Rows with partial historical features: 4,313
Rows with no historical features: 227

Demand preserved:
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/12_step2_historical_feature_panel_full.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/12_step2_historical_feature_model_view.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_

In [53]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 1
# Cell 42: Load and validate the chronological split source
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the earlier forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define the official Step 3 input files
# ------------------------------------------------------------

STEP2_MODEL_FEATURE_VIEW_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_model_view.csv"
)

PRODUCT_STRATEGY_FILE = (
    FORECAST_PREPARATION_DIR
    / "06_product_modelling_strategy_register.csv"
)

FORECASTING_POLICY_FILE = (
    FORECAST_PREPARATION_DIR
    / "05_forecasting_policy_parameters.csv"
)

required_step3_input_files = [
    STEP2_MODEL_FEATURE_VIEW_FILE,
    PRODUCT_STRATEGY_FILE,
    FORECASTING_POLICY_FILE
]

missing_step3_input_files = [
    file_path
    for file_path in required_step3_input_files
    if not file_path.exists()
]

if missing_step3_input_files:
    raise FileNotFoundError(
        "The following required Step 3 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_step3_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load the official inputs
# ------------------------------------------------------------

step3_source_df = pd.read_csv(
    STEP2_MODEL_FEATURE_VIEW_FILE,
    low_memory=False
)

step3_product_strategy_df = pd.read_csv(
    PRODUCT_STRATEGY_FILE,
    low_memory=False
)

step3_existing_policy_df = pd.read_csv(
    FORECASTING_POLICY_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Parse source dates and numeric fields
# ------------------------------------------------------------

date_columns = [
    "Date",
    "ProductFirstObservedDate"
]

for column in date_columns:

    step3_source_df[column] = pd.to_datetime(
        step3_source_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step3_source_df[
    date_columns
].isna().sum().sum() == 0


required_numeric_columns = [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]

for column in required_numeric_columns:

    step3_source_df[column] = pd.to_numeric(
        step3_source_df[column],
        errors="coerce"
    )

assert step3_source_df[
    required_numeric_columns
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 5. Standardise product identifiers
# ------------------------------------------------------------

for dataframe in [
    step3_source_df,
    step3_product_strategy_df
]:

    dataframe[
        "CanonicalProductID"
    ] = (
        dataframe[
            "CanonicalProductID"
        ]
        .astype("string")
        .str.strip()
    )

assert step3_source_df[
    "CanonicalProductID"
].isna().sum() == 0

assert step3_product_strategy_df[
    "CanonicalProductID"
].isna().sum() == 0


# ------------------------------------------------------------
# 6. Validate the completed Step 2 model feature view
# ------------------------------------------------------------

assert step3_source_df.shape == (
    43_774,
    58
), (
    "Unexpected Step 2 model feature-view dimensions.\n"
    f"Found: {step3_source_df.shape}"
)

assert step3_source_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_source_df[
    "Date"
].nunique() == 245

assert step3_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step3_source_df[
    "TotalDemand"
].isna().sum() == 0

assert (
    step3_source_df[
        "TotalDemand"
    ] < 0
).sum() == 0

assert step3_source_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 7. Confirm same-day leakage columns remain absent
# ------------------------------------------------------------

same_day_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

same_day_leakage_columns_found = sorted(
    same_day_leakage_columns.intersection(
        step3_source_df.columns
    )
)

assert len(
    same_day_leakage_columns_found
) == 0


# ------------------------------------------------------------
# 8. Validate the product strategy register
# ------------------------------------------------------------

required_strategy_columns = [
    "CanonicalProductID",
    "CanonicalProductName",
    "HistoryOperatingDays",
    "PositiveDemandDays",
    "TotalDemandUnits",
    "DemandPatternClass",
    "EvaluationCohort",
    "EvaluationCohortPriority",
    "EvaluationAction",
    "ForecastInProjectScope"
]

missing_strategy_columns = [
    column
    for column in required_strategy_columns
    if column not in step3_product_strategy_df.columns
]

if missing_strategy_columns:
    raise KeyError(
        "The following strategy columns are missing:\n"
        f"{missing_strategy_columns}"
    )

for column in [
    "HistoryOperatingDays",
    "PositiveDemandDays",
    "TotalDemandUnits",
    "EvaluationCohortPriority"
]:

    step3_product_strategy_df[column] = pd.to_numeric(
        step3_product_strategy_df[column],
        errors="coerce"
    )

assert len(step3_product_strategy_df) == 227

assert step3_product_strategy_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_product_strategy_df[
    "TotalDemandUnits"
].sum() == 116_158


# ------------------------------------------------------------
# 9. Confirm the expected evaluation cohorts
# ------------------------------------------------------------

cohort_count_lookup = (
    step3_product_strategy_df[
        "EvaluationCohort"
    ]
    .value_counts()
    .to_dict()
)

assert cohort_count_lookup.get(
    "MULTI_FOLD_BACKTEST_READY",
    0
) == 127

assert cohort_count_lookup.get(
    "SINGLE_HOLDOUT_BACKTEST_READY",
    0
) == 77

assert cohort_count_lookup.get(
    "LIMITED_HOLDOUT_ONLY",
    0
) == 14

assert cohort_count_lookup.get(
    "MINIMAL_EVIDENCE_FORECAST_ONLY",
    0
) == 9


# ------------------------------------------------------------
# 10. Reconstruct the operating calendar
# ------------------------------------------------------------

date_to_sequence_stability = (
    step3_source_df
    .groupby("Date")[
        "OperatingDaySequence"
    ]
    .nunique()
)

sequence_to_date_stability = (
    step3_source_df
    .groupby("OperatingDaySequence")[
        "Date"
    ]
    .nunique()
)

assert date_to_sequence_stability.max() == 1

assert sequence_to_date_stability.max() == 1


step3_operating_calendar_df = (
    step3_source_df[
        [
            "Date",
            "OperatingDaySequence"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "OperatingDaySequence",
        kind="stable"
    )
    .reset_index(drop=True)
)

assert len(step3_operating_calendar_df) == 245

assert step3_operating_calendar_df[
    "Date"
].nunique() == 245

assert step3_operating_calendar_df[
    "OperatingDaySequence"
].nunique() == 245

assert step3_operating_calendar_df[
    "Date"
].is_monotonic_increasing


minimum_operating_sequence = int(
    step3_operating_calendar_df[
        "OperatingDaySequence"
    ].min()
)

maximum_operating_sequence = int(
    step3_operating_calendar_df[
        "OperatingDaySequence"
    ].max()
)

expected_operating_sequences = np.arange(
    minimum_operating_sequence,
    maximum_operating_sequence + 1
)

assert np.array_equal(
    step3_operating_calendar_df[
        "OperatingDaySequence"
    ].astype(int).to_numpy(),
    expected_operating_sequences
)

assert minimum_operating_sequence == 1
assert maximum_operating_sequence == 245


# ------------------------------------------------------------
# 11. Validate strategy history against the source panel
# ------------------------------------------------------------

step3_product_history_check_df = (
    step3_source_df
    .assign(
        _PositiveDemandFlag=(
            step3_source_df[
                "TotalDemand"
            ] > 0
        ).astype(int)
    )
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        SourceHistoryOperatingDays=(
            "Date",
            "size"
        ),
        SourcePositiveDemandDays=(
            "_PositiveDemandFlag",
            "sum"
        ),
        SourceTotalDemandUnits=(
            "TotalDemand",
            "sum"
        ),
        SourceFirstOperatingSequence=(
            "OperatingDaySequence",
            "min"
        ),
        SourceFinalOperatingSequence=(
            "OperatingDaySequence",
            "max"
        )
    )
)

step3_product_history_check_df = (
    step3_product_history_check_df
    .merge(
        step3_product_strategy_df[
            [
                "CanonicalProductID",
                "HistoryOperatingDays",
                "PositiveDemandDays",
                "TotalDemandUnits",
                "EvaluationCohort"
            ]
        ],
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
)

history_length_mismatches = int(
    (
        step3_product_history_check_df[
            "SourceHistoryOperatingDays"
        ]
        != step3_product_history_check_df[
            "HistoryOperatingDays"
        ]
    ).sum()
)

positive_day_mismatches = int(
    (
        step3_product_history_check_df[
            "SourcePositiveDemandDays"
        ]
        != step3_product_history_check_df[
            "PositiveDemandDays"
        ]
    ).sum()
)

demand_total_mismatches = int(
    (
        step3_product_history_check_df[
            "SourceTotalDemandUnits"
        ]
        != step3_product_history_check_df[
            "TotalDemandUnits"
        ]
    ).sum()
)

products_not_continued_to_final_date = int(
    (
        step3_product_history_check_df[
            "SourceFinalOperatingSequence"
        ]
        != maximum_operating_sequence
    ).sum()
)

assert history_length_mismatches == 0
assert positive_day_mismatches == 0
assert demand_total_mismatches == 0
assert products_not_continued_to_final_date == 0


# ------------------------------------------------------------
# 12. Load and validate the existing forecasting policy
# ------------------------------------------------------------

step3_existing_policy_df[
    "Value"
] = pd.to_numeric(
    step3_existing_policy_df[
        "Value"
    ],
    errors="coerce"
)

assert step3_existing_policy_df[
    "Value"
].isna().sum() == 0


existing_policy_lookup = dict(
    zip(
        step3_existing_policy_df[
            "PolicyParameter"
        ],
        step3_existing_policy_df[
            "Value"
        ]
    )
)

assert int(
    existing_policy_lookup[
        "EvaluationHorizonOperatingDays"
    ]
) == 20

assert int(
    existing_policy_lookup[
        "MinimumTrainingOperatingDays"
    ]
) == 60

assert int(
    existing_policy_lookup[
        "RollingOriginFolds"
    ]
) == 3

assert int(
    existing_policy_lookup[
        "MinimumHistoryMultiFold"
    ]
) == 120

assert int(
    existing_policy_lookup[
        "MinimumHistorySingleHoldout"
    ]
) == 80

assert int(
    existing_policy_lookup[
        "MinimumHistoryLimitedHoldout"
    ]
) == 30


# ------------------------------------------------------------
# 13. Sort the official Step 3 source
# ------------------------------------------------------------

step3_source_df = (
    step3_source_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Print Cell 42 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 1 — "
    "CHRONOLOGICAL SPLIT SOURCE VALIDATED"
)
print("=" * 75)

print()
print("Step 3 source:")
print(f"Rows: {len(step3_source_df):,}")
print(f"Columns: {step3_source_df.shape[1]}")
print(
    "Canonical products:",
    step3_source_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step3_source_df[
        "Date"
    ].nunique()
)
print(
    "Date range:",
    step3_source_df[
        "Date"
    ].min().date(),
    "to",
    step3_source_df[
        "Date"
    ].max().date()
)

print()
print("Evaluation cohorts:")
for cohort_name, cohort_count in (
    step3_product_strategy_df[
        "EvaluationCohort"
    ]
    .value_counts()
    .items()
):
    print(
        f"- {cohort_name}: "
        f"{cohort_count}"
    )

print()
print("Source-strategy validation:")
print(
    "History-length mismatches:",
    history_length_mismatches
)
print(
    "Positive-demand-day mismatches:",
    positive_day_mismatches
)
print(
    "Demand-total mismatches:",
    demand_total_mismatches
)
print(
    "Products not continued to final date:",
    products_not_continued_to_final_date
)

print()
print("Leakage validation:")
print(
    "Same-day leakage columns found:",
    len(
        same_day_leakage_columns_found
    )
)
print(
    "Total demand units:",
    f"{step3_source_df['TotalDemand'].sum():,}"
)

print()
print("Cell 42 completed successfully.")

STEP 3, PART 1 — CHRONOLOGICAL SPLIT SOURCE VALIDATED

Step 3 source:
Rows: 43,774
Columns: 58
Canonical products: 227
Operating dates: 245
Date range: 2025-04-01 to 2026-03-30

Evaluation cohorts:
- MULTI_FOLD_BACKTEST_READY: 127
- SINGLE_HOLDOUT_BACKTEST_READY: 77
- LIMITED_HOLDOUT_ONLY: 14
- MINIMAL_EVIDENCE_FORECAST_ONLY: 9

Source-strategy validation:
History-length mismatches: 0
Positive-demand-day mismatches: 0
Demand-total mismatches: 0
Products not continued to final date: 0

Leakage validation:
Same-day leakage columns found: 0
Total demand units: 116,158

Cell 42 completed successfully.


In [54]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 1
# Cell 43: Freeze the chronological split policy and
# create the global evaluation-window calendar
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 42 objects exist
# ------------------------------------------------------------

required_cell_42_objects = [
    "step3_source_df",
    "step3_product_strategy_df",
    "step3_operating_calendar_df",
    "step3_product_history_check_df",
    "existing_policy_lookup",
    "minimum_operating_sequence",
    "maximum_operating_sequence"
]

missing_cell_42_objects = [
    object_name
    for object_name in required_cell_42_objects
    if object_name not in globals()
]

if missing_cell_42_objects:
    raise NameError(
        "The following Cell 42 objects are missing:\n"
        f"{missing_cell_42_objects}\n\n"
        "Run Cell 42 before running Cell 43."
    )


# ------------------------------------------------------------
# 2. Freeze the standard evaluation policy
# ------------------------------------------------------------

STANDARD_EVALUATION_HORIZON = int(
    existing_policy_lookup[
        "EvaluationHorizonOperatingDays"
    ]
)

STANDARD_MINIMUM_TRAINING_DAYS = int(
    existing_policy_lookup[
        "MinimumTrainingOperatingDays"
    ]
)

STANDARD_EVALUATION_WINDOW_COUNT = int(
    existing_policy_lookup[
        "RollingOriginFolds"
    ]
)

STANDARD_MODEL_SELECTION_WINDOW_COUNT = 2
STANDARD_FINAL_TEST_WINDOW_COUNT = 1

assert STANDARD_EVALUATION_HORIZON == 20
assert STANDARD_MINIMUM_TRAINING_DAYS == 60
assert STANDARD_EVALUATION_WINDOW_COUNT == 3


# ------------------------------------------------------------
# 3. Define the limited-evidence holdout policy
# ------------------------------------------------------------

LIMITED_EVALUATION_HORIZON = 5
LIMITED_MINIMUM_TRAINING_DAYS = 25

assert (
    LIMITED_EVALUATION_HORIZON
    + LIMITED_MINIMUM_TRAINING_DAYS
    == int(
        existing_policy_lookup[
            "MinimumHistoryLimitedHoldout"
        ]
    )
)


# ------------------------------------------------------------
# 4. Create sequence-to-date lookup
# ------------------------------------------------------------

sequence_to_date_lookup = (
    step3_operating_calendar_df
    .set_index(
        "OperatingDaySequence"
    )["Date"]
    .to_dict()
)


# ------------------------------------------------------------
# 5. Create three consecutive standard evaluation windows
# ------------------------------------------------------------

standard_evaluation_block_length = (
    STANDARD_EVALUATION_HORIZON
    * STANDARD_EVALUATION_WINDOW_COUNT
)

standard_evaluation_block_start = (
    maximum_operating_sequence
    - standard_evaluation_block_length
    + 1
)

standard_window_records = []

for window_order in range(
    1,
    STANDARD_EVALUATION_WINDOW_COUNT + 1
):

    evaluation_start_sequence = (
        standard_evaluation_block_start
        + (
            window_order - 1
        )
        * STANDARD_EVALUATION_HORIZON
    )

    evaluation_end_sequence = (
        evaluation_start_sequence
        + STANDARD_EVALUATION_HORIZON
        - 1
    )

    training_end_sequence = (
        evaluation_start_sequence - 1
    )

    is_final_holdout = (
        window_order
        == STANDARD_EVALUATION_WINDOW_COUNT
    )

    if is_final_holdout:

        window_id = (
            "STANDARD_FINAL_HOLDOUT"
        )

        window_purpose = (
            "FINAL_HOLDOUT_TEST"
        )

        eligible_cohorts = (
            "MULTI_FOLD_BACKTEST_READY|"
            "SINGLE_HOLDOUT_BACKTEST_READY"
        )

        may_be_used_for_model_selection = False

    else:

        window_id = (
            f"STANDARD_BACKTEST_FOLD_{window_order}"
        )

        window_purpose = (
            "BACKTEST_VALIDATION"
        )

        eligible_cohorts = (
            "MULTI_FOLD_BACKTEST_READY"
        )

        may_be_used_for_model_selection = True

    standard_window_records.append({
        "WindowID":
            window_id,

        "WindowOrder":
            window_order,

        "WindowScope":
            "STANDARD_EVALUATION",

        "WindowPurpose":
            window_purpose,

        "EligibleCohorts":
            eligible_cohorts,

        "GlobalTrainingStartSequence":
            minimum_operating_sequence,

        "TrainingEndSequence":
            training_end_sequence,

        "EvaluationStartSequence":
            evaluation_start_sequence,

        "EvaluationEndSequence":
            evaluation_end_sequence,

        "EvaluationHorizonOperatingDays":
            STANDARD_EVALUATION_HORIZON,

        "MinimumRequiredProductTrainingDays":
            STANDARD_MINIMUM_TRAINING_DAYS,

        "GlobalTrainingOperatingDates":
            (
                training_end_sequence
                - minimum_operating_sequence
                + 1
            ),

        "TrainingEndDate":
            sequence_to_date_lookup[
                training_end_sequence
            ],

        "EvaluationStartDate":
            sequence_to_date_lookup[
                evaluation_start_sequence
            ],

        "EvaluationEndDate":
            sequence_to_date_lookup[
                evaluation_end_sequence
            ],

        "MayBeUsedForModelSelection":
            may_be_used_for_model_selection,

        "ReservedFinalTest":
            is_final_holdout
    })


step3_standard_window_calendar_df = pd.DataFrame(
    standard_window_records
)


# ------------------------------------------------------------
# 6. Create the limited-evidence five-day holdout
# ------------------------------------------------------------

limited_evaluation_end_sequence = (
    maximum_operating_sequence
)

limited_evaluation_start_sequence = (
    limited_evaluation_end_sequence
    - LIMITED_EVALUATION_HORIZON
    + 1
)

limited_training_end_sequence = (
    limited_evaluation_start_sequence - 1
)


step3_limited_window_calendar_df = pd.DataFrame([
    {
        "WindowID":
            "LIMITED_FINAL_HOLDOUT",

        "WindowOrder":
            1,

        "WindowScope":
            "LIMITED_EVALUATION",

        "WindowPurpose":
            "LIMITED_FINAL_HOLDOUT_TEST",

        "EligibleCohorts":
            "LIMITED_HOLDOUT_ONLY",

        "GlobalTrainingStartSequence":
            minimum_operating_sequence,

        "TrainingEndSequence":
            limited_training_end_sequence,

        "EvaluationStartSequence":
            limited_evaluation_start_sequence,

        "EvaluationEndSequence":
            limited_evaluation_end_sequence,

        "EvaluationHorizonOperatingDays":
            LIMITED_EVALUATION_HORIZON,

        "MinimumRequiredProductTrainingDays":
            LIMITED_MINIMUM_TRAINING_DAYS,

        "GlobalTrainingOperatingDates":
            (
                limited_training_end_sequence
                - minimum_operating_sequence
                + 1
            ),

        "TrainingEndDate":
            sequence_to_date_lookup[
                limited_training_end_sequence
            ],

        "EvaluationStartDate":
            sequence_to_date_lookup[
                limited_evaluation_start_sequence
            ],

        "EvaluationEndDate":
            sequence_to_date_lookup[
                limited_evaluation_end_sequence
            ],

        "MayBeUsedForModelSelection":
            False,

        "ReservedFinalTest":
            True
    }
])


# ------------------------------------------------------------
# 7. Combine the global window calendar
# ------------------------------------------------------------

step3_global_window_calendar_df = pd.concat(
    [
        step3_standard_window_calendar_df,
        step3_limited_window_calendar_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 8. Create cohort-level split policy
# ------------------------------------------------------------

step3_cohort_split_policy_df = pd.DataFrame([
    {
        "EvaluationCohort":
            "MULTI_FOLD_BACKTEST_READY",

        "ProductCount":
            127,

        "EvaluationWindowIDs":
            (
                "STANDARD_BACKTEST_FOLD_1|"
                "STANDARD_BACKTEST_FOLD_2|"
                "STANDARD_FINAL_HOLDOUT"
            ),

        "ModelSelectionWindowCount":
            2,

        "FinalTestWindowCount":
            1,

        "EvaluationHorizonOperatingDays":
            STANDARD_EVALUATION_HORIZON,

        "MinimumTrainingOperatingDays":
            STANDARD_MINIMUM_TRAINING_DAYS,

        "IndependentEvaluationAvailable":
            True,

        "SplitTreatment":
            (
                "TWO_EXPANDING_WINDOW_VALIDATION_FOLDS_"
                "PLUS_ONE_RESERVED_FINAL_HOLDOUT"
            )
    },
    {
        "EvaluationCohort":
            "SINGLE_HOLDOUT_BACKTEST_READY",

        "ProductCount":
            77,

        "EvaluationWindowIDs":
            "STANDARD_FINAL_HOLDOUT",

        "ModelSelectionWindowCount":
            0,

        "FinalTestWindowCount":
            1,

        "EvaluationHorizonOperatingDays":
            STANDARD_EVALUATION_HORIZON,

        "MinimumTrainingOperatingDays":
            STANDARD_MINIMUM_TRAINING_DAYS,

        "IndependentEvaluationAvailable":
            True,

        "SplitTreatment":
            (
                "ONE_RESERVED_TWENTY_OPERATING_DAY_"
                "FINAL_HOLDOUT"
            )
    },
    {
        "EvaluationCohort":
            "LIMITED_HOLDOUT_ONLY",

        "ProductCount":
            14,

        "EvaluationWindowIDs":
            "LIMITED_FINAL_HOLDOUT",

        "ModelSelectionWindowCount":
            0,

        "FinalTestWindowCount":
            1,

        "EvaluationHorizonOperatingDays":
            LIMITED_EVALUATION_HORIZON,

        "MinimumTrainingOperatingDays":
            LIMITED_MINIMUM_TRAINING_DAYS,

        "IndependentEvaluationAvailable":
            True,

        "SplitTreatment":
            (
                "CAUTIOUS_FIVE_OPERATING_DAY_"
                "FINAL_HOLDOUT"
            )
    },
    {
        "EvaluationCohort":
            "MINIMAL_EVIDENCE_FORECAST_ONLY",

        "ProductCount":
            9,

        "EvaluationWindowIDs":
            "NONE",

        "ModelSelectionWindowCount":
            0,

        "FinalTestWindowCount":
            0,

        "EvaluationHorizonOperatingDays":
            0,

        "MinimumTrainingOperatingDays":
            0,

        "IndependentEvaluationAvailable":
            False,

        "SplitTreatment":
            (
                "NO_INDEPENDENT_PRODUCT_HOLDOUT_"
                "USE_POOLED_MODEL_AND_CONSERVATIVE_BASELINES"
            )
    }
])


# ------------------------------------------------------------
# 9. Create Step 3 split-policy parameter table
# ------------------------------------------------------------

step3_split_policy_parameters_df = pd.DataFrame({
    "PolicyParameter": [
        "SplitMethod",
        "RandomSplittingAllowed",
        "StandardEvaluationHorizonOperatingDays",
        "StandardMinimumTrainingOperatingDays",
        "StandardEvaluationWindowCount",
        "StandardModelSelectionWindowCount",
        "StandardFinalTestWindowCount",
        "LimitedEvaluationHorizonOperatingDays",
        "LimitedMinimumTrainingOperatingDays",
        "FinalHoldoutReservedUntilModelSelectionComplete"
    ],
    "Value": [
        "EXPANDING_WINDOW_CHRONOLOGICAL",
        False,
        STANDARD_EVALUATION_HORIZON,
        STANDARD_MINIMUM_TRAINING_DAYS,
        STANDARD_EVALUATION_WINDOW_COUNT,
        STANDARD_MODEL_SELECTION_WINDOW_COUNT,
        STANDARD_FINAL_TEST_WINDOW_COUNT,
        LIMITED_EVALUATION_HORIZON,
        LIMITED_MINIMUM_TRAINING_DAYS,
        True
    ]
})


# ------------------------------------------------------------
# 10. Validate the standard windows
# ------------------------------------------------------------

assert len(
    step3_standard_window_calendar_df
) == 3

assert (
    step3_standard_window_calendar_df[
        "EvaluationHorizonOperatingDays"
    ] == 20
).all()

assert (
    step3_standard_window_calendar_df[
        "EvaluationEndSequence"
    ]
    - step3_standard_window_calendar_df[
        "EvaluationStartSequence"
    ]
    + 1
    == 20
).all()

assert (
    step3_standard_window_calendar_df[
        "TrainingEndSequence"
    ]
    < step3_standard_window_calendar_df[
        "EvaluationStartSequence"
    ]
).all()

assert (
    step3_standard_window_calendar_df[
        "EvaluationStartSequence"
    ].iloc[1:].to_numpy()
    == (
        step3_standard_window_calendar_df[
            "EvaluationEndSequence"
        ].iloc[:-1].to_numpy()
        + 1
    )
).all()

assert (
    step3_standard_window_calendar_df[
        "EvaluationEndSequence"
    ].iloc[-1]
    == maximum_operating_sequence
)

assert (
    step3_standard_window_calendar_df[
        "ReservedFinalTest"
    ].sum()
    == 1
)

assert (
    step3_standard_window_calendar_df[
        "MayBeUsedForModelSelection"
    ].sum()
    == 2
)


# ------------------------------------------------------------
# 11. Validate the limited holdout
# ------------------------------------------------------------

assert len(
    step3_limited_window_calendar_df
) == 1

assert (
    limited_evaluation_end_sequence
    - limited_evaluation_start_sequence
    + 1
    == 5
)

assert (
    limited_training_end_sequence
    < limited_evaluation_start_sequence
)

standard_final_start = int(
    step3_standard_window_calendar_df.loc[
        step3_standard_window_calendar_df[
            "WindowID"
        ]
        == "STANDARD_FINAL_HOLDOUT",
        "EvaluationStartSequence"
    ].iloc[0]
)

standard_final_end = int(
    step3_standard_window_calendar_df.loc[
        step3_standard_window_calendar_df[
            "WindowID"
        ]
        == "STANDARD_FINAL_HOLDOUT",
        "EvaluationEndSequence"
    ].iloc[0]
)

assert (
    limited_evaluation_start_sequence
    >= standard_final_start
)

assert (
    limited_evaluation_end_sequence
    <= standard_final_end
)


# ------------------------------------------------------------
# 12. Validate product-specific training feasibility
# ------------------------------------------------------------

window_lookup = (
    step3_global_window_calendar_df
    .set_index("WindowID")
)

cohort_feasibility_records = []

cohort_window_mapping = {
    "MULTI_FOLD_BACKTEST_READY":
        "STANDARD_BACKTEST_FOLD_1",

    "SINGLE_HOLDOUT_BACKTEST_READY":
        "STANDARD_FINAL_HOLDOUT",

    "LIMITED_HOLDOUT_ONLY":
        "LIMITED_FINAL_HOLDOUT"
}

minimum_training_mapping = {
    "MULTI_FOLD_BACKTEST_READY":
        STANDARD_MINIMUM_TRAINING_DAYS,

    "SINGLE_HOLDOUT_BACKTEST_READY":
        STANDARD_MINIMUM_TRAINING_DAYS,

    "LIMITED_HOLDOUT_ONLY":
        LIMITED_MINIMUM_TRAINING_DAYS
}

for (
    evaluation_cohort,
    first_required_window
) in cohort_window_mapping.items():

    cohort_products_df = (
        step3_product_history_check_df.loc[
            step3_product_history_check_df[
                "EvaluationCohort"
            ] == evaluation_cohort
        ]
        .copy()
    )

    training_end_sequence = int(
        window_lookup.loc[
            first_required_window,
            "TrainingEndSequence"
        ]
    )

    evaluation_start_sequence = int(
        window_lookup.loc[
            first_required_window,
            "EvaluationStartSequence"
        ]
    )

    evaluation_end_sequence = int(
        window_lookup.loc[
            first_required_window,
            "EvaluationEndSequence"
        ]
    )

    cohort_products_df[
        "AvailableTrainingOperatingDays"
    ] = (
        training_end_sequence
        - cohort_products_df[
            "SourceFirstOperatingSequence"
        ]
        + 1
    )

    cohort_products_df[
        "AvailableEvaluationOperatingDays"
    ] = (
        evaluation_end_sequence
        - evaluation_start_sequence
        + 1
    )

    minimum_required_training_days = (
        minimum_training_mapping[
            evaluation_cohort
        ]
    )

    training_requirement_violations = int(
        (
            cohort_products_df[
                "AvailableTrainingOperatingDays"
            ]
            < minimum_required_training_days
        ).sum()
    )

    cohort_feasibility_records.append({
        "EvaluationCohort":
            evaluation_cohort,

        "ProductCount":
            len(cohort_products_df),

        "FirstRequiredWindow":
            first_required_window,

        "MinimumRequiredTrainingDays":
            minimum_required_training_days,

        "MinimumAvailableTrainingDays":
            int(
                cohort_products_df[
                    "AvailableTrainingOperatingDays"
                ].min()
            ),

        "MedianAvailableTrainingDays":
            float(
                cohort_products_df[
                    "AvailableTrainingOperatingDays"
                ].median()
            ),

        "MaximumAvailableTrainingDays":
            int(
                cohort_products_df[
                    "AvailableTrainingOperatingDays"
                ].max()
            ),

        "EvaluationOperatingDays":
            int(
                cohort_products_df[
                    "AvailableEvaluationOperatingDays"
                ].min()
            ),

        "TrainingRequirementViolations":
            training_requirement_violations,

        "FeasibilityStatus":
            (
                "PASSED"
                if training_requirement_violations == 0
                else "FAILED"
            )
    })


step3_cohort_window_feasibility_df = pd.DataFrame(
    cohort_feasibility_records
)

assert step3_cohort_window_feasibility_df[
    "ProductCount"
].sum() == (
    127 + 77 + 14
)

assert step3_cohort_window_feasibility_df[
    "TrainingRequirementViolations"
].sum() == 0

assert (
    step3_cohort_window_feasibility_df[
        "FeasibilityStatus"
    ] == "PASSED"
).all()


# ------------------------------------------------------------
# 13. Validate cohort policy counts
# ------------------------------------------------------------

assert step3_cohort_split_policy_df[
    "ProductCount"
].sum() == 227

assert (
    step3_cohort_split_policy_df.loc[
        step3_cohort_split_policy_df[
            "EvaluationCohort"
        ] == "MINIMAL_EVIDENCE_FORECAST_ONLY",
        "IndependentEvaluationAvailable"
    ].iloc[0]
    == False
)


# ------------------------------------------------------------
# 14. Print Cell 43 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 1 — "
    "CHRONOLOGICAL SPLIT POLICY FROZEN"
)
print("=" * 75)

print()
print("Standard evaluation windows:")
display(
    step3_standard_window_calendar_df[
        [
            "WindowID",
            "WindowPurpose",
            "TrainingEndSequence",
            "TrainingEndDate",
            "EvaluationStartSequence",
            "EvaluationStartDate",
            "EvaluationEndSequence",
            "EvaluationEndDate",
            "EvaluationHorizonOperatingDays",
            "MayBeUsedForModelSelection",
            "ReservedFinalTest"
        ]
    ]
)

print()
print("Limited-evidence evaluation window:")
display(
    step3_limited_window_calendar_df[
        [
            "WindowID",
            "WindowPurpose",
            "TrainingEndSequence",
            "TrainingEndDate",
            "EvaluationStartSequence",
            "EvaluationStartDate",
            "EvaluationEndSequence",
            "EvaluationEndDate",
            "EvaluationHorizonOperatingDays"
        ]
    ]
)

print()
print("Cohort split policy:")
display(step3_cohort_split_policy_df)

print()
print("Product-window feasibility:")
display(step3_cohort_window_feasibility_df)

print()
print("Split safeguards:")
print("Random splitting allowed:", False)
print(
    "Model-selection windows:",
    STANDARD_MODEL_SELECTION_WINDOW_COUNT
)
print(
    "Reserved standard final-test windows:",
    STANDARD_FINAL_TEST_WINDOW_COUNT
)
print(
    "Training requirement violations:",
    int(
        step3_cohort_window_feasibility_df[
            "TrainingRequirementViolations"
        ].sum()
    )
)

print()
print("Cell 43 completed successfully.")

STEP 3, PART 1 — CHRONOLOGICAL SPLIT POLICY FROZEN

Standard evaluation windows:


,WindowID,WindowPurpose,TrainingEndSequence,TrainingEndDate,EvaluationStartSequence,EvaluationStartDate,EvaluationEndSequence,EvaluationEndDate,EvaluationHorizonOperatingDays,MayBeUsedForModelSelection,ReservedFinalTest
0,STANDARD_BACKTEST_FOLD_1,BACKTEST_VALIDATION,185,2025-12-23,186,2026-01-05,205,2026-01-29,20,True,False
1,STANDARD_BACKTEST_FOLD_2,BACKTEST_VALIDATION,205,2026-01-29,206,2026-01-30,225,2026-02-27,20,True,False
2,STANDARD_FINAL_HOLDOUT,FINAL_HOLDOUT_TEST,225,2026-02-27,226,2026-03-02,245,2026-03-30,20,False,True



Limited-evidence evaluation window:


,WindowID,WindowPurpose,TrainingEndSequence,TrainingEndDate,EvaluationStartSequence,EvaluationStartDate,EvaluationEndSequence,EvaluationEndDate,EvaluationHorizonOperatingDays
0,LIMITED_FINAL_HOLDOUT,LIMITED_FINAL_HOLDOUT_TEST,240,2026-03-23,241,2026-03-24,245,2026-03-30,5



Cohort split policy:


,EvaluationCohort,ProductCount,EvaluationWindowIDs,ModelSelectionWindowCount,FinalTestWindowCount,EvaluationHorizonOperatingDays,MinimumTrainingOperatingDays,IndependentEvaluationAvailable,SplitTreatment
0,MULTI_FOLD_BACKTEST_READY,127,STANDARD_BACKTEST_FOLD_1|STANDARD_BACKTEST_FOL...,2,1,20,60,True,TWO_EXPANDING_WINDOW_VALIDATION_FOLDS_PLUS_ONE...
1,SINGLE_HOLDOUT_BACKTEST_READY,77,STANDARD_FINAL_HOLDOUT,0,1,20,60,True,ONE_RESERVED_TWENTY_OPERATING_DAY_FINAL_HOLDOUT
2,LIMITED_HOLDOUT_ONLY,14,LIMITED_FINAL_HOLDOUT,0,1,5,25,True,CAUTIOUS_FIVE_OPERATING_DAY_FINAL_HOLDOUT
3,MINIMAL_EVIDENCE_FORECAST_ONLY,9,NONE,0,0,0,0,False,NO_INDEPENDENT_PRODUCT_HOLDOUT_USE_POOLED_MODE...



Product-window feasibility:


,EvaluationCohort,ProductCount,FirstRequiredWindow,MinimumRequiredTrainingDays,MinimumAvailableTrainingDays,MedianAvailableTrainingDays,MaximumAvailableTrainingDays,EvaluationOperatingDays,TrainingRequirementViolations,FeasibilityStatus
0,MULTI_FOLD_BACKTEST_READY,127,STANDARD_BACKTEST_FOLD_1,60,121,185.0,185,20,0,PASSED
1,SINGLE_HOLDOUT_BACKTEST_READY,77,STANDARD_FINAL_HOLDOUT,60,60,90.0,225,20,0,PASSED
2,LIMITED_HOLDOUT_ONLY,14,LIMITED_FINAL_HOLDOUT,25,36,154.5,240,5,0,PASSED



Split safeguards:
Random splitting allowed: False
Model-selection windows: 2
Reserved standard final-test windows: 1
Training requirement violations: 0

Cell 43 completed successfully.


In [55]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 1
# Cell 44: Save chronological split policy outputs
# and update the Markdown handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Step 3 Part 1 objects exist
# ------------------------------------------------------------

required_step3_part1_objects = [
    "step3_operating_calendar_df",
    "step3_global_window_calendar_df",
    "step3_cohort_split_policy_df",
    "step3_split_policy_parameters_df",
    "step3_cohort_window_feasibility_df",
    "step3_product_history_check_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step3_part1_objects = [
    object_name
    for object_name in required_step3_part1_objects
    if object_name not in globals()
]

if missing_step3_part1_objects:
    raise NameError(
        "The following Step 3 Part 1 objects are missing:\n"
        f"{missing_step3_part1_objects}\n\n"
        "Run Cells 42 and 43 before running Cell 44."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Create the Part 1 validation summary
# ------------------------------------------------------------

step3_part1_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "SourceRows",
        "SourceColumns",
        "CanonicalProducts",
        "OperatingDates",
        "DuplicateProductDateRows",
        "TotalDemandUnits",
        "StandardEvaluationWindows",
        "StandardEvaluationHorizonDays",
        "StandardModelSelectionWindows",
        "ReservedStandardFinalTestWindows",
        "LimitedEvaluationWindows",
        "LimitedEvaluationHorizonDays",
        "RandomSplittingAllowed",
        "SameDayLeakageColumnsPresent",
        "HistoryLengthMismatches",
        "PositiveDemandDayMismatches",
        "DemandTotalMismatches",
        "ProductsNotContinuedToFinalDate",
        "TrainingRequirementViolations"
    ],
    "Value": [
        len(step3_source_df),
        step3_source_df.shape[1],
        step3_source_df[
            "CanonicalProductID"
        ].nunique(),
        step3_source_df[
            "Date"
        ].nunique(),
        int(
            step3_source_df[
                [
                    "Date",
                    "CanonicalProductID"
                ]
            ].duplicated().sum()
        ),
        int(
            step3_source_df[
                "TotalDemand"
            ].sum()
        ),
        len(
            step3_standard_window_calendar_df
        ),
        STANDARD_EVALUATION_HORIZON,
        STANDARD_MODEL_SELECTION_WINDOW_COUNT,
        STANDARD_FINAL_TEST_WINDOW_COUNT,
        len(
            step3_limited_window_calendar_df
        ),
        LIMITED_EVALUATION_HORIZON,
        False,
        len(
            same_day_leakage_columns_found
        ),
        history_length_mismatches,
        positive_day_mismatches,
        demand_total_mismatches,
        products_not_continued_to_final_date,
        int(
            step3_cohort_window_feasibility_df[
                "TrainingRequirementViolations"
            ].sum()
        )
    ]
})


# ------------------------------------------------------------
# 4. Validate the Part 1 summary
# ------------------------------------------------------------

part1_summary_lookup = dict(
    zip(
        step3_part1_validation_summary_df[
            "ValidationMetric"
        ],
        step3_part1_validation_summary_df[
            "Value"
        ]
    )
)

assert part1_summary_lookup[
    "SourceRows"
] == 43_774

assert part1_summary_lookup[
    "SourceColumns"
] == 58

assert part1_summary_lookup[
    "CanonicalProducts"
] == 227

assert part1_summary_lookup[
    "OperatingDates"
] == 245

assert part1_summary_lookup[
    "DuplicateProductDateRows"
] == 0

assert part1_summary_lookup[
    "TotalDemandUnits"
] == 116_158

assert part1_summary_lookup[
    "StandardEvaluationWindows"
] == 3

assert part1_summary_lookup[
    "StandardEvaluationHorizonDays"
] == 20

assert part1_summary_lookup[
    "StandardModelSelectionWindows"
] == 2

assert part1_summary_lookup[
    "ReservedStandardFinalTestWindows"
] == 1

assert part1_summary_lookup[
    "LimitedEvaluationHorizonDays"
] == 5

assert part1_summary_lookup[
    "SameDayLeakageColumnsPresent"
] == 0

assert part1_summary_lookup[
    "TrainingRequirementViolations"
] == 0


# ------------------------------------------------------------
# 5. Define Step 3 Part 1 output paths
# ------------------------------------------------------------

STEP3_OPERATING_CALENDAR_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_operating_calendar.csv"
)

STEP3_WINDOW_CALENDAR_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_global_evaluation_window_calendar.csv"
)

STEP3_COHORT_POLICY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_cohort_split_policy.csv"
)

STEP3_POLICY_PARAMETERS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_split_policy_parameters.csv"
)

STEP3_WINDOW_FEASIBILITY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_cohort_window_feasibility.csv"
)

STEP3_PRODUCT_HISTORY_CHECK_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_product_history_validation.csv"
)

STEP3_PART1_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "13_step3_part1_validation_summary.csv"
)


# ------------------------------------------------------------
# 6. Save all Step 3 Part 1 outputs
# ------------------------------------------------------------

step3_operating_calendar_df.to_csv(
    STEP3_OPERATING_CALENDAR_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_global_window_calendar_df.to_csv(
    STEP3_WINDOW_CALENDAR_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_cohort_split_policy_df.to_csv(
    STEP3_COHORT_POLICY_OUTPUT,
    index=False
)

step3_split_policy_parameters_df.to_csv(
    STEP3_POLICY_PARAMETERS_OUTPUT,
    index=False
)

step3_cohort_window_feasibility_df.to_csv(
    STEP3_WINDOW_FEASIBILITY_OUTPUT,
    index=False
)

step3_product_history_check_df.to_csv(
    STEP3_PRODUCT_HISTORY_CHECK_OUTPUT,
    index=False
)

step3_part1_validation_summary_df.to_csv(
    STEP3_PART1_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 7. Reload and validate saved outputs
# ------------------------------------------------------------

saved_step3_calendar = pd.read_csv(
    STEP3_OPERATING_CALENDAR_OUTPUT,
    low_memory=False
)

saved_step3_windows = pd.read_csv(
    STEP3_WINDOW_CALENDAR_OUTPUT,
    low_memory=False
)

saved_step3_cohort_policy = pd.read_csv(
    STEP3_COHORT_POLICY_OUTPUT,
    low_memory=False
)

saved_step3_policy_parameters = pd.read_csv(
    STEP3_POLICY_PARAMETERS_OUTPUT,
    low_memory=False
)

saved_step3_feasibility = pd.read_csv(
    STEP3_WINDOW_FEASIBILITY_OUTPUT,
    low_memory=False
)

saved_step3_product_history = pd.read_csv(
    STEP3_PRODUCT_HISTORY_CHECK_OUTPUT,
    low_memory=False
)

saved_step3_validation = pd.read_csv(
    STEP3_PART1_VALIDATION_OUTPUT,
    low_memory=False
)


assert len(saved_step3_calendar) == 245

assert saved_step3_calendar[
    "OperatingDaySequence"
].nunique() == 245

assert len(saved_step3_windows) == 4

assert saved_step3_windows[
    "WindowID"
].nunique() == 4

assert len(saved_step3_cohort_policy) == 4

assert saved_step3_cohort_policy[
    "ProductCount"
].sum() == 227

assert saved_step3_feasibility[
    "TrainingRequirementViolations"
].sum() == 0

assert len(saved_step3_product_history) == 227

assert len(saved_step3_validation) == 19


# ------------------------------------------------------------
# 8. Build Markdown window summary
# ------------------------------------------------------------

window_markdown_rows = "\n".join(
    (
        f"- `{row.WindowID}`: training through "
        f"{pd.Timestamp(row.TrainingEndDate).date()}, "
        f"evaluation from "
        f"{pd.Timestamp(row.EvaluationStartDate).date()} "
        f"to {pd.Timestamp(row.EvaluationEndDate).date()} "
        f"({int(row.EvaluationHorizonOperatingDays)} "
        f"operating days); purpose `{row.WindowPurpose}`"
    )
    for row in (
        step3_global_window_calendar_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 9. Update the Markdown handoff
# ------------------------------------------------------------

step3_part1_summary = f"""
**Status:** Completed and validated

### Purpose

Step 3 Part 1 froze the chronological evaluation policy and
created the global operating-date windows used for product-level
backtesting and final holdout evaluation.

Random train/test splitting is prohibited.

### Standard evaluation design

- Evaluation horizon: {STANDARD_EVALUATION_HORIZON} operating days
- Minimum standard training history: {STANDARD_MINIMUM_TRAINING_DAYS} operating days
- Total standard evaluation windows: {STANDARD_EVALUATION_WINDOW_COUNT}
- Model-selection validation windows: {STANDARD_MODEL_SELECTION_WINDOW_COUNT}
- Reserved final-test windows: {STANDARD_FINAL_TEST_WINDOW_COUNT}

The first two standard windows may be used for model comparison.
The final standard window is reserved and must not be used for
feature selection, hyperparameter tuning or model choice.

### Limited-evidence design

- Evaluation horizon: {LIMITED_EVALUATION_HORIZON} operating days
- Minimum retained training history: {LIMITED_MINIMUM_TRAINING_DAYS} operating days
- Applicable cohort: `LIMITED_HOLDOUT_ONLY`

### Evaluation windows

{window_markdown_rows}

### Cohort treatment

- Multi-fold backtest ready: 127 products
- Single-holdout backtest ready: 77 products
- Limited-holdout only: 14 products
- Minimal-evidence forecast only: 9 products

Minimal-evidence products remain in the forecasting project but
do not receive an unreliable independent product-level holdout
score.

### Validation

- Training requirement violations: 0
- History-length mismatches: 0
- Positive-demand-day mismatches: 0
- Demand-total mismatches: 0
- Products not continued to the final operating date: 0
- Same-day leakage columns present: 0
- Total demand units preserved: 116,158

### Saved Step 3 Part 1 outputs

- `13_step3_operating_calendar.csv`
- `13_step3_global_evaluation_window_calendar.csv`
- `13_step3_cohort_split_policy.csv`
- `13_step3_split_policy_parameters.csv`
- `13_step3_cohort_window_feasibility.csv`
- `13_step3_product_history_validation.csv`
- `13_step3_part1_validation_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_3_part_1",
    section_title=(
        "Forecasting Preparation — Step 3, Part 1"
    ),
    section_body=step3_part1_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 10. Print final Part 1 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 3, PART 1 COMPLETED"
)
print("=" * 75)

print()
print("Chronological split source:")
print(
    f"Rows validated: "
    f"{len(step3_source_df):,}"
)
print(
    "Columns validated:",
    step3_source_df.shape[1]
)
print(
    "Canonical products:",
    step3_source_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    step3_source_df[
        "Date"
    ].nunique()
)

print()
print("Standard evaluation design:")
print(
    "Twenty-day model-selection windows:",
    STANDARD_MODEL_SELECTION_WINDOW_COUNT
)
print(
    "Twenty-day reserved final-test windows:",
    STANDARD_FINAL_TEST_WINDOW_COUNT
)
print(
    "Standard evaluation windows total:",
    STANDARD_EVALUATION_WINDOW_COUNT
)

print()
print("Limited evaluation design:")
print(
    "Limited holdout horizon:",
    LIMITED_EVALUATION_HORIZON,
    "operating days"
)
print(
    "Limited minimum training history:",
    LIMITED_MINIMUM_TRAINING_DAYS,
    "operating days"
)

print()
print("Validation:")
print(
    "Random splitting allowed:",
    False
)
print(
    "Training requirement violations:",
    int(
        saved_step3_feasibility[
            "TrainingRequirementViolations"
        ].sum()
    )
)
print(
    "Same-day leakage columns present:",
    len(
        same_day_leakage_columns_found
    )
)
print(
    "Total demand units:",
    f"{step3_source_df['TotalDemand'].sum():,}"
)

print()
print("Saved files:")
print(f"1. {STEP3_OPERATING_CALENDAR_OUTPUT}")
print(f"2. {STEP3_WINDOW_CALENDAR_OUTPUT}")
print(f"3. {STEP3_COHORT_POLICY_OUTPUT}")
print(f"4. {STEP3_POLICY_PARAMETERS_OUTPUT}")
print(f"5. {STEP3_WINDOW_FEASIBILITY_OUTPUT}")
print(f"6. {STEP3_PRODUCT_HISTORY_CHECK_OUTPUT}")
print(f"7. {STEP3_PART1_VALIDATION_OUTPUT}")
print(f"8. {HANDOFF_FILE}")

print()
print(
    "All Step 3, Part 1 validation checks passed."
)

FORECASTING PREPARATION — STEP 3, PART 1 COMPLETED

Chronological split source:
Rows validated: 43,774
Columns validated: 58
Canonical products: 227
Operating dates: 245

Standard evaluation design:
Twenty-day model-selection windows: 2
Twenty-day reserved final-test windows: 1
Standard evaluation windows total: 3

Limited evaluation design:
Limited holdout horizon: 5 operating days
Limited minimum training history: 25 operating days

Validation:
Random splitting allowed: False
Training requirement violations: 0
Same-day leakage columns present: 0
Total demand units: 116,158

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/13_step3_operating_calendar.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/13_step3_global_evaluation_window_calendar.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/13_step3_cohort_split_policy.csv
4. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_prep

In [56]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 2
# Cell 45: Assign products to chronological evaluation windows
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the earlier forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define official Step 3 Part 2 input files
# ------------------------------------------------------------

STEP3_SOURCE_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_model_view.csv"
)

STEP3_PRODUCT_STRATEGY_FILE = (
    FORECAST_PREPARATION_DIR
    / "06_product_modelling_strategy_register.csv"
)

STEP3_WINDOW_CALENDAR_FILE = (
    FORECAST_PREPARATION_DIR
    / "13_step3_global_evaluation_window_calendar.csv"
)

STEP3_COHORT_POLICY_FILE = (
    FORECAST_PREPARATION_DIR
    / "13_step3_cohort_split_policy.csv"
)

STEP3_COMBINED_FEATURE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_combined_feature_contract.csv"
)

required_part2_input_files = [
    STEP3_SOURCE_FILE,
    STEP3_PRODUCT_STRATEGY_FILE,
    STEP3_WINDOW_CALENDAR_FILE,
    STEP3_COHORT_POLICY_FILE,
    STEP3_COMBINED_FEATURE_CONTRACT_FILE
]

missing_part2_input_files = [
    file_path
    for file_path in required_part2_input_files
    if not file_path.exists()
]

if missing_part2_input_files:
    raise FileNotFoundError(
        "The following Step 3 Part 2 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_part2_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load official inputs
# ------------------------------------------------------------

step3_part2_source_df = pd.read_csv(
    STEP3_SOURCE_FILE,
    low_memory=False
)

step3_part2_product_strategy_df = pd.read_csv(
    STEP3_PRODUCT_STRATEGY_FILE,
    low_memory=False
)

step3_part2_window_calendar_df = pd.read_csv(
    STEP3_WINDOW_CALENDAR_FILE,
    low_memory=False
)

step3_part2_cohort_policy_df = pd.read_csv(
    STEP3_COHORT_POLICY_FILE,
    low_memory=False
)

step3_part2_feature_contract_df = pd.read_csv(
    STEP3_COMBINED_FEATURE_CONTRACT_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Boolean parsing helper
# ------------------------------------------------------------

def parse_step3_boolean(series):
    """
    Convert common CSV Boolean representations into bool values.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalised_values = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    parsed_values = normalised_values.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if parsed_values.isna().sum() > 0:

        invalid_values = (
            series.loc[
                parsed_values.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unrecognised Boolean values found:\n"
            f"{invalid_values}"
        )

    return parsed_values.astype(bool)


# ------------------------------------------------------------
# 5. Parse source dates and numeric columns
# ------------------------------------------------------------

for column in [
    "Date",
    "ProductFirstObservedDate"
]:

    step3_part2_source_df[column] = pd.to_datetime(
        step3_part2_source_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step3_part2_source_df[
    [
        "Date",
        "ProductFirstObservedDate"
    ]
].isna().sum().sum() == 0


for column in [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]:

    step3_part2_source_df[column] = pd.to_numeric(
        step3_part2_source_df[column],
        errors="coerce"
    )

assert step3_part2_source_df[
    [
        "OperatingDaySequence",
        "ProductAgeOperatingDays",
        "TotalDemand"
    ]
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 6. Parse window dates and numeric columns
# ------------------------------------------------------------

window_date_columns = [
    "TrainingEndDate",
    "EvaluationStartDate",
    "EvaluationEndDate"
]

for column in window_date_columns:

    step3_part2_window_calendar_df[column] = pd.to_datetime(
        step3_part2_window_calendar_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step3_part2_window_calendar_df[
    window_date_columns
].isna().sum().sum() == 0


window_numeric_columns = [
    "WindowOrder",
    "GlobalTrainingStartSequence",
    "TrainingEndSequence",
    "EvaluationStartSequence",
    "EvaluationEndSequence",
    "EvaluationHorizonOperatingDays",
    "MinimumRequiredProductTrainingDays",
    "GlobalTrainingOperatingDates"
]

for column in window_numeric_columns:

    step3_part2_window_calendar_df[column] = pd.to_numeric(
        step3_part2_window_calendar_df[column],
        errors="coerce"
    )

assert step3_part2_window_calendar_df[
    window_numeric_columns
].isna().sum().sum() == 0


step3_part2_window_calendar_df[
    "MayBeUsedForModelSelection"
] = parse_step3_boolean(
    step3_part2_window_calendar_df[
        "MayBeUsedForModelSelection"
    ]
)

step3_part2_window_calendar_df[
    "ReservedFinalTest"
] = parse_step3_boolean(
    step3_part2_window_calendar_df[
        "ReservedFinalTest"
    ]
)


# ------------------------------------------------------------
# 7. Standardise product identifiers
# ------------------------------------------------------------

for dataframe in [
    step3_part2_source_df,
    step3_part2_product_strategy_df
]:

    dataframe[
        "CanonicalProductID"
    ] = (
        dataframe[
            "CanonicalProductID"
        ]
        .astype("string")
        .str.strip()
    )

assert step3_part2_source_df[
    "CanonicalProductID"
].isna().sum() == 0

assert step3_part2_product_strategy_df[
    "CanonicalProductID"
].isna().sum() == 0


# ------------------------------------------------------------
# 8. Validate loaded structures
# ------------------------------------------------------------

assert step3_part2_source_df.shape == (
    43_774,
    58
)

assert step3_part2_source_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_part2_source_df[
    "Date"
].nunique() == 245

assert step3_part2_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step3_part2_source_df[
    "TotalDemand"
].sum() == 116_158

assert len(
    step3_part2_product_strategy_df
) == 227

assert step3_part2_product_strategy_df[
    "CanonicalProductID"
].nunique() == 227

assert len(
    step3_part2_window_calendar_df
) == 4

assert step3_part2_window_calendar_df[
    "WindowID"
].nunique() == 4

assert len(
    step3_part2_cohort_policy_df
) == 4

assert step3_part2_cohort_policy_df[
    "ProductCount"
].sum() == 227

assert len(
    step3_part2_feature_contract_df
) == 58


# ------------------------------------------------------------
# 9. Identify the 31 historical demand features
# ------------------------------------------------------------

historical_feature_flag = parse_step3_boolean(
    step3_part2_feature_contract_df[
        "IsHistoricalDemandFeature"
    ]
)

step3_historical_feature_columns = (
    step3_part2_feature_contract_df.loc[
        historical_feature_flag,
        "Column"
    ]
    .tolist()
)

assert len(
    step3_historical_feature_columns
) == 31

assert all(
    column in step3_part2_source_df.columns
    for column in step3_historical_feature_columns
)


# ------------------------------------------------------------
# 10. Reconstruct product history boundaries
# ------------------------------------------------------------

step3_product_boundary_df = (
    step3_part2_source_df
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        SourceFirstOperatingSequence=(
            "OperatingDaySequence",
            "min"
        ),
        SourceFinalOperatingSequence=(
            "OperatingDaySequence",
            "max"
        ),
        SourceFirstDate=(
            "Date",
            "min"
        ),
        SourceFinalDate=(
            "Date",
            "max"
        ),
        SourceHistoryRows=(
            "Date",
            "size"
        )
    )
)

assert len(step3_product_boundary_df) == 227

assert (
    step3_product_boundary_df[
        "SourceFinalOperatingSequence"
    ] == 245
).all()


# ------------------------------------------------------------
# 11. Expand eligible cohorts from the window calendar
# ------------------------------------------------------------

window_cohort_mapping_df = (
    step3_part2_window_calendar_df
    .assign(
        EvaluationCohort=(
            step3_part2_window_calendar_df[
                "EligibleCohorts"
            ]
            .astype(str)
            .str.split(
                "|",
                regex=False
            )
        )
    )
    .explode("EvaluationCohort")
    .reset_index(drop=True)
)

window_cohort_mapping_df[
    "EvaluationCohort"
] = (
    window_cohort_mapping_df[
        "EvaluationCohort"
    ]
    .astype(str)
    .str.strip()
)

assert len(window_cohort_mapping_df) == 5


# ------------------------------------------------------------
# 12. Create product-window assignments
# ------------------------------------------------------------

step3_product_window_assignment_df = (
    step3_part2_product_strategy_df
    .merge(
        window_cohort_mapping_df,
        on="EvaluationCohort",
        how="inner",
        validate="many_to_many"
    )
    .merge(
        step3_product_boundary_df,
        on="CanonicalProductID",
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# 13. Add product-specific training boundaries
# ------------------------------------------------------------

step3_product_window_assignment_df[
    "ProductTrainingStartSequence"
] = (
    step3_product_window_assignment_df[
        "SourceFirstOperatingSequence"
    ]
)

step3_product_window_assignment_df[
    "ProductTrainingStartDate"
] = (
    step3_product_window_assignment_df[
        "SourceFirstDate"
    ]
)

step3_product_window_assignment_df[
    "AvailableTrainingOperatingDays"
] = (
    step3_product_window_assignment_df[
        "TrainingEndSequence"
    ]
    - step3_product_window_assignment_df[
        "ProductTrainingStartSequence"
    ]
    + 1
)

step3_product_window_assignment_df[
    "ExpectedEvaluationOperatingDays"
] = (
    step3_product_window_assignment_df[
        "EvaluationEndSequence"
    ]
    - step3_product_window_assignment_df[
        "EvaluationStartSequence"
    ]
    + 1
)


# ------------------------------------------------------------
# 14. Add temporal eligibility checks
# ------------------------------------------------------------

step3_product_window_assignment_df[
    "TrainingRequirementViolation"
] = (
    step3_product_window_assignment_df[
        "AvailableTrainingOperatingDays"
    ]
    < step3_product_window_assignment_df[
        "MinimumRequiredProductTrainingDays"
    ]
)

step3_product_window_assignment_df[
    "EvaluationStartsAfterTraining"
] = (
    step3_product_window_assignment_df[
        "EvaluationStartSequence"
    ]
    > step3_product_window_assignment_df[
        "TrainingEndSequence"
    ]
)

step3_product_window_assignment_df[
    "EvaluationStartsAfterProductIntroduction"
] = (
    step3_product_window_assignment_df[
        "EvaluationStartSequence"
    ]
    > step3_product_window_assignment_df[
        "SourceFirstOperatingSequence"
    ]
)

step3_product_window_assignment_df[
    "ProductContinuesThroughEvaluationEnd"
] = (
    step3_product_window_assignment_df[
        "SourceFinalOperatingSequence"
    ]
    >= step3_product_window_assignment_df[
        "EvaluationEndSequence"
    ]
)

step3_product_window_assignment_df[
    "ExpectedEvaluationWindowComplete"
] = (
    step3_product_window_assignment_df[
        "ExpectedEvaluationOperatingDays"
    ]
    == step3_product_window_assignment_df[
        "EvaluationHorizonOperatingDays"
    ]
)

step3_product_window_assignment_df[
    "ChronologicalAssignmentStatus"
] = np.where(
    (
        ~step3_product_window_assignment_df[
            "TrainingRequirementViolation"
        ]
        &
        step3_product_window_assignment_df[
            "EvaluationStartsAfterTraining"
        ]
        &
        step3_product_window_assignment_df[
            "EvaluationStartsAfterProductIntroduction"
        ]
        &
        step3_product_window_assignment_df[
            "ProductContinuesThroughEvaluationEnd"
        ]
        &
        step3_product_window_assignment_df[
            "ExpectedEvaluationWindowComplete"
        ]
    ),
    "ELIGIBLE",
    "REVIEW_REQUIRED"
)


# ------------------------------------------------------------
# 15. Create the no-independent-holdout register
# ------------------------------------------------------------

step3_no_independent_holdout_df = (
    step3_part2_product_strategy_df.loc[
        step3_part2_product_strategy_df[
            "EvaluationCohort"
        ]
        == "MINIMAL_EVIDENCE_FORECAST_ONLY"
    ]
    .copy()
    .sort_values(
        [
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

step3_no_independent_holdout_df[
    "IndependentHoldoutAvailable"
] = False

step3_no_independent_holdout_df[
    "SplitTreatment"
] = (
    "NO_INDEPENDENT_PRODUCT_LEVEL_HOLDOUT"
)

step3_no_independent_holdout_df[
    "Reason"
] = (
    "INSUFFICIENT_PRODUCT_LEVEL_HISTORY_OR_"
    "POSITIVE_DEMAND_EVIDENCE"
)

assert len(
    step3_no_independent_holdout_df
) == 9


# ------------------------------------------------------------
# 16. Validate assignment counts
# ------------------------------------------------------------

assert len(
    step3_product_window_assignment_df
) == 472

assert step3_product_window_assignment_df[
    [
        "CanonicalProductID",
        "WindowID"
    ]
].duplicated().sum() == 0


window_assignment_count_lookup = (
    step3_product_window_assignment_df[
        "WindowID"
    ]
    .value_counts()
    .to_dict()
)

assert window_assignment_count_lookup.get(
    "STANDARD_BACKTEST_FOLD_1",
    0
) == 127

assert window_assignment_count_lookup.get(
    "STANDARD_BACKTEST_FOLD_2",
    0
) == 127

assert window_assignment_count_lookup.get(
    "STANDARD_FINAL_HOLDOUT",
    0
) == 204

assert window_assignment_count_lookup.get(
    "LIMITED_FINAL_HOLDOUT",
    0
) == 14


cohort_assignment_count_lookup = (
    step3_product_window_assignment_df
    .groupby("EvaluationCohort")
    .size()
    .to_dict()
)

assert cohort_assignment_count_lookup.get(
    "MULTI_FOLD_BACKTEST_READY",
    0
) == 381

assert cohort_assignment_count_lookup.get(
    "SINGLE_HOLDOUT_BACKTEST_READY",
    0
) == 77

assert cohort_assignment_count_lookup.get(
    "LIMITED_HOLDOUT_ONLY",
    0
) == 14


# ------------------------------------------------------------
# 17. Validate all temporal assignment rules
# ------------------------------------------------------------

training_requirement_violations = int(
    step3_product_window_assignment_df[
        "TrainingRequirementViolation"
    ].sum()
)

evaluation_before_training_violations = int(
    (
        ~step3_product_window_assignment_df[
            "EvaluationStartsAfterTraining"
        ]
    ).sum()
)

product_start_violations = int(
    (
        ~step3_product_window_assignment_df[
            "EvaluationStartsAfterProductIntroduction"
        ]
    ).sum()
)

product_end_violations = int(
    (
        ~step3_product_window_assignment_df[
            "ProductContinuesThroughEvaluationEnd"
        ]
    ).sum()
)

evaluation_window_length_violations = int(
    (
        ~step3_product_window_assignment_df[
            "ExpectedEvaluationWindowComplete"
        ]
    ).sum()
)

assignments_requiring_review = int(
    (
        step3_product_window_assignment_df[
            "ChronologicalAssignmentStatus"
        ]
        != "ELIGIBLE"
    ).sum()
)

assert training_requirement_violations == 0
assert evaluation_before_training_violations == 0
assert product_start_violations == 0
assert product_end_violations == 0
assert evaluation_window_length_violations == 0
assert assignments_requiring_review == 0


# ------------------------------------------------------------
# 18. Confirm all 227 products are accounted for
# ------------------------------------------------------------

assigned_product_ids = set(
    step3_product_window_assignment_df[
        "CanonicalProductID"
    ]
)

no_holdout_product_ids = set(
    step3_no_independent_holdout_df[
        "CanonicalProductID"
    ]
)

assert assigned_product_ids.isdisjoint(
    no_holdout_product_ids
)

assert len(assigned_product_ids) == 218

assert len(no_holdout_product_ids) == 9

assert (
    assigned_product_ids
    | no_holdout_product_ids
) == set(
    step3_part2_product_strategy_df[
        "CanonicalProductID"
    ]
)


# ------------------------------------------------------------
# 19. Sort the official assignment register
# ------------------------------------------------------------

step3_product_window_assignment_df = (
    step3_product_window_assignment_df
    .sort_values(
        [
            "EvaluationCohortPriority",
            "CanonicalProductID",
            "EvaluationStartSequence"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 20. Print Cell 45 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 2 — "
    "PRODUCT-WINDOW ASSIGNMENTS CREATED"
)
print("=" * 75)

print()
print("Product-window assignments:")
print(
    "Total assignments:",
    len(step3_product_window_assignment_df)
)
print(
    "Products with independent evaluation:",
    len(assigned_product_ids)
)
print(
    "Products without independent holdout:",
    len(no_holdout_product_ids)
)

print()
print("Assignments by window:")
display(
    step3_product_window_assignment_df[
        "WindowID"
    ]
    .value_counts()
    .rename_axis("WindowID")
    .reset_index(name="AssignmentCount")
)

print()
print("Assignments by cohort:")
display(
    step3_product_window_assignment_df
    .groupby(
        "EvaluationCohort",
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        ProductWindowAssignments=(
            "WindowID",
            "size"
        )
    )
)

print()
print("Temporal validation:")
print(
    "Training requirement violations:",
    training_requirement_violations
)
print(
    "Evaluation-before-training violations:",
    evaluation_before_training_violations
)
print(
    "Product-start violations:",
    product_start_violations
)
print(
    "Product-end violations:",
    product_end_violations
)
print(
    "Evaluation-window-length violations:",
    evaluation_window_length_violations
)
print(
    "Assignments requiring review:",
    assignments_requiring_review
)

print()
print("Cell 45 completed successfully.")

STEP 3, PART 2 — PRODUCT-WINDOW ASSIGNMENTS CREATED

Product-window assignments:
Total assignments: 472
Products with independent evaluation: 218
Products without independent holdout: 9

Assignments by window:


,WindowID,AssignmentCount
0,STANDARD_FINAL_HOLDOUT,204
1,STANDARD_BACKTEST_FOLD_1,127
2,STANDARD_BACKTEST_FOLD_2,127
3,LIMITED_FINAL_HOLDOUT,14



Assignments by cohort:


,EvaluationCohort,ProductCount,ProductWindowAssignments
0,LIMITED_HOLDOUT_ONLY,14,14
1,MULTI_FOLD_BACKTEST_READY,127,381
2,SINGLE_HOLDOUT_BACKTEST_READY,77,77



Temporal validation:
Training requirement violations: 0
Evaluation-before-training violations: 0
Product-start violations: 0
Product-end violations: 0
Evaluation-window-length violations: 0
Assignments requiring review: 0

Cell 45 completed successfully.


In [57]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 2
# Cell 46: Validate actual product-window row coverage,
# demand coverage and historical-feature readiness
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 45 objects exist
# ------------------------------------------------------------

required_cell_45_objects = [
    "step3_part2_source_df",
    "step3_product_window_assignment_df",
    "step3_no_independent_holdout_df",
    "step3_historical_feature_columns"
]

missing_cell_45_objects = [
    object_name
    for object_name in required_cell_45_objects
    if object_name not in globals()
]

if missing_cell_45_objects:
    raise NameError(
        "The following Cell 45 objects are missing:\n"
        f"{missing_cell_45_objects}\n\n"
        "Run Cell 45 before running Cell 46."
    )


# ------------------------------------------------------------
# 2. Prepare one ordered source group per product
# ------------------------------------------------------------

step3_source_by_product = {
    str(canonical_product_id): (
        product_group
        .sort_values(
            "OperatingDaySequence",
            kind="stable"
        )
        .reset_index(drop=True)
    )
    for (
        canonical_product_id,
        product_group
    ) in step3_part2_source_df.groupby(
        "CanonicalProductID",
        sort=False
    )
}

assert len(step3_source_by_product) == 227


# ------------------------------------------------------------
# 3. Calculate actual coverage for every assignment
# ------------------------------------------------------------

product_window_coverage_records = []

for assignment in (
    step3_product_window_assignment_df
    .itertuples(index=False)
):

    canonical_product_id = str(
        assignment.CanonicalProductID
    )

    product_rows = (
        step3_source_by_product[
            canonical_product_id
        ]
    )

    training_rows = (
        product_rows.loc[
            product_rows[
                "OperatingDaySequence"
            ]
            <= int(
                assignment.TrainingEndSequence
            )
        ]
        .copy()
    )

    evaluation_rows = (
        product_rows.loc[
            product_rows[
                "OperatingDaySequence"
            ].between(
                int(
                    assignment.EvaluationStartSequence
                ),
                int(
                    assignment.EvaluationEndSequence
                )
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Historical feature readiness in evaluation rows
    # --------------------------------------------------------

    evaluation_complete_feature_mask = (
        evaluation_rows[
            step3_historical_feature_columns
        ]
        .notna()
        .all(axis=1)
    )

    evaluation_missing_feature_cells = int(
        evaluation_rows[
            step3_historical_feature_columns
        ]
        .isna()
        .sum()
        .sum()
    )


    # --------------------------------------------------------
    # Training coverage
    # --------------------------------------------------------

    training_row_count = len(training_rows)

    training_positive_demand_days = int(
        (
            training_rows[
                "TotalDemand"
            ] > 0
        ).sum()
    )

    training_zero_demand_days = int(
        (
            training_rows[
                "TotalDemand"
            ] == 0
        ).sum()
    )

    training_total_demand_units = float(
        training_rows[
            "TotalDemand"
        ].sum()
    )


    # --------------------------------------------------------
    # Evaluation coverage
    # --------------------------------------------------------

    evaluation_row_count = len(
        evaluation_rows
    )

    evaluation_positive_demand_days = int(
        (
            evaluation_rows[
                "TotalDemand"
            ] > 0
        ).sum()
    )

    evaluation_zero_demand_days = int(
        (
            evaluation_rows[
                "TotalDemand"
            ] == 0
        ).sum()
    )

    evaluation_total_demand_units = float(
        evaluation_rows[
            "TotalDemand"
        ].sum()
    )


    # --------------------------------------------------------
    # Actual temporal boundaries
    # --------------------------------------------------------

    actual_training_start_sequence = int(
        training_rows[
            "OperatingDaySequence"
        ].min()
    )

    actual_training_end_sequence = int(
        training_rows[
            "OperatingDaySequence"
        ].max()
    )

    actual_evaluation_start_sequence = int(
        evaluation_rows[
            "OperatingDaySequence"
        ].min()
    )

    actual_evaluation_end_sequence = int(
        evaluation_rows[
            "OperatingDaySequence"
        ].max()
    )


    product_window_coverage_records.append({
        "CanonicalProductID":
            canonical_product_id,

        "CanonicalProductName":
            assignment.CanonicalProductName,

        "EvaluationCohort":
            assignment.EvaluationCohort,

        "DemandPatternClass":
            assignment.DemandPatternClass,

        "WindowID":
            assignment.WindowID,

        "WindowPurpose":
            assignment.WindowPurpose,

        "MayBeUsedForModelSelection":
            bool(
                assignment.MayBeUsedForModelSelection
            ),

        "ReservedFinalTest":
            bool(
                assignment.ReservedFinalTest
            ),

        "ExpectedTrainingRows":
            int(
                assignment.AvailableTrainingOperatingDays
            ),

        "ActualTrainingRows":
            training_row_count,

        "TrainingPositiveDemandDays":
            training_positive_demand_days,

        "TrainingZeroDemandDays":
            training_zero_demand_days,

        "TrainingTotalDemandUnits":
            training_total_demand_units,

        "ExpectedEvaluationRows":
            int(
                assignment.ExpectedEvaluationOperatingDays
            ),

        "ActualEvaluationRows":
            evaluation_row_count,

        "EvaluationPositiveDemandDays":
            evaluation_positive_demand_days,

        "EvaluationZeroDemandDays":
            evaluation_zero_demand_days,

        "EvaluationTotalDemandUnits":
            evaluation_total_demand_units,

        "EvaluationAllZeroDemand":
            bool(
                evaluation_total_demand_units == 0
            ),

        "EvaluationRowsWithAllHistoricalFeatures":
            int(
                evaluation_complete_feature_mask.sum()
            ),

        "EvaluationRowsWithMissingHistoricalFeatures":
            int(
                (
                    ~evaluation_complete_feature_mask
                ).sum()
            ),

        "EvaluationHistoricalFeatureMissingCells":
            evaluation_missing_feature_cells,

        "ExpectedTrainingStartSequence":
            int(
                assignment.ProductTrainingStartSequence
            ),

        "ActualTrainingStartSequence":
            actual_training_start_sequence,

        "ExpectedTrainingEndSequence":
            int(
                assignment.TrainingEndSequence
            ),

        "ActualTrainingEndSequence":
            actual_training_end_sequence,

        "ExpectedEvaluationStartSequence":
            int(
                assignment.EvaluationStartSequence
            ),

        "ActualEvaluationStartSequence":
            actual_evaluation_start_sequence,

        "ExpectedEvaluationEndSequence":
            int(
                assignment.EvaluationEndSequence
            ),

        "ActualEvaluationEndSequence":
            actual_evaluation_end_sequence,

        "ActualTrainingStartDate":
            training_rows["Date"].min(),

        "ActualTrainingEndDate":
            training_rows["Date"].max(),

        "ActualEvaluationStartDate":
            evaluation_rows["Date"].min(),

        "ActualEvaluationEndDate":
            evaluation_rows["Date"].max()
    })


step3_product_window_coverage_df = pd.DataFrame(
    product_window_coverage_records
)

assert len(
    step3_product_window_coverage_df
) == 472


# ------------------------------------------------------------
# 4. Create detailed validation flags
# ------------------------------------------------------------

step3_product_window_coverage_df[
    "TrainingRowCountMismatch"
] = (
    step3_product_window_coverage_df[
        "ActualTrainingRows"
    ]
    != step3_product_window_coverage_df[
        "ExpectedTrainingRows"
    ]
)

step3_product_window_coverage_df[
    "EvaluationRowCountMismatch"
] = (
    step3_product_window_coverage_df[
        "ActualEvaluationRows"
    ]
    != step3_product_window_coverage_df[
        "ExpectedEvaluationRows"
    ]
)

step3_product_window_coverage_df[
    "TrainingStartSequenceMismatch"
] = (
    step3_product_window_coverage_df[
        "ActualTrainingStartSequence"
    ]
    != step3_product_window_coverage_df[
        "ExpectedTrainingStartSequence"
    ]
)

step3_product_window_coverage_df[
    "TrainingEndSequenceMismatch"
] = (
    step3_product_window_coverage_df[
        "ActualTrainingEndSequence"
    ]
    != step3_product_window_coverage_df[
        "ExpectedTrainingEndSequence"
    ]
)

step3_product_window_coverage_df[
    "EvaluationStartSequenceMismatch"
] = (
    step3_product_window_coverage_df[
        "ActualEvaluationStartSequence"
    ]
    != step3_product_window_coverage_df[
        "ExpectedEvaluationStartSequence"
    ]
)

step3_product_window_coverage_df[
    "EvaluationEndSequenceMismatch"
] = (
    step3_product_window_coverage_df[
        "ActualEvaluationEndSequence"
    ]
    != step3_product_window_coverage_df[
        "ExpectedEvaluationEndSequence"
    ]
)

step3_product_window_coverage_df[
    "ChronologicalOverlapViolation"
] = (
    step3_product_window_coverage_df[
        "ActualTrainingEndSequence"
    ]
    >= step3_product_window_coverage_df[
        "ActualEvaluationStartSequence"
    ]
)

step3_product_window_coverage_df[
    "EvaluationFeatureReadinessViolation"
] = (
    step3_product_window_coverage_df[
        "EvaluationRowsWithMissingHistoricalFeatures"
    ] > 0
)


# ------------------------------------------------------------
# 5. Calculate validation totals
# ------------------------------------------------------------

training_row_count_mismatches = int(
    step3_product_window_coverage_df[
        "TrainingRowCountMismatch"
    ].sum()
)

evaluation_row_count_mismatches = int(
    step3_product_window_coverage_df[
        "EvaluationRowCountMismatch"
    ].sum()
)

training_start_mismatches = int(
    step3_product_window_coverage_df[
        "TrainingStartSequenceMismatch"
    ].sum()
)

training_end_mismatches = int(
    step3_product_window_coverage_df[
        "TrainingEndSequenceMismatch"
    ].sum()
)

evaluation_start_mismatches = int(
    step3_product_window_coverage_df[
        "EvaluationStartSequenceMismatch"
    ].sum()
)

evaluation_end_mismatches = int(
    step3_product_window_coverage_df[
        "EvaluationEndSequenceMismatch"
    ].sum()
)

chronological_overlap_violations = int(
    step3_product_window_coverage_df[
        "ChronologicalOverlapViolation"
    ].sum()
)

evaluation_feature_readiness_violations = int(
    step3_product_window_coverage_df[
        "EvaluationFeatureReadinessViolation"
    ].sum()
)

evaluation_missing_historical_cells = int(
    step3_product_window_coverage_df[
        "EvaluationHistoricalFeatureMissingCells"
    ].sum()
)


assert training_row_count_mismatches == 0
assert evaluation_row_count_mismatches == 0
assert training_start_mismatches == 0
assert training_end_mismatches == 0
assert evaluation_start_mismatches == 0
assert evaluation_end_mismatches == 0
assert chronological_overlap_violations == 0
assert evaluation_feature_readiness_violations == 0
assert evaluation_missing_historical_cells == 0


# ------------------------------------------------------------
# 6. Create product-window demand audit
# ------------------------------------------------------------

step3_zero_demand_evaluation_audit_df = (
    step3_product_window_coverage_df.loc[
        step3_product_window_coverage_df[
            "EvaluationAllZeroDemand"
        ]
    ]
    .copy()
    .sort_values(
        [
            "WindowID",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

step3_zero_demand_evaluation_audit_df[
    "AuditInterpretation"
] = (
    "VALID_ZERO_DEMAND_EVALUATION_WINDOW"
)

step3_zero_demand_evaluation_audit_df[
    "RemovedFromEvaluation"
] = False


# ------------------------------------------------------------
# 7. Create evaluation feature-readiness audit
# ------------------------------------------------------------

step3_evaluation_feature_readiness_audit_df = (
    step3_product_window_coverage_df.loc[
        step3_product_window_coverage_df[
            "EvaluationFeatureReadinessViolation"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(
    step3_evaluation_feature_readiness_audit_df
) == 0


# ------------------------------------------------------------
# 8. Create window-level summary
# ------------------------------------------------------------

step3_window_assignment_summary_df = (
    step3_product_window_coverage_df
    .groupby(
        [
            "WindowID",
            "WindowPurpose"
        ],
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        ProductWindowAssignments=(
            "CanonicalProductID",
            "size"
        ),
        TotalTrainingRows=(
            "ActualTrainingRows",
            "sum"
        ),
        TotalEvaluationRows=(
            "ActualEvaluationRows",
            "sum"
        ),
        EvaluationPositiveDemandRows=(
            "EvaluationPositiveDemandDays",
            "sum"
        ),
        EvaluationZeroDemandRows=(
            "EvaluationZeroDemandDays",
            "sum"
        ),
        EvaluationTotalDemandUnits=(
            "EvaluationTotalDemandUnits",
            "sum"
        ),
        AllZeroDemandProductWindows=(
            "EvaluationAllZeroDemand",
            "sum"
        ),
        EvaluationRowsWithMissingHistoricalFeatures=(
            "EvaluationRowsWithMissingHistoricalFeatures",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 9. Create cohort-level summary
# ------------------------------------------------------------

step3_cohort_assignment_summary_df = (
    step3_product_window_coverage_df
    .groupby(
        "EvaluationCohort",
        as_index=False
    )
    .agg(
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        ProductWindowAssignments=(
            "WindowID",
            "size"
        ),
        EvaluationRows=(
            "ActualEvaluationRows",
            "sum"
        ),
        EvaluationPositiveDemandRows=(
            "EvaluationPositiveDemandDays",
            "sum"
        ),
        EvaluationZeroDemandRows=(
            "EvaluationZeroDemandDays",
            "sum"
        ),
        EvaluationTotalDemandUnits=(
            "EvaluationTotalDemandUnits",
            "sum"
        ),
        AllZeroDemandProductWindows=(
            "EvaluationAllZeroDemand",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 10. Validate summary totals
# ------------------------------------------------------------

assert step3_window_assignment_summary_df[
    "ProductWindowAssignments"
].sum() == 472

assert step3_cohort_assignment_summary_df[
    "ProductWindowAssignments"
].sum() == 472

assert step3_window_assignment_summary_df[
    "EvaluationRowsWithMissingHistoricalFeatures"
].sum() == 0

assert (
    step3_product_window_coverage_df[
        "TrainingPositiveDemandDays"
    ]
    + step3_product_window_coverage_df[
        "TrainingZeroDemandDays"
    ]
    == step3_product_window_coverage_df[
        "ActualTrainingRows"
    ]
).all()

assert (
    step3_product_window_coverage_df[
        "EvaluationPositiveDemandDays"
    ]
    + step3_product_window_coverage_df[
        "EvaluationZeroDemandDays"
    ]
    == step3_product_window_coverage_df[
        "ActualEvaluationRows"
    ]
).all()


# ------------------------------------------------------------
# 11. Print Cell 46 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 2 — "
    "PRODUCT-WINDOW COVERAGE VALIDATED"
)
print("=" * 75)

print()
print("Window-level coverage:")
display(step3_window_assignment_summary_df)

print()
print("Cohort-level coverage:")
display(step3_cohort_assignment_summary_df)

print()
print("Validation:")
print(
    "Training-row-count mismatches:",
    training_row_count_mismatches
)
print(
    "Evaluation-row-count mismatches:",
    evaluation_row_count_mismatches
)
print(
    "Training-boundary mismatches:",
    (
        training_start_mismatches
        + training_end_mismatches
    )
)
print(
    "Evaluation-boundary mismatches:",
    (
        evaluation_start_mismatches
        + evaluation_end_mismatches
    )
)
print(
    "Chronological overlap violations:",
    chronological_overlap_violations
)
print(
    "Evaluation feature-readiness violations:",
    evaluation_feature_readiness_violations
)
print(
    "Missing historical-feature cells "
    "inside evaluation windows:",
    evaluation_missing_historical_cells
)

print()
print("Intermittent-demand audit:")
print(
    "All-zero-demand product-window assignments:",
    len(
        step3_zero_demand_evaluation_audit_df
    )
)
print(
    "All-zero windows removed:",
    0
)

print()
print("Cell 46 completed successfully.")

STEP 3, PART 2 — PRODUCT-WINDOW COVERAGE VALIDATED

Window-level coverage:


,WindowID,WindowPurpose,ProductCount,ProductWindowAssignments,TotalTrainingRows,TotalEvaluationRows,EvaluationPositiveDemandRows,EvaluationZeroDemandRows,EvaluationTotalDemandUnits,AllZeroDemandProductWindows,EvaluationRowsWithMissingHistoricalFeatures
0,LIMITED_FINAL_HOLDOUT,LIMITED_FINAL_HOLDOUT_TEST,14,14,2104,70,3,67,61.0,13,0
1,STANDARD_BACKTEST_FOLD_1,BACKTEST_VALIDATION,127,127,22999,2540,361,2179,3309.0,105,0
2,STANDARD_BACKTEST_FOLD_2,BACKTEST_VALIDATION,127,127,25539,2540,397,2143,4808.0,105,0
3,STANDARD_FINAL_HOLDOUT,FINAL_HOLDOUT_TEST,204,204,36524,4080,1287,2793,13736.0,121,0



Cohort-level coverage:


,EvaluationCohort,ProductCount,ProductWindowAssignments,EvaluationRows,EvaluationPositiveDemandRows,EvaluationZeroDemandRows,EvaluationTotalDemandUnits,AllZeroDemandProductWindows
0,LIMITED_HOLDOUT_ONLY,14,14,70,3,67,61.0,13
1,MULTI_FOLD_BACKTEST_READY,127,381,7620,1166,6454,12879.0,315
2,SINGLE_HOLDOUT_BACKTEST_READY,77,77,1540,879,661,8974.0,16



Validation:
Training-row-count mismatches: 0
Evaluation-row-count mismatches: 0
Training-boundary mismatches: 0
Evaluation-boundary mismatches: 0
Chronological overlap violations: 0
Evaluation feature-readiness violations: 0
Missing historical-feature cells inside evaluation windows: 0

Intermittent-demand audit:
All-zero-demand product-window assignments: 344
All-zero windows removed: 0

Cell 46 completed successfully.


In [58]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 2
# Cell 47: Save product-window assignments and update handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 2 objects exist
# ------------------------------------------------------------

required_step3_part2_objects = [
    "step3_product_window_assignment_df",
    "step3_product_window_coverage_df",
    "step3_window_assignment_summary_df",
    "step3_cohort_assignment_summary_df",
    "step3_zero_demand_evaluation_audit_df",
    "step3_evaluation_feature_readiness_audit_df",
    "step3_no_independent_holdout_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step3_part2_objects = [
    object_name
    for object_name in required_step3_part2_objects
    if object_name not in globals()
]

if missing_step3_part2_objects:
    raise NameError(
        "The following Step 3 Part 2 objects are missing:\n"
        f"{missing_step3_part2_objects}\n\n"
        "Run Cells 45 and 46 before running Cell 47."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if required
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Create Part 2 validation summary
# ------------------------------------------------------------

step3_part2_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "ProductWindowAssignments",
        "ProductsWithIndependentEvaluation",
        "ProductsWithoutIndependentHoldout",
        "MultiFoldProductWindowAssignments",
        "SingleHoldoutProductWindowAssignments",
        "LimitedHoldoutProductWindowAssignments",
        "StandardFold1Assignments",
        "StandardFold2Assignments",
        "StandardFinalHoldoutAssignments",
        "LimitedFinalHoldoutAssignments",
        "TrainingRequirementViolations",
        "TrainingRowCountMismatches",
        "EvaluationRowCountMismatches",
        "TrainingBoundaryMismatches",
        "EvaluationBoundaryMismatches",
        "ChronologicalOverlapViolations",
        "EvaluationFeatureReadinessViolations",
        "EvaluationHistoricalFeatureMissingCells",
        "AllZeroDemandProductWindows",
        "AllZeroWindowsRemoved",
        "TotalDemandUnitsInSource"
    ],
    "Value": [
        len(
            step3_product_window_assignment_df
        ),
        step3_product_window_assignment_df[
            "CanonicalProductID"
        ].nunique(),
        len(
            step3_no_independent_holdout_df
        ),
        int(
            (
                step3_product_window_assignment_df[
                    "EvaluationCohort"
                ]
                == "MULTI_FOLD_BACKTEST_READY"
            ).sum()
        ),
        int(
            (
                step3_product_window_assignment_df[
                    "EvaluationCohort"
                ]
                == "SINGLE_HOLDOUT_BACKTEST_READY"
            ).sum()
        ),
        int(
            (
                step3_product_window_assignment_df[
                    "EvaluationCohort"
                ]
                == "LIMITED_HOLDOUT_ONLY"
            ).sum()
        ),
        window_assignment_count_lookup[
            "STANDARD_BACKTEST_FOLD_1"
        ],
        window_assignment_count_lookup[
            "STANDARD_BACKTEST_FOLD_2"
        ],
        window_assignment_count_lookup[
            "STANDARD_FINAL_HOLDOUT"
        ],
        window_assignment_count_lookup[
            "LIMITED_FINAL_HOLDOUT"
        ],
        training_requirement_violations,
        training_row_count_mismatches,
        evaluation_row_count_mismatches,
        (
            training_start_mismatches
            + training_end_mismatches
        ),
        (
            evaluation_start_mismatches
            + evaluation_end_mismatches
        ),
        chronological_overlap_violations,
        evaluation_feature_readiness_violations,
        evaluation_missing_historical_cells,
        len(
            step3_zero_demand_evaluation_audit_df
        ),
        0,
        int(
            step3_part2_source_df[
                "TotalDemand"
            ].sum()
        )
    ]
})


# ------------------------------------------------------------
# 4. Validate the summary
# ------------------------------------------------------------

part2_summary_lookup = dict(
    zip(
        step3_part2_validation_summary_df[
            "ValidationMetric"
        ],
        step3_part2_validation_summary_df[
            "Value"
        ]
    )
)

assert part2_summary_lookup[
    "ProductWindowAssignments"
] == 472

assert part2_summary_lookup[
    "ProductsWithIndependentEvaluation"
] == 218

assert part2_summary_lookup[
    "ProductsWithoutIndependentHoldout"
] == 9

assert part2_summary_lookup[
    "MultiFoldProductWindowAssignments"
] == 381

assert part2_summary_lookup[
    "SingleHoldoutProductWindowAssignments"
] == 77

assert part2_summary_lookup[
    "LimitedHoldoutProductWindowAssignments"
] == 14

assert part2_summary_lookup[
    "TrainingRequirementViolations"
] == 0

assert part2_summary_lookup[
    "TrainingRowCountMismatches"
] == 0

assert part2_summary_lookup[
    "EvaluationRowCountMismatches"
] == 0

assert part2_summary_lookup[
    "ChronologicalOverlapViolations"
] == 0

assert part2_summary_lookup[
    "EvaluationFeatureReadinessViolations"
] == 0

assert part2_summary_lookup[
    "EvaluationHistoricalFeatureMissingCells"
] == 0

assert part2_summary_lookup[
    "AllZeroWindowsRemoved"
] == 0

assert part2_summary_lookup[
    "TotalDemandUnitsInSource"
] == 116_158


# ------------------------------------------------------------
# 5. Define Step 3 Part 2 output paths
# ------------------------------------------------------------

STEP3_PRODUCT_WINDOW_ASSIGNMENT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_product_window_assignment_register.csv"
)

STEP3_PRODUCT_WINDOW_COVERAGE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_product_window_coverage_audit.csv"
)

STEP3_WINDOW_ASSIGNMENT_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_window_assignment_summary.csv"
)

STEP3_COHORT_ASSIGNMENT_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_cohort_assignment_summary.csv"
)

STEP3_ZERO_DEMAND_EVALUATION_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_zero_demand_evaluation_audit.csv"
)

STEP3_EVALUATION_FEATURE_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_evaluation_feature_readiness_audit.csv"
)

STEP3_NO_HOLDOUT_REGISTER_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_no_independent_holdout_register.csv"
)

STEP3_PART2_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "14_step3_part2_validation_summary.csv"
)


# ------------------------------------------------------------
# 6. Save Part 2 outputs
# ------------------------------------------------------------

step3_product_window_assignment_df.to_csv(
    STEP3_PRODUCT_WINDOW_ASSIGNMENT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_product_window_coverage_df.to_csv(
    STEP3_PRODUCT_WINDOW_COVERAGE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_window_assignment_summary_df.to_csv(
    STEP3_WINDOW_ASSIGNMENT_SUMMARY_OUTPUT,
    index=False
)

step3_cohort_assignment_summary_df.to_csv(
    STEP3_COHORT_ASSIGNMENT_SUMMARY_OUTPUT,
    index=False
)

step3_zero_demand_evaluation_audit_df.to_csv(
    STEP3_ZERO_DEMAND_EVALUATION_AUDIT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_evaluation_feature_readiness_audit_df.to_csv(
    STEP3_EVALUATION_FEATURE_READINESS_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_no_independent_holdout_df.to_csv(
    STEP3_NO_HOLDOUT_REGISTER_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part2_validation_summary_df.to_csv(
    STEP3_PART2_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 7. Reload saved outputs
# ------------------------------------------------------------

saved_step3_assignments = pd.read_csv(
    STEP3_PRODUCT_WINDOW_ASSIGNMENT_OUTPUT,
    low_memory=False
)

saved_step3_coverage = pd.read_csv(
    STEP3_PRODUCT_WINDOW_COVERAGE_OUTPUT,
    low_memory=False
)

saved_step3_window_summary = pd.read_csv(
    STEP3_WINDOW_ASSIGNMENT_SUMMARY_OUTPUT,
    low_memory=False
)

saved_step3_cohort_summary = pd.read_csv(
    STEP3_COHORT_ASSIGNMENT_SUMMARY_OUTPUT,
    low_memory=False
)

saved_step3_zero_demand = pd.read_csv(
    STEP3_ZERO_DEMAND_EVALUATION_AUDIT_OUTPUT,
    low_memory=False
)

saved_step3_feature_readiness = pd.read_csv(
    STEP3_EVALUATION_FEATURE_READINESS_OUTPUT,
    low_memory=False
)

saved_step3_no_holdout = pd.read_csv(
    STEP3_NO_HOLDOUT_REGISTER_OUTPUT,
    low_memory=False
)

saved_step3_part2_validation = pd.read_csv(
    STEP3_PART2_VALIDATION_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 8. Validate saved outputs
# ------------------------------------------------------------

assert len(saved_step3_assignments) == 472

assert saved_step3_assignments[
    [
        "CanonicalProductID",
        "WindowID"
    ]
].duplicated().sum() == 0

assert saved_step3_assignments[
    "CanonicalProductID"
].nunique() == 218

assert len(saved_step3_coverage) == 472

assert saved_step3_coverage[
    "TrainingRowCountMismatch"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_step3_coverage[
    "EvaluationRowCountMismatch"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_step3_coverage[
    "ChronologicalOverlapViolation"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_step3_window_summary[
    "ProductWindowAssignments"
].sum() == 472

assert saved_step3_cohort_summary[
    "ProductWindowAssignments"
].sum() == 472

assert len(
    saved_step3_zero_demand
) == len(
    step3_zero_demand_evaluation_audit_df
)

assert len(saved_step3_feature_readiness) == 0

assert len(saved_step3_no_holdout) == 9

assert len(
    saved_step3_part2_validation
) == 21


# ------------------------------------------------------------
# 9. Build Markdown window summary
# ------------------------------------------------------------

assignment_markdown_rows = "\n".join(
    (
        f"- `{row.WindowID}`: "
        f"{int(row.ProductCount)} products, "
        f"{int(row.TotalEvaluationRows):,} evaluation rows, "
        f"{float(row.EvaluationTotalDemandUnits):,.0f} "
        f"demand units, "
        f"{int(row.AllZeroDemandProductWindows)} "
        f"all-zero product windows"
    )
    for row in (
        step3_window_assignment_summary_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 10. Update Markdown handoff
# ------------------------------------------------------------

step3_part2_summary = f"""
**Status:** Completed and validated

### Purpose

Step 3 Part 2 assigned each eligible canonical product to its
approved chronological evaluation windows and validated the actual
training and evaluation coverage.

### Assignment structure

- Product-window assignments: {len(saved_step3_assignments):,}
- Products with independent evaluation: {saved_step3_assignments["CanonicalProductID"].nunique()}
- Products without independent product-level holdout: {len(saved_step3_no_holdout)}
- Multi-fold assignments: 381
- Single-holdout assignments: 77
- Limited-holdout assignments: 14

### Assignments by evaluation window

{assignment_markdown_rows}

### Temporal validation

- Training requirement violations: 0
- Training-row-count mismatches: 0
- Evaluation-row-count mismatches: 0
- Training-boundary mismatches: 0
- Evaluation-boundary mismatches: 0
- Chronological overlap violations: 0

Every training period ends before its corresponding evaluation
period begins.

### Historical-feature readiness

- Historical features required: 31
- Evaluation rows with missing historical features: 0
- Historical-feature missing cells inside evaluation windows: 0

### Zero-demand evaluation windows

- All-zero-demand product-window assignments: {len(saved_step3_zero_demand)}
- All-zero windows removed: 0

An all-zero evaluation window is valid for intermittent-demand
products and remains part of the evaluation data.

### Minimal-evidence products

The 9 minimal-evidence products remain in forecasting scope but
do not receive an unreliable independent product-level holdout.
They will be handled through pooled/global models and conservative
baselines.

### Saved Step 3 Part 2 outputs

- `14_step3_product_window_assignment_register.csv`
- `14_step3_product_window_coverage_audit.csv`
- `14_step3_window_assignment_summary.csv`
- `14_step3_cohort_assignment_summary.csv`
- `14_step3_zero_demand_evaluation_audit.csv`
- `14_step3_evaluation_feature_readiness_audit.csv`
- `14_step3_no_independent_holdout_register.csv`
- `14_step3_part2_validation_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_3_part_2",
    section_title=(
        "Forecasting Preparation — Step 3, Part 2"
    ),
    section_body=step3_part2_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 11. Print final Part 2 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 3, PART 2 COMPLETED"
)
print("=" * 75)

print()
print("Product-window assignment register:")
print(
    "Assignments saved:",
    len(saved_step3_assignments)
)
print(
    "Products with independent evaluation:",
    saved_step3_assignments[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Products without independent holdout:",
    len(saved_step3_no_holdout)
)

print()
print("Assignments:")
print(
    "Multi-fold assignments:",
    381
)
print(
    "Single-holdout assignments:",
    77
)
print(
    "Limited-holdout assignments:",
    14
)

print()
print("Temporal validation:")
print(
    "Training-row-count mismatches:",
    training_row_count_mismatches
)
print(
    "Evaluation-row-count mismatches:",
    evaluation_row_count_mismatches
)
print(
    "Chronological overlap violations:",
    chronological_overlap_violations
)

print()
print("Historical-feature readiness:")
print(
    "Evaluation feature-readiness violations:",
    evaluation_feature_readiness_violations
)
print(
    "Missing historical-feature cells "
    "inside evaluation windows:",
    evaluation_missing_historical_cells
)

print()
print("Intermittent-demand evaluation:")
print(
    "All-zero-demand product windows:",
    len(saved_step3_zero_demand)
)
print(
    "All-zero windows removed:",
    0
)

print()
print("Saved files:")
print(f"1. {STEP3_PRODUCT_WINDOW_ASSIGNMENT_OUTPUT}")
print(f"2. {STEP3_PRODUCT_WINDOW_COVERAGE_OUTPUT}")
print(f"3. {STEP3_WINDOW_ASSIGNMENT_SUMMARY_OUTPUT}")
print(f"4. {STEP3_COHORT_ASSIGNMENT_SUMMARY_OUTPUT}")
print(f"5. {STEP3_ZERO_DEMAND_EVALUATION_AUDIT_OUTPUT}")
print(f"6. {STEP3_EVALUATION_FEATURE_READINESS_OUTPUT}")
print(f"7. {STEP3_NO_HOLDOUT_REGISTER_OUTPUT}")
print(f"8. {STEP3_PART2_VALIDATION_OUTPUT}")
print(f"9. {HANDOFF_FILE}")

print()
print(
    "All Step 3, Part 2 validation checks passed."
)

FORECASTING PREPARATION — STEP 3, PART 2 COMPLETED

Product-window assignment register:
Assignments saved: 472
Products with independent evaluation: 218
Products without independent holdout: 9

Assignments:
Multi-fold assignments: 381
Single-holdout assignments: 77
Limited-holdout assignments: 14

Temporal validation:
Training-row-count mismatches: 0
Evaluation-row-count mismatches: 0
Chronological overlap violations: 0

Historical-feature readiness:
Evaluation feature-readiness violations: 0
Missing historical-feature cells inside evaluation windows: 0

Intermittent-demand evaluation:
All-zero-demand product windows: 344
All-zero windows removed: 0

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/14_step3_product_window_assignment_register.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/14_step3_product_window_coverage_audit.csv
3. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/14_step3_win

In [59]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 3
# Cell 48: Create row-level chronological split assignments
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the previous forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define official Part 3 input files
# ------------------------------------------------------------

STEP3_ROW_SOURCE_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_model_view.csv"
)

STEP3_ASSIGNMENT_REGISTER_FILE = (
    FORECAST_PREPARATION_DIR
    / "14_step3_product_window_assignment_register.csv"
)

STEP3_COVERAGE_AUDIT_FILE = (
    FORECAST_PREPARATION_DIR
    / "14_step3_product_window_coverage_audit.csv"
)

STEP3_NO_HOLDOUT_REGISTER_FILE = (
    FORECAST_PREPARATION_DIR
    / "14_step3_no_independent_holdout_register.csv"
)

STEP3_FEATURE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_combined_feature_contract.csv"
)

required_part3_input_files = [
    STEP3_ROW_SOURCE_FILE,
    STEP3_ASSIGNMENT_REGISTER_FILE,
    STEP3_COVERAGE_AUDIT_FILE,
    STEP3_NO_HOLDOUT_REGISTER_FILE,
    STEP3_FEATURE_CONTRACT_FILE
]

missing_part3_input_files = [
    file_path
    for file_path in required_part3_input_files
    if not file_path.exists()
]

if missing_part3_input_files:
    raise FileNotFoundError(
        "The following required Step 3 Part 3 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_part3_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load official inputs
# ------------------------------------------------------------

step3_row_source_df = pd.read_csv(
    STEP3_ROW_SOURCE_FILE,
    low_memory=False
)

step3_assignment_register_df = pd.read_csv(
    STEP3_ASSIGNMENT_REGISTER_FILE,
    low_memory=False
)

step3_coverage_audit_df = pd.read_csv(
    STEP3_COVERAGE_AUDIT_FILE,
    low_memory=False
)

step3_no_holdout_register_df = pd.read_csv(
    STEP3_NO_HOLDOUT_REGISTER_FILE,
    low_memory=False
)

step3_feature_contract_df = pd.read_csv(
    STEP3_FEATURE_CONTRACT_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Boolean parsing helper
# ------------------------------------------------------------

def parse_step3_part3_boolean(series):
    """
    Convert common CSV Boolean representations to bool.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalised_values = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    parsed_values = normalised_values.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if parsed_values.isna().sum() > 0:

        invalid_values = (
            series.loc[
                parsed_values.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unrecognised Boolean values were found:\n"
            f"{invalid_values}"
        )

    return parsed_values.astype(bool)


# ------------------------------------------------------------
# 5. Parse source dates and numeric fields
# ------------------------------------------------------------

for column in [
    "Date",
    "ProductFirstObservedDate"
]:

    step3_row_source_df[column] = pd.to_datetime(
        step3_row_source_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step3_row_source_df[
    [
        "Date",
        "ProductFirstObservedDate"
    ]
].isna().sum().sum() == 0


for column in [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]:

    step3_row_source_df[column] = pd.to_numeric(
        step3_row_source_df[column],
        errors="coerce"
    )

assert step3_row_source_df[
    [
        "OperatingDaySequence",
        "ProductAgeOperatingDays",
        "TotalDemand"
    ]
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 6. Standardise product identifiers
# ------------------------------------------------------------

for dataframe in [
    step3_row_source_df,
    step3_assignment_register_df,
    step3_coverage_audit_df,
    step3_no_holdout_register_df
]:

    dataframe[
        "CanonicalProductID"
    ] = (
        dataframe[
            "CanonicalProductID"
        ]
        .astype("string")
        .str.strip()
    )


# ------------------------------------------------------------
# 7. Parse assignment numeric and Boolean fields
# ------------------------------------------------------------

assignment_numeric_columns = [
    "WindowOrder",
    "ProductTrainingStartSequence",
    "TrainingEndSequence",
    "EvaluationStartSequence",
    "EvaluationEndSequence",
    "AvailableTrainingOperatingDays",
    "ExpectedEvaluationOperatingDays"
]

for column in assignment_numeric_columns:

    step3_assignment_register_df[column] = pd.to_numeric(
        step3_assignment_register_df[column],
        errors="coerce"
    )

assert step3_assignment_register_df[
    assignment_numeric_columns
].isna().sum().sum() == 0


for column in [
    "MayBeUsedForModelSelection",
    "ReservedFinalTest"
]:

    step3_assignment_register_df[column] = (
        parse_step3_part3_boolean(
            step3_assignment_register_df[column]
        )
    )


# ------------------------------------------------------------
# 8. Validate loaded structures
# ------------------------------------------------------------

assert step3_row_source_df.shape == (
    43_774,
    58
)

assert step3_row_source_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_row_source_df[
    "Date"
].nunique() == 245

assert step3_row_source_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert step3_row_source_df[
    "TotalDemand"
].sum() == 116_158

assert len(
    step3_assignment_register_df
) == 472

assert step3_assignment_register_df[
    [
        "CanonicalProductID",
        "WindowID"
    ]
].duplicated().sum() == 0

assert len(
    step3_no_holdout_register_df
) == 9

assert len(
    step3_feature_contract_df
) == 58


# ------------------------------------------------------------
# 9. Identify the 31 historical demand features
# ------------------------------------------------------------

historical_feature_flag = (
    parse_step3_part3_boolean(
        step3_feature_contract_df[
            "IsHistoricalDemandFeature"
        ]
    )
)

step3_historical_feature_columns = (
    step3_feature_contract_df.loc[
        historical_feature_flag,
        "Column"
    ]
    .tolist()
)

assert len(
    step3_historical_feature_columns
) == 31

assert all(
    column in step3_row_source_df.columns
    for column in step3_historical_feature_columns
)


# ------------------------------------------------------------
# 10. Create source groups for efficient row extraction
# ------------------------------------------------------------

step3_source_by_product = {
    str(canonical_product_id): (
        product_group
        .sort_values(
            "OperatingDaySequence",
            kind="stable"
        )
        .reset_index(drop=True)
    )
    for (
        canonical_product_id,
        product_group
    ) in step3_row_source_df.groupby(
        "CanonicalProductID",
        sort=False
    )
}

assert len(step3_source_by_product) == 227


# ------------------------------------------------------------
# 11. Define the row-level manifest columns
# ------------------------------------------------------------

source_manifest_columns = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]

manifest_records = []


# ------------------------------------------------------------
# 12. Create TRAIN, VALIDATION and FINAL_TEST rows
# ------------------------------------------------------------

for assignment in (
    step3_assignment_register_df
    .itertuples(index=False)
):

    canonical_product_id = str(
        assignment.CanonicalProductID
    )

    product_rows = (
        step3_source_by_product[
            canonical_product_id
        ]
    )

    training_rows = (
        product_rows.loc[
            product_rows[
                "OperatingDaySequence"
            ].between(
                int(
                    assignment.ProductTrainingStartSequence
                ),
                int(
                    assignment.TrainingEndSequence
                )
            )
        ]
        .copy()
    )

    evaluation_rows = (
        product_rows.loc[
            product_rows[
                "OperatingDaySequence"
            ].between(
                int(
                    assignment.EvaluationStartSequence
                ),
                int(
                    assignment.EvaluationEndSequence
                )
            )
        ]
        .copy()
    )

    if bool(
        assignment.MayBeUsedForModelSelection
    ):

        evaluation_split_role = (
            "VALIDATION"
        )

    elif bool(
        assignment.ReservedFinalTest
    ):

        evaluation_split_role = (
            "FINAL_TEST"
        )

    else:

        raise ValueError(
            "An evaluation window is neither a model-selection "
            "window nor a reserved final-test window:\n"
            f"{assignment.WindowID}"
        )


    # --------------------------------------------------------
    # Training rows
    # --------------------------------------------------------

    training_manifest = (
        training_rows[
            source_manifest_columns
        ]
        .copy()
    )

    training_manifest[
        "HistoricalFeatureAvailableCount"
    ] = (
        training_rows[
            step3_historical_feature_columns
        ]
        .notna()
        .sum(axis=1)
        .astype(int)
        .to_numpy()
    )

    training_manifest[
        "AllHistoricalFeaturesAvailable"
    ] = (
        training_manifest[
            "HistoricalFeatureAvailableCount"
        ]
        == len(
            step3_historical_feature_columns
        )
    )

    training_manifest[
        "EvaluationCohort"
    ] = assignment.EvaluationCohort

    training_manifest[
        "DemandPatternClass"
    ] = assignment.DemandPatternClass

    training_manifest[
        "WindowID"
    ] = assignment.WindowID

    training_manifest[
        "WindowOrder"
    ] = int(
        assignment.WindowOrder
    )

    training_manifest[
        "WindowPurpose"
    ] = assignment.WindowPurpose

    training_manifest[
        "SplitRole"
    ] = "TRAIN"

    training_manifest[
        "MayBeUsedForModelSelection"
    ] = bool(
        assignment.MayBeUsedForModelSelection
    )

    training_manifest[
        "ReservedFinalTest"
    ] = bool(
        assignment.ReservedFinalTest
    )

    training_manifest[
        "IndependentEvaluationAvailable"
    ] = True

    training_manifest[
        "RowEligibleForScoring"
    ] = False

    training_manifest[
        "AssignmentSource"
    ] = (
        "INDEPENDENT_EVALUATION_WINDOW"
    )

    training_manifest[
        "SplitContextID"
    ] = (
        canonical_product_id
        + "::"
        + str(
            assignment.WindowID
        )
    )


    # --------------------------------------------------------
    # Evaluation rows
    # --------------------------------------------------------

    evaluation_manifest = (
        evaluation_rows[
            source_manifest_columns
        ]
        .copy()
    )

    evaluation_manifest[
        "HistoricalFeatureAvailableCount"
    ] = (
        evaluation_rows[
            step3_historical_feature_columns
        ]
        .notna()
        .sum(axis=1)
        .astype(int)
        .to_numpy()
    )

    evaluation_manifest[
        "AllHistoricalFeaturesAvailable"
    ] = (
        evaluation_manifest[
            "HistoricalFeatureAvailableCount"
        ]
        == len(
            step3_historical_feature_columns
        )
    )

    evaluation_manifest[
        "EvaluationCohort"
    ] = assignment.EvaluationCohort

    evaluation_manifest[
        "DemandPatternClass"
    ] = assignment.DemandPatternClass

    evaluation_manifest[
        "WindowID"
    ] = assignment.WindowID

    evaluation_manifest[
        "WindowOrder"
    ] = int(
        assignment.WindowOrder
    )

    evaluation_manifest[
        "WindowPurpose"
    ] = assignment.WindowPurpose

    evaluation_manifest[
        "SplitRole"
    ] = evaluation_split_role

    evaluation_manifest[
        "MayBeUsedForModelSelection"
    ] = bool(
        assignment.MayBeUsedForModelSelection
    )

    evaluation_manifest[
        "ReservedFinalTest"
    ] = bool(
        assignment.ReservedFinalTest
    )

    evaluation_manifest[
        "IndependentEvaluationAvailable"
    ] = True

    evaluation_manifest[
        "RowEligibleForScoring"
    ] = True

    evaluation_manifest[
        "AssignmentSource"
    ] = (
        "INDEPENDENT_EVALUATION_WINDOW"
    )

    evaluation_manifest[
        "SplitContextID"
    ] = (
        canonical_product_id
        + "::"
        + str(
            assignment.WindowID
        )
    )

    manifest_records.extend([
        training_manifest,
        evaluation_manifest
    ])


# ------------------------------------------------------------
# 13. Combine independent evaluation contexts
# ------------------------------------------------------------

step3_independent_split_manifest_df = pd.concat(
    manifest_records,
    ignore_index=True
)

expected_independent_manifest_rows = int(
    (
        step3_assignment_register_df[
            "AvailableTrainingOperatingDays"
        ]
        + step3_assignment_register_df[
            "ExpectedEvaluationOperatingDays"
        ]
    ).sum()
)

assert len(
    step3_independent_split_manifest_df
) == expected_independent_manifest_rows


# ------------------------------------------------------------
# 14. Create FORECAST_ONLY_TRAINING rows
# ------------------------------------------------------------

forecast_only_manifest_records = []

for product_record in (
    step3_no_holdout_register_df
    .itertuples(index=False)
):

    canonical_product_id = str(
        product_record.CanonicalProductID
    )

    product_rows = (
        step3_source_by_product[
            canonical_product_id
        ]
    )

    forecast_only_manifest = (
        product_rows[
            source_manifest_columns
        ]
        .copy()
    )

    forecast_only_manifest[
        "HistoricalFeatureAvailableCount"
    ] = (
        product_rows[
            step3_historical_feature_columns
        ]
        .notna()
        .sum(axis=1)
        .astype(int)
        .to_numpy()
    )

    forecast_only_manifest[
        "AllHistoricalFeaturesAvailable"
    ] = (
        forecast_only_manifest[
            "HistoricalFeatureAvailableCount"
        ]
        == len(
            step3_historical_feature_columns
        )
    )

    forecast_only_manifest[
        "EvaluationCohort"
    ] = (
        product_record.EvaluationCohort
    )

    forecast_only_manifest[
        "DemandPatternClass"
    ] = (
        product_record.DemandPatternClass
    )

    forecast_only_manifest[
        "WindowID"
    ] = (
        "FORECAST_ONLY_FULL_HISTORY"
    )

    forecast_only_manifest[
        "WindowOrder"
    ] = 99

    forecast_only_manifest[
        "WindowPurpose"
    ] = (
        "FORECAST_MODEL_FIT_ONLY"
    )

    forecast_only_manifest[
        "SplitRole"
    ] = (
        "FORECAST_ONLY_TRAINING"
    )

    forecast_only_manifest[
        "MayBeUsedForModelSelection"
    ] = False

    forecast_only_manifest[
        "ReservedFinalTest"
    ] = False

    forecast_only_manifest[
        "IndependentEvaluationAvailable"
    ] = False

    forecast_only_manifest[
        "RowEligibleForScoring"
    ] = False

    forecast_only_manifest[
        "AssignmentSource"
    ] = (
        "NO_INDEPENDENT_HOLDOUT"
    )

    forecast_only_manifest[
        "SplitContextID"
    ] = (
        canonical_product_id
        + "::FORECAST_ONLY_FULL_HISTORY"
    )

    forecast_only_manifest_records.append(
        forecast_only_manifest
    )


step3_forecast_only_split_manifest_df = pd.concat(
    forecast_only_manifest_records,
    ignore_index=True
)

expected_forecast_only_rows = int(
    step3_row_source_df.loc[
        step3_row_source_df[
            "CanonicalProductID"
        ].isin(
            step3_no_holdout_register_df[
                "CanonicalProductID"
            ]
        )
    ].shape[0]
)

assert len(
    step3_forecast_only_split_manifest_df
) == expected_forecast_only_rows


# ------------------------------------------------------------
# 15. Create the complete row-level split manifest
# ------------------------------------------------------------

step3_complete_row_split_manifest_df = pd.concat(
    [
        step3_independent_split_manifest_df,
        step3_forecast_only_split_manifest_df
    ],
    ignore_index=True
)

step3_complete_row_split_manifest_df = (
    step3_complete_row_split_manifest_df
    .sort_values(
        [
            "WindowOrder",
            "WindowID",
            "CanonicalProductID",
            "OperatingDaySequence",
            "SplitRole"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 16. Validate manifest structure
# ------------------------------------------------------------

assert step3_complete_row_split_manifest_df[
    [
        "CanonicalProductID",
        "WindowID",
        "Date"
    ]
].duplicated().sum() == 0

assert step3_complete_row_split_manifest_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_complete_row_split_manifest_df[
    "SplitContextID"
].nunique() == (
    472 + 9
)

assert set(
    step3_complete_row_split_manifest_df[
        "SplitRole"
    ].unique()
) == {
    "TRAIN",
    "VALIDATION",
    "FINAL_TEST",
    "FORECAST_ONLY_TRAINING"
}


# ------------------------------------------------------------
# 17. Validate evaluation feature readiness
# ------------------------------------------------------------

evaluation_role_mask = (
    step3_complete_row_split_manifest_df[
        "SplitRole"
    ].isin([
        "VALIDATION",
        "FINAL_TEST"
    ])
)

evaluation_rows_missing_history = int(
    (
        ~step3_complete_row_split_manifest_df.loc[
            evaluation_role_mask,
            "AllHistoricalFeaturesAvailable"
        ]
    ).sum()
)

assert evaluation_rows_missing_history == 0


# ------------------------------------------------------------
# 18. Confirm role-policy consistency
# ------------------------------------------------------------

validation_policy_violations = int(
    (
        step3_complete_row_split_manifest_df[
            "SplitRole"
        ].eq("VALIDATION")
        &
        (
            ~step3_complete_row_split_manifest_df[
                "MayBeUsedForModelSelection"
            ]
            |
            step3_complete_row_split_manifest_df[
                "ReservedFinalTest"
            ]
        )
    ).sum()
)

final_test_policy_violations = int(
    (
        step3_complete_row_split_manifest_df[
            "SplitRole"
        ].eq("FINAL_TEST")
        &
        (
            step3_complete_row_split_manifest_df[
                "MayBeUsedForModelSelection"
            ]
            |
            ~step3_complete_row_split_manifest_df[
                "ReservedFinalTest"
            ]
        )
    ).sum()
)

forecast_only_policy_violations = int(
    (
        step3_complete_row_split_manifest_df[
            "SplitRole"
        ].eq(
            "FORECAST_ONLY_TRAINING"
        )
        &
        (
            step3_complete_row_split_manifest_df[
                "IndependentEvaluationAvailable"
            ]
            |
            step3_complete_row_split_manifest_df[
                "RowEligibleForScoring"
            ]
        )
    ).sum()
)

assert validation_policy_violations == 0
assert final_test_policy_violations == 0
assert forecast_only_policy_violations == 0


# ------------------------------------------------------------
# 19. Confirm leakage columns are absent
# ------------------------------------------------------------

same_day_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

manifest_leakage_columns_found = sorted(
    same_day_leakage_columns.intersection(
        step3_complete_row_split_manifest_df.columns
    )
)

assert len(
    manifest_leakage_columns_found
) == 0


# ------------------------------------------------------------
# 20. Print Cell 48 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 3 — "
    "ROW-LEVEL SPLIT MANIFEST CREATED"
)
print("=" * 75)

print()
print("Split contexts:")
print(
    "Independent evaluation contexts:",
    step3_independent_split_manifest_df[
        "SplitContextID"
    ].nunique()
)
print(
    "Forecast-only contexts:",
    step3_forecast_only_split_manifest_df[
        "SplitContextID"
    ].nunique()
)
print(
    "Total split contexts:",
    step3_complete_row_split_manifest_df[
        "SplitContextID"
    ].nunique()
)

print()
print("Manifest rows:")
print(
    "Independent evaluation manifest rows:",
    f"{len(step3_independent_split_manifest_df):,}"
)
print(
    "Forecast-only manifest rows:",
    f"{len(step3_forecast_only_split_manifest_df):,}"
)
print(
    "Complete manifest rows:",
    f"{len(step3_complete_row_split_manifest_df):,}"
)

print()
print("Rows by split role:")
display(
    step3_complete_row_split_manifest_df[
        "SplitRole"
    ]
    .value_counts()
    .rename_axis("SplitRole")
    .reset_index(name="RowCount")
)

print()
print("Validation:")
print(
    "Duplicate product-window-date rows:",
    step3_complete_row_split_manifest_df[
        [
            "CanonicalProductID",
            "WindowID",
            "Date"
        ]
    ].duplicated().sum()
)
print(
    "Evaluation rows missing historical features:",
    evaluation_rows_missing_history
)
print(
    "Validation policy violations:",
    validation_policy_violations
)
print(
    "Final-test policy violations:",
    final_test_policy_violations
)
print(
    "Forecast-only policy violations:",
    forecast_only_policy_violations
)
print(
    "Same-day leakage columns found:",
    len(
        manifest_leakage_columns_found
    )
)

print()
print("Cell 48 completed successfully.")

STEP 3, PART 3 — ROW-LEVEL SPLIT MANIFEST CREATED

Split contexts:
Independent evaluation contexts: 472
Forecast-only contexts: 9
Total split contexts: 481

Manifest rows:
Independent evaluation manifest rows: 96,396
Forecast-only manifest rows: 996
Complete manifest rows: 97,392

Rows by split role:


,SplitRole,RowCount
0,TRAIN,87166
1,VALIDATION,5080
2,FINAL_TEST,4150
3,FORECAST_ONLY_TRAINING,996



Validation:
Duplicate product-window-date rows: 0
Evaluation rows missing historical features: 0
Validation policy violations: 0
Final-test policy violations: 0
Forecast-only policy violations: 0
Same-day leakage columns found: 0

Cell 48 completed successfully.


In [60]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 3
# Cell 49: Independently validate row-level split assignments
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 48 objects exist
# ------------------------------------------------------------

required_cell_48_objects = [
    "step3_row_source_df",
    "step3_assignment_register_df",
    "step3_coverage_audit_df",
    "step3_independent_split_manifest_df",
    "step3_forecast_only_split_manifest_df",
    "step3_complete_row_split_manifest_df",
    "step3_historical_feature_columns"
]

missing_cell_48_objects = [
    object_name
    for object_name in required_cell_48_objects
    if object_name not in globals()
]

if missing_cell_48_objects:
    raise NameError(
        "The following Cell 48 objects are missing:\n"
        f"{missing_cell_48_objects}\n\n"
        "Run Cell 48 before running Cell 49."
    )


# ------------------------------------------------------------
# 2. Create training and evaluation row counts
# ------------------------------------------------------------

independent_training_counts_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "SplitRole"
        ] == "TRAIN"
    ]
    .groupby(
        [
            "CanonicalProductID",
            "WindowID"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "ManifestTrainingRows"
        }
    )
)

independent_evaluation_counts_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "SplitRole"
        ].isin([
            "VALIDATION",
            "FINAL_TEST"
        ])
    ]
    .groupby(
        [
            "CanonicalProductID",
            "WindowID"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "ManifestEvaluationRows"
        }
    )
)


# ------------------------------------------------------------
# 3. Compare manifest counts to the Part 2 coverage audit
# ------------------------------------------------------------

step3_manifest_coverage_comparison_df = (
    step3_coverage_audit_df[
        [
            "CanonicalProductID",
            "WindowID",
            "ActualTrainingRows",
            "ActualEvaluationRows"
        ]
    ]
    .merge(
        independent_training_counts_df,
        on=[
            "CanonicalProductID",
            "WindowID"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        independent_evaluation_counts_df,
        on=[
            "CanonicalProductID",
            "WindowID"
        ],
        how="left",
        validate="one_to_one"
    )
)

step3_manifest_coverage_comparison_df[
    "TrainingRowCountMismatch"
] = (
    step3_manifest_coverage_comparison_df[
        "ActualTrainingRows"
    ]
    != step3_manifest_coverage_comparison_df[
        "ManifestTrainingRows"
    ]
)

step3_manifest_coverage_comparison_df[
    "EvaluationRowCountMismatch"
] = (
    step3_manifest_coverage_comparison_df[
        "ActualEvaluationRows"
    ]
    != step3_manifest_coverage_comparison_df[
        "ManifestEvaluationRows"
    ]
)

manifest_training_count_mismatches = int(
    step3_manifest_coverage_comparison_df[
        "TrainingRowCountMismatch"
    ].sum()
)

manifest_evaluation_count_mismatches = int(
    step3_manifest_coverage_comparison_df[
        "EvaluationRowCountMismatch"
    ].sum()
)

assert manifest_training_count_mismatches == 0
assert manifest_evaluation_count_mismatches == 0


# ------------------------------------------------------------
# 4. Validate within-window chronological boundaries
# ------------------------------------------------------------

manifest_training_boundaries_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "SplitRole"
        ] == "TRAIN"
    ]
    .groupby(
        [
            "CanonicalProductID",
            "WindowID"
        ],
        as_index=False
    )
    .agg(
        ManifestTrainingStartSequence=(
            "OperatingDaySequence",
            "min"
        ),
        ManifestTrainingEndSequence=(
            "OperatingDaySequence",
            "max"
        ),
        ManifestTrainingStartDate=(
            "Date",
            "min"
        ),
        ManifestTrainingEndDate=(
            "Date",
            "max"
        )
    )
)

manifest_evaluation_boundaries_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "SplitRole"
        ].isin([
            "VALIDATION",
            "FINAL_TEST"
        ])
    ]
    .groupby(
        [
            "CanonicalProductID",
            "WindowID"
        ],
        as_index=False
    )
    .agg(
        ManifestEvaluationStartSequence=(
            "OperatingDaySequence",
            "min"
        ),
        ManifestEvaluationEndSequence=(
            "OperatingDaySequence",
            "max"
        ),
        ManifestEvaluationStartDate=(
            "Date",
            "min"
        ),
        ManifestEvaluationEndDate=(
            "Date",
            "max"
        )
    )
)


step3_manifest_boundary_validation_df = (
    step3_assignment_register_df[
        [
            "CanonicalProductID",
            "WindowID",
            "ProductTrainingStartSequence",
            "TrainingEndSequence",
            "EvaluationStartSequence",
            "EvaluationEndSequence"
        ]
    ]
    .merge(
        manifest_training_boundaries_df,
        on=[
            "CanonicalProductID",
            "WindowID"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        manifest_evaluation_boundaries_df,
        on=[
            "CanonicalProductID",
            "WindowID"
        ],
        how="left",
        validate="one_to_one"
    )
)


step3_manifest_boundary_validation_df[
    "TrainingStartMismatch"
] = (
    step3_manifest_boundary_validation_df[
        "ProductTrainingStartSequence"
    ]
    != step3_manifest_boundary_validation_df[
        "ManifestTrainingStartSequence"
    ]
)

step3_manifest_boundary_validation_df[
    "TrainingEndMismatch"
] = (
    step3_manifest_boundary_validation_df[
        "TrainingEndSequence"
    ]
    != step3_manifest_boundary_validation_df[
        "ManifestTrainingEndSequence"
    ]
)

step3_manifest_boundary_validation_df[
    "EvaluationStartMismatch"
] = (
    step3_manifest_boundary_validation_df[
        "EvaluationStartSequence"
    ]
    != step3_manifest_boundary_validation_df[
        "ManifestEvaluationStartSequence"
    ]
)

step3_manifest_boundary_validation_df[
    "EvaluationEndMismatch"
] = (
    step3_manifest_boundary_validation_df[
        "EvaluationEndSequence"
    ]
    != step3_manifest_boundary_validation_df[
        "ManifestEvaluationEndSequence"
    ]
)

step3_manifest_boundary_validation_df[
    "ChronologicalOverlapViolation"
] = (
    step3_manifest_boundary_validation_df[
        "ManifestTrainingEndSequence"
    ]
    >= step3_manifest_boundary_validation_df[
        "ManifestEvaluationStartSequence"
    ]
)


manifest_boundary_mismatches = int(
    (
        step3_manifest_boundary_validation_df[
            [
                "TrainingStartMismatch",
                "TrainingEndMismatch",
                "EvaluationStartMismatch",
                "EvaluationEndMismatch"
            ]
        ]
        .sum()
        .sum()
    )
)

manifest_overlap_violations = int(
    step3_manifest_boundary_validation_df[
        "ChronologicalOverlapViolation"
    ].sum()
)

assert manifest_boundary_mismatches == 0
assert manifest_overlap_violations == 0


# ------------------------------------------------------------
# 5. Create clean role-specific manifests
# ------------------------------------------------------------

step3_model_selection_manifest_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "WindowPurpose"
        ]
        == "BACKTEST_VALIDATION"
    ]
    .copy()
    .reset_index(drop=True)
)

step3_final_test_manifest_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "ReservedFinalTest"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

step3_forecast_only_training_manifest_df = (
    step3_forecast_only_split_manifest_df
    .copy()
    .reset_index(drop=True)
)


assert set(
    step3_model_selection_manifest_df[
        "SplitRole"
    ].unique()
) == {
    "TRAIN",
    "VALIDATION"
}

assert set(
    step3_final_test_manifest_df[
        "SplitRole"
    ].unique()
) == {
    "TRAIN",
    "FINAL_TEST"
}

assert set(
    step3_forecast_only_training_manifest_df[
        "SplitRole"
    ].unique()
) == {
    "FORECAST_ONLY_TRAINING"
}


# ------------------------------------------------------------
# 6. Validate all source rows are represented
# ------------------------------------------------------------

unique_manifest_source_rows_df = (
    step3_complete_row_split_manifest_df[
        [
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence",
            "TotalDemand"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Date",
            "CanonicalProductID"
        ]
    )
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

source_row_comparison_df = (
    step3_row_source_df[
        [
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence",
            "TotalDemand"
        ]
    ]
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert len(
    unique_manifest_source_rows_df
) == 43_774

pd.testing.assert_frame_equal(
    source_row_comparison_df,
    unique_manifest_source_rows_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

assert unique_manifest_source_rows_df[
    "TotalDemand"
].sum() == 116_158


# ------------------------------------------------------------
# 7. Validate the forecast-only products
# ------------------------------------------------------------

forecast_only_source_counts_df = (
    step3_row_source_df.loc[
        step3_row_source_df[
            "CanonicalProductID"
        ].isin(
            step3_no_holdout_register_df[
                "CanonicalProductID"
            ]
        )
    ]
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "SourceRows"
        }
    )
)

forecast_only_manifest_counts_df = (
    step3_forecast_only_training_manifest_df
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "ManifestRows"
        }
    )
)

step3_forecast_only_coverage_validation_df = (
    forecast_only_source_counts_df
    .merge(
        forecast_only_manifest_counts_df,
        on="CanonicalProductID",
        how="outer",
        validate="one_to_one"
    )
)

step3_forecast_only_coverage_validation_df[
    "RowCountMismatch"
] = (
    step3_forecast_only_coverage_validation_df[
        "SourceRows"
    ]
    != step3_forecast_only_coverage_validation_df[
        "ManifestRows"
    ]
)

forecast_only_row_count_mismatches = int(
    step3_forecast_only_coverage_validation_df[
        "RowCountMismatch"
    ].sum()
)

assert forecast_only_row_count_mismatches == 0


# ------------------------------------------------------------
# 8. Recreate the all-zero evaluation-window audit
# ------------------------------------------------------------

evaluation_manifest_rows_df = (
    step3_independent_split_manifest_df.loc[
        step3_independent_split_manifest_df[
            "SplitRole"
        ].isin([
            "VALIDATION",
            "FINAL_TEST"
        ])
    ]
    .copy()
)

step3_row_manifest_zero_demand_audit_df = (
    evaluation_manifest_rows_df
    .assign(
        PositiveDemandFlag=(
            evaluation_manifest_rows_df[
                "TotalDemand"
            ] > 0
        ).astype(int)
    )
    .groupby(
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "EvaluationCohort",
            "WindowID",
            "SplitRole"
        ],
        as_index=False
    )
    .agg(
        EvaluationRows=(
            "Date",
            "size"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        EvaluationDemandUnits=(
            "TotalDemand",
            "sum"
        )
    )
)

step3_row_manifest_zero_demand_audit_df[
    "AllZeroDemandWindow"
] = (
    step3_row_manifest_zero_demand_audit_df[
        "EvaluationDemandUnits"
    ] == 0
)

step3_row_manifest_zero_demand_audit_df = (
    step3_row_manifest_zero_demand_audit_df.loc[
        step3_row_manifest_zero_demand_audit_df[
            "AllZeroDemandWindow"
        ]
    ]
    .copy()
    .sort_values(
        [
            "WindowID",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

step3_row_manifest_zero_demand_audit_df[
    "RemovedFromEvaluation"
] = False

assert len(
    step3_row_manifest_zero_demand_audit_df
) == 344


# ------------------------------------------------------------
# 9. Create split-role summary
# ------------------------------------------------------------

step3_split_role_summary_df = (
    step3_complete_row_split_manifest_df
    .assign(
        PositiveDemandFlag=(
            step3_complete_row_split_manifest_df[
                "TotalDemand"
            ] > 0
        ).astype(int),
        ZeroDemandFlag=(
            step3_complete_row_split_manifest_df[
                "TotalDemand"
            ] == 0
        ).astype(int),
        MissingHistoricalFeatureFlag=(
            ~step3_complete_row_split_manifest_df[
                "AllHistoricalFeaturesAvailable"
            ]
        ).astype(int)
    )
    .groupby(
        "SplitRole",
        as_index=False
    )
    .agg(
        RowCount=(
            "Date",
            "size"
        ),
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        SplitContextCount=(
            "SplitContextID",
            "nunique"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        ZeroDemandRows=(
            "ZeroDemandFlag",
            "sum"
        ),
        ContextDemandUnits=(
            "TotalDemand",
            "sum"
        ),
        RowsWithAllHistoricalFeatures=(
            "AllHistoricalFeaturesAvailable",
            "sum"
        ),
        RowsWithMissingHistoricalFeatures=(
            "MissingHistoricalFeatureFlag",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 10. Create window-level split summary
# ------------------------------------------------------------

step3_window_split_summary_df = (
    step3_complete_row_split_manifest_df
    .assign(
        PositiveDemandFlag=(
            step3_complete_row_split_manifest_df[
                "TotalDemand"
            ] > 0
        ).astype(int)
    )
    .groupby(
        [
            "WindowID",
            "WindowPurpose",
            "SplitRole"
        ],
        as_index=False
    )
    .agg(
        RowCount=(
            "Date",
            "size"
        ),
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        ContextDemandUnits=(
            "TotalDemand",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 11. Audit expected cross-window reuse
# ------------------------------------------------------------

step3_cross_window_reuse_detail_df = (
    step3_independent_split_manifest_df
    .groupby(
        [
            "CanonicalProductID",
            "Date"
        ],
        as_index=False
    )
    .agg(
        SplitContextCount=(
            "SplitContextID",
            "nunique"
        ),
        SplitRoleCount=(
            "SplitRole",
            "nunique"
        ),
        AppearsAsTraining=(
            "SplitRole",
            lambda series:
            bool(
                series.eq("TRAIN").any()
            )
        ),
        AppearsAsValidation=(
            "SplitRole",
            lambda series:
            bool(
                series.eq("VALIDATION").any()
            )
        ),
        AppearsAsFinalTest=(
            "SplitRole",
            lambda series:
            bool(
                series.eq("FINAL_TEST").any()
            )
        )
    )
)

rows_reused_across_windows = int(
    (
        step3_cross_window_reuse_detail_df[
            "SplitContextCount"
        ] > 1
    ).sum()
)

rows_with_multiple_roles_across_windows = int(
    (
        step3_cross_window_reuse_detail_df[
            "SplitRoleCount"
        ] > 1
    ).sum()
)

rows_used_for_evaluation_then_later_training = int(
    (
        step3_cross_window_reuse_detail_df[
            "AppearsAsTraining"
        ]
        &
        (
            step3_cross_window_reuse_detail_df[
                "AppearsAsValidation"
            ]
            |
            step3_cross_window_reuse_detail_df[
                "AppearsAsFinalTest"
            ]
        )
    ).sum()
)

step3_cross_window_reuse_summary_df = pd.DataFrame([
    {
        "AuditMetric":
            "IndependentManifestRows",
        "Value":
            len(
                step3_independent_split_manifest_df
            ),
        "Interpretation":
            "Rows include repeated source rows across expanding windows."
    },
    {
        "AuditMetric":
            "UniqueIndependentSourceProductDateRows",
        "Value":
            step3_cross_window_reuse_detail_df.shape[0],
        "Interpretation":
            "Distinct source product-date rows represented."
    },
    {
        "AuditMetric":
            "RowsReusedAcrossMultipleWindows",
        "Value":
            rows_reused_across_windows,
        "Interpretation":
            "Expected under expanding-window backtesting."
    },
    {
        "AuditMetric":
            "RowsWithMultipleRolesAcrossWindows",
        "Value":
            rows_with_multiple_roles_across_windows,
        "Interpretation":
            "A prior validation row may later become training data."
    },
    {
        "AuditMetric":
            "RowsUsedForEvaluationAndTrainingAcrossDifferentWindows",
        "Value":
            rows_used_for_evaluation_then_later_training,
        "Interpretation":
            "Expected chronological expanding-window behaviour."
    },
    {
        "AuditMetric":
            "WithinWindowDuplicateProductDateRows",
        "Value":
            int(
                step3_independent_split_manifest_df[
                    [
                        "CanonicalProductID",
                        "WindowID",
                        "Date"
                    ]
                ].duplicated().sum()
            ),
        "Interpretation":
            "Must remain zero."
    }
])

assert step3_cross_window_reuse_summary_df.loc[
    step3_cross_window_reuse_summary_df[
        "AuditMetric"
    ] == "WithinWindowDuplicateProductDateRows",
    "Value"
].iloc[0] == 0


# ------------------------------------------------------------
# 12. Create Part 3 validation summary
# ------------------------------------------------------------

step3_part3_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "IndependentProductWindowContexts",
        "ForecastOnlyProductContexts",
        "TotalSplitContexts",
        "ProductsRepresented",
        "IndependentEvaluationProducts",
        "ForecastOnlyProducts",
        "UniqueSourceRowsRepresented",
        "UniqueSourceDemandUnitsRepresented",
        "ManifestTrainingCountMismatches",
        "ManifestEvaluationCountMismatches",
        "ManifestBoundaryMismatches",
        "ManifestChronologicalOverlapViolations",
        "ForecastOnlyRowCountMismatches",
        "EvaluationRowsMissingHistoricalFeatures",
        "ValidationPolicyViolations",
        "FinalTestPolicyViolations",
        "ForecastOnlyPolicyViolations",
        "WithinWindowDuplicateProductDateRows",
        "AllZeroDemandEvaluationWindows",
        "AllZeroEvaluationWindowsRemoved",
        "SameDayLeakageColumnsPresent"
    ],
    "Value": [
        step3_independent_split_manifest_df[
            "SplitContextID"
        ].nunique(),
        step3_forecast_only_split_manifest_df[
            "SplitContextID"
        ].nunique(),
        step3_complete_row_split_manifest_df[
            "SplitContextID"
        ].nunique(),
        step3_complete_row_split_manifest_df[
            "CanonicalProductID"
        ].nunique(),
        step3_independent_split_manifest_df[
            "CanonicalProductID"
        ].nunique(),
        step3_forecast_only_split_manifest_df[
            "CanonicalProductID"
        ].nunique(),
        len(
            unique_manifest_source_rows_df
        ),
        int(
            unique_manifest_source_rows_df[
                "TotalDemand"
            ].sum()
        ),
        manifest_training_count_mismatches,
        manifest_evaluation_count_mismatches,
        manifest_boundary_mismatches,
        manifest_overlap_violations,
        forecast_only_row_count_mismatches,
        evaluation_rows_missing_history,
        validation_policy_violations,
        final_test_policy_violations,
        forecast_only_policy_violations,
        int(
            step3_complete_row_split_manifest_df[
                [
                    "CanonicalProductID",
                    "WindowID",
                    "Date"
                ]
            ].duplicated().sum()
        ),
        len(
            step3_row_manifest_zero_demand_audit_df
        ),
        0,
        len(
            manifest_leakage_columns_found
        )
    ]
})


# ------------------------------------------------------------
# 13. Validate Part 3 summary
# ------------------------------------------------------------

part3_summary_lookup = dict(
    zip(
        step3_part3_validation_summary_df[
            "ValidationMetric"
        ],
        step3_part3_validation_summary_df[
            "Value"
        ]
    )
)

assert part3_summary_lookup[
    "IndependentProductWindowContexts"
] == 472

assert part3_summary_lookup[
    "ForecastOnlyProductContexts"
] == 9

assert part3_summary_lookup[
    "TotalSplitContexts"
] == 481

assert part3_summary_lookup[
    "ProductsRepresented"
] == 227

assert part3_summary_lookup[
    "IndependentEvaluationProducts"
] == 218

assert part3_summary_lookup[
    "ForecastOnlyProducts"
] == 9

assert part3_summary_lookup[
    "UniqueSourceRowsRepresented"
] == 43_774

assert part3_summary_lookup[
    "UniqueSourceDemandUnitsRepresented"
] == 116_158

assert part3_summary_lookup[
    "ManifestTrainingCountMismatches"
] == 0

assert part3_summary_lookup[
    "ManifestEvaluationCountMismatches"
] == 0

assert part3_summary_lookup[
    "ManifestBoundaryMismatches"
] == 0

assert part3_summary_lookup[
    "ManifestChronologicalOverlapViolations"
] == 0

assert part3_summary_lookup[
    "ForecastOnlyRowCountMismatches"
] == 0

assert part3_summary_lookup[
    "EvaluationRowsMissingHistoricalFeatures"
] == 0

assert part3_summary_lookup[
    "AllZeroDemandEvaluationWindows"
] == 344

assert part3_summary_lookup[
    "AllZeroEvaluationWindowsRemoved"
] == 0

assert part3_summary_lookup[
    "SameDayLeakageColumnsPresent"
] == 0


# ------------------------------------------------------------
# 14. Print Cell 49 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 3 — "
    "ROW-LEVEL SPLIT VALIDATION PASSED"
)
print("=" * 75)

print()
print("Split-role summary:")
display(step3_split_role_summary_df)

print()
print("Window and role summary:")
display(step3_window_split_summary_df)

print()
print("Coverage validation:")
print(
    "Training-count mismatches:",
    manifest_training_count_mismatches
)
print(
    "Evaluation-count mismatches:",
    manifest_evaluation_count_mismatches
)
print(
    "Boundary mismatches:",
    manifest_boundary_mismatches
)
print(
    "Chronological overlap violations:",
    manifest_overlap_violations
)
print(
    "Forecast-only count mismatches:",
    forecast_only_row_count_mismatches
)

print()
print("Source representation:")
print(
    "Unique source rows represented:",
    f"{len(unique_manifest_source_rows_df):,}"
)
print(
    "Unique source demand units represented:",
    f"{unique_manifest_source_rows_df['TotalDemand'].sum():,}"
)

print()
print("Intermittent-demand audit:")
print(
    "All-zero evaluation windows retained:",
    len(
        step3_row_manifest_zero_demand_audit_df
    )
)
print(
    "All-zero evaluation windows removed:",
    0
)

print()
print("Cross-window reuse:")
display(step3_cross_window_reuse_summary_df)

print()
print("Cell 49 completed successfully.")

STEP 3, PART 3 — ROW-LEVEL SPLIT VALIDATION PASSED

Split-role summary:


,SplitRole,RowCount,ProductCount,SplitContextCount,PositiveDemandRows,ZeroDemandRows,ContextDemandUnits,RowsWithAllHistoricalFeatures,RowsWithMissingHistoricalFeatures
0,FINAL_TEST,4150,218,218,1290,2860,13797,4150,0
1,FORECAST_ONLY_TRAINING,996,9,9,36,960,65,816,180
2,TRAIN,87166,218,472,32663,54503,218359,77726,9440
3,VALIDATION,5080,127,254,758,4322,8117,5080,0



Window and role summary:


,WindowID,WindowPurpose,SplitRole,RowCount,ProductCount,PositiveDemandRows,ContextDemandUnits
0,FORECAST_ONLY_FULL_HISTORY,FORECAST_MODEL_FIT_ONLY,FORECAST_ONLY_TRAINING,996,9,36,65
1,LIMITED_FINAL_HOLDOUT,LIMITED_FINAL_HOLDOUT_TEST,FINAL_TEST,70,14,3,61
2,LIMITED_FINAL_HOLDOUT,LIMITED_FINAL_HOLDOUT_TEST,TRAIN,2104,14,61,708
3,STANDARD_BACKTEST_FOLD_1,BACKTEST_VALIDATION,TRAIN,22999,127,9245,56377
4,STANDARD_BACKTEST_FOLD_1,BACKTEST_VALIDATION,VALIDATION,2540,127,361,3309
5,STANDARD_BACKTEST_FOLD_2,BACKTEST_VALIDATION,TRAIN,25539,127,9606,59686
6,STANDARD_BACKTEST_FOLD_2,BACKTEST_VALIDATION,VALIDATION,2540,127,397,4808
7,STANDARD_FINAL_HOLDOUT,FINAL_HOLDOUT_TEST,FINAL_TEST,4080,204,1287,13736
8,STANDARD_FINAL_HOLDOUT,FINAL_HOLDOUT_TEST,TRAIN,36524,204,13751,101588



Coverage validation:
Training-count mismatches: 0
Evaluation-count mismatches: 0
Boundary mismatches: 0
Chronological overlap violations: 0
Forecast-only count mismatches: 0

Source representation:
Unique source rows represented: 43,774
Unique source demand units represented: 116,158

Intermittent-demand audit:
All-zero evaluation windows retained: 344
All-zero evaluation windows removed: 0

Cross-window reuse:


,AuditMetric,Value,Interpretation
0,IndependentManifestRows,96396,Rows include repeated source rows across expan...
1,UniqueIndependentSourceProductDateRows,42778,Distinct source product-date rows represented.
2,RowsReusedAcrossMultipleWindows,28079,Expected under expanding-window backtesting.
3,RowsWithMultipleRolesAcrossWindows,5080,A prior validation row may later become traini...
4,RowsUsedForEvaluationAndTrainingAcrossDifferen...,5080,Expected chronological expanding-window behavi...
5,WithinWindowDuplicateProductDateRows,0,Must remain zero.



Cell 49 completed successfully.


In [61]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 3
# Cell 50: Save row-level split manifests and update handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 3 objects exist
# ------------------------------------------------------------

required_step3_part3_objects = [
    "step3_complete_row_split_manifest_df",
    "step3_independent_split_manifest_df",
    "step3_model_selection_manifest_df",
    "step3_final_test_manifest_df",
    "step3_forecast_only_training_manifest_df",
    "step3_manifest_coverage_comparison_df",
    "step3_manifest_boundary_validation_df",
    "step3_split_role_summary_df",
    "step3_window_split_summary_df",
    "step3_cross_window_reuse_summary_df",
    "step3_row_manifest_zero_demand_audit_df",
    "step3_part3_validation_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step3_part3_objects = [
    object_name
    for object_name in required_step3_part3_objects
    if object_name not in globals()
]

if missing_step3_part3_objects:
    raise NameError(
        "The following Step 3 Part 3 objects are missing:\n"
        f"{missing_step3_part3_objects}\n\n"
        "Run Cells 48 and 49 before running Cell 50."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff objects if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Part 3 output paths
# ------------------------------------------------------------

STEP3_COMPLETE_ROW_MANIFEST_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_complete_row_split_manifest.csv"
)

STEP3_INDEPENDENT_ROW_MANIFEST_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_independent_evaluation_split_manifest.csv"
)

STEP3_MODEL_SELECTION_MANIFEST_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_model_selection_split_manifest.csv"
)

STEP3_FINAL_TEST_MANIFEST_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_final_test_split_manifest.csv"
)

STEP3_FORECAST_ONLY_MANIFEST_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_forecast_only_training_manifest.csv"
)

STEP3_MANIFEST_COVERAGE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_manifest_coverage_validation.csv"
)

STEP3_MANIFEST_BOUNDARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_manifest_boundary_validation.csv"
)

STEP3_SPLIT_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_split_role_summary.csv"
)

STEP3_WINDOW_SPLIT_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_window_split_summary.csv"
)

STEP3_CROSS_WINDOW_REUSE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_cross_window_reuse_summary.csv"
)

STEP3_ROW_ZERO_DEMAND_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_row_manifest_zero_demand_evaluation_audit.csv"
)

STEP3_PART3_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "15_step3_part3_validation_summary.csv"
)


# ------------------------------------------------------------
# 4. Save all Part 3 outputs
# ------------------------------------------------------------

step3_complete_row_split_manifest_df.to_csv(
    STEP3_COMPLETE_ROW_MANIFEST_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_independent_split_manifest_df.to_csv(
    STEP3_INDEPENDENT_ROW_MANIFEST_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_model_selection_manifest_df.to_csv(
    STEP3_MODEL_SELECTION_MANIFEST_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_final_test_manifest_df.to_csv(
    STEP3_FINAL_TEST_MANIFEST_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_forecast_only_training_manifest_df.to_csv(
    STEP3_FORECAST_ONLY_MANIFEST_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_manifest_coverage_comparison_df.to_csv(
    STEP3_MANIFEST_COVERAGE_OUTPUT,
    index=False
)

step3_manifest_boundary_validation_df.to_csv(
    STEP3_MANIFEST_BOUNDARY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_split_role_summary_df.to_csv(
    STEP3_SPLIT_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_window_split_summary_df.to_csv(
    STEP3_WINDOW_SPLIT_SUMMARY_OUTPUT,
    index=False
)

step3_cross_window_reuse_summary_df.to_csv(
    STEP3_CROSS_WINDOW_REUSE_OUTPUT,
    index=False
)

step3_row_manifest_zero_demand_audit_df.to_csv(
    STEP3_ROW_ZERO_DEMAND_AUDIT_OUTPUT,
    index=False
)

step3_part3_validation_summary_df.to_csv(
    STEP3_PART3_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 5. Reload major saved outputs
# ------------------------------------------------------------

saved_complete_manifest = pd.read_csv(
    STEP3_COMPLETE_ROW_MANIFEST_OUTPUT,
    low_memory=False
)

saved_independent_manifest = pd.read_csv(
    STEP3_INDEPENDENT_ROW_MANIFEST_OUTPUT,
    low_memory=False
)

saved_model_selection_manifest = pd.read_csv(
    STEP3_MODEL_SELECTION_MANIFEST_OUTPUT,
    low_memory=False
)

saved_final_test_manifest = pd.read_csv(
    STEP3_FINAL_TEST_MANIFEST_OUTPUT,
    low_memory=False
)

saved_forecast_only_manifest = pd.read_csv(
    STEP3_FORECAST_ONLY_MANIFEST_OUTPUT,
    low_memory=False
)

saved_manifest_coverage = pd.read_csv(
    STEP3_MANIFEST_COVERAGE_OUTPUT,
    low_memory=False
)

saved_manifest_boundaries = pd.read_csv(
    STEP3_MANIFEST_BOUNDARY_OUTPUT,
    low_memory=False
)

saved_split_role_summary = pd.read_csv(
    STEP3_SPLIT_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_window_split_summary = pd.read_csv(
    STEP3_WINDOW_SPLIT_SUMMARY_OUTPUT,
    low_memory=False
)

saved_cross_window_reuse = pd.read_csv(
    STEP3_CROSS_WINDOW_REUSE_OUTPUT,
    low_memory=False
)

saved_zero_demand_audit = pd.read_csv(
    STEP3_ROW_ZERO_DEMAND_AUDIT_OUTPUT,
    low_memory=False
)

saved_part3_validation = pd.read_csv(
    STEP3_PART3_VALIDATION_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 6. Validate saved outputs
# ------------------------------------------------------------

assert len(
    saved_complete_manifest
) == len(
    step3_complete_row_split_manifest_df
)

assert len(
    saved_independent_manifest
) == len(
    step3_independent_split_manifest_df
)

assert saved_complete_manifest[
    "CanonicalProductID"
].nunique() == 227

assert saved_complete_manifest[
    "SplitContextID"
].nunique() == 481

assert saved_complete_manifest[
    [
        "CanonicalProductID",
        "WindowID",
        "Date"
    ]
].duplicated().sum() == 0

assert saved_independent_manifest[
    "CanonicalProductID"
].nunique() == 218

assert saved_forecast_only_manifest[
    "CanonicalProductID"
].nunique() == 9

assert set(
    saved_model_selection_manifest[
        "SplitRole"
    ].unique()
) == {
    "TRAIN",
    "VALIDATION"
}

assert set(
    saved_final_test_manifest[
        "SplitRole"
    ].unique()
) == {
    "TRAIN",
    "FINAL_TEST"
}

assert set(
    saved_forecast_only_manifest[
        "SplitRole"
    ].unique()
) == {
    "FORECAST_ONLY_TRAINING"
}

assert saved_manifest_coverage[
    "TrainingRowCountMismatch"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_manifest_coverage[
    "EvaluationRowCountMismatch"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_manifest_boundaries[
    "ChronologicalOverlapViolation"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert len(
    saved_zero_demand_audit
) == 344

assert len(
    saved_part3_validation
) == 21


# ------------------------------------------------------------
# 7. Build Markdown split-role summary
# ------------------------------------------------------------

split_role_markdown = "\n".join(
    (
        f"- `{row.SplitRole}`: "
        f"{int(row.RowCount):,} manifest rows, "
        f"{int(row.ProductCount)} products, "
        f"{int(row.SplitContextCount)} split contexts"
    )
    for row in (
        step3_split_role_summary_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 8. Update the Markdown handoff
# ------------------------------------------------------------

step3_part3_summary = f"""
**Status:** Completed and validated

### Purpose

Step 3 Part 3 converted the approved product-window assignments
into row-level chronological split manifests.

### Split roles

{split_role_markdown}

### Split contexts

- Independent product-window contexts: 472
- Forecast-only product contexts: 9
- Total split contexts: 481
- Products represented: 227
- Products with independent evaluation: 218
- Products without independent holdout: 9

### Row-level split policy

- `TRAIN`: training observations belonging to a specific evaluation window
- `VALIDATION`: observations used only in the two model-selection folds
- `FINAL_TEST`: observations reserved for standard or limited final testing
- `FORECAST_ONLY_TRAINING`: full available histories for minimal-evidence products

A source row may appear in multiple expanding-window contexts.
For example, an earlier validation row can become training data
for a later window. No source row appears in multiple roles inside
the same product-window context.

### Validation

- Training-count mismatches: 0
- Evaluation-count mismatches: 0
- Boundary mismatches: 0
- Chronological overlap violations: 0
- Forecast-only row-count mismatches: 0
- Evaluation rows missing historical features: 0
- Same-day leakage columns present: 0
- Unique source rows represented: 43,774
- Unique source demand units represented: 116,158

### Intermittent-demand evaluation

- All-zero-demand evaluation windows retained: 344
- All-zero-demand evaluation windows removed: 0

### Saved Step 3 Part 3 outputs

- `15_step3_complete_row_split_manifest.csv`
- `15_step3_independent_evaluation_split_manifest.csv`
- `15_step3_model_selection_split_manifest.csv`
- `15_step3_final_test_split_manifest.csv`
- `15_step3_forecast_only_training_manifest.csv`
- `15_step3_manifest_coverage_validation.csv`
- `15_step3_manifest_boundary_validation.csv`
- `15_step3_split_role_summary.csv`
- `15_step3_window_split_summary.csv`
- `15_step3_cross_window_reuse_summary.csv`
- `15_step3_row_manifest_zero_demand_evaluation_audit.csv`
- `15_step3_part3_validation_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_3_part_3",
    section_title=(
        "Forecasting Preparation — Step 3, Part 3"
    ),
    section_body=step3_part3_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 9. Print final completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 3, PART 3 COMPLETED"
)
print("=" * 75)

print()
print("Split contexts:")
print("Independent contexts: 472")
print("Forecast-only contexts: 9")
print("Total contexts: 481")

print()
print("Product coverage:")
print("Products represented: 227")
print("Products with independent evaluation: 218")
print("Forecast-only products: 9")

print()
print("Row-level manifests:")
print(
    "Complete manifest rows:",
    f"{len(saved_complete_manifest):,}"
)
print(
    "Independent-evaluation manifest rows:",
    f"{len(saved_independent_manifest):,}"
)
print(
    "Model-selection manifest rows:",
    f"{len(saved_model_selection_manifest):,}"
)
print(
    "Final-test manifest rows:",
    f"{len(saved_final_test_manifest):,}"
)
print(
    "Forecast-only training rows:",
    f"{len(saved_forecast_only_manifest):,}"
)

print()
print("Validation:")
print(
    "Training-count mismatches:",
    manifest_training_count_mismatches
)
print(
    "Evaluation-count mismatches:",
    manifest_evaluation_count_mismatches
)
print(
    "Boundary mismatches:",
    manifest_boundary_mismatches
)
print(
    "Chronological overlap violations:",
    manifest_overlap_violations
)
print(
    "Evaluation rows missing historical features:",
    evaluation_rows_missing_history
)

print()
print("Source representation:")
print("Unique source rows represented: 43,774")
print("Unique source demand units represented: 116,158")

print()
print("Intermittent-demand evaluation:")
print(
    "All-zero evaluation windows retained:",
    len(saved_zero_demand_audit)
)
print("All-zero evaluation windows removed: 0")

print()
print("Saved files:")
print(f"1. {STEP3_COMPLETE_ROW_MANIFEST_OUTPUT}")
print(f"2. {STEP3_INDEPENDENT_ROW_MANIFEST_OUTPUT}")
print(f"3. {STEP3_MODEL_SELECTION_MANIFEST_OUTPUT}")
print(f"4. {STEP3_FINAL_TEST_MANIFEST_OUTPUT}")
print(f"5. {STEP3_FORECAST_ONLY_MANIFEST_OUTPUT}")
print(f"6. {STEP3_MANIFEST_COVERAGE_OUTPUT}")
print(f"7. {STEP3_MANIFEST_BOUNDARY_OUTPUT}")
print(f"8. {STEP3_SPLIT_ROLE_SUMMARY_OUTPUT}")
print(f"9. {STEP3_WINDOW_SPLIT_SUMMARY_OUTPUT}")
print(f"10. {STEP3_CROSS_WINDOW_REUSE_OUTPUT}")
print(f"11. {STEP3_ROW_ZERO_DEMAND_AUDIT_OUTPUT}")
print(f"12. {STEP3_PART3_VALIDATION_OUTPUT}")
print(f"13. {HANDOFF_FILE}")

print()
print(
    "All Step 3, Part 3 validation checks passed."
)

FORECASTING PREPARATION — STEP 3, PART 3 COMPLETED

Split contexts:
Independent contexts: 472
Forecast-only contexts: 9
Total contexts: 481

Product coverage:
Products represented: 227
Products with independent evaluation: 218
Forecast-only products: 9

Row-level manifests:
Complete manifest rows: 97,392
Independent-evaluation manifest rows: 96,396
Model-selection manifest rows: 53,618
Final-test manifest rows: 42,778
Forecast-only training rows: 996

Validation:
Training-count mismatches: 0
Evaluation-count mismatches: 0
Boundary mismatches: 0
Chronological overlap violations: 0
Evaluation rows missing historical features: 0

Source representation:
Unique source rows represented: 43,774
Unique source demand units represented: 116,158

Intermittent-demand evaluation:
All-zero evaluation windows retained: 344
All-zero evaluation windows removed: 0

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/15_step3_complete_row_split_manifest.csv
2. /Users/r

In [62]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 4
# Cell 51: Load and validate the completed row-level
# chronological split system
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the previous forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


# ------------------------------------------------------------
# 2. Define official Step 3 Part 4 input files
# ------------------------------------------------------------

STEP3_PART4_COMPLETE_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_complete_row_split_manifest.csv"
)

STEP3_PART4_INDEPENDENT_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_independent_evaluation_split_manifest.csv"
)

STEP3_PART4_MODEL_SELECTION_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_model_selection_split_manifest.csv"
)

STEP3_PART4_FINAL_TEST_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_final_test_split_manifest.csv"
)

STEP3_PART4_FORECAST_ONLY_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_forecast_only_training_manifest.csv"
)

STEP3_PART4_SOURCE_FEATURE_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_model_view.csv"
)

STEP3_PART4_FEATURE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_combined_feature_contract.csv"
)

STEP3_PART4_ASSIGNMENT_REGISTER_FILE = (
    FORECAST_PREPARATION_DIR
    / "14_step3_product_window_assignment_register.csv"
)

STEP3_PART4_ZERO_DEMAND_AUDIT_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_row_manifest_zero_demand_evaluation_audit.csv"
)

required_part4_input_files = [
    STEP3_PART4_COMPLETE_MANIFEST_FILE,
    STEP3_PART4_INDEPENDENT_MANIFEST_FILE,
    STEP3_PART4_MODEL_SELECTION_MANIFEST_FILE,
    STEP3_PART4_FINAL_TEST_MANIFEST_FILE,
    STEP3_PART4_FORECAST_ONLY_MANIFEST_FILE,
    STEP3_PART4_SOURCE_FEATURE_FILE,
    STEP3_PART4_FEATURE_CONTRACT_FILE,
    STEP3_PART4_ASSIGNMENT_REGISTER_FILE,
    STEP3_PART4_ZERO_DEMAND_AUDIT_FILE
]

missing_part4_input_files = [
    file_path
    for file_path in required_part4_input_files
    if not file_path.exists()
]

if missing_part4_input_files:
    raise FileNotFoundError(
        "The following Step 3 Part 4 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_part4_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load official inputs
# ------------------------------------------------------------

step3_part4_complete_manifest_df = pd.read_csv(
    STEP3_PART4_COMPLETE_MANIFEST_FILE,
    low_memory=False
)

step3_part4_independent_manifest_df = pd.read_csv(
    STEP3_PART4_INDEPENDENT_MANIFEST_FILE,
    low_memory=False
)

step3_part4_model_selection_manifest_df = pd.read_csv(
    STEP3_PART4_MODEL_SELECTION_MANIFEST_FILE,
    low_memory=False
)

step3_part4_final_test_manifest_df = pd.read_csv(
    STEP3_PART4_FINAL_TEST_MANIFEST_FILE,
    low_memory=False
)

step3_part4_forecast_only_manifest_df = pd.read_csv(
    STEP3_PART4_FORECAST_ONLY_MANIFEST_FILE,
    low_memory=False
)

step3_part4_source_feature_df = pd.read_csv(
    STEP3_PART4_SOURCE_FEATURE_FILE,
    low_memory=False
)

step3_part4_feature_contract_df = pd.read_csv(
    STEP3_PART4_FEATURE_CONTRACT_FILE,
    low_memory=False
)

step3_part4_assignment_register_df = pd.read_csv(
    STEP3_PART4_ASSIGNMENT_REGISTER_FILE,
    low_memory=False
)

step3_part4_previous_zero_audit_df = pd.read_csv(
    STEP3_PART4_ZERO_DEMAND_AUDIT_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Boolean parsing helper
# ------------------------------------------------------------

def parse_step3_part4_boolean(series):
    """
    Convert common CSV Boolean representations into bool values.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalised_values = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    converted_values = normalised_values.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if converted_values.isna().sum() > 0:

        invalid_values = (
            series.loc[
                converted_values.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unrecognised Boolean values were found:\n"
            f"{invalid_values}"
        )

    return converted_values.astype(bool)


# ------------------------------------------------------------
# 5. Parse manifest dates
# ------------------------------------------------------------

manifest_dataframes = [
    step3_part4_complete_manifest_df,
    step3_part4_independent_manifest_df,
    step3_part4_model_selection_manifest_df,
    step3_part4_final_test_manifest_df,
    step3_part4_forecast_only_manifest_df
]

for dataframe in manifest_dataframes:

    dataframe["Date"] = pd.to_datetime(
        dataframe["Date"],
        format="%Y-%m-%d",
        errors="coerce"
    )

    assert dataframe["Date"].isna().sum() == 0


# ------------------------------------------------------------
# 6. Parse source dates
# ------------------------------------------------------------

for column in [
    "Date",
    "ProductFirstObservedDate"
]:

    step3_part4_source_feature_df[column] = pd.to_datetime(
        step3_part4_source_feature_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step3_part4_source_feature_df[
    [
        "Date",
        "ProductFirstObservedDate"
    ]
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 7. Standardise product identifiers
# ------------------------------------------------------------

identifier_dataframes = (
    manifest_dataframes
    + [
        step3_part4_source_feature_df,
        step3_part4_assignment_register_df,
        step3_part4_previous_zero_audit_df
    ]
)

for dataframe in identifier_dataframes:

    dataframe[
        "CanonicalProductID"
    ] = (
        dataframe[
            "CanonicalProductID"
        ]
        .astype("string")
        .str.strip()
    )


# ------------------------------------------------------------
# 8. Parse important numeric fields
# ------------------------------------------------------------

for dataframe in manifest_dataframes:

    for column in [
        "OperatingDaySequence",
        "ProductAgeOperatingDays",
        "TotalDemand",
        "HistoricalFeatureAvailableCount",
        "WindowOrder"
    ]:

        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce"
        )

    assert dataframe[
        [
            "OperatingDaySequence",
            "ProductAgeOperatingDays",
            "TotalDemand",
            "HistoricalFeatureAvailableCount",
            "WindowOrder"
        ]
    ].isna().sum().sum() == 0


for column in [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]:

    step3_part4_source_feature_df[column] = pd.to_numeric(
        step3_part4_source_feature_df[column],
        errors="coerce"
    )

assert step3_part4_source_feature_df[
    [
        "OperatingDaySequence",
        "ProductAgeOperatingDays",
        "TotalDemand"
    ]
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 9. Parse manifest Boolean fields
# ------------------------------------------------------------

manifest_boolean_columns = [
    "AllHistoricalFeaturesAvailable",
    "MayBeUsedForModelSelection",
    "ReservedFinalTest",
    "IndependentEvaluationAvailable",
    "RowEligibleForScoring"
]

for dataframe in manifest_dataframes:

    for column in manifest_boolean_columns:

        dataframe[column] = parse_step3_part4_boolean(
            dataframe[column]
        )


# ------------------------------------------------------------
# 10. Validate the completed manifest structures
# ------------------------------------------------------------

assert len(
    step3_part4_complete_manifest_df
) == 97_392

assert len(
    step3_part4_independent_manifest_df
) == 96_396

assert len(
    step3_part4_model_selection_manifest_df
) == 53_618

assert len(
    step3_part4_final_test_manifest_df
) == 42_778

assert len(
    step3_part4_forecast_only_manifest_df
) == 996

assert step3_part4_complete_manifest_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_part4_independent_manifest_df[
    "CanonicalProductID"
].nunique() == 218

assert step3_part4_forecast_only_manifest_df[
    "CanonicalProductID"
].nunique() == 9

assert step3_part4_complete_manifest_df[
    "SplitContextID"
].nunique() == 481

assert step3_part4_independent_manifest_df[
    "SplitContextID"
].nunique() == 472

assert step3_part4_forecast_only_manifest_df[
    "SplitContextID"
].nunique() == 9

assert step3_part4_complete_manifest_df[
    [
        "CanonicalProductID",
        "WindowID",
        "Date"
    ]
].duplicated().sum() == 0


# ------------------------------------------------------------
# 11. Validate the source feature dataset
# ------------------------------------------------------------

assert step3_part4_source_feature_df.shape == (
    43_774,
    58
)

assert step3_part4_source_feature_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_part4_source_feature_df[
    "Date"
].nunique() == 245

assert step3_part4_source_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert np.isclose(
    step3_part4_source_feature_df[
        "TotalDemand"
    ].sum(),
    116_158
)


# ------------------------------------------------------------
# 12. Identify the 31 historical demand features
# ------------------------------------------------------------

historical_feature_flag = parse_step3_part4_boolean(
    step3_part4_feature_contract_df[
        "IsHistoricalDemandFeature"
    ]
)

step3_part4_historical_feature_columns = (
    step3_part4_feature_contract_df.loc[
        historical_feature_flag,
        "Column"
    ]
    .tolist()
)

assert len(
    step3_part4_historical_feature_columns
) == 31

assert all(
    column in step3_part4_source_feature_df.columns
    for column in step3_part4_historical_feature_columns
)


# ------------------------------------------------------------
# 13. Confirm all source rows are represented
# ------------------------------------------------------------

step3_part4_unique_manifest_source_df = (
    step3_part4_complete_manifest_df[
        [
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence",
            "TotalDemand"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Date",
            "CanonicalProductID"
        ]
    )
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

step3_part4_source_comparison_df = (
    step3_part4_source_feature_df[
        [
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence",
            "TotalDemand"
        ]
    ]
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert len(
    step3_part4_unique_manifest_source_df
) == 43_774

pd.testing.assert_frame_equal(
    step3_part4_source_comparison_df,
    step3_part4_unique_manifest_source_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

assert np.isclose(
    step3_part4_unique_manifest_source_df[
        "TotalDemand"
    ].sum(),
    116_158
)


# ------------------------------------------------------------
# 14. Extract all scoring rows
# ------------------------------------------------------------

step3_part4_scoring_rows_df = (
    step3_part4_complete_manifest_df.loc[
        step3_part4_complete_manifest_df[
            "RowEligibleForScoring"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

validation_scoring_rows = int(
    (
        step3_part4_scoring_rows_df[
            "SplitRole"
        ] == "VALIDATION"
    ).sum()
)

final_test_scoring_rows = int(
    (
        step3_part4_scoring_rows_df[
            "SplitRole"
        ] == "FINAL_TEST"
    ).sum()
)

total_scoring_rows = len(
    step3_part4_scoring_rows_df
)

assert validation_scoring_rows == 5_080
assert final_test_scoring_rows == 4_150
assert total_scoring_rows == 9_230


# ------------------------------------------------------------
# 15. Independently check scoring-row feature readiness
# ------------------------------------------------------------

scoring_feature_source_columns = (
    [
        "Date",
        "CanonicalProductID",
        "OperatingDaySequence",
        "TotalDemand"
    ]
    + step3_part4_historical_feature_columns
)

step3_part4_scoring_feature_check_df = (
    step3_part4_scoring_rows_df[
        [
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence",
            "TotalDemand",
            "SplitContextID",
            "WindowID",
            "SplitRole",
            "AllHistoricalFeaturesAvailable",
            "HistoricalFeatureAvailableCount"
        ]
    ]
    .merge(
        step3_part4_source_feature_df[
            scoring_feature_source_columns
        ].rename(
            columns={
                "TotalDemand":
                    "SourceTotalDemand"
            }
        ),
        on=[
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# 16. Validate target and historical features
# ------------------------------------------------------------

scoring_target_mismatches = int(
    (
        ~np.isclose(
            step3_part4_scoring_feature_check_df[
                "TotalDemand"
            ],
            step3_part4_scoring_feature_check_df[
                "SourceTotalDemand"
            ],
            rtol=1e-12,
            atol=1e-12
        )
    ).sum()
)

scoring_missing_historical_cells = int(
    step3_part4_scoring_feature_check_df[
        step3_part4_historical_feature_columns
    ]
    .isna()
    .sum()
    .sum()
)

step3_part4_scoring_feature_check_df[
    "IndependentAvailableHistoricalFeatureCount"
] = (
    step3_part4_scoring_feature_check_df[
        step3_part4_historical_feature_columns
    ]
    .notna()
    .sum(axis=1)
    .astype(int)
)

step3_part4_scoring_feature_check_df[
    "IndependentAllHistoricalFeaturesAvailable"
] = (
    step3_part4_scoring_feature_check_df[
        "IndependentAvailableHistoricalFeatureCount"
    ] == 31
)

scoring_rows_missing_any_historical_feature = int(
    (
        ~step3_part4_scoring_feature_check_df[
            "IndependentAllHistoricalFeaturesAvailable"
        ]
    ).sum()
)

manifest_feature_count_mismatches = int(
    (
        step3_part4_scoring_feature_check_df[
            "HistoricalFeatureAvailableCount"
        ]
        != step3_part4_scoring_feature_check_df[
            "IndependentAvailableHistoricalFeatureCount"
        ]
    ).sum()
)

manifest_feature_flag_mismatches = int(
    (
        step3_part4_scoring_feature_check_df[
            "AllHistoricalFeaturesAvailable"
        ]
        != step3_part4_scoring_feature_check_df[
            "IndependentAllHistoricalFeaturesAvailable"
        ]
    ).sum()
)

assert scoring_target_mismatches == 0
assert scoring_missing_historical_cells == 0
assert scoring_rows_missing_any_historical_feature == 0
assert manifest_feature_count_mismatches == 0
assert manifest_feature_flag_mismatches == 0


# ------------------------------------------------------------
# 17. Confirm same-day leakage columns remain absent
# ------------------------------------------------------------

same_day_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

part4_manifest_leakage_columns_found = sorted(
    same_day_leakage_columns.intersection(
        step3_part4_complete_manifest_df.columns
    )
)

part4_source_leakage_columns_found = sorted(
    same_day_leakage_columns.intersection(
        step3_part4_source_feature_df.columns
    )
)

assert len(
    part4_manifest_leakage_columns_found
) == 0

assert len(
    part4_source_leakage_columns_found
) == 0


# ------------------------------------------------------------
# 18. Create scoring feature-readiness audit
# ------------------------------------------------------------

step3_part4_evaluation_feature_readiness_audit_df = (
    step3_part4_scoring_feature_check_df.loc[
        (
            ~step3_part4_scoring_feature_check_df[
                "IndependentAllHistoricalFeaturesAvailable"
            ]
        )
        |
        (
            step3_part4_scoring_feature_check_df[
                "HistoricalFeatureAvailableCount"
            ]
            != 31
        )
        |
        (
            ~step3_part4_scoring_feature_check_df[
                "AllHistoricalFeaturesAvailable"
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(
    step3_part4_evaluation_feature_readiness_audit_df
) == 0


# ------------------------------------------------------------
# 19. Print Cell 51 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 4 — "
    "COMPLETED SPLIT SYSTEM LOADED AND VALIDATED"
)
print("=" * 75)

print()
print("Manifest structure:")
print(
    "Complete manifest rows:",
    f"{len(step3_part4_complete_manifest_df):,}"
)
print(
    "Independent-evaluation rows:",
    f"{len(step3_part4_independent_manifest_df):,}"
)
print(
    "Forecast-only rows:",
    f"{len(step3_part4_forecast_only_manifest_df):,}"
)
print(
    "Total split contexts:",
    step3_part4_complete_manifest_df[
        "SplitContextID"
    ].nunique()
)
print(
    "Products represented:",
    step3_part4_complete_manifest_df[
        "CanonicalProductID"
    ].nunique()
)

print()
print("Scoring rows:")
print(
    "Validation scoring rows:",
    f"{validation_scoring_rows:,}"
)
print(
    "Final-test scoring rows:",
    f"{final_test_scoring_rows:,}"
)
print(
    "Total scoring rows:",
    f"{total_scoring_rows:,}"
)

print()
print("Feature readiness:")
print(
    "Historical features required:",
    len(
        step3_part4_historical_feature_columns
    )
)
print(
    "Scoring rows missing any historical feature:",
    scoring_rows_missing_any_historical_feature
)
print(
    "Missing historical-feature cells:",
    scoring_missing_historical_cells
)
print(
    "Manifest feature-count mismatches:",
    manifest_feature_count_mismatches
)
print(
    "Manifest feature-flag mismatches:",
    manifest_feature_flag_mismatches
)

print()
print("Source representation:")
print(
    "Unique source rows:",
    f"{len(step3_part4_unique_manifest_source_df):,}"
)
print(
    "Unique source demand units:",
    f"{step3_part4_unique_manifest_source_df['TotalDemand'].sum():,.0f}"
)

print()
print("Cell 51 completed successfully.")

STEP 3, PART 4 — COMPLETED SPLIT SYSTEM LOADED AND VALIDATED

Manifest structure:
Complete manifest rows: 97,392
Independent-evaluation rows: 96,396
Forecast-only rows: 996
Total split contexts: 481
Products represented: 227

Scoring rows:
Validation scoring rows: 5,080
Final-test scoring rows: 4,150
Total scoring rows: 9,230

Feature readiness:
Historical features required: 31
Scoring rows missing any historical feature: 0
Missing historical-feature cells: 0
Manifest feature-count mismatches: 0
Manifest feature-flag mismatches: 0

Source representation:
Unique source rows: 43,774
Unique source demand units: 116,158

Cell 51 completed successfully.


In [63]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 4
# Cell 52: Audit chronology, final-test isolation,
# product coverage and demand distribution
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 51 objects exist
# ------------------------------------------------------------

required_cell_51_objects = [
    "step3_part4_complete_manifest_df",
    "step3_part4_independent_manifest_df",
    "step3_part4_model_selection_manifest_df",
    "step3_part4_final_test_manifest_df",
    "step3_part4_forecast_only_manifest_df",
    "step3_part4_scoring_rows_df",
    "step3_part4_scoring_feature_check_df",
    "step3_part4_historical_feature_columns"
]

missing_cell_51_objects = [
    object_name
    for object_name in required_cell_51_objects
    if object_name not in globals()
]

if missing_cell_51_objects:
    raise NameError(
        "The following Cell 51 objects are missing:\n"
        f"{missing_cell_51_objects}\n\n"
        "Run Cell 51 before running Cell 52."
    )


# ------------------------------------------------------------
# 2. Add demand flags for summaries
# ------------------------------------------------------------

step3_part4_summary_source_df = (
    step3_part4_complete_manifest_df
    .copy()
)

step3_part4_summary_source_df[
    "PositiveDemandFlag"
] = (
    step3_part4_summary_source_df[
        "TotalDemand"
    ] > 0
).astype(int)

step3_part4_summary_source_df[
    "ZeroDemandFlag"
] = (
    step3_part4_summary_source_df[
        "TotalDemand"
    ] == 0
).astype(int)

step3_part4_summary_source_df[
    "MissingHistoricalFeatureFlag"
] = (
    ~step3_part4_summary_source_df[
        "AllHistoricalFeaturesAvailable"
    ]
).astype(int)


# ------------------------------------------------------------
# 3. Create split-role demand summary
# ------------------------------------------------------------

step3_part4_split_role_demand_summary_df = (
    step3_part4_summary_source_df
    .groupby(
        "SplitRole",
        as_index=False
    )
    .agg(
        ManifestRows=(
            "Date",
            "size"
        ),
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        SplitContextCount=(
            "SplitContextID",
            "nunique"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        ZeroDemandRows=(
            "ZeroDemandFlag",
            "sum"
        ),
        ContextWeightedDemandUnits=(
            "TotalDemand",
            "sum"
        ),
        RowsWithAllHistoricalFeatures=(
            "AllHistoricalFeaturesAvailable",
            "sum"
        ),
        RowsWithMissingHistoricalFeatures=(
            "MissingHistoricalFeatureFlag",
            "sum"
        )
    )
)

step3_part4_split_role_demand_summary_df[
    "PositiveDemandRowPercentage"
] = (
    step3_part4_split_role_demand_summary_df[
        "PositiveDemandRows"
    ]
    / step3_part4_split_role_demand_summary_df[
        "ManifestRows"
    ]
    * 100
).round(2)

step3_part4_split_role_demand_summary_df[
    "ZeroDemandRowPercentage"
] = (
    step3_part4_split_role_demand_summary_df[
        "ZeroDemandRows"
    ]
    / step3_part4_split_role_demand_summary_df[
        "ManifestRows"
    ]
    * 100
).round(2)


# ------------------------------------------------------------
# 4. Create window-role demand summary
# ------------------------------------------------------------

step3_part4_window_role_demand_summary_df = (
    step3_part4_summary_source_df
    .groupby(
        [
            "WindowID",
            "WindowPurpose",
            "SplitRole"
        ],
        as_index=False
    )
    .agg(
        ManifestRows=(
            "Date",
            "size"
        ),
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        SplitContextCount=(
            "SplitContextID",
            "nunique"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        ZeroDemandRows=(
            "ZeroDemandFlag",
            "sum"
        ),
        ContextWeightedDemandUnits=(
            "TotalDemand",
            "sum"
        ),
        RowsWithMissingHistoricalFeatures=(
            "MissingHistoricalFeatureFlag",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 5. Create cohort-role demand summary
# ------------------------------------------------------------

step3_part4_cohort_role_demand_summary_df = (
    step3_part4_summary_source_df
    .groupby(
        [
            "EvaluationCohort",
            "SplitRole"
        ],
        as_index=False
    )
    .agg(
        ManifestRows=(
            "Date",
            "size"
        ),
        ProductCount=(
            "CanonicalProductID",
            "nunique"
        ),
        SplitContextCount=(
            "SplitContextID",
            "nunique"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        ZeroDemandRows=(
            "ZeroDemandFlag",
            "sum"
        ),
        ContextWeightedDemandUnits=(
            "TotalDemand",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 6. Create product-role coverage audit
# ------------------------------------------------------------

step3_part4_product_role_coverage_df = (
    step3_part4_summary_source_df
    .groupby(
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "EvaluationCohort",
            "DemandPatternClass"
        ],
        as_index=False
    )
    .agg(
        SplitContextCount=(
            "SplitContextID",
            "nunique"
        ),
        ManifestRows=(
            "Date",
            "size"
        ),
        TrainingRows=(
            "SplitRole",
            lambda series:
            int(
                series.eq("TRAIN").sum()
            )
        ),
        ValidationRows=(
            "SplitRole",
            lambda series:
            int(
                series.eq("VALIDATION").sum()
            )
        ),
        FinalTestRows=(
            "SplitRole",
            lambda series:
            int(
                series.eq("FINAL_TEST").sum()
            )
        ),
        ForecastOnlyTrainingRows=(
            "SplitRole",
            lambda series:
            int(
                series.eq(
                    "FORECAST_ONLY_TRAINING"
                ).sum()
            )
        )
    )
)

step3_part4_product_role_coverage_df[
    "HasIndependentEvaluation"
] = (
    (
        step3_part4_product_role_coverage_df[
            "ValidationRows"
        ] > 0
    )
    |
    (
        step3_part4_product_role_coverage_df[
            "FinalTestRows"
        ] > 0
    )
)

step3_part4_product_role_coverage_df[
    "IsForecastOnlyProduct"
] = (
    step3_part4_product_role_coverage_df[
        "ForecastOnlyTrainingRows"
    ] > 0
)

assert len(
    step3_part4_product_role_coverage_df
) == 227

assert step3_part4_product_role_coverage_df[
    "HasIndependentEvaluation"
].sum() == 218

assert step3_part4_product_role_coverage_df[
    "IsForecastOnlyProduct"
].sum() == 9


# ------------------------------------------------------------
# 7. Create independent-context chronology audit
# ------------------------------------------------------------

independent_training_boundaries_df = (
    step3_part4_independent_manifest_df.loc[
        step3_part4_independent_manifest_df[
            "SplitRole"
        ] == "TRAIN"
    ]
    .groupby(
        [
            "CanonicalProductID",
            "WindowID",
            "SplitContextID"
        ],
        as_index=False
    )
    .agg(
        TrainingRows=(
            "Date",
            "size"
        ),
        TrainingStartSequence=(
            "OperatingDaySequence",
            "min"
        ),
        TrainingEndSequence=(
            "OperatingDaySequence",
            "max"
        ),
        TrainingStartDate=(
            "Date",
            "min"
        ),
        TrainingEndDate=(
            "Date",
            "max"
        )
    )
)

independent_evaluation_boundaries_df = (
    step3_part4_independent_manifest_df.loc[
        step3_part4_independent_manifest_df[
            "SplitRole"
        ].isin([
            "VALIDATION",
            "FINAL_TEST"
        ])
    ]
    .groupby(
        [
            "CanonicalProductID",
            "WindowID",
            "SplitContextID"
        ],
        as_index=False
    )
    .agg(
        EvaluationRows=(
            "Date",
            "size"
        ),
        EvaluationStartSequence=(
            "OperatingDaySequence",
            "min"
        ),
        EvaluationEndSequence=(
            "OperatingDaySequence",
            "max"
        ),
        EvaluationStartDate=(
            "Date",
            "min"
        ),
        EvaluationEndDate=(
            "Date",
            "max"
        ),
        EvaluationRoleCount=(
            "SplitRole",
            "nunique"
        ),
        EvaluationRole=(
            "SplitRole",
            "first"
        )
    )
)

step3_part4_context_integrity_audit_df = (
    independent_training_boundaries_df
    .merge(
        independent_evaluation_boundaries_df,
        on=[
            "CanonicalProductID",
            "WindowID",
            "SplitContextID"
        ],
        how="outer",
        validate="one_to_one"
    )
)

step3_part4_context_integrity_audit_df[
    "ChronologicalOverlapViolation"
] = (
    step3_part4_context_integrity_audit_df[
        "TrainingEndSequence"
    ]
    >= step3_part4_context_integrity_audit_df[
        "EvaluationStartSequence"
    ]
)

step3_part4_context_integrity_audit_df[
    "MissingTrainingRole"
] = (
    step3_part4_context_integrity_audit_df[
        "TrainingRows"
    ].isna()
)

step3_part4_context_integrity_audit_df[
    "MissingEvaluationRole"
] = (
    step3_part4_context_integrity_audit_df[
        "EvaluationRows"
    ].isna()
)

step3_part4_context_integrity_audit_df[
    "MultipleEvaluationRoles"
] = (
    step3_part4_context_integrity_audit_df[
        "EvaluationRoleCount"
    ] != 1
)

context_overlap_violations = int(
    step3_part4_context_integrity_audit_df[
        "ChronologicalOverlapViolation"
    ].sum()
)

missing_training_contexts = int(
    step3_part4_context_integrity_audit_df[
        "MissingTrainingRole"
    ].sum()
)

missing_evaluation_contexts = int(
    step3_part4_context_integrity_audit_df[
        "MissingEvaluationRole"
    ].sum()
)

multiple_evaluation_role_contexts = int(
    step3_part4_context_integrity_audit_df[
        "MultipleEvaluationRoles"
    ].sum()
)

assert len(
    step3_part4_context_integrity_audit_df
) == 472

assert context_overlap_violations == 0
assert missing_training_contexts == 0
assert missing_evaluation_contexts == 0
assert multiple_evaluation_role_contexts == 0


# ------------------------------------------------------------
# 8. Audit model-selection and final-test isolation
# ------------------------------------------------------------

model_selection_context_ids = set(
    step3_part4_model_selection_manifest_df[
        "SplitContextID"
    ]
)

final_test_context_ids = set(
    step3_part4_final_test_manifest_df[
        "SplitContextID"
    ]
)

context_overlap_between_model_selection_and_test = len(
    model_selection_context_ids.intersection(
        final_test_context_ids
    )
)

model_selection_rows_with_final_test_role = int(
    (
        step3_part4_model_selection_manifest_df[
            "SplitRole"
        ] == "FINAL_TEST"
    ).sum()
)

final_test_rows_with_validation_role = int(
    (
        step3_part4_final_test_manifest_df[
            "SplitRole"
        ] == "VALIDATION"
    ).sum()
)

validation_rows_marked_reserved_final_test = int(
    (
        step3_part4_complete_manifest_df[
            "SplitRole"
        ].eq("VALIDATION")
        &
        step3_part4_complete_manifest_df[
            "ReservedFinalTest"
        ]
    ).sum()
)

validation_rows_not_marked_for_model_selection = int(
    (
        step3_part4_complete_manifest_df[
            "SplitRole"
        ].eq("VALIDATION")
        &
        (
            ~step3_part4_complete_manifest_df[
                "MayBeUsedForModelSelection"
            ]
        )
    ).sum()
)

final_test_rows_marked_for_model_selection = int(
    (
        step3_part4_complete_manifest_df[
            "SplitRole"
        ].eq("FINAL_TEST")
        &
        step3_part4_complete_manifest_df[
            "MayBeUsedForModelSelection"
        ]
    ).sum()
)

final_test_rows_not_reserved = int(
    (
        step3_part4_complete_manifest_df[
            "SplitRole"
        ].eq("FINAL_TEST")
        &
        (
            ~step3_part4_complete_manifest_df[
                "ReservedFinalTest"
            ]
        )
    ).sum()
)

training_rows_eligible_for_scoring = int(
    (
        step3_part4_complete_manifest_df[
            "SplitRole"
        ].isin([
            "TRAIN",
            "FORECAST_ONLY_TRAINING"
        ])
        &
        step3_part4_complete_manifest_df[
            "RowEligibleForScoring"
        ]
    ).sum()
)

scoring_rows_not_in_evaluation_roles = int(
    (
        step3_part4_complete_manifest_df[
            "RowEligibleForScoring"
        ]
        &
        (
            ~step3_part4_complete_manifest_df[
                "SplitRole"
            ].isin([
                "VALIDATION",
                "FINAL_TEST"
            ])
        )
    ).sum()
)


step3_part4_final_test_isolation_audit_df = pd.DataFrame([
    {
        "IsolationCheck":
            "MODEL_SELECTION_CONTEXT_COUNT",
        "Value":
            len(model_selection_context_ids),
        "ExpectedValue":
            254,
        "ViolationCount":
            int(
                len(model_selection_context_ids)
                != 254
            )
    },
    {
        "IsolationCheck":
            "FINAL_TEST_CONTEXT_COUNT",
        "Value":
            len(final_test_context_ids),
        "ExpectedValue":
            218,
        "ViolationCount":
            int(
                len(final_test_context_ids)
                != 218
            )
    },
    {
        "IsolationCheck":
            "CONTEXT_OVERLAP_BETWEEN_MODEL_SELECTION_AND_FINAL_TEST",
        "Value":
            context_overlap_between_model_selection_and_test,
        "ExpectedValue":
            0,
        "ViolationCount":
            context_overlap_between_model_selection_and_test
    },
    {
        "IsolationCheck":
            "MODEL_SELECTION_ROWS_WITH_FINAL_TEST_ROLE",
        "Value":
            model_selection_rows_with_final_test_role,
        "ExpectedValue":
            0,
        "ViolationCount":
            model_selection_rows_with_final_test_role
    },
    {
        "IsolationCheck":
            "FINAL_TEST_ROWS_WITH_VALIDATION_ROLE",
        "Value":
            final_test_rows_with_validation_role,
        "ExpectedValue":
            0,
        "ViolationCount":
            final_test_rows_with_validation_role
    },
    {
        "IsolationCheck":
            "VALIDATION_ROWS_MARKED_RESERVED_FINAL_TEST",
        "Value":
            validation_rows_marked_reserved_final_test,
        "ExpectedValue":
            0,
        "ViolationCount":
            validation_rows_marked_reserved_final_test
    },
    {
        "IsolationCheck":
            "VALIDATION_ROWS_NOT_MARKED_FOR_MODEL_SELECTION",
        "Value":
            validation_rows_not_marked_for_model_selection,
        "ExpectedValue":
            0,
        "ViolationCount":
            validation_rows_not_marked_for_model_selection
    },
    {
        "IsolationCheck":
            "FINAL_TEST_ROWS_MARKED_FOR_MODEL_SELECTION",
        "Value":
            final_test_rows_marked_for_model_selection,
        "ExpectedValue":
            0,
        "ViolationCount":
            final_test_rows_marked_for_model_selection
    },
    {
        "IsolationCheck":
            "FINAL_TEST_ROWS_NOT_RESERVED",
        "Value":
            final_test_rows_not_reserved,
        "ExpectedValue":
            0,
        "ViolationCount":
            final_test_rows_not_reserved
    },
    {
        "IsolationCheck":
            "TRAINING_ROWS_ELIGIBLE_FOR_SCORING",
        "Value":
            training_rows_eligible_for_scoring,
        "ExpectedValue":
            0,
        "ViolationCount":
            training_rows_eligible_for_scoring
    },
    {
        "IsolationCheck":
            "SCORING_ROWS_OUTSIDE_EVALUATION_ROLES",
        "Value":
            scoring_rows_not_in_evaluation_roles,
        "ExpectedValue":
            0,
        "ViolationCount":
            scoring_rows_not_in_evaluation_roles
    }
])

final_test_isolation_violations = int(
    step3_part4_final_test_isolation_audit_df[
        "ViolationCount"
    ].sum()
)

assert len(model_selection_context_ids) == 254
assert len(final_test_context_ids) == 218
assert final_test_isolation_violations == 0


# ------------------------------------------------------------
# 9. Recreate all-zero evaluation-window audit
# ------------------------------------------------------------

step3_part4_zero_demand_window_audit_df = (
    step3_part4_scoring_rows_df
    .assign(
        PositiveDemandFlag=(
            step3_part4_scoring_rows_df[
                "TotalDemand"
            ] > 0
        ).astype(int)
    )
    .groupby(
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "EvaluationCohort",
            "DemandPatternClass",
            "WindowID",
            "WindowPurpose",
            "SplitContextID",
            "SplitRole"
        ],
        as_index=False
    )
    .agg(
        EvaluationRows=(
            "Date",
            "size"
        ),
        PositiveDemandRows=(
            "PositiveDemandFlag",
            "sum"
        ),
        ZeroDemandRows=(
            "TotalDemand",
            lambda series:
            int(
                series.eq(0).sum()
            )
        ),
        EvaluationDemandUnits=(
            "TotalDemand",
            "sum"
        ),
        EvaluationStartDate=(
            "Date",
            "min"
        ),
        EvaluationEndDate=(
            "Date",
            "max"
        )
    )
)

step3_part4_zero_demand_window_audit_df[
    "AllZeroDemandWindow"
] = (
    step3_part4_zero_demand_window_audit_df[
        "EvaluationDemandUnits"
    ] == 0
)

step3_part4_zero_demand_window_audit_df = (
    step3_part4_zero_demand_window_audit_df.loc[
        step3_part4_zero_demand_window_audit_df[
            "AllZeroDemandWindow"
        ]
    ]
    .copy()
    .sort_values(
        [
            "WindowID",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

step3_part4_zero_demand_window_audit_df[
    "RemovedFromEvaluation"
] = False

step3_part4_zero_demand_window_audit_df[
    "AuditInterpretation"
] = (
    "VALID_INTERMITTENT_DEMAND_EVALUATION_WINDOW"
)

assert len(
    step3_part4_zero_demand_window_audit_df
) == 344


# ------------------------------------------------------------
# 10. Compare against the previous zero-demand audit
# ------------------------------------------------------------

previous_zero_context_ids = set(
    step3_part4_previous_zero_audit_df[
        "CanonicalProductID"
    ].astype(str)
    + "::"
    + step3_part4_previous_zero_audit_df[
        "WindowID"
    ].astype(str)
)

current_zero_context_ids = set(
    step3_part4_zero_demand_window_audit_df[
        "SplitContextID"
    ].astype(str)
)

zero_demand_audit_context_mismatches = len(
    previous_zero_context_ids.symmetric_difference(
        current_zero_context_ids
    )
)

assert zero_demand_audit_context_mismatches == 0

assert not step3_part4_zero_demand_window_audit_df[
    "RemovedFromEvaluation"
].any()


# ------------------------------------------------------------
# 11. Create final Part 4 validation summary
# ------------------------------------------------------------

step3_part4_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "CompleteManifestRows",
        "IndependentEvaluationManifestRows",
        "ModelSelectionManifestRows",
        "FinalTestManifestRows",
        "ForecastOnlyTrainingRows",
        "TotalSplitContexts",
        "IndependentEvaluationContexts",
        "ForecastOnlyContexts",
        "ProductsRepresented",
        "ProductsWithIndependentEvaluation",
        "ForecastOnlyProducts",
        "UniqueSourceRowsRepresented",
        "UniqueSourceDemandUnitsRepresented",
        "ValidationScoringRows",
        "FinalTestScoringRows",
        "TotalScoringRows",
        "HistoricalFeaturesRequiredForScoring",
        "ScoringRowsMissingHistoricalFeatures",
        "ScoringHistoricalFeatureMissingCells",
        "ContextChronologicalOverlapViolations",
        "MissingTrainingContexts",
        "MissingEvaluationContexts",
        "MultipleEvaluationRoleContexts",
        "FinalTestIsolationViolations",
        "AllZeroDemandEvaluationWindows",
        "AllZeroEvaluationWindowsRemoved",
        "ZeroDemandAuditContextMismatches",
        "SameDayLeakageColumnsPresent"
    ],
    "Value": [
        len(
            step3_part4_complete_manifest_df
        ),
        len(
            step3_part4_independent_manifest_df
        ),
        len(
            step3_part4_model_selection_manifest_df
        ),
        len(
            step3_part4_final_test_manifest_df
        ),
        len(
            step3_part4_forecast_only_manifest_df
        ),
        step3_part4_complete_manifest_df[
            "SplitContextID"
        ].nunique(),
        step3_part4_independent_manifest_df[
            "SplitContextID"
        ].nunique(),
        step3_part4_forecast_only_manifest_df[
            "SplitContextID"
        ].nunique(),
        step3_part4_complete_manifest_df[
            "CanonicalProductID"
        ].nunique(),
        step3_part4_independent_manifest_df[
            "CanonicalProductID"
        ].nunique(),
        step3_part4_forecast_only_manifest_df[
            "CanonicalProductID"
        ].nunique(),
        len(
            step3_part4_unique_manifest_source_df
        ),
        int(
            step3_part4_unique_manifest_source_df[
                "TotalDemand"
            ].sum()
        ),
        validation_scoring_rows,
        final_test_scoring_rows,
        total_scoring_rows,
        len(
            step3_part4_historical_feature_columns
        ),
        scoring_rows_missing_any_historical_feature,
        scoring_missing_historical_cells,
        context_overlap_violations,
        missing_training_contexts,
        missing_evaluation_contexts,
        multiple_evaluation_role_contexts,
        final_test_isolation_violations,
        len(
            step3_part4_zero_demand_window_audit_df
        ),
        0,
        zero_demand_audit_context_mismatches,
        (
            len(
                part4_manifest_leakage_columns_found
            )
            + len(
                part4_source_leakage_columns_found
            )
        )
    ]
})


# ------------------------------------------------------------
# 12. Validate the Part 4 summary
# ------------------------------------------------------------

part4_summary_lookup = dict(
    zip(
        step3_part4_validation_summary_df[
            "ValidationMetric"
        ],
        step3_part4_validation_summary_df[
            "Value"
        ]
    )
)

assert part4_summary_lookup[
    "CompleteManifestRows"
] == 97_392

assert part4_summary_lookup[
    "IndependentEvaluationManifestRows"
] == 96_396

assert part4_summary_lookup[
    "ModelSelectionManifestRows"
] == 53_618

assert part4_summary_lookup[
    "FinalTestManifestRows"
] == 42_778

assert part4_summary_lookup[
    "ForecastOnlyTrainingRows"
] == 996

assert part4_summary_lookup[
    "TotalSplitContexts"
] == 481

assert part4_summary_lookup[
    "ProductsRepresented"
] == 227

assert part4_summary_lookup[
    "UniqueSourceRowsRepresented"
] == 43_774

assert part4_summary_lookup[
    "UniqueSourceDemandUnitsRepresented"
] == 116_158

assert part4_summary_lookup[
    "ValidationScoringRows"
] == 5_080

assert part4_summary_lookup[
    "FinalTestScoringRows"
] == 4_150

assert part4_summary_lookup[
    "TotalScoringRows"
] == 9_230

assert part4_summary_lookup[
    "HistoricalFeaturesRequiredForScoring"
] == 31

assert part4_summary_lookup[
    "ScoringRowsMissingHistoricalFeatures"
] == 0

assert part4_summary_lookup[
    "ScoringHistoricalFeatureMissingCells"
] == 0

assert part4_summary_lookup[
    "ContextChronologicalOverlapViolations"
] == 0

assert part4_summary_lookup[
    "FinalTestIsolationViolations"
] == 0

assert part4_summary_lookup[
    "AllZeroDemandEvaluationWindows"
] == 344

assert part4_summary_lookup[
    "AllZeroEvaluationWindowsRemoved"
] == 0

assert part4_summary_lookup[
    "ZeroDemandAuditContextMismatches"
] == 0

assert part4_summary_lookup[
    "SameDayLeakageColumnsPresent"
] == 0


# ------------------------------------------------------------
# 13. Print Cell 52 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 4 — "
    "FINAL SPLIT AND DEMAND-COVERAGE AUDIT PASSED"
)
print("=" * 75)

print()
print("Split-role demand summary:")
display(
    step3_part4_split_role_demand_summary_df
)

print()
print("Window-role demand summary:")
display(
    step3_part4_window_role_demand_summary_df
)

print()
print("Cohort-role demand summary:")
display(
    step3_part4_cohort_role_demand_summary_df
)

print()
print("Chronological integrity:")
print(
    "Independent contexts audited:",
    len(
        step3_part4_context_integrity_audit_df
    )
)
print(
    "Chronological overlap violations:",
    context_overlap_violations
)
print(
    "Missing training contexts:",
    missing_training_contexts
)
print(
    "Missing evaluation contexts:",
    missing_evaluation_contexts
)
print(
    "Multiple evaluation-role contexts:",
    multiple_evaluation_role_contexts
)

print()
print("Final-test isolation:")
display(
    step3_part4_final_test_isolation_audit_df
)

print()
print("Scoring feature readiness:")
print(
    "Validation scoring rows:",
    f"{validation_scoring_rows:,}"
)
print(
    "Final-test scoring rows:",
    f"{final_test_scoring_rows:,}"
)
print(
    "Scoring rows missing historical features:",
    scoring_rows_missing_any_historical_feature
)
print(
    "Missing historical-feature cells:",
    scoring_missing_historical_cells
)

print()
print("Intermittent-demand evaluation:")
print(
    "All-zero evaluation windows retained:",
    len(
        step3_part4_zero_demand_window_audit_df
    )
)
print(
    "All-zero evaluation windows removed:",
    0
)
print(
    "Zero-demand audit context mismatches:",
    zero_demand_audit_context_mismatches
)

print()
print("Cell 52 completed successfully.")

STEP 3, PART 4 — FINAL SPLIT AND DEMAND-COVERAGE AUDIT PASSED

Split-role demand summary:


,SplitRole,ManifestRows,ProductCount,SplitContextCount,PositiveDemandRows,ZeroDemandRows,ContextWeightedDemandUnits,RowsWithAllHistoricalFeatures,RowsWithMissingHistoricalFeatures,PositiveDemandRowPercentage,ZeroDemandRowPercentage
0,FINAL_TEST,4150,218,218,1290,2860,13797,4150,0,31.08,68.92
1,FORECAST_ONLY_TRAINING,996,9,9,36,960,65,816,180,3.61,96.39
2,TRAIN,87166,218,472,32663,54503,218359,77726,9440,37.47,62.53
3,VALIDATION,5080,127,254,758,4322,8117,5080,0,14.92,85.08



Window-role demand summary:


,WindowID,WindowPurpose,SplitRole,ManifestRows,ProductCount,SplitContextCount,PositiveDemandRows,ZeroDemandRows,ContextWeightedDemandUnits,RowsWithMissingHistoricalFeatures
0,FORECAST_ONLY_FULL_HISTORY,FORECAST_MODEL_FIT_ONLY,FORECAST_ONLY_TRAINING,996,9,9,36,960,65,180
1,LIMITED_FINAL_HOLDOUT,LIMITED_FINAL_HOLDOUT_TEST,FINAL_TEST,70,14,14,3,67,61,0
2,LIMITED_FINAL_HOLDOUT,LIMITED_FINAL_HOLDOUT_TEST,TRAIN,2104,14,14,61,2043,708,280
3,STANDARD_BACKTEST_FOLD_1,BACKTEST_VALIDATION,TRAIN,22999,127,127,9245,13754,56377,2540
4,STANDARD_BACKTEST_FOLD_1,BACKTEST_VALIDATION,VALIDATION,2540,127,127,361,2179,3309,0
5,STANDARD_BACKTEST_FOLD_2,BACKTEST_VALIDATION,TRAIN,25539,127,127,9606,15933,59686,2540
6,STANDARD_BACKTEST_FOLD_2,BACKTEST_VALIDATION,VALIDATION,2540,127,127,397,2143,4808,0
7,STANDARD_FINAL_HOLDOUT,FINAL_HOLDOUT_TEST,FINAL_TEST,4080,204,204,1287,2793,13736,0
8,STANDARD_FINAL_HOLDOUT,FINAL_HOLDOUT_TEST,TRAIN,36524,204,204,13751,22773,101588,4080



Cohort-role demand summary:


,EvaluationCohort,SplitRole,ManifestRows,ProductCount,SplitContextCount,PositiveDemandRows,ZeroDemandRows,ContextWeightedDemandUnits
0,LIMITED_HOLDOUT_ONLY,FINAL_TEST,70,14,14,3,67,61
1,LIMITED_HOLDOUT_ONLY,TRAIN,2104,14,14,61,2043,708
2,MINIMAL_EVIDENCE_FORECAST_ONLY,FORECAST_ONLY_TRAINING,996,9,9,36,960,65
3,MULTI_FOLD_BACKTEST_READY,FINAL_TEST,2540,127,127,408,2132,4762
4,MULTI_FOLD_BACKTEST_READY,TRAIN,76617,127,381,28854,47763,180557
5,MULTI_FOLD_BACKTEST_READY,VALIDATION,5080,127,254,758,4322,8117
6,SINGLE_HOLDOUT_BACKTEST_READY,FINAL_TEST,1540,77,77,879,661,8974
7,SINGLE_HOLDOUT_BACKTEST_READY,TRAIN,8445,77,77,3748,4697,37094



Chronological integrity:
Independent contexts audited: 472
Chronological overlap violations: 0
Missing training contexts: 0
Missing evaluation contexts: 0
Multiple evaluation-role contexts: 0

Final-test isolation:


,IsolationCheck,Value,ExpectedValue,ViolationCount
0,MODEL_SELECTION_CONTEXT_COUNT,254,254,0
1,FINAL_TEST_CONTEXT_COUNT,218,218,0
2,CONTEXT_OVERLAP_BETWEEN_MODEL_SELECTION_AND_FI...,0,0,0
3,MODEL_SELECTION_ROWS_WITH_FINAL_TEST_ROLE,0,0,0
4,FINAL_TEST_ROWS_WITH_VALIDATION_ROLE,0,0,0
5,VALIDATION_ROWS_MARKED_RESERVED_FINAL_TEST,0,0,0
6,VALIDATION_ROWS_NOT_MARKED_FOR_MODEL_SELECTION,0,0,0
7,FINAL_TEST_ROWS_MARKED_FOR_MODEL_SELECTION,0,0,0
8,FINAL_TEST_ROWS_NOT_RESERVED,0,0,0
9,TRAINING_ROWS_ELIGIBLE_FOR_SCORING,0,0,0



Scoring feature readiness:
Validation scoring rows: 5,080
Final-test scoring rows: 4,150
Scoring rows missing historical features: 0
Missing historical-feature cells: 0

Intermittent-demand evaluation:
All-zero evaluation windows retained: 344
All-zero evaluation windows removed: 0
Zero-demand audit context mismatches: 0

Cell 52 completed successfully.


In [64]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 4
# Cell 53: Save final split-audit outputs and update handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 4 objects exist
# ------------------------------------------------------------

required_step3_part4_objects = [
    "step3_part4_split_role_demand_summary_df",
    "step3_part4_window_role_demand_summary_df",
    "step3_part4_cohort_role_demand_summary_df",
    "step3_part4_product_role_coverage_df",
    "step3_part4_context_integrity_audit_df",
    "step3_part4_final_test_isolation_audit_df",
    "step3_part4_evaluation_feature_readiness_audit_df",
    "step3_part4_zero_demand_window_audit_df",
    "step3_part4_validation_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step3_part4_objects = [
    object_name
    for object_name in required_step3_part4_objects
    if object_name not in globals()
]

if missing_step3_part4_objects:
    raise NameError(
        "The following Step 3 Part 4 objects are missing:\n"
        f"{missing_step3_part4_objects}\n\n"
        "Run Cells 51 and 52 before running Cell 53."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff helper if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Step 3 Part 4 output paths
# ------------------------------------------------------------

STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_split_role_demand_summary.csv"
)

STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_window_role_demand_summary.csv"
)

STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_cohort_role_demand_summary.csv"
)

STEP3_PART4_PRODUCT_COVERAGE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_product_role_coverage_audit.csv"
)

STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_context_integrity_audit.csv"
)

STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_final_test_isolation_audit.csv"
)

STEP3_PART4_FEATURE_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_evaluation_feature_readiness_audit.csv"
)

STEP3_PART4_ZERO_DEMAND_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_zero_demand_evaluation_window_audit.csv"
)

STEP3_PART4_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_part4_validation_summary.csv"
)


# ------------------------------------------------------------
# 4. Save all Part 4 outputs
# ------------------------------------------------------------

step3_part4_split_role_demand_summary_df.to_csv(
    STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_part4_window_role_demand_summary_df.to_csv(
    STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_part4_cohort_role_demand_summary_df.to_csv(
    STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_part4_product_role_coverage_df.to_csv(
    STEP3_PART4_PRODUCT_COVERAGE_OUTPUT,
    index=False
)

step3_part4_context_integrity_audit_df.to_csv(
    STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part4_final_test_isolation_audit_df.to_csv(
    STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT,
    index=False
)

step3_part4_evaluation_feature_readiness_audit_df.to_csv(
    STEP3_PART4_FEATURE_READINESS_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part4_zero_demand_window_audit_df.to_csv(
    STEP3_PART4_ZERO_DEMAND_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part4_validation_summary_df.to_csv(
    STEP3_PART4_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 5. Reload saved outputs
# ------------------------------------------------------------

saved_part4_split_role_summary = pd.read_csv(
    STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_part4_window_role_summary = pd.read_csv(
    STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_part4_cohort_role_summary = pd.read_csv(
    STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_part4_product_coverage = pd.read_csv(
    STEP3_PART4_PRODUCT_COVERAGE_OUTPUT,
    low_memory=False
)

saved_part4_context_integrity = pd.read_csv(
    STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT,
    low_memory=False
)

saved_part4_final_test_isolation = pd.read_csv(
    STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT,
    low_memory=False
)

saved_part4_feature_readiness = pd.read_csv(
    STEP3_PART4_FEATURE_READINESS_OUTPUT,
    low_memory=False
)

saved_part4_zero_demand = pd.read_csv(
    STEP3_PART4_ZERO_DEMAND_OUTPUT,
    low_memory=False
)

saved_part4_validation = pd.read_csv(
    STEP3_PART4_VALIDATION_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 6. Validate saved outputs
# ------------------------------------------------------------

assert saved_part4_split_role_summary[
    "ManifestRows"
].sum() == 97_392

assert saved_part4_product_coverage[
    "CanonicalProductID"
].nunique() == 227

assert len(
    saved_part4_context_integrity
) == 472

assert saved_part4_context_integrity[
    "ChronologicalOverlapViolation"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_part4_context_integrity[
    "MissingTrainingRole"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_part4_context_integrity[
    "MissingEvaluationRole"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_part4_final_test_isolation[
    "ViolationCount"
].sum() == 0

assert len(
    saved_part4_feature_readiness
) == 0

assert len(
    saved_part4_zero_demand
) == 344

assert saved_part4_zero_demand[
    "RemovedFromEvaluation"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert len(
    saved_part4_validation
) == 28


# ------------------------------------------------------------
# 7. Build Markdown demand summary
# ------------------------------------------------------------

split_role_markdown_rows = "\n".join(
    (
        f"- `{row.SplitRole}`: "
        f"{int(row.ManifestRows):,} rows, "
        f"{int(row.ProductCount)} products, "
        f"{int(row.PositiveDemandRows):,} positive-demand rows, "
        f"{int(row.ZeroDemandRows):,} zero-demand rows"
    )
    for row in (
        step3_part4_split_role_demand_summary_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 8. Update the Markdown handoff
# ------------------------------------------------------------

step3_part4_summary = f"""
**Status:** Completed and validated

### Purpose

Step 3 Part 4 performed the final quality-control audit of the
chronological split system before creation of the frozen modelling
datasets.

### Manifest structure

- Complete manifest rows: {len(step3_part4_complete_manifest_df):,}
- Independent-evaluation manifest rows: {len(step3_part4_independent_manifest_df):,}
- Model-selection manifest rows: {len(step3_part4_model_selection_manifest_df):,}
- Final-test manifest rows: {len(step3_part4_final_test_manifest_df):,}
- Forecast-only training rows: {len(step3_part4_forecast_only_manifest_df):,}
- Total split contexts: 481
- Products represented: 227

### Split-role demand coverage

{split_role_markdown_rows}

The demand-unit totals in split-role summaries are context-weighted
because source observations are intentionally reused across
expanding-window contexts.

### Unique source representation

- Unique source rows represented: 43,774
- Unique source demand units represented: 116,158
- Missing source rows: 0

### Chronological integrity

- Independent contexts audited: 472
- Chronological overlap violations: 0
- Missing training contexts: 0
- Missing evaluation contexts: 0
- Contexts with multiple evaluation roles: 0

Every independent evaluation context has a training period followed
strictly by one validation or final-test period.

### Final-test isolation

- Model-selection contexts: 254
- Reserved final-test contexts: 218
- Context overlap between model selection and final test: 0
- Final-test rows marked for model selection: 0
- Validation rows marked as reserved final test: 0
- Training rows eligible for scoring: 0
- Total final-test isolation violations: 0

The reserved final-test contexts must not be used for feature
selection, model selection or hyperparameter tuning.

### Scoring feature readiness

- Validation scoring rows: {validation_scoring_rows:,}
- Final-test scoring rows: {final_test_scoring_rows:,}
- Total scoring rows: {total_scoring_rows:,}
- Historical features required: 31
- Scoring rows missing historical features: 0
- Missing historical-feature cells: 0

### Intermittent-demand evaluation

- All-zero-demand evaluation windows retained: 344
- All-zero-demand evaluation windows removed: 0
- Zero-demand audit context mismatches: 0

All-zero evaluation windows remain valid and must be included when
evaluating intermittent-demand performance.

### Saved Step 3 Part 4 outputs

- `16_step3_split_role_demand_summary.csv`
- `16_step3_window_role_demand_summary.csv`
- `16_step3_cohort_role_demand_summary.csv`
- `16_step3_product_role_coverage_audit.csv`
- `16_step3_context_integrity_audit.csv`
- `16_step3_final_test_isolation_audit.csv`
- `16_step3_evaluation_feature_readiness_audit.csv`
- `16_step3_zero_demand_evaluation_window_audit.csv`
- `16_step3_part4_validation_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_3_part_4",
    section_title=(
        "Forecasting Preparation — Step 3, Part 4"
    ),
    section_body=step3_part4_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 9. Print final Part 4 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 3, PART 4 COMPLETED"
)
print("=" * 75)

print()
print("Manifest and product coverage:")
print(
    "Complete manifest rows:",
    f"{len(step3_part4_complete_manifest_df):,}"
)
print(
    "Total split contexts:",
    step3_part4_complete_manifest_df[
        "SplitContextID"
    ].nunique()
)
print(
    "Products represented:",
    step3_part4_complete_manifest_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Unique source rows represented:",
    f"{len(step3_part4_unique_manifest_source_df):,}"
)
print(
    "Unique source demand units:",
    f"{step3_part4_unique_manifest_source_df['TotalDemand'].sum():,.0f}"
)

print()
print("Chronological integrity:")
print(
    "Contexts audited:",
    len(
        step3_part4_context_integrity_audit_df
    )
)
print(
    "Chronological overlap violations:",
    context_overlap_violations
)
print(
    "Missing training contexts:",
    missing_training_contexts
)
print(
    "Missing evaluation contexts:",
    missing_evaluation_contexts
)

print()
print("Final-test isolation:")
print(
    "Model-selection contexts:",
    len(model_selection_context_ids)
)
print(
    "Final-test contexts:",
    len(final_test_context_ids)
)
print(
    "Final-test isolation violations:",
    final_test_isolation_violations
)

print()
print("Scoring readiness:")
print(
    "Validation scoring rows:",
    f"{validation_scoring_rows:,}"
)
print(
    "Final-test scoring rows:",
    f"{final_test_scoring_rows:,}"
)
print(
    "Scoring rows missing historical features:",
    scoring_rows_missing_any_historical_feature
)
print(
    "Historical-feature missing cells:",
    scoring_missing_historical_cells
)

print()
print("Intermittent-demand evaluation:")
print(
    "All-zero evaluation windows retained:",
    len(saved_part4_zero_demand)
)
print(
    "All-zero evaluation windows removed:",
    0
)

print()
print("Saved files:")
print(f"1. {STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT}")
print(f"2. {STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT}")
print(f"3. {STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT}")
print(f"4. {STEP3_PART4_PRODUCT_COVERAGE_OUTPUT}")
print(f"5. {STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT}")
print(f"6. {STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT}")
print(f"7. {STEP3_PART4_FEATURE_READINESS_OUTPUT}")
print(f"8. {STEP3_PART4_ZERO_DEMAND_OUTPUT}")
print(f"9. {STEP3_PART4_VALIDATION_OUTPUT}")
print(f"10. {HANDOFF_FILE}")

print()
print(
    "All Step 3, Part 4 validation checks passed."
)

FORECASTING PREPARATION — STEP 3, PART 4 COMPLETED

Manifest and product coverage:
Complete manifest rows: 97,392
Total split contexts: 481
Products represented: 227
Unique source rows represented: 43,774
Unique source demand units: 116,158

Chronological integrity:
Contexts audited: 472
Chronological overlap violations: 0
Missing training contexts: 0
Missing evaluation contexts: 0

Final-test isolation:
Model-selection contexts: 254
Final-test contexts: 218
Final-test isolation violations: 0

Scoring readiness:
Validation scoring rows: 5,080
Final-test scoring rows: 4,150
Scoring rows missing historical features: 0
Historical-feature missing cells: 0

Intermittent-demand evaluation:
All-zero evaluation windows retained: 344
All-zero evaluation windows removed: 0

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/16_step3_split_role_demand_summary.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/16_step3_window_role_de

In [69]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 5
# Cell 54: Create the final frozen model-ready datasets
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Confirm the forecasting-preparation directory exists
# ------------------------------------------------------------

if "FORECAST_PREPARATION_DIR" not in globals():
    raise NameError(
        "FORECAST_PREPARATION_DIR is missing. "
        "Run the previous forecasting-preparation cells first."
    )

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )

EDEN_DATASETS_DIR = FORECAST_PREPARATION_DIR.parent

assert EDEN_DATASETS_DIR.exists()


# ------------------------------------------------------------
# 2. Define the official Part 5 input files
# ------------------------------------------------------------

STEP3_PART5_SOURCE_FEATURE_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_historical_feature_model_view.csv"
)

STEP3_PART5_MODEL_SELECTION_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_model_selection_split_manifest.csv"
)

STEP3_PART5_FINAL_TEST_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_final_test_split_manifest.csv"
)

STEP3_PART5_FORECAST_ONLY_MANIFEST_FILE = (
    FORECAST_PREPARATION_DIR
    / "15_step3_forecast_only_training_manifest.csv"
)

STEP3_PART5_FEATURE_CONTRACT_FILE = (
    FORECAST_PREPARATION_DIR
    / "12_step2_combined_feature_contract.csv"
)

STEP3_PART5_PART4_VALIDATION_FILE = (
    FORECAST_PREPARATION_DIR
    / "16_step3_part4_validation_summary.csv"
)

required_part5_input_files = [
    STEP3_PART5_SOURCE_FEATURE_FILE,
    STEP3_PART5_MODEL_SELECTION_MANIFEST_FILE,
    STEP3_PART5_FINAL_TEST_MANIFEST_FILE,
    STEP3_PART5_FORECAST_ONLY_MANIFEST_FILE,
    STEP3_PART5_FEATURE_CONTRACT_FILE,
    STEP3_PART5_PART4_VALIDATION_FILE
]

missing_part5_input_files = [
    file_path
    for file_path in required_part5_input_files
    if not file_path.exists()
]

if missing_part5_input_files:
    raise FileNotFoundError(
        "The following required Step 3 Part 5 files are missing:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_part5_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load the official inputs
# ------------------------------------------------------------

step3_part5_source_feature_df = pd.read_csv(
    STEP3_PART5_SOURCE_FEATURE_FILE,
    low_memory=False
)

step3_part5_model_selection_manifest_df = pd.read_csv(
    STEP3_PART5_MODEL_SELECTION_MANIFEST_FILE,
    low_memory=False
)

step3_part5_final_test_manifest_df = pd.read_csv(
    STEP3_PART5_FINAL_TEST_MANIFEST_FILE,
    low_memory=False
)

step3_part5_forecast_only_manifest_df = pd.read_csv(
    STEP3_PART5_FORECAST_ONLY_MANIFEST_FILE,
    low_memory=False
)

step3_part5_feature_contract_df = pd.read_csv(
    STEP3_PART5_FEATURE_CONTRACT_FILE,
    low_memory=False
)

step3_part5_part4_validation_df = pd.read_csv(
    STEP3_PART5_PART4_VALIDATION_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# 4. Boolean parsing helper
# ------------------------------------------------------------

def parse_step3_part5_boolean(series):
    """
    Convert common CSV Boolean representations into bool values.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalised_values = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    converted_values = normalised_values.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if converted_values.isna().sum() > 0:

        invalid_values = (
            series.loc[
                converted_values.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unrecognised Boolean values were found:\n"
            f"{invalid_values}"
        )

    return converted_values.astype(bool)


# ------------------------------------------------------------
# 5. Parse the source feature dataset
# ------------------------------------------------------------

for column in [
    "Date",
    "ProductFirstObservedDate"
]:

    step3_part5_source_feature_df[column] = pd.to_datetime(
        step3_part5_source_feature_df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

assert step3_part5_source_feature_df[
    [
        "Date",
        "ProductFirstObservedDate"
    ]
].isna().sum().sum() == 0


for column in [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand"
]:

    step3_part5_source_feature_df[column] = pd.to_numeric(
        step3_part5_source_feature_df[column],
        errors="coerce"
    )

assert step3_part5_source_feature_df[
    [
        "OperatingDaySequence",
        "ProductAgeOperatingDays",
        "TotalDemand"
    ]
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 6. Parse the row-level manifests
# ------------------------------------------------------------

part5_manifest_dataframes = [
    step3_part5_model_selection_manifest_df,
    step3_part5_final_test_manifest_df,
    step3_part5_forecast_only_manifest_df
]

manifest_boolean_columns = [
    "AllHistoricalFeaturesAvailable",
    "MayBeUsedForModelSelection",
    "ReservedFinalTest",
    "IndependentEvaluationAvailable",
    "RowEligibleForScoring"
]

manifest_numeric_columns = [
    "OperatingDaySequence",
    "ProductAgeOperatingDays",
    "TotalDemand",
    "HistoricalFeatureAvailableCount",
    "WindowOrder"
]

for dataframe in part5_manifest_dataframes:

    dataframe["Date"] = pd.to_datetime(
        dataframe["Date"],
        format="%Y-%m-%d",
        errors="coerce"
    )

    assert dataframe["Date"].isna().sum() == 0

    dataframe[
        "CanonicalProductID"
    ] = (
        dataframe[
            "CanonicalProductID"
        ]
        .astype("string")
        .str.strip()
    )

    for column in manifest_numeric_columns:

        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce"
        )

    assert dataframe[
        manifest_numeric_columns
    ].isna().sum().sum() == 0

    for column in manifest_boolean_columns:

        dataframe[column] = parse_step3_part5_boolean(
            dataframe[column]
        )


step3_part5_source_feature_df[
    "CanonicalProductID"
] = (
    step3_part5_source_feature_df[
        "CanonicalProductID"
    ]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# 7. Validate the completed Part 4 inputs
# ------------------------------------------------------------

assert step3_part5_source_feature_df.shape == (
    43_774,
    58
)

assert len(
    step3_part5_model_selection_manifest_df
) == 53_618

assert len(
    step3_part5_final_test_manifest_df
) == 42_778

assert len(
    step3_part5_forecast_only_manifest_df
) == 996

assert step3_part5_source_feature_df[
    "CanonicalProductID"
].nunique() == 227

assert step3_part5_source_feature_df[
    "Date"
].nunique() == 245

assert step3_part5_source_feature_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert np.isclose(
    step3_part5_source_feature_df[
        "TotalDemand"
    ].sum(),
    116_158
)


# ------------------------------------------------------------
# 8. Freeze the feature roles
# ------------------------------------------------------------

required_contract_boolean_columns = [
    "DirectModelInputAllowed",
    "IsHistoricalDemandFeature",
    "IsIdentifier",
    "IsSupportColumn",
    "IsForecastTarget",
    "UsesCurrentRowTarget",
    "UsesFutureTarget"
]

for column in required_contract_boolean_columns:

    step3_part5_feature_contract_df[column] = (
        parse_step3_part5_boolean(
            step3_part5_feature_contract_df[column]
        )
    )


direct_predictor_columns = (
    step3_part5_feature_contract_df.loc[
        step3_part5_feature_contract_df[
            "DirectModelInputAllowed"
        ],
        "Column"
    ]
    .tolist()
)

historical_feature_columns = (
    step3_part5_feature_contract_df.loc[
        step3_part5_feature_contract_df[
            "IsHistoricalDemandFeature"
        ],
        "Column"
    ]
    .tolist()
)

identifier_columns = (
    step3_part5_feature_contract_df.loc[
        step3_part5_feature_contract_df[
            "IsIdentifier"
        ],
        "Column"
    ]
    .tolist()
)

support_columns = (
    step3_part5_feature_contract_df.loc[
        step3_part5_feature_contract_df[
            "IsSupportColumn"
        ],
        "Column"
    ]
    .tolist()
)

target_columns = (
    step3_part5_feature_contract_df.loc[
        step3_part5_feature_contract_df[
            "IsForecastTarget"
        ],
        "Column"
    ]
    .tolist()
)

assert len(direct_predictor_columns) == 53
assert len(historical_feature_columns) == 31
assert len(identifier_columns) == 2
assert len(support_columns) == 2
assert target_columns == ["TotalDemand"]

TARGET_COLUMN = "TotalDemand"

current_date_predictor_columns = [
    column
    for column in direct_predictor_columns
    if column not in historical_feature_columns
]

assert len(current_date_predictor_columns) == 22
assert TARGET_COLUMN not in direct_predictor_columns

assert all(
    column in step3_part5_source_feature_df.columns
    for column in direct_predictor_columns
)

assert all(
    column in step3_part5_source_feature_df.columns
    for column in historical_feature_columns
)


# ------------------------------------------------------------
# 9. Confirm historical features remain leakage safe
# ------------------------------------------------------------

historical_contract_rows = (
    step3_part5_feature_contract_df.loc[
        step3_part5_feature_contract_df[
            "IsHistoricalDemandFeature"
        ]
    ]
)

assert not historical_contract_rows[
    "UsesCurrentRowTarget"
].any()

assert not historical_contract_rows[
    "UsesFutureTarget"
].any()


# ------------------------------------------------------------
# 10. Define row-join keys
# ------------------------------------------------------------

ROW_JOIN_KEYS = [
    "Date",
    "CanonicalProductID",
    "OperatingDaySequence"
]


# ------------------------------------------------------------
# 11. Validate common manifest fields against the source
# ------------------------------------------------------------

def validate_manifest_against_source(
    manifest_df,
    dataset_name
):
    """
    Confirm the target and identity fields in a manifest agree
    with the canonical 58-column feature source.
    """

    shared_validation_columns = [
        "CanonicalProductName",
        "ProductAgeOperatingDays",
        "TotalDemand"
    ]

    comparison_df = (
        manifest_df[
            ROW_JOIN_KEYS
            + shared_validation_columns
        ]
        .merge(
            step3_part5_source_feature_df[
                ROW_JOIN_KEYS
                + shared_validation_columns
            ],
            on=ROW_JOIN_KEYS,
            how="left",
            validate="many_to_one",
            suffixes=(
                "_Manifest",
                "_Source"
            )
        )
    )

    missing_source_matches = int(
        comparison_df[
            "CanonicalProductName_Source"
        ].isna().sum()
    )

    name_mismatches = int(
        (
            comparison_df[
                "CanonicalProductName_Manifest"
            ].astype(str)
            != comparison_df[
                "CanonicalProductName_Source"
            ].astype(str)
        ).sum()
    )

    product_age_mismatches = int(
        (
            comparison_df[
                "ProductAgeOperatingDays_Manifest"
            ]
            != comparison_df[
                "ProductAgeOperatingDays_Source"
            ]
        ).sum()
    )

    target_mismatches = int(
        (
            ~np.isclose(
                comparison_df[
                    "TotalDemand_Manifest"
                ],
                comparison_df[
                    "TotalDemand_Source"
                ],
                rtol=1e-12,
                atol=1e-12
            )
        ).sum()
    )

    assert missing_source_matches == 0, (
        f"{dataset_name} contains rows that do not match "
        "the canonical feature source."
    )

    assert name_mismatches == 0
    assert product_age_mismatches == 0
    assert target_mismatches == 0


validate_manifest_against_source(
    step3_part5_model_selection_manifest_df,
    "MODEL_SELECTION_MANIFEST"
)

validate_manifest_against_source(
    step3_part5_final_test_manifest_df,
    "FINAL_TEST_MANIFEST"
)

validate_manifest_against_source(
    step3_part5_forecast_only_manifest_df,
    "FORECAST_ONLY_MANIFEST"
)


# ------------------------------------------------------------
# 12. Attach all 58 source columns to each manifest row
# ------------------------------------------------------------

def attach_source_features(
    manifest_df
):
    """
    Add the complete canonical feature row to every split-context
    row without duplicating columns already held in the source.
    """

    manifest_metadata_columns = [
        column
        for column in manifest_df.columns
        if (
            column not in
            step3_part5_source_feature_df.columns
            and column not in ROW_JOIN_KEYS
        )
    ]

    enriched_df = (
        manifest_df[
            ROW_JOIN_KEYS
            + manifest_metadata_columns
        ]
        .merge(
            step3_part5_source_feature_df,
            on=ROW_JOIN_KEYS,
            how="left",
            validate="many_to_one"
        )
    )

    ordered_columns = (
        ROW_JOIN_KEYS
        + manifest_metadata_columns
        + [
            column
            for column in
            step3_part5_source_feature_df.columns
            if column not in ROW_JOIN_KEYS
        ]
    )

    enriched_df = (
        enriched_df[
            ordered_columns
        ]
        .sort_values(
            [
                "WindowOrder",
                "WindowID",
                "CanonicalProductID",
                "OperatingDaySequence",
                "SplitRole"
            ],
            kind="stable"
        )
        .reset_index(drop=True)
    )

    return enriched_df


step3_model_selection_context_df = attach_source_features(
    step3_part5_model_selection_manifest_df
)

step3_reserved_final_test_context_df = attach_source_features(
    step3_part5_final_test_manifest_df
)

step3_forecast_only_context_df = attach_source_features(
    step3_part5_forecast_only_manifest_df
)


# ------------------------------------------------------------
# 13. Split model-selection training and validation rows
# ------------------------------------------------------------

step3_model_selection_training_df = (
    step3_model_selection_context_df.loc[
        step3_model_selection_context_df[
            "SplitRole"
        ] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)

step3_model_selection_validation_df = (
    step3_model_selection_context_df.loc[
        step3_model_selection_context_df[
            "SplitRole"
        ] == "VALIDATION"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Split reserved final-test training and scoring rows
# ------------------------------------------------------------

step3_reserved_final_test_training_df = (
    step3_reserved_final_test_context_df.loc[
        step3_reserved_final_test_context_df[
            "SplitRole"
        ] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)

step3_reserved_final_test_scoring_with_target_df = (
    step3_reserved_final_test_context_df.loc[
        step3_reserved_final_test_context_df[
            "SplitRole"
        ] == "FINAL_TEST"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. Physically separate final-test features and target
# ------------------------------------------------------------

step3_reserved_final_test_scoring_features_df = (
    step3_reserved_final_test_scoring_with_target_df
    .drop(
        columns=[
            TARGET_COLUMN
        ]
    )
    .copy()
)

final_test_target_vault_columns = [
    "SplitContextID",
    "WindowID",
    "WindowPurpose",
    "EvaluationCohort",
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "OperatingDaySequence",
    TARGET_COLUMN
]

step3_reserved_final_test_target_vault_df = (
    step3_reserved_final_test_scoring_with_target_df[
        final_test_target_vault_columns
    ]
    .copy()
)

assert TARGET_COLUMN not in (
    step3_reserved_final_test_scoring_features_df.columns
)

assert TARGET_COLUMN in (
    step3_reserved_final_test_target_vault_df.columns
)


# ------------------------------------------------------------
# 16. Prepare forecast-only post-evaluation training data
# ------------------------------------------------------------

step3_post_evaluation_forecast_only_training_df = (
    step3_forecast_only_context_df.loc[
        step3_forecast_only_context_df[
            "SplitRole"
        ]
        == "FORECAST_ONLY_TRAINING"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 17. Freeze the full-history final-fit dataset
# ------------------------------------------------------------

step3_post_evaluation_full_history_final_fit_df = (
    step3_part5_source_feature_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 18. Validate fixed row counts
# ------------------------------------------------------------

assert len(
    step3_model_selection_training_df
) == 48_538

assert len(
    step3_model_selection_validation_df
) == 5_080

assert len(
    step3_reserved_final_test_training_df
) == 38_628

assert len(
    step3_reserved_final_test_scoring_features_df
) == 4_150

assert len(
    step3_reserved_final_test_target_vault_df
) == 4_150

assert len(
    step3_post_evaluation_forecast_only_training_df
) == 996

assert len(
    step3_post_evaluation_full_history_final_fit_df
) == 43_774


# ------------------------------------------------------------
# 19. Validate fixed product and context counts
# ------------------------------------------------------------

assert step3_model_selection_training_df[
    "CanonicalProductID"
].nunique() == 127

assert step3_model_selection_validation_df[
    "CanonicalProductID"
].nunique() == 127

assert step3_model_selection_training_df[
    "SplitContextID"
].nunique() == 254

assert step3_model_selection_validation_df[
    "SplitContextID"
].nunique() == 254


assert step3_reserved_final_test_training_df[
    "CanonicalProductID"
].nunique() == 218

assert step3_reserved_final_test_scoring_features_df[
    "CanonicalProductID"
].nunique() == 218

assert step3_reserved_final_test_training_df[
    "SplitContextID"
].nunique() == 218

assert step3_reserved_final_test_scoring_features_df[
    "SplitContextID"
].nunique() == 218


assert step3_post_evaluation_forecast_only_training_df[
    "CanonicalProductID"
].nunique() == 9

assert step3_post_evaluation_forecast_only_training_df[
    "SplitContextID"
].nunique() == 9


# ------------------------------------------------------------
# 20. Validate scoring feature readiness
# ------------------------------------------------------------

assert step3_model_selection_validation_df[
    historical_feature_columns
].isna().sum().sum() == 0

assert step3_reserved_final_test_scoring_features_df[
    historical_feature_columns
].isna().sum().sum() == 0

assert step3_model_selection_validation_df[
    "AllHistoricalFeaturesAvailable"
].all()

assert step3_reserved_final_test_scoring_features_df[
    "AllHistoricalFeaturesAvailable"
].all()


# ------------------------------------------------------------
# 21. Freeze the feature contract
# ------------------------------------------------------------

step3_frozen_model_feature_contract_df = (
    step3_part5_feature_contract_df
    .copy()
    .sort_values(
        "ColumnOrder",
        kind="stable"
    )
    .reset_index(drop=True)
)

step3_frozen_model_feature_contract_df[
    "FrozenForModelTraining"
] = True

step3_frozen_model_feature_contract_df[
    "ForecastingPreparationVersion"
] = (
    "STEP_3_PART_5_FINAL"
)


# ------------------------------------------------------------
# 22. Freeze the operational forecasting protocol
# ------------------------------------------------------------

step3_forecasting_protocol_contract_df = pd.DataFrame({
    "ProtocolParameter": [
        "ForecastGranularity",
        "ForecastTarget",
        "OperationalForecastMode",
        "ForecastStepOperatingDays",
        "StandardEvaluationBlockOperatingDays",
        "LimitedEvaluationBlockOperatingDays",
        "PriorActualDemandUpdatePolicy",
        "RandomSplittingAllowed",
        "ValidationMayBeUsedForModelSelection",
        "FinalTestMayBeUsedForModelSelection",
        "FinalTestTargetStoredSeparately",
        "AllZeroEvaluationWindowsIncluded",
        "PredictorCount",
        "HistoricalPredictorCount",
        "CurrentDatePredictorCount",
        "MissingValueTreatmentPolicy",
        "CategoricalEncodingPolicy",
        "ForecastOnlyDatasetUsage",
        "FullHistoryDatasetUsage",
        "FinalTestAccessPolicy"
    ],
    "Value": [
        "CANONICAL_PRODUCT_BY_OPERATING_DATE",
        "TotalDemand",
        "ROLLING_ONE_OPERATING_DAY_AHEAD",
        1,
        20,
        5,
        (
            "ACTUAL_DEMAND_FROM_A_COMPLETED_OPERATING_DAY_"
            "MAY_UPDATE_FEATURES_FOR_THE_NEXT_OPERATING_DAY"
        ),
        False,
        True,
        False,
        True,
        True,
        len(direct_predictor_columns),
        len(historical_feature_columns),
        len(current_date_predictor_columns),
        (
            "FIT_MISSING_VALUE_TREATMENT_ON_TRAINING_ROWS_"
            "ONLY_WITHIN_EACH_SPLIT_CONTEXT"
        ),
        (
            "FIT_CATEGORICAL_ENCODING_ON_TRAINING_ROWS_"
            "ONLY_WITHIN_EACH_SPLIT_CONTEXT"
        ),
        (
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
        ),
        (
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
        ),
        (
            "DO_NOT_OPEN_FINAL_TEST_TARGET_VAULT_UNTIL_"
            "MODEL_SELECTION_AND_HYPERPARAMETERS_ARE_LOCKED"
        )
    ]
})


# ------------------------------------------------------------
# 23. Print Cell 54 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 5 — "
    "FINAL MODEL-READY DATASETS CREATED"
)
print("=" * 75)

print()
print("Frozen predictor contract:")
print(
    "Direct predictor columns:",
    len(direct_predictor_columns)
)
print(
    "Current-date predictors:",
    len(current_date_predictor_columns)
)
print(
    "Historical predictors:",
    len(historical_feature_columns)
)
print(
    "Forecast target:",
    TARGET_COLUMN
)

print()
print("Model-selection datasets:")
print(
    "Training rows:",
    f"{len(step3_model_selection_training_df):,}"
)
print(
    "Validation rows:",
    f"{len(step3_model_selection_validation_df):,}"
)
print(
    "Split contexts:",
    step3_model_selection_training_df[
        "SplitContextID"
    ].nunique()
)

print()
print("Reserved final-test datasets:")
print(
    "Training rows:",
    f"{len(step3_reserved_final_test_training_df):,}"
)
print(
    "Scoring-feature rows:",
    f"{len(step3_reserved_final_test_scoring_features_df):,}"
)
print(
    "Target-vault rows:",
    f"{len(step3_reserved_final_test_target_vault_df):,}"
)
print(
    "Target present in scoring-feature dataset:",
    TARGET_COLUMN
    in step3_reserved_final_test_scoring_features_df.columns
)

print()
print("Post-evaluation datasets:")
print(
    "Forecast-only training rows:",
    f"{len(step3_post_evaluation_forecast_only_training_df):,}"
)
print(
    "Full-history final-fit rows:",
    f"{len(step3_post_evaluation_full_history_final_fit_df):,}"
)

print()
print("Cell 54 completed successfully.")

STEP 3, PART 5 — FINAL MODEL-READY DATASETS CREATED

Frozen predictor contract:
Direct predictor columns: 53
Current-date predictors: 22
Historical predictors: 31
Forecast target: TotalDemand

Model-selection datasets:
Training rows: 48,538
Validation rows: 5,080
Split contexts: 254

Reserved final-test datasets:
Training rows: 38,628
Scoring-feature rows: 4,150
Target-vault rows: 4,150
Target present in scoring-feature dataset: False

Post-evaluation datasets:
Forecast-only training rows: 996
Full-history final-fit rows: 43,774

Cell 54 completed successfully.


In [70]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 5
# Cell 55: Independently validate the final frozen datasets
# ============================================================

# ------------------------------------------------------------
# 1. Confirm Cell 54 objects exist
# ------------------------------------------------------------

required_cell_54_objects = [
    "step3_model_selection_training_df",
    "step3_model_selection_validation_df",
    "step3_reserved_final_test_training_df",
    "step3_reserved_final_test_scoring_features_df",
    "step3_reserved_final_test_target_vault_df",
    "step3_post_evaluation_forecast_only_training_df",
    "step3_post_evaluation_full_history_final_fit_df",
    "step3_frozen_model_feature_contract_df",
    "step3_forecasting_protocol_contract_df",
    "direct_predictor_columns",
    "historical_feature_columns",
    "current_date_predictor_columns",
    "TARGET_COLUMN"
]

missing_cell_54_objects = [
    object_name
    for object_name in required_cell_54_objects
    if object_name not in globals()
]

if missing_cell_54_objects:
    raise NameError(
        "The following Cell 54 objects are missing:\n"
        f"{missing_cell_54_objects}\n\n"
        "Run Cell 54 before running Cell 55."
    )


# ------------------------------------------------------------
# 2. Create a corrected feature-parity validation helper
# ------------------------------------------------------------

def validate_dataset_feature_parity(
    dataset_df,
    dataset_name,
    compare_target=True
):
    """
    Confirm every frozen row contains the exact canonical source
    features belonging to its product-date.

    Column order is explicitly aligned before comparison.
    """

    source_columns_to_compare = (
        step3_part5_source_feature_df
        .columns
        .tolist()
    )

    if not compare_target:

        source_columns_to_compare = [
            column
            for column in source_columns_to_compare
            if column != TARGET_COLUMN
        ]

    # Final comparison order must be identical for both frames.
    comparison_columns = (
        ["SplitContextID"]
        + source_columns_to_compare
    )

    # Build expected rows from the canonical source dataset.
    expected_df = (
        dataset_df[
            [
                "SplitContextID"
            ]
            + ROW_JOIN_KEYS
        ]
        .merge(
            step3_part5_source_feature_df[
                source_columns_to_compare
            ],
            on=ROW_JOIN_KEYS,
            how="left",
            validate="many_to_one"
        )
    )

    # Build actual rows from the frozen dataset.
    actual_df = (
        dataset_df[
            comparison_columns
        ]
        .copy()
    )

    # Explicitly align the expected column order.
    expected_df = expected_df[
        comparison_columns
    ]

    sort_columns = [
        "SplitContextID",
        "CanonicalProductID",
        "OperatingDaySequence",
        "Date"
    ]

    expected_df = (
        expected_df
        .sort_values(
            sort_columns,
            kind="stable"
        )
        .reset_index(drop=True)
    )

    actual_df = (
        actual_df
        .sort_values(
            sort_columns,
            kind="stable"
        )
        .reset_index(drop=True)
    )

    try:

        pd.testing.assert_frame_equal(
            actual_df,
            expected_df,
            check_dtype=False,
            check_exact=False,
            check_like=False,
            rtol=1e-12,
            atol=1e-12
        )

    except AssertionError as error:

        raise AssertionError(
            f"Feature-parity validation failed for "
            f"{dataset_name}.\n\n{error}"
        ) from error

    return 0


# ------------------------------------------------------------
# 3. Validate exact source-feature parity
# ------------------------------------------------------------

model_selection_training_parity_mismatches = (
    validate_dataset_feature_parity(
        step3_model_selection_training_df,
        "MODEL_SELECTION_TRAINING"
    )
)

model_selection_validation_parity_mismatches = (
    validate_dataset_feature_parity(
        step3_model_selection_validation_df,
        "MODEL_SELECTION_VALIDATION"
    )
)

final_test_training_parity_mismatches = (
    validate_dataset_feature_parity(
        step3_reserved_final_test_training_df,
        "FINAL_TEST_TRAINING"
    )
)

final_test_feature_parity_mismatches = (
    validate_dataset_feature_parity(
        step3_reserved_final_test_scoring_features_df,
        "FINAL_TEST_SCORING_FEATURES",
        compare_target=False
    )
)

forecast_only_parity_mismatches = (
    validate_dataset_feature_parity(
        step3_post_evaluation_forecast_only_training_df,
        "FORECAST_ONLY_TRAINING"
    )
)

total_feature_parity_mismatches = (
    model_selection_training_parity_mismatches
    + model_selection_validation_parity_mismatches
    + final_test_training_parity_mismatches
    + final_test_feature_parity_mismatches
    + forecast_only_parity_mismatches
)

assert total_feature_parity_mismatches == 0


# ------------------------------------------------------------
# 4. Validate full-history source parity
# ------------------------------------------------------------

full_history_expected_df = (
    step3_part5_source_feature_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

full_history_actual_df = (
    step3_post_evaluation_full_history_final_fit_df
    .sort_values(
        [
            "Date",
            "CanonicalProductID"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    full_history_actual_df,
    full_history_expected_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 5. Validate the final-test target vault
# ------------------------------------------------------------

final_test_target_check_df = (
    step3_reserved_final_test_scoring_features_df[
        [
            "SplitContextID",
            "WindowID",
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence"
        ]
    ]
    .merge(
        step3_reserved_final_test_target_vault_df[
            [
                "SplitContextID",
                "WindowID",
                "Date",
                "CanonicalProductID",
                "OperatingDaySequence",
                TARGET_COLUMN
            ]
        ],
        on=[
            "SplitContextID",
            "WindowID",
            "Date",
            "CanonicalProductID",
            "OperatingDaySequence"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        step3_part5_source_feature_df[
            ROW_JOIN_KEYS
            + [
                TARGET_COLUMN
            ]
        ].rename(
            columns={
                TARGET_COLUMN:
                    "SourceTotalDemand"
            }
        ),
        on=ROW_JOIN_KEYS,
        how="left",
        validate="many_to_one"
    )
)

final_test_target_vault_missing_rows = int(
    final_test_target_check_df[
        TARGET_COLUMN
    ].isna().sum()
)

final_test_target_vault_mismatches = int(
    (
        ~np.isclose(
            final_test_target_check_df[
                TARGET_COLUMN
            ],
            final_test_target_check_df[
                "SourceTotalDemand"
            ],
            rtol=1e-12,
            atol=1e-12
        )
    ).sum()
)

assert final_test_target_vault_missing_rows == 0
assert final_test_target_vault_mismatches == 0


# ------------------------------------------------------------
# 6. Create a context chronology helper
# ------------------------------------------------------------

def create_context_chronology_audit(
    training_df,
    evaluation_df,
    evaluation_role,
    expected_context_count
):

    training_boundaries_df = (
        training_df
        .groupby(
            [
                "SplitContextID",
                "CanonicalProductID",
                "WindowID"
            ],
            as_index=False
        )
        .agg(
            TrainingRows=(
                "Date",
                "size"
            ),
            TrainingStartSequence=(
                "OperatingDaySequence",
                "min"
            ),
            TrainingEndSequence=(
                "OperatingDaySequence",
                "max"
            ),
            TrainingStartDate=(
                "Date",
                "min"
            ),
            TrainingEndDate=(
                "Date",
                "max"
            )
        )
    )

    evaluation_boundaries_df = (
        evaluation_df
        .groupby(
            [
                "SplitContextID",
                "CanonicalProductID",
                "WindowID"
            ],
            as_index=False
        )
        .agg(
            EvaluationRows=(
                "Date",
                "size"
            ),
            EvaluationStartSequence=(
                "OperatingDaySequence",
                "min"
            ),
            EvaluationEndSequence=(
                "OperatingDaySequence",
                "max"
            ),
            EvaluationStartDate=(
                "Date",
                "min"
            ),
            EvaluationEndDate=(
                "Date",
                "max"
            )
        )
    )

    audit_df = (
        training_boundaries_df
        .merge(
            evaluation_boundaries_df,
            on=[
                "SplitContextID",
                "CanonicalProductID",
                "WindowID"
            ],
            how="outer",
            validate="one_to_one"
        )
    )

    audit_df[
        "EvaluationRole"
    ] = evaluation_role

    audit_df[
        "MissingTrainingContext"
    ] = audit_df[
        "TrainingRows"
    ].isna()

    audit_df[
        "MissingEvaluationContext"
    ] = audit_df[
        "EvaluationRows"
    ].isna()

    audit_df[
        "ChronologicalOverlapViolation"
    ] = (
        audit_df[
            "TrainingEndSequence"
        ]
        >= audit_df[
            "EvaluationStartSequence"
        ]
    )

    assert len(audit_df) == expected_context_count

    assert not audit_df[
        "MissingTrainingContext"
    ].any()

    assert not audit_df[
        "MissingEvaluationContext"
    ].any()

    assert not audit_df[
        "ChronologicalOverlapViolation"
    ].any()

    return audit_df


step3_model_selection_chronology_audit_df = (
    create_context_chronology_audit(
        step3_model_selection_training_df,
        step3_model_selection_validation_df,
        "VALIDATION",
        254
    )
)

step3_final_test_chronology_audit_df = (
    create_context_chronology_audit(
        step3_reserved_final_test_training_df,
        step3_reserved_final_test_scoring_features_df,
        "FINAL_TEST",
        218
    )
)


# ------------------------------------------------------------
# 7. Validate model-selection/final-test isolation
# ------------------------------------------------------------

model_selection_context_ids = set(
    step3_model_selection_validation_df[
        "SplitContextID"
    ]
)

final_test_context_ids = set(
    step3_reserved_final_test_scoring_features_df[
        "SplitContextID"
    ]
)

context_isolation_violations = len(
    model_selection_context_ids.intersection(
        final_test_context_ids
    )
)

model_selection_product_date_keys = pd.MultiIndex.from_frame(
    step3_model_selection_context_df[
        [
            "Date",
            "CanonicalProductID"
        ]
    ]
    .drop_duplicates()
)

final_test_scoring_product_date_keys = pd.MultiIndex.from_frame(
    step3_reserved_final_test_scoring_features_df[
        [
            "Date",
            "CanonicalProductID"
        ]
    ]
    .drop_duplicates()
)

final_test_scoring_rows_found_in_model_selection = int(
    final_test_scoring_product_date_keys.isin(
        model_selection_product_date_keys
    ).sum()
)

assert context_isolation_violations == 0
assert final_test_scoring_rows_found_in_model_selection == 0


# ------------------------------------------------------------
# 8. Validate role and access-policy flags
# ------------------------------------------------------------

assert (
    step3_model_selection_training_df[
        "SplitRole"
    ] == "TRAIN"
).all()

assert (
    step3_model_selection_validation_df[
        "SplitRole"
    ] == "VALIDATION"
).all()

assert (
    step3_reserved_final_test_training_df[
        "SplitRole"
    ] == "TRAIN"
).all()

assert (
    step3_reserved_final_test_scoring_features_df[
        "SplitRole"
    ] == "FINAL_TEST"
).all()

assert (
    step3_post_evaluation_forecast_only_training_df[
        "SplitRole"
    ] == "FORECAST_ONLY_TRAINING"
).all()


assert step3_model_selection_training_df[
    "MayBeUsedForModelSelection"
].all()

assert step3_model_selection_validation_df[
    "MayBeUsedForModelSelection"
].all()

assert not step3_reserved_final_test_training_df[
    "MayBeUsedForModelSelection"
].any()

assert not step3_reserved_final_test_scoring_features_df[
    "MayBeUsedForModelSelection"
].any()

assert step3_reserved_final_test_training_df[
    "ReservedFinalTest"
].all()

assert step3_reserved_final_test_scoring_features_df[
    "ReservedFinalTest"
].all()


# ------------------------------------------------------------
# 9. Confirm the final-test target is physically absent
# ------------------------------------------------------------

final_test_target_columns_in_feature_file = int(
    TARGET_COLUMN
    in step3_reserved_final_test_scoring_features_df.columns
)

assert final_test_target_columns_in_feature_file == 0


# ------------------------------------------------------------
# 10. Validate predictor and target availability
# ------------------------------------------------------------

training_and_validation_datasets = {
    "MODEL_SELECTION_TRAINING":
        step3_model_selection_training_df,

    "MODEL_SELECTION_VALIDATION":
        step3_model_selection_validation_df,

    "FINAL_TEST_TRAINING":
        step3_reserved_final_test_training_df,

    "FORECAST_ONLY_POST_EVALUATION_TRAINING":
        step3_post_evaluation_forecast_only_training_df,

    "FULL_HISTORY_POST_EVALUATION_FINAL_FIT":
        step3_post_evaluation_full_history_final_fit_df
}

for (
    dataset_name,
    dataset_df
) in training_and_validation_datasets.items():

    assert all(
        column in dataset_df.columns
        for column in direct_predictor_columns
    ), (
        f"{dataset_name} is missing one or more "
        "frozen predictor columns."
    )

    assert TARGET_COLUMN in dataset_df.columns

    assert dataset_df[
        TARGET_COLUMN
    ].isna().sum() == 0


assert all(
    column
    in step3_reserved_final_test_scoring_features_df.columns
    for column in direct_predictor_columns
)

assert TARGET_COLUMN not in (
    step3_reserved_final_test_scoring_features_df.columns
)


# ------------------------------------------------------------
# 11. Confirm same-day leakage fields remain absent
# ------------------------------------------------------------

same_day_leakage_columns = {
    "NormalDemand",
    "BulkDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource"
}

frozen_datasets_for_leakage_check = {
    **training_and_validation_datasets,
    "FINAL_TEST_SCORING_FEATURES":
        step3_reserved_final_test_scoring_features_df
}

frozen_leakage_records = []

for (
    dataset_name,
    dataset_df
) in frozen_datasets_for_leakage_check.items():

    leakage_columns_found = sorted(
        same_day_leakage_columns.intersection(
            dataset_df.columns
        )
    )

    frozen_leakage_records.append({
        "DatasetName":
            dataset_name,
        "LeakageColumnCount":
            len(leakage_columns_found),
        "LeakageColumns":
            "|".join(
                leakage_columns_found
            )
    })

step3_final_leakage_audit_df = pd.DataFrame(
    frozen_leakage_records
)

assert step3_final_leakage_audit_df[
    "LeakageColumnCount"
].sum() == 0


# ------------------------------------------------------------
# 12. Create predictor missingness audit
# ------------------------------------------------------------

predictor_missingness_datasets = {
    "MODEL_SELECTION_TRAINING":
        step3_model_selection_training_df,

    "MODEL_SELECTION_VALIDATION":
        step3_model_selection_validation_df,

    "FINAL_TEST_TRAINING":
        step3_reserved_final_test_training_df,

    "FINAL_TEST_SCORING_FEATURES":
        step3_reserved_final_test_scoring_features_df,

    "FORECAST_ONLY_POST_EVALUATION_TRAINING":
        step3_post_evaluation_forecast_only_training_df,

    "FULL_HISTORY_POST_EVALUATION_FINAL_FIT":
        step3_post_evaluation_full_history_final_fit_df
}

feature_group_mapping = {
    "CURRENT_DATE_PREDICTORS":
        current_date_predictor_columns,

    "HISTORICAL_PREDICTORS":
        historical_feature_columns,

    "ALL_DIRECT_PREDICTORS":
        direct_predictor_columns
}

predictor_missingness_records = []

for (
    dataset_name,
    dataset_df
) in predictor_missingness_datasets.items():

    for (
        feature_group,
        feature_columns
    ) in feature_group_mapping.items():

        missing_cell_count = int(
            dataset_df[
                feature_columns
            ]
            .isna()
            .sum()
            .sum()
        )

        rows_with_any_missing = int(
            dataset_df[
                feature_columns
            ]
            .isna()
            .any(axis=1)
            .sum()
        )

        rows_with_complete_group = int(
            (
                ~dataset_df[
                    feature_columns
                ]
                .isna()
                .any(axis=1)
            ).sum()
        )

        predictor_missingness_records.append({
            "DatasetName":
                dataset_name,
            "FeatureGroup":
                feature_group,
            "FeatureCount":
                len(feature_columns),
            "RowCount":
                len(dataset_df),
            "MissingCellCount":
                missing_cell_count,
            "RowsWithAnyMissing":
                rows_with_any_missing,
            "RowsWithCompleteFeatureGroup":
                rows_with_complete_group,
            "MissingnessTreatmentApplied":
                False,
            "TreatmentPolicy":
                (
                    "FIT_INSIDE_TRAINING_PIPELINE_ONLY"
                )
        })


step3_final_predictor_missingness_audit_df = pd.DataFrame(
    predictor_missingness_records
)


# ------------------------------------------------------------
# 13. Validate scoring historical-feature completeness
# ------------------------------------------------------------

scoring_historical_missing_cells = int(
    step3_model_selection_validation_df[
        historical_feature_columns
    ].isna().sum().sum()
    +
    step3_reserved_final_test_scoring_features_df[
        historical_feature_columns
    ].isna().sum().sum()
)

assert scoring_historical_missing_cells == 0


# ------------------------------------------------------------
# 14. Create final dataset summary
# ------------------------------------------------------------

step3_final_dataset_summary_df = pd.DataFrame([
    {
        "DatasetName":
            "MODEL_SELECTION_TRAINING",
        "Rows":
            len(
                step3_model_selection_training_df
            ),
        "Columns":
            step3_model_selection_training_df.shape[1],
        "Products":
            step3_model_selection_training_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            step3_model_selection_training_df[
                "SplitContextID"
            ].nunique(),
        "ScoringRows":
            0,
        "TargetIncluded":
            True,
        "AllowedBeforeModelSelectionLock":
            True,
        "UsageStage":
            "MODEL_SELECTION_TRAINING"
    },
    {
        "DatasetName":
            "MODEL_SELECTION_VALIDATION",
        "Rows":
            len(
                step3_model_selection_validation_df
            ),
        "Columns":
            step3_model_selection_validation_df.shape[1],
        "Products":
            step3_model_selection_validation_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            step3_model_selection_validation_df[
                "SplitContextID"
            ].nunique(),
        "ScoringRows":
            len(
                step3_model_selection_validation_df
            ),
        "TargetIncluded":
            True,
        "AllowedBeforeModelSelectionLock":
            True,
        "UsageStage":
            "MODEL_SELECTION_SCORING"
    },
    {
        "DatasetName":
            "RESERVED_FINAL_TEST_TRAINING",
        "Rows":
            len(
                step3_reserved_final_test_training_df
            ),
        "Columns":
            step3_reserved_final_test_training_df.shape[1],
        "Products":
            step3_reserved_final_test_training_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            step3_reserved_final_test_training_df[
                "SplitContextID"
            ].nunique(),
        "ScoringRows":
            0,
        "TargetIncluded":
            True,
        "AllowedBeforeModelSelectionLock":
            False,
        "UsageStage":
            "AFTER_MODEL_SELECTION_LOCK"
    },
    {
        "DatasetName":
            "RESERVED_FINAL_TEST_SCORING_FEATURES",
        "Rows":
            len(
                step3_reserved_final_test_scoring_features_df
            ),
        "Columns":
            step3_reserved_final_test_scoring_features_df.shape[1],
        "Products":
            step3_reserved_final_test_scoring_features_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            step3_reserved_final_test_scoring_features_df[
                "SplitContextID"
            ].nunique(),
        "ScoringRows":
            len(
                step3_reserved_final_test_scoring_features_df
            ),
        "TargetIncluded":
            False,
        "AllowedBeforeModelSelectionLock":
            False,
        "UsageStage":
            "AFTER_MODEL_SELECTION_LOCK"
    },
    {
        "DatasetName":
            "RESERVED_FINAL_TEST_TARGET_VAULT",
        "Rows":
            len(
                step3_reserved_final_test_target_vault_df
            ),
        "Columns":
            step3_reserved_final_test_target_vault_df.shape[1],
        "Products":
            step3_reserved_final_test_target_vault_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            step3_reserved_final_test_target_vault_df[
                "SplitContextID"
            ].nunique(),
        "ScoringRows":
            len(
                step3_reserved_final_test_target_vault_df
            ),
        "TargetIncluded":
            True,
        "AllowedBeforeModelSelectionLock":
            False,
        "UsageStage":
            "OPEN_ONLY_AFTER_FINAL_TEST_PREDICTIONS_SAVED"
    },
    {
        "DatasetName":
            "FORECAST_ONLY_POST_EVALUATION_TRAINING",
        "Rows":
            len(
                step3_post_evaluation_forecast_only_training_df
            ),
        "Columns":
            step3_post_evaluation_forecast_only_training_df.shape[1],
        "Products":
            step3_post_evaluation_forecast_only_training_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            step3_post_evaluation_forecast_only_training_df[
                "SplitContextID"
            ].nunique(),
        "ScoringRows":
            0,
        "TargetIncluded":
            True,
        "AllowedBeforeModelSelectionLock":
            False,
        "UsageStage":
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
    },
    {
        "DatasetName":
            "FULL_HISTORY_POST_EVALUATION_FINAL_FIT",
        "Rows":
            len(
                step3_post_evaluation_full_history_final_fit_df
            ),
        "Columns":
            step3_post_evaluation_full_history_final_fit_df.shape[1],
        "Products":
            step3_post_evaluation_full_history_final_fit_df[
                "CanonicalProductID"
            ].nunique(),
        "SplitContexts":
            0,
        "ScoringRows":
            0,
        "TargetIncluded":
            True,
        "AllowedBeforeModelSelectionLock":
            False,
        "UsageStage":
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
    }
])


# ------------------------------------------------------------
# 15. Create the final validation summary
# ------------------------------------------------------------

step3_part5_validation_summary_df = pd.DataFrame({
    "ValidationMetric": [
        "DirectPredictorColumns",
        "CurrentDatePredictorColumns",
        "HistoricalPredictorColumns",
        "ForecastTargetColumns",
        "ModelSelectionTrainingRows",
        "ModelSelectionValidationRows",
        "ModelSelectionContexts",
        "ReservedFinalTestTrainingRows",
        "ReservedFinalTestScoringRows",
        "ReservedFinalTestTargetVaultRows",
        "ReservedFinalTestContexts",
        "ForecastOnlyTrainingRows",
        "ForecastOnlyContexts",
        "FullHistoryFinalFitRows",
        "FullHistoryFinalFitColumns",
        "FullHistoryProducts",
        "FullHistoryOperatingDates",
        "UniqueFullHistoryDemandUnits",
        "FeatureParityMismatches",
        "FinalTestTargetVaultMissingRows",
        "FinalTestTargetVaultMismatches",
        "ModelSelectionChronologyViolations",
        "FinalTestChronologyViolations",
        "ModelSelectionFinalTestContextOverlap",
        "FinalTestScoringRowsInModelSelectionData",
        "FinalTestTargetPresentInFeatureFile",
        "ScoringHistoricalFeatureMissingCells",
        "SameDayLeakageColumnsPresent",
        "RandomSplittingAllowed",
        "AllZeroEvaluationWindowsRetained"
    ],
    "Value": [
        len(direct_predictor_columns),
        len(current_date_predictor_columns),
        len(historical_feature_columns),
        len(target_columns),
        len(step3_model_selection_training_df),
        len(step3_model_selection_validation_df),
        step3_model_selection_training_df[
            "SplitContextID"
        ].nunique(),
        len(step3_reserved_final_test_training_df),
        len(step3_reserved_final_test_scoring_features_df),
        len(step3_reserved_final_test_target_vault_df),
        step3_reserved_final_test_training_df[
            "SplitContextID"
        ].nunique(),
        len(
            step3_post_evaluation_forecast_only_training_df
        ),
        step3_post_evaluation_forecast_only_training_df[
            "SplitContextID"
        ].nunique(),
        len(
            step3_post_evaluation_full_history_final_fit_df
        ),
        step3_post_evaluation_full_history_final_fit_df.shape[1],
        step3_post_evaluation_full_history_final_fit_df[
            "CanonicalProductID"
        ].nunique(),
        step3_post_evaluation_full_history_final_fit_df[
            "Date"
        ].nunique(),
        int(
            step3_post_evaluation_full_history_final_fit_df[
                "TotalDemand"
            ].sum()
        ),
        total_feature_parity_mismatches,
        final_test_target_vault_missing_rows,
        final_test_target_vault_mismatches,
        int(
            step3_model_selection_chronology_audit_df[
                "ChronologicalOverlapViolation"
            ].sum()
        ),
        int(
            step3_final_test_chronology_audit_df[
                "ChronologicalOverlapViolation"
            ].sum()
        ),
        context_isolation_violations,
        final_test_scoring_rows_found_in_model_selection,
        final_test_target_columns_in_feature_file,
        scoring_historical_missing_cells,
        int(
            step3_final_leakage_audit_df[
                "LeakageColumnCount"
            ].sum()
        ),
        False,
        344
    ]
})


# ------------------------------------------------------------
# 16. Validate the final summary
# ------------------------------------------------------------

part5_summary_lookup = dict(
    zip(
        step3_part5_validation_summary_df[
            "ValidationMetric"
        ],
        step3_part5_validation_summary_df[
            "Value"
        ]
    )
)

assert part5_summary_lookup[
    "DirectPredictorColumns"
] == 53

assert part5_summary_lookup[
    "HistoricalPredictorColumns"
] == 31

assert part5_summary_lookup[
    "ModelSelectionTrainingRows"
] == 48_538

assert part5_summary_lookup[
    "ModelSelectionValidationRows"
] == 5_080

assert part5_summary_lookup[
    "ReservedFinalTestTrainingRows"
] == 38_628

assert part5_summary_lookup[
    "ReservedFinalTestScoringRows"
] == 4_150

assert part5_summary_lookup[
    "ReservedFinalTestTargetVaultRows"
] == 4_150

assert part5_summary_lookup[
    "ForecastOnlyTrainingRows"
] == 996

assert part5_summary_lookup[
    "FullHistoryFinalFitRows"
] == 43_774

assert part5_summary_lookup[
    "FullHistoryFinalFitColumns"
] == 58

assert part5_summary_lookup[
    "UniqueFullHistoryDemandUnits"
] == 116_158

assert part5_summary_lookup[
    "FeatureParityMismatches"
] == 0

assert part5_summary_lookup[
    "FinalTestTargetVaultMismatches"
] == 0

assert part5_summary_lookup[
    "ModelSelectionChronologyViolations"
] == 0

assert part5_summary_lookup[
    "FinalTestChronologyViolations"
] == 0

assert part5_summary_lookup[
    "ModelSelectionFinalTestContextOverlap"
] == 0

assert part5_summary_lookup[
    "FinalTestScoringRowsInModelSelectionData"
] == 0

assert part5_summary_lookup[
    "FinalTestTargetPresentInFeatureFile"
] == 0

assert part5_summary_lookup[
    "ScoringHistoricalFeatureMissingCells"
] == 0

assert part5_summary_lookup[
    "SameDayLeakageColumnsPresent"
] == 0


# ------------------------------------------------------------
# 17. Print Cell 55 results
# ------------------------------------------------------------

print("=" * 75)
print(
    "STEP 3, PART 5 — "
    "FINAL DATASET VALIDATION PASSED"
)
print("=" * 75)

print()
print("Final dataset summary:")
display(step3_final_dataset_summary_df)

print()
print("Predictor missingness audit:")
display(step3_final_predictor_missingness_audit_df)

print()
print("Chronological validation:")
print(
    "Model-selection contexts audited:",
    len(
        step3_model_selection_chronology_audit_df
    )
)
print(
    "Model-selection overlap violations:",
    int(
        step3_model_selection_chronology_audit_df[
            "ChronologicalOverlapViolation"
        ].sum()
    )
)
print(
    "Final-test contexts audited:",
    len(
        step3_final_test_chronology_audit_df
    )
)
print(
    "Final-test overlap violations:",
    int(
        step3_final_test_chronology_audit_df[
            "ChronologicalOverlapViolation"
        ].sum()
    )
)

print()
print("Final-test isolation:")
print(
    "Context overlap with model selection:",
    context_isolation_violations
)
print(
    "Final-test scoring rows found in "
    "model-selection data:",
    final_test_scoring_rows_found_in_model_selection
)
print(
    "Target present in final-test feature file:",
    bool(
        final_test_target_columns_in_feature_file
    )
)
print(
    "Target-vault mismatches:",
    final_test_target_vault_mismatches
)

print()
print("Leakage validation:")
display(step3_final_leakage_audit_df)

print()
print("Cell 55 completed successfully.")

STEP 3, PART 5 — FINAL DATASET VALIDATION PASSED

Final dataset summary:


,DatasetName,Rows,Columns,Products,SplitContexts,ScoringRows,TargetIncluded,AllowedBeforeModelSelectionLock,UsageStage
0,MODEL_SELECTION_TRAINING,48538,72,127,254,0,True,True,MODEL_SELECTION_TRAINING
1,MODEL_SELECTION_VALIDATION,5080,72,127,254,5080,True,True,MODEL_SELECTION_SCORING
2,RESERVED_FINAL_TEST_TRAINING,38628,72,218,218,0,True,False,AFTER_MODEL_SELECTION_LOCK
3,RESERVED_FINAL_TEST_SCORING_FEATURES,4150,71,218,218,4150,False,False,AFTER_MODEL_SELECTION_LOCK
4,RESERVED_FINAL_TEST_TARGET_VAULT,4150,9,218,218,4150,True,False,OPEN_ONLY_AFTER_FINAL_TEST_PREDICTIONS_SAVED
5,FORECAST_ONLY_POST_EVALUATION_TRAINING,996,72,9,9,0,True,False,POST_FINAL_TEST_PRODUCTION_FIT_ONLY
6,FULL_HISTORY_POST_EVALUATION_FINAL_FIT,43774,58,227,0,0,True,False,POST_FINAL_TEST_PRODUCTION_FIT_ONLY



Predictor missingness audit:


,DatasetName,FeatureGroup,FeatureCount,RowCount,MissingCellCount,RowsWithAnyMissing,RowsWithCompleteFeatureGroup,MissingnessTreatmentApplied,TreatmentPolicy
0,MODEL_SELECTION_TRAINING,CURRENT_DATE_PREDICTORS,22,48538,276396,48538,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
1,MODEL_SELECTION_TRAINING,HISTORICAL_PREDICTORS,31,48538,16764,5080,43458,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
2,MODEL_SELECTION_TRAINING,ALL_DIRECT_PREDICTORS,53,48538,293160,48538,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
3,MODEL_SELECTION_VALIDATION,CURRENT_DATE_PREDICTORS,22,5080,28920,5080,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
4,MODEL_SELECTION_VALIDATION,HISTORICAL_PREDICTORS,31,5080,0,0,5080,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
5,MODEL_SELECTION_VALIDATION,ALL_DIRECT_PREDICTORS,53,5080,28920,5080,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
6,FINAL_TEST_TRAINING,CURRENT_DATE_PREDICTORS,22,38628,220326,38628,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
7,FINAL_TEST_TRAINING,HISTORICAL_PREDICTORS,31,38628,14388,4360,34268,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
8,FINAL_TEST_TRAINING,ALL_DIRECT_PREDICTORS,53,38628,234714,38628,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY
9,FINAL_TEST_SCORING_FEATURES,CURRENT_DATE_PREDICTORS,22,4150,23670,4150,0,False,FIT_INSIDE_TRAINING_PIPELINE_ONLY



Chronological validation:
Model-selection contexts audited: 254
Model-selection overlap violations: 0
Final-test contexts audited: 218
Final-test overlap violations: 0

Final-test isolation:
Context overlap with model selection: 0
Final-test scoring rows found in model-selection data: 0
Target present in final-test feature file: False
Target-vault mismatches: 0

Leakage validation:


,DatasetName,LeakageColumnCount,LeakageColumns
0,MODEL_SELECTION_TRAINING,0,
1,MODEL_SELECTION_VALIDATION,0,
2,FINAL_TEST_TRAINING,0,
3,FORECAST_ONLY_POST_EVALUATION_TRAINING,0,
4,FULL_HISTORY_POST_EVALUATION_FINAL_FIT,0,
5,FINAL_TEST_SCORING_FEATURES,0,



Cell 55 completed successfully.


In [71]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 4
# Cell 53: Save final split-audit outputs and update handoff
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required Part 4 objects exist
# ------------------------------------------------------------

required_step3_part4_objects = [
    "step3_part4_split_role_demand_summary_df",
    "step3_part4_window_role_demand_summary_df",
    "step3_part4_cohort_role_demand_summary_df",
    "step3_part4_product_role_coverage_df",
    "step3_part4_context_integrity_audit_df",
    "step3_part4_final_test_isolation_audit_df",
    "step3_part4_evaluation_feature_readiness_audit_df",
    "step3_part4_zero_demand_window_audit_df",
    "step3_part4_validation_summary_df",
    "FORECAST_PREPARATION_DIR"
]

missing_step3_part4_objects = [
    object_name
    for object_name in required_step3_part4_objects
    if object_name not in globals()
]

if missing_step3_part4_objects:
    raise NameError(
        "The following Step 3 Part 4 objects are missing:\n"
        f"{missing_step3_part4_objects}\n\n"
        "Run Cells 51 and 52 before running Cell 53."
    )


# ------------------------------------------------------------
# 2. Restore Markdown handoff helper if necessary
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(end_marker)
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 3. Define Step 3 Part 4 output paths
# ------------------------------------------------------------

STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_split_role_demand_summary.csv"
)

STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_window_role_demand_summary.csv"
)

STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_cohort_role_demand_summary.csv"
)

STEP3_PART4_PRODUCT_COVERAGE_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_product_role_coverage_audit.csv"
)

STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_context_integrity_audit.csv"
)

STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_final_test_isolation_audit.csv"
)

STEP3_PART4_FEATURE_READINESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_evaluation_feature_readiness_audit.csv"
)

STEP3_PART4_ZERO_DEMAND_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_zero_demand_evaluation_window_audit.csv"
)

STEP3_PART4_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "16_step3_part4_validation_summary.csv"
)


# ------------------------------------------------------------
# 4. Save all Part 4 outputs
# ------------------------------------------------------------

step3_part4_split_role_demand_summary_df.to_csv(
    STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_part4_window_role_demand_summary_df.to_csv(
    STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_part4_cohort_role_demand_summary_df.to_csv(
    STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT,
    index=False
)

step3_part4_product_role_coverage_df.to_csv(
    STEP3_PART4_PRODUCT_COVERAGE_OUTPUT,
    index=False
)

step3_part4_context_integrity_audit_df.to_csv(
    STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part4_final_test_isolation_audit_df.to_csv(
    STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT,
    index=False
)

step3_part4_evaluation_feature_readiness_audit_df.to_csv(
    STEP3_PART4_FEATURE_READINESS_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part4_zero_demand_window_audit_df.to_csv(
    STEP3_PART4_ZERO_DEMAND_OUTPUT,
    index=False,
    date_format="%Y-%m-%d"
)

step3_part4_validation_summary_df.to_csv(
    STEP3_PART4_VALIDATION_OUTPUT,
    index=False
)


# ------------------------------------------------------------
# 5. Reload saved outputs
# ------------------------------------------------------------

saved_part4_split_role_summary = pd.read_csv(
    STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_part4_window_role_summary = pd.read_csv(
    STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_part4_cohort_role_summary = pd.read_csv(
    STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT,
    low_memory=False
)

saved_part4_product_coverage = pd.read_csv(
    STEP3_PART4_PRODUCT_COVERAGE_OUTPUT,
    low_memory=False
)

saved_part4_context_integrity = pd.read_csv(
    STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT,
    low_memory=False
)

saved_part4_final_test_isolation = pd.read_csv(
    STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT,
    low_memory=False
)

saved_part4_feature_readiness = pd.read_csv(
    STEP3_PART4_FEATURE_READINESS_OUTPUT,
    low_memory=False
)

saved_part4_zero_demand = pd.read_csv(
    STEP3_PART4_ZERO_DEMAND_OUTPUT,
    low_memory=False
)

saved_part4_validation = pd.read_csv(
    STEP3_PART4_VALIDATION_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 6. Validate saved outputs
# ------------------------------------------------------------

assert saved_part4_split_role_summary[
    "ManifestRows"
].sum() == 97_392

assert saved_part4_product_coverage[
    "CanonicalProductID"
].nunique() == 227

assert len(
    saved_part4_context_integrity
) == 472

assert saved_part4_context_integrity[
    "ChronologicalOverlapViolation"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_part4_context_integrity[
    "MissingTrainingRole"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_part4_context_integrity[
    "MissingEvaluationRole"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert saved_part4_final_test_isolation[
    "ViolationCount"
].sum() == 0

assert len(
    saved_part4_feature_readiness
) == 0

assert len(
    saved_part4_zero_demand
) == 344

assert saved_part4_zero_demand[
    "RemovedFromEvaluation"
].astype(str).str.lower().isin(
    ["true", "1"]
).sum() == 0

assert len(
    saved_part4_validation
) == 28


# ------------------------------------------------------------
# 7. Build Markdown demand summary
# ------------------------------------------------------------

split_role_markdown_rows = "\n".join(
    (
        f"- `{row.SplitRole}`: "
        f"{int(row.ManifestRows):,} rows, "
        f"{int(row.ProductCount)} products, "
        f"{int(row.PositiveDemandRows):,} positive-demand rows, "
        f"{int(row.ZeroDemandRows):,} zero-demand rows"
    )
    for row in (
        step3_part4_split_role_demand_summary_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 8. Update the Markdown handoff
# ------------------------------------------------------------

step3_part4_summary = f"""
**Status:** Completed and validated

### Purpose

Step 3 Part 4 performed the final quality-control audit of the
chronological split system before creation of the frozen modelling
datasets.

### Manifest structure

- Complete manifest rows: {len(step3_part4_complete_manifest_df):,}
- Independent-evaluation manifest rows: {len(step3_part4_independent_manifest_df):,}
- Model-selection manifest rows: {len(step3_part4_model_selection_manifest_df):,}
- Final-test manifest rows: {len(step3_part4_final_test_manifest_df):,}
- Forecast-only training rows: {len(step3_part4_forecast_only_manifest_df):,}
- Total split contexts: 481
- Products represented: 227

### Split-role demand coverage

{split_role_markdown_rows}

The demand-unit totals in split-role summaries are context-weighted
because source observations are intentionally reused across
expanding-window contexts.

### Unique source representation

- Unique source rows represented: 43,774
- Unique source demand units represented: 116,158
- Missing source rows: 0

### Chronological integrity

- Independent contexts audited: 472
- Chronological overlap violations: 0
- Missing training contexts: 0
- Missing evaluation contexts: 0
- Contexts with multiple evaluation roles: 0

Every independent evaluation context has a training period followed
strictly by one validation or final-test period.

### Final-test isolation

- Model-selection contexts: 254
- Reserved final-test contexts: 218
- Context overlap between model selection and final test: 0
- Final-test rows marked for model selection: 0
- Validation rows marked as reserved final test: 0
- Training rows eligible for scoring: 0
- Total final-test isolation violations: 0

The reserved final-test contexts must not be used for feature
selection, model selection or hyperparameter tuning.

### Scoring feature readiness

- Validation scoring rows: {validation_scoring_rows:,}
- Final-test scoring rows: {final_test_scoring_rows:,}
- Total scoring rows: {total_scoring_rows:,}
- Historical features required: 31
- Scoring rows missing historical features: 0
- Missing historical-feature cells: 0

### Intermittent-demand evaluation

- All-zero-demand evaluation windows retained: 344
- All-zero-demand evaluation windows removed: 0
- Zero-demand audit context mismatches: 0

All-zero evaluation windows remain valid and must be included when
evaluating intermittent-demand performance.

### Saved Step 3 Part 4 outputs

- `16_step3_split_role_demand_summary.csv`
- `16_step3_window_role_demand_summary.csv`
- `16_step3_cohort_role_demand_summary.csv`
- `16_step3_product_role_coverage_audit.csv`
- `16_step3_context_integrity_audit.csv`
- `16_step3_final_test_isolation_audit.csv`
- `16_step3_evaluation_feature_readiness_audit.csv`
- `16_step3_zero_demand_evaluation_window_audit.csv`
- `16_step3_part4_validation_summary.csv`
"""

upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_3_part_4",
    section_title=(
        "Forecasting Preparation — Step 3, Part 4"
    ),
    section_body=step3_part4_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 9. Print final Part 4 completion summary
# ------------------------------------------------------------

print("=" * 75)
print(
    "FORECASTING PREPARATION — "
    "STEP 3, PART 4 COMPLETED"
)
print("=" * 75)

print()
print("Manifest and product coverage:")
print(
    "Complete manifest rows:",
    f"{len(step3_part4_complete_manifest_df):,}"
)
print(
    "Total split contexts:",
    step3_part4_complete_manifest_df[
        "SplitContextID"
    ].nunique()
)
print(
    "Products represented:",
    step3_part4_complete_manifest_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Unique source rows represented:",
    f"{len(step3_part4_unique_manifest_source_df):,}"
)
print(
    "Unique source demand units:",
    f"{step3_part4_unique_manifest_source_df['TotalDemand'].sum():,.0f}"
)

print()
print("Chronological integrity:")
print(
    "Contexts audited:",
    len(
        step3_part4_context_integrity_audit_df
    )
)
print(
    "Chronological overlap violations:",
    context_overlap_violations
)
print(
    "Missing training contexts:",
    missing_training_contexts
)
print(
    "Missing evaluation contexts:",
    missing_evaluation_contexts
)

print()
print("Final-test isolation:")
print(
    "Model-selection contexts:",
    len(model_selection_context_ids)
)
print(
    "Final-test contexts:",
    len(final_test_context_ids)
)
print(
    "Final-test isolation violations:",
    final_test_isolation_violations
)

print()
print("Scoring readiness:")
print(
    "Validation scoring rows:",
    f"{validation_scoring_rows:,}"
)
print(
    "Final-test scoring rows:",
    f"{final_test_scoring_rows:,}"
)
print(
    "Scoring rows missing historical features:",
    scoring_rows_missing_any_historical_feature
)
print(
    "Historical-feature missing cells:",
    scoring_missing_historical_cells
)

print()
print("Intermittent-demand evaluation:")
print(
    "All-zero evaluation windows retained:",
    len(saved_part4_zero_demand)
)
print(
    "All-zero evaluation windows removed:",
    0
)

print()
print("Saved files:")
print(f"1. {STEP3_PART4_SPLIT_ROLE_SUMMARY_OUTPUT}")
print(f"2. {STEP3_PART4_WINDOW_ROLE_SUMMARY_OUTPUT}")
print(f"3. {STEP3_PART4_COHORT_ROLE_SUMMARY_OUTPUT}")
print(f"4. {STEP3_PART4_PRODUCT_COVERAGE_OUTPUT}")
print(f"5. {STEP3_PART4_CONTEXT_INTEGRITY_OUTPUT}")
print(f"6. {STEP3_PART4_FINAL_TEST_ISOLATION_OUTPUT}")
print(f"7. {STEP3_PART4_FEATURE_READINESS_OUTPUT}")
print(f"8. {STEP3_PART4_ZERO_DEMAND_OUTPUT}")
print(f"9. {STEP3_PART4_VALIDATION_OUTPUT}")
print(f"10. {HANDOFF_FILE}")

print()
print(
    "All Step 3, Part 4 validation checks passed."
)

FORECASTING PREPARATION — STEP 3, PART 4 COMPLETED

Manifest and product coverage:
Complete manifest rows: 97,392
Total split contexts: 481
Products represented: 227
Unique source rows represented: 43,774
Unique source demand units: 116,158

Chronological integrity:
Contexts audited: 472
Chronological overlap violations: 0
Missing training contexts: 0
Missing evaluation contexts: 0

Final-test isolation:
Model-selection contexts: 254
Final-test contexts: 218
Final-test isolation violations: 0

Scoring readiness:
Validation scoring rows: 5,080
Final-test scoring rows: 4,150
Scoring rows missing historical features: 0
Historical-feature missing cells: 0

Intermittent-demand evaluation:
All-zero evaluation windows retained: 344
All-zero evaluation windows removed: 0

Saved files:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/16_step3_split_role_demand_summary.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/16_step3_window_role_de

In [72]:
# ============================================================
# FORECASTING PREPARATION
# STEP 3 — PART 5
# Cell 56: Save, reload, hash and freeze the final
# modelling-ready datasets
# ============================================================

import hashlib
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Confirm that Cells 54 and 55 completed successfully
# ------------------------------------------------------------

required_cell_56_objects = [
    # Final model-selection datasets
    "step3_model_selection_training_df",
    "step3_model_selection_validation_df",

    # Reserved final-test datasets
    "step3_reserved_final_test_training_df",
    "step3_reserved_final_test_scoring_features_df",
    "step3_reserved_final_test_target_vault_df",

    # Post-evaluation datasets
    "step3_post_evaluation_forecast_only_training_df",
    "step3_post_evaluation_full_history_final_fit_df",

    # Frozen contracts and audits
    "step3_frozen_model_feature_contract_df",
    "step3_forecasting_protocol_contract_df",
    "step3_final_predictor_missingness_audit_df",
    "step3_final_leakage_audit_df",
    "step3_final_dataset_summary_df",
    "step3_part5_validation_summary_df",

    # Validation objects
    "step3_model_selection_chronology_audit_df",
    "step3_final_test_chronology_audit_df",
    "total_feature_parity_mismatches",
    "context_isolation_violations",
    "final_test_target_vault_mismatches",
    "scoring_historical_missing_cells",

    # Feature definitions
    "direct_predictor_columns",
    "current_date_predictor_columns",
    "historical_feature_columns",
    "TARGET_COLUMN",

    # Project directory
    "FORECAST_PREPARATION_DIR"
]

missing_cell_56_objects = [
    object_name
    for object_name in required_cell_56_objects
    if object_name not in globals()
]

if missing_cell_56_objects:
    raise NameError(
        "The following required objects are missing:\n"
        f"{missing_cell_56_objects}\n\n"
        "Run the corrected Cells 54 and 55 before running Cell 56."
    )


# ------------------------------------------------------------
# 2. Confirm the output directories
# ------------------------------------------------------------

FORECAST_PREPARATION_DIR = Path(
    FORECAST_PREPARATION_DIR
)

if not FORECAST_PREPARATION_DIR.exists():
    raise FileNotFoundError(
        "The forecasting-preparation directory does not exist:\n"
        f"{FORECAST_PREPARATION_DIR}"
    )


if "EDEN_DATASETS_DIR" not in globals():
    EDEN_DATASETS_DIR = (
        FORECAST_PREPARATION_DIR.parent
    )
else:
    EDEN_DATASETS_DIR = Path(
        EDEN_DATASETS_DIR
    )


if not EDEN_DATASETS_DIR.exists():
    raise FileNotFoundError(
        "The eden_datasets directory does not exist:\n"
        f"{EDEN_DATASETS_DIR}"
    )


# ------------------------------------------------------------
# 3. Restore the Markdown handoff path and helper if needed
# ------------------------------------------------------------

if "HANDOFF_FILE" not in globals():

    HANDOFF_FILE = (
        FORECAST_PREPARATION_DIR
        / "FORECASTING_PREPARATION_HANDOFF.md"
    )

else:

    HANDOFF_FILE = Path(
        HANDOFF_FILE
    )


if "upsert_markdown_section" not in globals():

    def upsert_markdown_section(
        file_path,
        section_id,
        section_title,
        section_body
    ):
        """
        Insert or replace a controlled Markdown section.
        """

        file_path = Path(file_path)

        start_marker = (
            f"<!-- START:{section_id} -->"
        )

        end_marker = (
            f"<!-- END:{section_id} -->"
        )

        section_block = (
            f"{start_marker}\n"
            f"## {section_title}\n\n"
            f"{section_body.strip()}\n"
            f"{end_marker}\n"
        )

        if file_path.exists():

            existing_text = file_path.read_text(
                encoding="utf-8"
            )

        else:

            existing_text = (
                "# Eden Forecasting Preparation Handoff\n\n"
            )

        if (
            start_marker in existing_text
            and end_marker in existing_text
        ):

            section_start = existing_text.index(
                start_marker
            )

            section_end = (
                existing_text.index(
                    end_marker
                )
                + len(end_marker)
            )

            updated_text = (
                existing_text[:section_start]
                + section_block.rstrip()
                + existing_text[section_end:]
            )

        else:

            updated_text = (
                existing_text.rstrip()
                + "\n\n"
                + section_block
            )

        file_path.write_text(
            updated_text.rstrip() + "\n",
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 4. Reconfirm the critical Cell 55 validation results
# ------------------------------------------------------------

model_selection_overlap_violations = int(
    step3_model_selection_chronology_audit_df[
        "ChronologicalOverlapViolation"
    ].sum()
)

final_test_overlap_violations = int(
    step3_final_test_chronology_audit_df[
        "ChronologicalOverlapViolation"
    ].sum()
)

same_day_leakage_columns_present = int(
    step3_final_leakage_audit_df[
        "LeakageColumnCount"
    ].sum()
)


assert total_feature_parity_mismatches == 0

assert model_selection_overlap_violations == 0

assert final_test_overlap_violations == 0

assert context_isolation_violations == 0

assert final_test_target_vault_mismatches == 0

assert scoring_historical_missing_cells == 0

assert same_day_leakage_columns_present == 0

assert TARGET_COLUMN not in (
    step3_reserved_final_test_scoring_features_df.columns
)

assert TARGET_COLUMN in (
    step3_reserved_final_test_target_vault_df.columns
)


# ------------------------------------------------------------
# 5. Define final output paths
# ------------------------------------------------------------

STEP3_FINAL_MODEL_SELECTION_TRAINING_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_model_selection_training_dataset.csv"
)

STEP3_FINAL_MODEL_SELECTION_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_model_selection_validation_dataset.csv"
)

STEP3_FINAL_TEST_TRAINING_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_reserved_final_test_training_dataset.csv"
)

STEP3_FINAL_TEST_FEATURES_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_reserved_final_test_scoring_features.csv"
)

STEP3_FINAL_TEST_TARGET_VAULT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_reserved_final_test_target_vault.csv"
)

STEP3_FINAL_FORECAST_ONLY_TRAINING_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_post_evaluation_forecast_only_training_dataset.csv"
)


FINAL_CANONICAL_DATASET_NAME = (
    "UL_EDEN_forecasting_preparation_final_model_dataset.csv"
)

FINAL_CANONICAL_DATASET_IN_PREPARATION = (
    FORECAST_PREPARATION_DIR
    / FINAL_CANONICAL_DATASET_NAME
)

FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS = (
    EDEN_DATASETS_DIR
    / FINAL_CANONICAL_DATASET_NAME
)


STEP3_FINAL_FEATURE_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_frozen_model_feature_contract.csv"
)

STEP3_FINAL_PROTOCOL_CONTRACT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_forecasting_protocol_contract.csv"
)

STEP3_FINAL_PREDICTOR_MISSINGNESS_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_final_predictor_missingness_audit.csv"
)

STEP3_FINAL_LEAKAGE_AUDIT_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_final_leakage_audit.csv"
)

STEP3_FINAL_DATASET_SUMMARY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_final_dataset_summary.csv"
)

STEP3_FINAL_VALIDATION_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_forecasting_preparation_final_validation_summary.csv"
)

STEP3_FINAL_DATASET_INVENTORY_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_final_dataset_inventory.csv"
)

STEP3_FINAL_FILE_HASH_OUTPUT = (
    FORECAST_PREPARATION_DIR
    / "17_final_file_hash_manifest.csv"
)


# ------------------------------------------------------------
# 6. Controlled atomic CSV-saving helper
# ------------------------------------------------------------

def save_csv_atomic(
    dataframe,
    output_path,
    date_format=None
):
    """
    Save to a temporary file first and then replace the output.
    This avoids leaving a partially written final file.
    """

    output_path = Path(output_path)

    temporary_path = output_path.with_name(
        output_path.name + ".temporary"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        date_format=date_format
    )

    temporary_path.replace(
        output_path
    )

    if not output_path.exists():
        raise FileNotFoundError(
            "The output file was not created:\n"
            f"{output_path}"
        )


# ------------------------------------------------------------
# 7. Save the final model-selection datasets
# ------------------------------------------------------------

save_csv_atomic(
    step3_model_selection_training_df,
    STEP3_FINAL_MODEL_SELECTION_TRAINING_OUTPUT,
    date_format="%Y-%m-%d"
)

save_csv_atomic(
    step3_model_selection_validation_df,
    STEP3_FINAL_MODEL_SELECTION_VALIDATION_OUTPUT,
    date_format="%Y-%m-%d"
)


# ------------------------------------------------------------
# 8. Save the reserved final-test datasets
# ------------------------------------------------------------

save_csv_atomic(
    step3_reserved_final_test_training_df,
    STEP3_FINAL_TEST_TRAINING_OUTPUT,
    date_format="%Y-%m-%d"
)

save_csv_atomic(
    step3_reserved_final_test_scoring_features_df,
    STEP3_FINAL_TEST_FEATURES_OUTPUT,
    date_format="%Y-%m-%d"
)

save_csv_atomic(
    step3_reserved_final_test_target_vault_df,
    STEP3_FINAL_TEST_TARGET_VAULT_OUTPUT,
    date_format="%Y-%m-%d"
)


# ------------------------------------------------------------
# 9. Save the forecast-only post-evaluation dataset
# ------------------------------------------------------------

save_csv_atomic(
    step3_post_evaluation_forecast_only_training_df,
    STEP3_FINAL_FORECAST_ONLY_TRAINING_OUTPUT,
    date_format="%Y-%m-%d"
)


# ------------------------------------------------------------
# 10. Save the final canonical dataset in both locations
# ------------------------------------------------------------

save_csv_atomic(
    step3_post_evaluation_full_history_final_fit_df,
    FINAL_CANONICAL_DATASET_IN_PREPARATION,
    date_format="%Y-%m-%d"
)

save_csv_atomic(
    step3_post_evaluation_full_history_final_fit_df,
    FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS,
    date_format="%Y-%m-%d"
)


# ------------------------------------------------------------
# 11. Save the frozen contracts and final audits
# ------------------------------------------------------------

save_csv_atomic(
    step3_frozen_model_feature_contract_df,
    STEP3_FINAL_FEATURE_CONTRACT_OUTPUT
)

save_csv_atomic(
    step3_forecasting_protocol_contract_df,
    STEP3_FINAL_PROTOCOL_CONTRACT_OUTPUT
)

save_csv_atomic(
    step3_final_predictor_missingness_audit_df,
    STEP3_FINAL_PREDICTOR_MISSINGNESS_OUTPUT
)

save_csv_atomic(
    step3_final_leakage_audit_df,
    STEP3_FINAL_LEAKAGE_AUDIT_OUTPUT
)

save_csv_atomic(
    step3_final_dataset_summary_df,
    STEP3_FINAL_DATASET_SUMMARY_OUTPUT
)

save_csv_atomic(
    step3_part5_validation_summary_df,
    STEP3_FINAL_VALIDATION_OUTPUT
)


# ------------------------------------------------------------
# 12. Create the final saved-dataset inventory
# ------------------------------------------------------------

step3_final_dataset_inventory_df = pd.DataFrame([
    {
        "DatasetName":
            "MODEL_SELECTION_TRAINING",

        "FilePath":
            str(
                STEP3_FINAL_MODEL_SELECTION_TRAINING_OUTPUT
            ),

        "Rows":
            len(
                step3_model_selection_training_df
            ),

        "Columns":
            step3_model_selection_training_df.shape[1],

        "Products":
            step3_model_selection_training_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            step3_model_selection_training_df[
                "SplitContextID"
            ].nunique(),

        "TargetIncluded":
            TARGET_COLUMN
            in step3_model_selection_training_df.columns,

        "UsagePolicy":
            "MODEL_SELECTION_TRAINING"
    },
    {
        "DatasetName":
            "MODEL_SELECTION_VALIDATION",

        "FilePath":
            str(
                STEP3_FINAL_MODEL_SELECTION_VALIDATION_OUTPUT
            ),

        "Rows":
            len(
                step3_model_selection_validation_df
            ),

        "Columns":
            step3_model_selection_validation_df.shape[1],

        "Products":
            step3_model_selection_validation_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            step3_model_selection_validation_df[
                "SplitContextID"
            ].nunique(),

        "TargetIncluded":
            TARGET_COLUMN
            in step3_model_selection_validation_df.columns,

        "UsagePolicy":
            "MODEL_SELECTION_SCORING"
    },
    {
        "DatasetName":
            "RESERVED_FINAL_TEST_TRAINING",

        "FilePath":
            str(
                STEP3_FINAL_TEST_TRAINING_OUTPUT
            ),

        "Rows":
            len(
                step3_reserved_final_test_training_df
            ),

        "Columns":
            step3_reserved_final_test_training_df.shape[1],

        "Products":
            step3_reserved_final_test_training_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            step3_reserved_final_test_training_df[
                "SplitContextID"
            ].nunique(),

        "TargetIncluded":
            TARGET_COLUMN
            in step3_reserved_final_test_training_df.columns,

        "UsagePolicy":
            "OPEN_AFTER_MODEL_SELECTION_IS_LOCKED"
    },
    {
        "DatasetName":
            "RESERVED_FINAL_TEST_SCORING_FEATURES",

        "FilePath":
            str(
                STEP3_FINAL_TEST_FEATURES_OUTPUT
            ),

        "Rows":
            len(
                step3_reserved_final_test_scoring_features_df
            ),

        "Columns":
            step3_reserved_final_test_scoring_features_df.shape[1],

        "Products":
            step3_reserved_final_test_scoring_features_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            step3_reserved_final_test_scoring_features_df[
                "SplitContextID"
            ].nunique(),

        "TargetIncluded":
            TARGET_COLUMN
            in step3_reserved_final_test_scoring_features_df.columns,

        "UsagePolicy":
            (
                "TARGET_EXCLUDED_OPEN_AFTER_"
                "MODEL_SELECTION_IS_LOCKED"
            )
    },
    {
        "DatasetName":
            "RESERVED_FINAL_TEST_TARGET_VAULT",

        "FilePath":
            str(
                STEP3_FINAL_TEST_TARGET_VAULT_OUTPUT
            ),

        "Rows":
            len(
                step3_reserved_final_test_target_vault_df
            ),

        "Columns":
            step3_reserved_final_test_target_vault_df.shape[1],

        "Products":
            step3_reserved_final_test_target_vault_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            step3_reserved_final_test_target_vault_df[
                "SplitContextID"
            ].nunique(),

        "TargetIncluded":
            TARGET_COLUMN
            in step3_reserved_final_test_target_vault_df.columns,

        "UsagePolicy":
            (
                "OPEN_ONLY_AFTER_FINAL_TEST_"
                "PREDICTIONS_HAVE_BEEN_SAVED"
            )
    },
    {
        "DatasetName":
            "FORECAST_ONLY_POST_EVALUATION_TRAINING",

        "FilePath":
            str(
                STEP3_FINAL_FORECAST_ONLY_TRAINING_OUTPUT
            ),

        "Rows":
            len(
                step3_post_evaluation_forecast_only_training_df
            ),

        "Columns":
            step3_post_evaluation_forecast_only_training_df.shape[1],

        "Products":
            step3_post_evaluation_forecast_only_training_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            step3_post_evaluation_forecast_only_training_df[
                "SplitContextID"
            ].nunique(),

        "TargetIncluded":
            TARGET_COLUMN
            in step3_post_evaluation_forecast_only_training_df.columns,

        "UsagePolicy":
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
    },
    {
        "DatasetName":
            "FINAL_CANONICAL_DATASET_PREPARATION_COPY",

        "FilePath":
            str(
                FINAL_CANONICAL_DATASET_IN_PREPARATION
            ),

        "Rows":
            len(
                step3_post_evaluation_full_history_final_fit_df
            ),

        "Columns":
            step3_post_evaluation_full_history_final_fit_df.shape[1],

        "Products":
            step3_post_evaluation_full_history_final_fit_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            0,

        "TargetIncluded":
            TARGET_COLUMN
            in step3_post_evaluation_full_history_final_fit_df.columns,

        "UsagePolicy":
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
    },
    {
        "DatasetName":
            "FINAL_CANONICAL_DATASET_EDEN_DATASETS_COPY",

        "FilePath":
            str(
                FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS
            ),

        "Rows":
            len(
                step3_post_evaluation_full_history_final_fit_df
            ),

        "Columns":
            step3_post_evaluation_full_history_final_fit_df.shape[1],

        "Products":
            step3_post_evaluation_full_history_final_fit_df[
                "CanonicalProductID"
            ].nunique(),

        "SplitContexts":
            0,

        "TargetIncluded":
            TARGET_COLUMN
            in step3_post_evaluation_full_history_final_fit_df.columns,

        "UsagePolicy":
            "POST_FINAL_TEST_PRODUCTION_FIT_ONLY"
    }
])


save_csv_atomic(
    step3_final_dataset_inventory_df,
    STEP3_FINAL_DATASET_INVENTORY_OUTPUT
)


# ------------------------------------------------------------
# 13. Reload the major saved datasets
# ------------------------------------------------------------

saved_model_selection_training_df = pd.read_csv(
    STEP3_FINAL_MODEL_SELECTION_TRAINING_OUTPUT,
    low_memory=False
)

saved_model_selection_validation_df = pd.read_csv(
    STEP3_FINAL_MODEL_SELECTION_VALIDATION_OUTPUT,
    low_memory=False
)

saved_final_test_training_df = pd.read_csv(
    STEP3_FINAL_TEST_TRAINING_OUTPUT,
    low_memory=False
)

saved_final_test_features_df = pd.read_csv(
    STEP3_FINAL_TEST_FEATURES_OUTPUT,
    low_memory=False
)

saved_final_test_target_vault_df = pd.read_csv(
    STEP3_FINAL_TEST_TARGET_VAULT_OUTPUT,
    low_memory=False
)

saved_forecast_only_training_df = pd.read_csv(
    STEP3_FINAL_FORECAST_ONLY_TRAINING_OUTPUT,
    low_memory=False
)

saved_final_canonical_preparation_df = pd.read_csv(
    FINAL_CANONICAL_DATASET_IN_PREPARATION,
    low_memory=False
)

saved_final_canonical_root_df = pd.read_csv(
    FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS,
    low_memory=False
)

saved_frozen_feature_contract_df = pd.read_csv(
    STEP3_FINAL_FEATURE_CONTRACT_OUTPUT,
    low_memory=False
)

saved_forecasting_protocol_contract_df = pd.read_csv(
    STEP3_FINAL_PROTOCOL_CONTRACT_OUTPUT,
    low_memory=False
)

saved_final_predictor_missingness_df = pd.read_csv(
    STEP3_FINAL_PREDICTOR_MISSINGNESS_OUTPUT,
    low_memory=False
)

saved_final_leakage_audit_df = pd.read_csv(
    STEP3_FINAL_LEAKAGE_AUDIT_OUTPUT,
    low_memory=False
)

saved_final_dataset_summary_df = pd.read_csv(
    STEP3_FINAL_DATASET_SUMMARY_OUTPUT,
    low_memory=False
)

saved_final_validation_summary_df = pd.read_csv(
    STEP3_FINAL_VALIDATION_OUTPUT,
    low_memory=False
)

saved_final_dataset_inventory_df = pd.read_csv(
    STEP3_FINAL_DATASET_INVENTORY_OUTPUT,
    low_memory=False
)


# ------------------------------------------------------------
# 14. Validate the reloaded row counts
# ------------------------------------------------------------

assert len(
    saved_model_selection_training_df
) == 48_538

assert len(
    saved_model_selection_validation_df
) == 5_080

assert len(
    saved_final_test_training_df
) == 38_628

assert len(
    saved_final_test_features_df
) == 4_150

assert len(
    saved_final_test_target_vault_df
) == 4_150

assert len(
    saved_forecast_only_training_df
) == 996

assert saved_final_canonical_preparation_df.shape == (
    43_774,
    58
)

assert saved_final_canonical_root_df.shape == (
    43_774,
    58
)


# ------------------------------------------------------------
# 15. Validate product, date and demand preservation
# ------------------------------------------------------------

assert saved_final_canonical_preparation_df[
    "CanonicalProductID"
].nunique() == 227

assert saved_final_canonical_preparation_df[
    "Date"
].nunique() == 245

assert saved_final_canonical_preparation_df[
    [
        "Date",
        "CanonicalProductID"
    ]
].duplicated().sum() == 0

assert np.isclose(
    saved_final_canonical_preparation_df[
        TARGET_COLUMN
    ].sum(),
    116_158
)

assert np.isclose(
    saved_final_canonical_root_df[
        TARGET_COLUMN
    ].sum(),
    116_158
)


# ------------------------------------------------------------
# 16. Validate model-selection files
# ------------------------------------------------------------

assert saved_model_selection_training_df[
    "CanonicalProductID"
].nunique() == 127

assert saved_model_selection_validation_df[
    "CanonicalProductID"
].nunique() == 127

assert saved_model_selection_training_df[
    "SplitContextID"
].nunique() == 254

assert saved_model_selection_validation_df[
    "SplitContextID"
].nunique() == 254

assert TARGET_COLUMN in (
    saved_model_selection_training_df.columns
)

assert TARGET_COLUMN in (
    saved_model_selection_validation_df.columns
)


# ------------------------------------------------------------
# 17. Validate reserved final-test isolation
# ------------------------------------------------------------

assert saved_final_test_training_df[
    "CanonicalProductID"
].nunique() == 218

assert saved_final_test_features_df[
    "CanonicalProductID"
].nunique() == 218

assert saved_final_test_training_df[
    "SplitContextID"
].nunique() == 218

assert saved_final_test_features_df[
    "SplitContextID"
].nunique() == 218

assert TARGET_COLUMN in (
    saved_final_test_training_df.columns
)

assert TARGET_COLUMN not in (
    saved_final_test_features_df.columns
)

assert TARGET_COLUMN in (
    saved_final_test_target_vault_df.columns
)


# ------------------------------------------------------------
# 18. Validate the forecast-only dataset
# ------------------------------------------------------------

assert saved_forecast_only_training_df[
    "CanonicalProductID"
].nunique() == 9

assert saved_forecast_only_training_df[
    "SplitContextID"
].nunique() == 9

assert TARGET_COLUMN in (
    saved_forecast_only_training_df.columns
)


# ------------------------------------------------------------
# 19. Validate predictor-column availability
# ------------------------------------------------------------

datasets_requiring_predictors = {
    "MODEL_SELECTION_TRAINING":
        saved_model_selection_training_df,

    "MODEL_SELECTION_VALIDATION":
        saved_model_selection_validation_df,

    "RESERVED_FINAL_TEST_TRAINING":
        saved_final_test_training_df,

    "RESERVED_FINAL_TEST_SCORING_FEATURES":
        saved_final_test_features_df,

    "FORECAST_ONLY_TRAINING":
        saved_forecast_only_training_df,

    "FINAL_CANONICAL_DATASET":
        saved_final_canonical_preparation_df
}

for (
    dataset_name,
    dataset_df
) in datasets_requiring_predictors.items():

    missing_predictors = [
        column
        for column in direct_predictor_columns
        if column not in dataset_df.columns
    ]

    assert len(missing_predictors) == 0, (
        f"{dataset_name} is missing predictors:\n"
        f"{missing_predictors}"
    )


# ------------------------------------------------------------
# 20. Validate saved contracts and audits
# ------------------------------------------------------------

assert len(
    saved_frozen_feature_contract_df
) == 58

assert len(
    direct_predictor_columns
) == 53

assert len(
    current_date_predictor_columns
) == 22

assert len(
    historical_feature_columns
) == 31

assert saved_final_leakage_audit_df[
    "LeakageColumnCount"
].sum() == 0

assert len(
    saved_final_dataset_inventory_df
) == 8


# ------------------------------------------------------------
# 21. SHA-256 helper
# ------------------------------------------------------------

def calculate_sha256(
    file_path
):
    """
    Calculate the SHA-256 hash of a file.
    """

    file_path = Path(file_path)

    sha256_digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:

        for data_block in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b""
        ):
            sha256_digest.update(
                data_block
            )

    return sha256_digest.hexdigest()


# ------------------------------------------------------------
# 22. Create the final file-hash manifest
# ------------------------------------------------------------

files_to_hash = [
    (
        "MODEL_SELECTION_TRAINING",
        STEP3_FINAL_MODEL_SELECTION_TRAINING_OUTPUT
    ),
    (
        "MODEL_SELECTION_VALIDATION",
        STEP3_FINAL_MODEL_SELECTION_VALIDATION_OUTPUT
    ),
    (
        "RESERVED_FINAL_TEST_TRAINING",
        STEP3_FINAL_TEST_TRAINING_OUTPUT
    ),
    (
        "RESERVED_FINAL_TEST_SCORING_FEATURES",
        STEP3_FINAL_TEST_FEATURES_OUTPUT
    ),
    (
        "RESERVED_FINAL_TEST_TARGET_VAULT",
        STEP3_FINAL_TEST_TARGET_VAULT_OUTPUT
    ),
    (
        "FORECAST_ONLY_POST_EVALUATION_TRAINING",
        STEP3_FINAL_FORECAST_ONLY_TRAINING_OUTPUT
    ),
    (
        "FINAL_CANONICAL_PREPARATION_COPY",
        FINAL_CANONICAL_DATASET_IN_PREPARATION
    ),
    (
        "FINAL_CANONICAL_EDEN_DATASETS_COPY",
        FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS
    ),
    (
        "FROZEN_MODEL_FEATURE_CONTRACT",
        STEP3_FINAL_FEATURE_CONTRACT_OUTPUT
    ),
    (
        "FORECASTING_PROTOCOL_CONTRACT",
        STEP3_FINAL_PROTOCOL_CONTRACT_OUTPUT
    ),
    (
        "FINAL_PREDICTOR_MISSINGNESS_AUDIT",
        STEP3_FINAL_PREDICTOR_MISSINGNESS_OUTPUT
    ),
    (
        "FINAL_LEAKAGE_AUDIT",
        STEP3_FINAL_LEAKAGE_AUDIT_OUTPUT
    ),
    (
        "FINAL_DATASET_SUMMARY",
        STEP3_FINAL_DATASET_SUMMARY_OUTPUT
    ),
    (
        "FINAL_VALIDATION_SUMMARY",
        STEP3_FINAL_VALIDATION_OUTPUT
    ),
    (
        "FINAL_DATASET_INVENTORY",
        STEP3_FINAL_DATASET_INVENTORY_OUTPUT
    )
]


file_hash_records = []

for (
    file_label,
    file_path
) in files_to_hash:

    file_path = Path(
        file_path
    )

    if not file_path.exists():
        raise FileNotFoundError(
            "A file expected for hashing is missing:\n"
            f"{file_path}"
        )

    file_hash_records.append({
        "FileLabel":
            file_label,

        "FilePath":
            str(file_path),

        "FileSizeBytes":
            file_path.stat().st_size,

        "SHA256":
            calculate_sha256(
                file_path
            )
    })


step3_final_file_hash_manifest_df = pd.DataFrame(
    file_hash_records
)


save_csv_atomic(
    step3_final_file_hash_manifest_df,
    STEP3_FINAL_FILE_HASH_OUTPUT
)


# ------------------------------------------------------------
# 23. Confirm both final canonical copies are identical
# ------------------------------------------------------------

preparation_copy_hash = (
    step3_final_file_hash_manifest_df.loc[
        step3_final_file_hash_manifest_df[
            "FileLabel"
        ]
        == "FINAL_CANONICAL_PREPARATION_COPY",
        "SHA256"
    ]
    .iloc[0]
)

eden_datasets_copy_hash = (
    step3_final_file_hash_manifest_df.loc[
        step3_final_file_hash_manifest_df[
            "FileLabel"
        ]
        == "FINAL_CANONICAL_EDEN_DATASETS_COPY",
        "SHA256"
    ]
    .iloc[0]
)

canonical_copy_hashes_identical = (
    preparation_copy_hash
    == eden_datasets_copy_hash
)

assert canonical_copy_hashes_identical


# ------------------------------------------------------------
# 24. Build the Markdown final-dataset inventory
# ------------------------------------------------------------

final_dataset_markdown_rows = "\n".join(
    (
        f"- `{row.DatasetName}`: "
        f"{int(row.Rows):,} rows, "
        f"{int(row.Columns)} columns, "
        f"{int(row.Products)} products; "
        f"usage `{row.UsagePolicy}`"
    )
    for row in (
        step3_final_dataset_inventory_df
        .itertuples()
    )
)


# ------------------------------------------------------------
# 25. Update the Markdown handoff
# ------------------------------------------------------------

step3_part5_handoff_summary = f"""
**Status:** Completed, validated, saved and frozen

### Purpose

Step 3 Part 5 created the final model-ready datasets, physically
separated the reserved final-test target from its feature dataset,
froze the model feature contract, created integrity hashes and
completed the complete forecasting-preparation phase.

### Frozen feature contract

- Direct predictor columns: {len(direct_predictor_columns)}
- Current-date predictor columns: {len(current_date_predictor_columns)}
- Historical demand predictor columns: {len(historical_feature_columns)}
- Forecast target: `{TARGET_COLUMN}`
- Feature-parity mismatches: {total_feature_parity_mismatches}
- Same-day leakage columns present: {same_day_leakage_columns_present}

### Model-selection datasets

- Training rows: {len(saved_model_selection_training_df):,}
- Validation rows: {len(saved_model_selection_validation_df):,}
- Products: 127
- Split contexts: 254
- Chronological overlap violations: {model_selection_overlap_violations}

### Reserved final-test datasets

- Training rows: {len(saved_final_test_training_df):,}
- Scoring-feature rows: {len(saved_final_test_features_df):,}
- Target-vault rows: {len(saved_final_test_target_vault_df):,}
- Products: 218
- Split contexts: 218
- Model-selection/final-test context overlap: {context_isolation_violations}
- Target present in scoring-feature file: no
- Target-vault mismatches: {final_test_target_vault_mismatches}
- Final-test chronological overlap violations: {final_test_overlap_violations}

The final-test target vault must remain unopened until the model,
features and hyperparameters have been locked and final-test
predictions have been saved.

### Post-evaluation datasets

- Forecast-only training rows: {len(saved_forecast_only_training_df):,}
- Forecast-only products: 9
- Final canonical rows: {len(saved_final_canonical_preparation_df):,}
- Final canonical columns: {saved_final_canonical_preparation_df.shape[1]}
- Canonical products: 227
- Operating dates: 245
- Total demand units: 116,158

### Final frozen datasets

{final_dataset_markdown_rows}

### Final canonical dataset locations

- `{FINAL_CANONICAL_DATASET_IN_PREPARATION}`
- `{FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS}`

The SHA-256 hashes of the two canonical copies are identical.

### Integrity and protocol files

- `17_frozen_model_feature_contract.csv`
- `17_forecasting_protocol_contract.csv`
- `17_final_predictor_missingness_audit.csv`
- `17_final_leakage_audit.csv`
- `17_final_dataset_summary.csv`
- `17_forecasting_preparation_final_validation_summary.csv`
- `17_final_dataset_inventory.csv`
- `17_final_file_hash_manifest.csv`

### Forecasting-preparation completion

Forecasting Preparation Steps 1, 2 and 3 are complete.

The next project phase is model training, chronological model
selection, reserved final testing and post-evaluation final fitting.
"""


upsert_markdown_section(
    HANDOFF_FILE,
    section_id="forecast_step_3_part_5",
    section_title=(
        "Forecasting Preparation — Step 3, Part 5"
    ),
    section_body=step3_part5_handoff_summary
)

assert HANDOFF_FILE.exists()


# ------------------------------------------------------------
# 26. Print the final completion summary
# ------------------------------------------------------------

print("=" * 78)
print(
    "FORECASTING PREPARATION — "
    "STEP 3, PART 5 COMPLETED"
)
print("=" * 78)

print()
print("Frozen model-selection datasets:")
print(
    "Training rows:",
    f"{len(saved_model_selection_training_df):,}"
)
print(
    "Validation rows:",
    f"{len(saved_model_selection_validation_df):,}"
)
print(
    "Model-selection contexts:",
    saved_model_selection_training_df[
        "SplitContextID"
    ].nunique()
)

print()
print("Frozen reserved final-test datasets:")
print(
    "Training rows:",
    f"{len(saved_final_test_training_df):,}"
)
print(
    "Scoring-feature rows:",
    f"{len(saved_final_test_features_df):,}"
)
print(
    "Target-vault rows:",
    f"{len(saved_final_test_target_vault_df):,}"
)
print(
    "Final-test contexts:",
    saved_final_test_features_df[
        "SplitContextID"
    ].nunique()
)
print(
    "Target present in scoring-feature file:",
    TARGET_COLUMN
    in saved_final_test_features_df.columns
)

print()
print("Post-evaluation datasets:")
print(
    "Forecast-only training rows:",
    f"{len(saved_forecast_only_training_df):,}"
)
print(
    "Final canonical rows:",
    f"{len(saved_final_canonical_preparation_df):,}"
)
print(
    "Final canonical columns:",
    saved_final_canonical_preparation_df.shape[1]
)
print(
    "Canonical products:",
    saved_final_canonical_preparation_df[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Operating dates:",
    saved_final_canonical_preparation_df[
        "Date"
    ].nunique()
)
print(
    "Total demand units:",
    f"{saved_final_canonical_preparation_df[TARGET_COLUMN].sum():,.0f}"
)

print()
print("Final validation:")
print(
    "Feature-parity mismatches:",
    total_feature_parity_mismatches
)
print(
    "Chronological overlap violations:",
    (
        model_selection_overlap_violations
        + final_test_overlap_violations
    )
)
print(
    "Model-selection/final-test context overlap:",
    context_isolation_violations
)
print(
    "Final-test target-vault mismatches:",
    final_test_target_vault_mismatches
)
print(
    "Scoring historical-feature missing cells:",
    scoring_historical_missing_cells
)
print(
    "Same-day leakage columns present:",
    same_day_leakage_columns_present
)

print()
print("Final canonical dataset saved in:")
print(
    f"1. {FINAL_CANONICAL_DATASET_IN_PREPARATION}"
)
print(
    f"2. {FINAL_CANONICAL_DATASET_IN_EDEN_DATASETS}"
)
print(
    "Canonical-copy hashes identical:",
    canonical_copy_hashes_identical
)

print()
print("Final integrity files:")
print(
    f"1. {STEP3_FINAL_FEATURE_CONTRACT_OUTPUT}"
)
print(
    f"2. {STEP3_FINAL_PROTOCOL_CONTRACT_OUTPUT}"
)
print(
    f"3. {STEP3_FINAL_PREDICTOR_MISSINGNESS_OUTPUT}"
)
print(
    f"4. {STEP3_FINAL_LEAKAGE_AUDIT_OUTPUT}"
)
print(
    f"5. {STEP3_FINAL_DATASET_SUMMARY_OUTPUT}"
)
print(
    f"6. {STEP3_FINAL_VALIDATION_OUTPUT}"
)
print(
    f"7. {STEP3_FINAL_DATASET_INVENTORY_OUTPUT}"
)
print(
    f"8. {STEP3_FINAL_FILE_HASH_OUTPUT}"
)
print(
    f"9. {HANDOFF_FILE}"
)

print()
print(
    "All Step 3, Part 5 validation checks passed."
)
print(
    "FORECASTING PREPARATION STEP 3 IS COMPLETE."
)
print(
    "THE COMPLETE FORECASTING-PREPARATION PHASE IS READY."
)

FORECASTING PREPARATION — STEP 3, PART 5 COMPLETED

Frozen model-selection datasets:
Training rows: 48,538
Validation rows: 5,080
Model-selection contexts: 254

Frozen reserved final-test datasets:
Training rows: 38,628
Scoring-feature rows: 4,150
Target-vault rows: 4,150
Final-test contexts: 218
Target present in scoring-feature file: False

Post-evaluation datasets:
Forecast-only training rows: 996
Final canonical rows: 43,774
Final canonical columns: 58
Canonical products: 227
Operating dates: 245
Total demand units: 116,158

Final validation:
Feature-parity mismatches: 0
Chronological overlap violations: 0
Model-selection/final-test context overlap: 0
Final-test target-vault mismatches: 0
Scoring historical-feature missing cells: 0
Same-day leakage columns present: 0

Final canonical dataset saved in:
1. /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/UL_EDEN_forecasting_preparation_final_model_dataset.csv
2. /Users/ryansmac/Desktop/Meng Project/eden_dataset